<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NBA_STABLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
SPORT = "basketball_nba"
SIMS = 50000

# NBA variance
SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200

print("✅ NBA Engine Settings Loaded")

✅ NBA Engine Settings Loaded


In [2]:
DAYS_BACK = 30

In [3]:
# NBA Possessions
games_hist["home_poss"] = (
    games_hist["home_fga"]
    - games_hist["home_oreb"]
    + games_hist["home_tov"]
    + 0.44 * games_hist["home_fta"]
)

NameError: name 'games_hist' is not defined

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

ODDS_API_KEY = os.getenv("ODDS_API_KEY")  # if set in Colab secrets
# If not set, paste your key here:
# ODDS_API_KEY = "PASTE_KEY"

SLATE_DATE = "2026-03-01"   # change as needed
SPORT = "basketball_nba"
print("✅ Loaded:", SPORT, SLATE_DATE, "key?", bool(ODDS_API_KEY))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("✅ ODDS_API_KEY set?", bool(os.getenv("ODDS_API_KEY")))

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

# pulls from env (set via Secrets or the one-liner cell)
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-01"   # change if needed

if not ODDS_API_KEY:
    raise ValueError("ODDS_API_KEY not found. Set it in env or Colab Secrets.")

print("✅ Ready:", SPORT, SLATE_DATE)

In [ ]:
def pull_market_data(
    sport: str,
    date: str,
    regions="us",
    markets="h2h,spreads,totals",
    odds_format="american",
    date_filter=True,
):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
    }

    if date_filter:
        day = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        params["commenceTimeFrom"] = day.isoformat().replace("+00:00", "Z")
        params["commenceTimeTo"] = (day + timedelta(days=1)).isoformat().replace("+00:00", "Z")

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        t = g.get("commence_time")

        bks = g.get("bookmakers", [])
        book = bks[0] if bks else None

        h2h = spreads = totals = None
        if book:
            for m in book.get("markets", []):
                if m.get("key") == "h2h":
                    h2h = m
                elif m.get("key") == "spreads":
                    spreads = m
                elif m.get("key") == "totals":
                    totals = m

        def pick_outcome(mkt, name):
            if not mkt: return None
            for o in mkt.get("outcomes", []):
                if o.get("name") == name:
                    return o
            return None

        home_ml = away_ml = np.nan
        if h2h:
            oh = pick_outcome(h2h, home)
            oa = pick_outcome(h2h, away)
            if oh: home_ml = oh.get("price", np.nan)
            if oa: away_ml = oa.get("price", np.nan)

        spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = np.nan
        if spreads:
            oh = pick_outcome(spreads, home)
            oa = pick_outcome(spreads, away)
            if oh:
                spread_home_line = oh.get("point", np.nan)
                spread_home_odds = oh.get("price", np.nan)
            if oa:
                spread_away_line = oa.get("point", np.nan)
                spread_away_odds = oa.get("price", np.nan)

        total_line = over_odds = under_odds = np.nan
        if totals:
            oo = pick_outcome(totals, "Over")
            ou = pick_outcome(totals, "Under")
            if oo:
                total_line = oo.get("point", np.nan)
                over_odds = oo.get("price", np.nan)
            if ou:
                under_odds = ou.get("price", np.nan)

        rows.append({
            "home_team": home,
            "away_team": away,
            "commence_time": t,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,
            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,
        })

    df = pd.DataFrame(rows)
    print(f"✅ pulled {len(df)} NBA games for {date}")
    print("ML non-null:", int(df["home_ml"].notna().sum()), "/", len(df))
    print("Spread non-null:", int(df["spread_home_line"].notna().sum()), "/", len(df))
    print("Total non-null:", int(df["total_line"].notna().sum()), "/", len(df))
    return df

games_df = pull_market_data(SPORT, SLATE_DATE, date_filter=True)
games_df.head()

In [ ]:
def pull_completed_scores(sport: str, days_from: int):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": int(days_from)}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g.get("home_team")
        away = g.get("away_team")
        ct = g.get("commence_time")
        scores = {s["name"]: s["score"] for s in g.get("scores", [])} if g.get("scores") else {}
        hs = scores.get(home)
        as_ = scores.get(away)
        if hs is None or as_ is None:
            continue
        rows.append({
            "home_team": home,
            "away_team": away,
            "home_score": float(hs),
            "away_score": float(as_),
            "date": ct
        })

    return pd.DataFrame(rows)

# Pull multiple windows (stitch together)
pieces = []
for d in [3, 6, 10, 14, 21, 28]:
    try:
        df_part = pull_completed_scores(SPORT, d)
        pieces.append(df_part)
        print(f"daysFrom {d}: {len(df_part)} games")
    except Exception as e:
        print("⚠️", e)

historical_df = pd.concat(pieces, ignore_index=True).drop_duplicates(
    subset=["home_team","away_team","home_score","away_score","date"]
)

print("✅ Unique historical games pulled:", len(historical_df))
historical_df.head()

In [ ]:
PPP_LEAGUE = 1.13
HOME_EDGE_PTS = 2.0

hist = historical_df.copy()
hist["total_pts"] = hist["home_score"] + hist["away_score"]

# Possession proxy
hist["poss_proxy"] = hist["total_pts"] / PPP_LEAGUE

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "points_scored": hist["home_score"],
    "points_allowed": hist["away_score"],
    "poss": hist["poss_proxy"],
})

away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "points_scored": hist["away_score"],
    "points_allowed": hist["home_score"],
    "poss": hist["poss_proxy"],
})

all_stats = pd.concat([home_rows, away_rows], ignore_index=True)

team_overall = all_stats.groupby("team").agg(
    poss=("poss","mean"),
    ppp_for=("points_scored", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean()),
    ppp_against=("points_allowed", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean())
).reset_index()

team_overall = team_overall.fillna(team_overall.mean(numeric_only=True))

print("✅ Team baseline built:", team_overall.shape)
team_overall.head()

In [ ]:
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

games_df = g

print("Margin model range:", float(g["mean_margin_model"].min()), "to", float(g["mean_margin_model"].max()))
print("Total model range:", float(g["mean_total_model"].min()), "to", float(g["mean_total_model"].max()))

In [ ]:
BLEND_MODEL = 0.4
BLEND_MARKET = 0.6

games_df["market_margin"] = -games_df["spread_home_line"]

games_df["mean_margin"] = (
    BLEND_MODEL * games_df["mean_margin_model"] +
    BLEND_MARKET * games_df["market_margin"]
)

games_df["mean_total"] = (
    BLEND_MODEL * games_df["mean_total_model"] +
    BLEND_MARKET * games_df["total_line"]
)

print("✅ NBA blend applied")
print("Blended margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Blended total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

In [ ]:
import numpy as np
import pandas as pd

# =========================
# NBA SIM SETTINGS
# =========================
SIMS = 50000
SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# ensure numeric
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# BUILD CARDS
# =========================
# ML (home side only for simplicity)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CORRELATION CAP: max 2 per game
# =========================
kept = []
counts = {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Plays after cap:", len(card))
top = card.head(20).copy()

top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

In [ ]:
SD_MARGIN = 14.5
SD_TOTAL  = 18.0
print("✅ Using widened NBA variance:", SD_MARGIN, SD_TOTAL)

In [ ]:
import numpy as np
import pandas as pd

SIMS = 50000
UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig
ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])
sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])
to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# cap: max 2 plays per game
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Plays after cap:", len(card))
top = card.head(20).copy()

top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_card_widevar.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

In [ ]:
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

games_df["market_margin"] = -pd.to_numeric(games_df["spread_home_line"], errors="coerce")
games_df["mean_margin"] = (
    BLEND_MODEL * pd.to_numeric(games_df["mean_margin_model"], errors="coerce") +
    BLEND_MARKET * games_df["market_margin"]
)

games_df["mean_total"] = (
    BLEND_MODEL * pd.to_numeric(games_df["mean_total_model"], errors="coerce") +
    BLEND_MARKET * pd.to_numeric(games_df["total_line"], errors="coerce")
)

print("✅ Strong NBA blend applied (25/75)")
print("Blended margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Blended total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

In [ ]:
# ======================================
# BACK-TO-BACK DETECTION
# ======================================

from datetime import datetime, timedelta

# pull yesterday’s completed games
yesterday = (datetime.strptime(SLATE_DATE, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")

recent_games = historical_df.copy()
recent_games["date"] = pd.to_datetime(recent_games["date"]).dt.date

yesterday_date = datetime.strptime(yesterday, "%Y-%m-%d").date()

teams_played_yesterday = set(
    recent_games.loc[recent_games["date"] == yesterday_date, "home_team"]
).union(
    set(recent_games.loc[recent_games["date"] == yesterday_date, "away_team"])
)

games_df["home_b2b"] = games_df["home_team"].isin(teams_played_yesterday)
games_df["away_b2b"] = games_df["away_team"].isin(teams_played_yesterday)

# apply fatigue penalty
B2B_PENALTY = 1.5

games_df["mean_margin"] = games_df["mean_margin"] \
    - games_df["home_b2b"] * B2B_PENALTY \
    + games_df["away_b2b"] * B2B_PENALTY

print("✅ B2B adjustment applied")
print("Home B2B:", games_df["home_b2b"].sum())
print("Away B2B:", games_df["away_b2b"].sum())

In [ ]:
# ======================================
# BLOWOUT ADJUSTMENT
# ======================================

BLOWOUT_THRESHOLD = 8

large_spread = games_df["spread_home_line"].abs() >= BLOWOUT_THRESHOLD

# reduce margin magnitude slightly
games_df.loc[large_spread, "mean_margin"] *= 0.90

# reduce totals slightly (garbage time compression)
games_df.loc[large_spread, "mean_total"] -= 1.5

print("✅ Blowout adjustment applied")
print("Large spread games:", large_spread.sum())

In [ ]:
Run NBA Stable Engine — STRICT

Settings:
• Sims: 50,000
• Variance: SD_MARGIN=14.5 | SD_TOTAL=18.0
• Market Blend: 25% Model / 75% Market
• B2B Adjustment: ON (−1.5 pts fatigue)
• Blowout Compression: ON (≥8 spread reduce margin 10%, total −1.5)
• Correlation Filter: Max 2 plays per game
• Exposure Rule: No ML + Spread on same team unless both ≥6% EV
• Unit Cap: 1.0u
• Min Unit: 0.25u
• Max Juice: −200
• Remove 0 EV plays

Process:
1) Use blended mean_margin + mean_total
2) Run Monte Carlo simulation (50k)
3) Calculate win probability vs market implied
4) Compute EV
5) Apply Kelly sizing (capped 1u)
6) Remove correlated exposure >2 per game
7) Sort by EV descending
8) Export top 20 to CSV
9) Generate Discord-ready output

Output:
• matchup
• bet
• units
• Sim Win%
• EV%

Date: 2026-03-01

In [4]:
import numpy as np
import pandas as pd

# =========================
# FINAL NBA RUN CELL
# (uses current games_df with all adjustments already applied)
# =========================

SIMS = 50000

# If you already set these earlier, leave them; otherwise these are your calibrated defaults
SD_MARGIN = globals().get("SD_MARGIN", 14.5)
SD_TOTAL  = globals().get("SD_TOTAL", 18.0)

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# ensure numeric
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# BUILD MARKET CARDS
# =========================
# ML (home only)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CAP: max 2 plays per game
# =========================
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Sims:", SIMS, "| SD_MARGIN:", SD_MARGIN, "| SD_TOTAL:", SD_TOTAL)
print("✅ Plays after cap:", len(card))

top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_final_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

NameError: name 'games_df' is not defined

In [ ]:
import numpy as np
import pandas as pd

# Uses your existing games_df that already has:
# mean_margin, mean_total, spread_home_line, spread_away_line, odds, etc
SIMS = 50000
SD_MARGIN = globals().get("SD_MARGIN", 14.5)
SD_TOTAL  = globals().get("SD_TOTAL", 18.0)

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_away_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

# Spread cover: home covers its listed spread (home_line)
# home covers if margin + home_line > 0  <=> margin > -home_line
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]

df["p_home_win"] = (margins > 0).mean(axis=1)
df["p_over"]  = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"] = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# ML (home only)
# =========================
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# =========================
# SPREAD (FIXED LINE SIGN)
# =========================
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()

sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])

# ✅ Use the actual market line from feed (NO NEGATION)
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], sp["spread_away_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])

sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].apply(fmt_line) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# =========================
# TOTAL (best side)
# =========================
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    return x[["matchup","market","bet","units","p_win","ev","commence_time"]]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CAP: max 2 plays per game
# =========================
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

# =========================
# DISCORD TEXT
# =========================
top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ FIXED spread signs. Plays after cap:", len(top))
print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_FIXED_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"   # <-- 3/2 run

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

# Market blend (NBA)
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

# Context layers
B2B_PENALTY = 1.5
BLOWOUT_THRESHOLD = 8
BLOWOUT_MARGIN_MULT = 0.90
BLOWOUT_TOTAL_BUMP = -1.5

ODDS_API_KEY = os.getenv("ODDS_API_KEY")
if not ODDS_API_KEY:
    raise ValueError("ODDS_API_KEY not found. Set it in Colab Secrets or env var.")

print("✅ Settings loaded:", SPORT, SLATE_DATE, "| sims:", SIMS)

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

print("✅ ODDS_API_KEY set in environment")

In [ ]:
def pull_market_data(
    sport: str,
    date: str,
    regions="us",
    markets="h2h,spreads,totals",
    odds_format="american",
    date_filter=True,
):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
    }

    if date_filter:
        day = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        params["commenceTimeFrom"] = day.isoformat().replace("+00:00", "Z")
        params["commenceTimeTo"] = (day + timedelta(days=1)).isoformat().replace("+00:00", "Z")

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        t = g.get("commence_time")

        bks = g.get("bookmakers", [])
        book = bks[0] if bks else None

        h2h = spreads = totals = None
        if book:
            for m in book.get("markets", []):
                if m.get("key") == "h2h":
                    h2h = m
                elif m.get("key") == "spreads":
                    spreads = m
                elif m.get("key") == "totals":
                    totals = m

        def pick_outcome(mkt, name):
            if not mkt: return None
            for o in mkt.get("outcomes", []):
                if o.get("name") == name:
                    return o
            return None

        home_ml = away_ml = np.nan
        if h2h:
            oh = pick_outcome(h2h, home)
            oa = pick_outcome(h2h, away)
            if oh: home_ml = oh.get("price", np.nan)
            if oa: away_ml = oa.get("price", np.nan)

        spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = np.nan
        if spreads:
            oh = pick_outcome(spreads, home)
            oa = pick_outcome(spreads, away)
            if oh:
                spread_home_line = oh.get("point", np.nan)
                spread_home_odds = oh.get("price", np.nan)
            if oa:
                spread_away_line = oa.get("point", np.nan)
                spread_away_odds = oa.get("price", np.nan)

        total_line = over_odds = under_odds = np.nan
        if totals:
            oo = pick_outcome(totals, "Over")
            ou = pick_outcome(totals, "Under")
            if oo:
                total_line = oo.get("point", np.nan)
                over_odds = oo.get("price", np.nan)
            if ou:
                under_odds = ou.get("price", np.nan)

        rows.append({
            "home_team": home,
            "away_team": away,
            "commence_time": t,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,
            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,
        })

    df = pd.DataFrame(rows)
    print(f"✅ pulled {len(df)} NBA games for {date}")
    print("ML non-null:", int(df["home_ml"].notna().sum()), "/", len(df))
    print("Spread non-null:", int(df["spread_home_line"].notna().sum()), "/", len(df))
    print("Total non-null:", int(df["total_line"].notna().sum()), "/", len(df))
    return df

games_df = pull_market_data(SPORT, SLATE_DATE, date_filter=True)
games_df.head()

In [ ]:
def pull_completed_scores_safe(sport: str, days_list):
    all_parts = []
    for d in days_list:
        try:
            url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
            params = {"apiKey": ODDS_API_KEY, "daysFrom": int(d)}
            r = requests.get(url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"⚠ daysFrom {d} failed — skipping")
                continue

            data = r.json()
            rows = []
            for g in data:
                if not g.get("completed"):
                    continue
                home = g.get("home_team")
                away = g.get("away_team")
                ct = g.get("commence_time")
                scores = {s["name"]: s["score"] for s in g.get("scores", [])} if g.get("scores") else {}
                hs = scores.get(home)
                as_ = scores.get(away)
                if hs is None or as_ is None:
                    continue
                rows.append({
                    "home_team": home,
                    "away_team": away,
                    "home_score": float(hs),
                    "away_score": float(as_),
                    "date": ct
                })

            part_df = pd.DataFrame(rows)
            print(f"✅ daysFrom {d}: {len(part_df)} games")
            all_parts.append(part_df)

        except Exception as e:
            print(f"⚠ daysFrom {d} error:", str(e))
            continue

    if not all_parts:
        raise ValueError("No historical score data pulled.")
    return pd.concat(all_parts, ignore_index=True).drop_duplicates()

# 🔒 Only use safe values your plan supports
historical_df = pull_completed_scores_safe(SPORT, [3, 6, 10, 14])

print("✅ Unique historical games pulled:", len(historical_df))
historical_df.head()

In [ ]:
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

g["market_margin"] = -pd.to_numeric(g["spread_home_line"], errors="coerce")

g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * pd.to_numeric(g["total_line"], errors="coerce")

games_df = g

print("✅ Blend applied (25/75)")
print("Margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

In [ ]:
import pandas as pd
import numpy as np

# ======================================
# ONE-CELL FIX: build team_overall + projections + 25/75 blend
# ======================================

# 1) Ensure historical_df exists
if "historical_df" not in globals():
    raise NameError("historical_df not found. Run the completed scores pull cell first.")

# 2) Build team_overall if missing
if "team_overall" not in globals() or not isinstance(globals().get("team_overall"), pd.DataFrame):
    PPP_LEAGUE = 1.13
    HOME_EDGE_PTS = 2.0

    hist = historical_df.copy()
    hist["total_pts"] = hist["home_score"] + hist["away_score"]
    hist["poss_proxy"] = hist["total_pts"] / PPP_LEAGUE

    home_rows = pd.DataFrame({
        "team": hist["home_team"],
        "points_scored": hist["home_score"],
        "points_allowed": hist["away_score"],
        "poss": hist["poss_proxy"],
    })
    away_rows = pd.DataFrame({
        "team": hist["away_team"],
        "points_scored": hist["away_score"],
        "points_allowed": hist["home_score"],
        "poss": hist["poss_proxy"],
    })

    all_stats = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = all_stats.groupby("team").agg(
        poss=("poss","mean"),
        ppp_for=("points_scored", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean()),
        ppp_against=("points_allowed", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean())
    ).reset_index()

    team_overall = team_overall.fillna(team_overall.mean(numeric_only=True))
    print("✅ team_overall built:", team_overall.shape)

# 3) Ensure games_df exists
if "games_df" not in globals():
    raise NameError("games_df not found. Run the market pull cell first.")

g = games_df.copy()

# 4) Merge team metrics into slate
g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

# fill any missing with slate means
for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")
    g[c] = g[c].fillna(g[c].mean())

# 5) PPP projection layer
PPP_LEAGUE = 1.13
HOME_EDGE_PTS = 2.0

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

# 6) 25/75 market blend (NBA standard in our pipeline)
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

g["spread_home_line"] = pd.to_numeric(g["spread_home_line"], errors="coerce")
g["total_line"] = pd.to_numeric(g["total_line"], errors="coerce")

g["market_margin"] = -g["spread_home_line"]

g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * g["total_line"]

games_df = g

print("✅ Projection + blend rebuilt")
print("Margin range:", float(games_df['mean_margin'].min()), "to", float(games_df['mean_margin'].max()))
print("Total range:", float(games_df['mean_total'].min()), "to", float(games_df['mean_total'].max()))
games_df[["home_team","away_team","spread_home_line","total_line","mean_margin","mean_total"]].head()

In [ ]:
# B2B: teams that played yesterday (based on completed scores list)
recent_games = historical_df.copy()
recent_games["date"] = pd.to_datetime(recent_games["date"]).dt.date

yesterday = (datetime.strptime(SLATE_DATE, "%Y-%m-%d") - timedelta(days=1)).date()

teams_played_yday = set(recent_games.loc[recent_games["date"] == yesterday, "home_team"]).union(
    set(recent_games.loc[recent_games["date"] == yesterday, "away_team"])
)

games_df["home_b2b"] = games_df["home_team"].isin(teams_played_yday)
games_df["away_b2b"] = games_df["away_team"].isin(teams_played_yday)

games_df["mean_margin"] = games_df["mean_margin"] - games_df["home_b2b"]*B2B_PENALTY + games_df["away_b2b"]*B2B_PENALTY

# Blowout compression
large_spread = games_df["spread_home_line"].abs() >= BLOWOUT_THRESHOLD
games_df.loc[large_spread, "mean_margin"] *= BLOWOUT_MARGIN_MULT
games_df.loc[large_spread, "mean_total"] += BLOWOUT_TOTAL_BUMP

print("✅ B2B + blowout applied")
print("Home B2B:", int(games_df["home_b2b"].sum()), "| Away B2B:", int(games_df["away_b2b"].sum()))
print("Large spread games:", int(large_spread.sum()))

In [ ]:
rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_away_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home only)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig
ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["pick_team"] = ml_card["home_team"]
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side) — FIXED sign (use feed lines as-is)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], sp["spread_away_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["pick_team"] = sp_card["team"]
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].apply(fmt_line) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["pick_team"] = ""  # totals not tied to a team for double-dip rule
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    return x[["matchup","market","bet","pick_team","units","p_win","ev","commence_time"]]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# cap: max 2 plays per matchup (side+total or ML+total, etc.)
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1
card = pd.DataFrame(kept).reset_index(drop=True)

# rule: no ML + Spread on same team (within same matchup) -> drop lower EV
def enforce_no_ml_spread_double_dip(df_card: pd.DataFrame) -> pd.DataFrame:
    out_rows = []
    for matchup, grp in df_card.groupby("matchup", sort=False):
        ml_rows = grp[grp["market"]=="ML"]
        sp_rows = grp[grp["market"]=="Spread"]
        if len(ml_rows) and len(sp_rows):
            ml_team = ml_rows.iloc[0]["pick_team"]
            sp_team = sp_rows.iloc[0]["pick_team"]
            if ml_team == sp_team and ml_team != "":
                # keep the higher EV one, drop the other
                keep_idx = grp["ev"].astype(float).idxmax()
                grp = grp.loc[[keep_idx] + [i for i in grp.index if i!=keep_idx and grp.loc[i,"market"]=="Total"]]
                grp = grp.sort_values("ev", ascending=False)
        out_rows.append(grp)
    return pd.concat(out_rows, ignore_index=True).sort_values("ev", ascending=False).reset_index(drop=True)

card = enforce_no_ml_spread_double_dip(card)

top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(lambda x: f"{round(float(x)*100,1)}%") +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ Final plays:", len(top))
print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_FINAL_CARD.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

In [ ]:
# ======================================
# HARD SLATE LOCK — NBA 3/2 ONLY
# ======================================

TARGET_DATE = "2026-03-02"

import pandas as pd

# Ensure datetime
games_df["commence_time"] = pd.to_datetime(games_df["commence_time"], utc=True)

# Convert to Eastern (NBA logic)
games_df["commence_et"] = games_df["commence_time"].dt.tz_convert("US/Eastern")

games_df["game_date"] = games_df["commence_et"].dt.strftime("%Y-%m-%d")

# STRICT FILTER
before = len(games_df)
games_df = games_df[games_df["game_date"] == TARGET_DATE].copy()
after = len(games_df)

print("=================================")
print("SLATE LOCK APPLIED")
print("Before:", before)
print("After :", after)
print("Target:", TARGET_DATE)
print("=================================")

games_df[["away_team","home_team","commence_et"]]

In [ ]:
import requests, pandas as pd
from datetime import datetime, timedelta
import pytz

TARGET_DATE_ET = "2026-03-02"   # the slate date you mean (Eastern day)

ET = pytz.timezone("US/Eastern")
start_et = ET.localize(datetime.strptime(TARGET_DATE_ET, "%Y-%m-%d"))
end_et = start_et + timedelta(days=1)

start_utc = start_et.astimezone(pytz.utc)
end_utc = end_et.astimezone(pytz.utc)

print("ET window:", start_et, "to", end_et)
print("UTC window:", start_utc, "to", end_utc)

url = f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)

data = r.json()

rows = []
for g in data:
    home = g.get("home_team")
    away = g.get("away_team")
    t = g.get("commence_time")

    bks = g.get("bookmakers", [])
    book = bks[0] if bks else None

    h2h = spreads = totals = None
    if book:
        for m in book.get("markets", []):
            if m.get("key") == "h2h":
                h2h = m
            elif m.get("key") == "spreads":
                spreads = m
            elif m.get("key") == "totals":
                totals = m

    def pick_outcome(mkt, name):
        if not mkt: return None
        for o in mkt.get("outcomes", []):
            if o.get("name") == name:
                return o
        return None

    home_ml = away_ml = float("nan")
    if h2h:
        oh = pick_outcome(h2h, home)
        oa = pick_outcome(h2h, away)
        if oh: home_ml = oh.get("price", float("nan"))
        if oa: away_ml = oa.get("price", float("nan"))

    spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = float("nan")
    if spreads:
        oh = pick_outcome(spreads, home)
        oa = pick_outcome(spreads, away)
        if oh:
            spread_home_line = oh.get("point", float("nan"))
            spread_home_odds = oh.get("price", float("nan"))
        if oa:
            spread_away_line = oa.get("point", float("nan"))
            spread_away_odds = oa.get("price", float("nan"))

    total_line = over_odds = under_odds = float("nan")
    if totals:
        oo = pick_outcome(totals, "Over")
        ou = pick_outcome(totals, "Under")
        if oo:
            total_line = oo.get("point", float("nan"))
            over_odds = oo.get("price", float("nan"))
        if ou:
            under_odds = ou.get("price", float("nan"))

    rows.append({
        "home_team": home,
        "away_team": away,
        "commence_time": t,
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": spread_home_line,
        "spread_home_odds": spread_home_odds,
        "spread_away_line": spread_away_line,
        "spread_away_odds": spread_away_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds,
    })

games_df = pd.DataFrame(rows)

# Add ET time + filter (should already be correct, but we keep it consistent)
games_df["commence_time"] = pd.to_datetime(games_df["commence_time"], utc=True)
games_df["commence_et"] = games_df["commence_time"].dt.tz_convert("US/Eastern")
games_df["game_date"] = games_df["commence_et"].dt.strftime("%Y-%m-%d")
games_df = games_df[games_df["game_date"] == TARGET_DATE_ET].copy()

print(f"✅ ET-slate pull complete. Games: {len(games_df)}")
games_df[["away_team","home_team","commence_et","home_ml","spread_home_line","total_line"]].sort_values("commence_et")

In [ ]:
# ======================================
# NBA STABLE RUN — STRICT (3/2)
# ======================================

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL = 18.0

HOME_EDGE = 2.0
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

g = games_df.copy()

# ===== PROJECTION LAYER =====
g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_pts"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_pts"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_pts"] - g["away_pts"] + HOME_EDGE
g["mean_total_model"] = g["home_pts"] + g["away_pts"]

# ===== MARKET BLEND =====
g["market_margin"] = -g["spread_home_line"]

g["mean_margin"] = (
    BLEND_MODEL * g["mean_margin_model"] +
    BLEND_MARKET * g["market_margin"]
)

g["mean_total"] = (
    BLEND_MODEL * g["mean_total_model"] +
    BLEND_MARKET * g["total_line"]
)

# ===== B2B ADJUSTMENT =====
if "home_b2b" in g.columns:
    g.loc[g["home_b2b"]==1, "mean_margin"] -= 1.5
if "away_b2b" in g.columns:
    g.loc[g["away_b2b"]==1, "mean_margin"] += 1.5

# ===== SIM ENGINE =====
results = []

for _, row in g.iterrows():
    margin_sims = np.random.normal(row["mean_margin"], SD_MARGIN, SIMS)
    total_sims  = np.random.normal(row["mean_total"], SD_TOTAL, SIMS)

    # Moneyline
    home_win_prob = (margin_sims > 0).mean()
    away_win_prob = 1 - home_win_prob

    # Spread
    spread_prob = (margin_sims > row["spread_home_line"]).mean()

    # Total
    over_prob = (total_sims > row["total_line"]).mean()

    results.append({
        "away_team": row["away_team"],
        "home_team": row["home_team"],
        "home_ml": row["home_ml"],
        "away_ml": row["away_ml"],
        "spread_home_line": row["spread_home_line"],
        "spread_home_odds": row["spread_home_odds"],
        "total_line": row["total_line"],
        "over_odds": row["over_odds"],
        "under_odds": row["under_odds"],
        "home_win_prob": home_win_prob,
        "away_win_prob": away_win_prob,
        "spread_prob": spread_prob,
        "over_prob": over_prob
    })

sim_df = pd.DataFrame(results)

# ===== EV + KELLY =====
def american_to_decimal(odds):
    return 1 + (odds/100 if odds > 0 else 100/abs(odds))

def ev_calc(prob, odds):
    dec = american_to_decimal(odds)
    return prob*dec - 1

def kelly(prob, odds):
    dec = american_to_decimal(odds)
    b = dec - 1
    return max((prob*b - (1-prob))/b, 0)

plays = []

for _, r in sim_df.iterrows():

    # Home ML
    ev = ev_calc(r["home_win_prob"], r["home_ml"])
    if ev > 0:
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "ML",
            "bet": f"{r['home_team']} ML ({r['home_ml']})",
            "prob": r["home_win_prob"],
            "ev": ev,
            "units": min(kelly(r["home_win_prob"], r["home_ml"]), 1.0)
        })

    # Spread
    ev = ev_calc(r["spread_prob"], r["spread_home_odds"])
    if ev > 0:
        line = r["spread_home_line"]
        sign = "+" if line > 0 else ""
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Spread",
            "bet": f"{r['home_team']} {sign}{line} ({r['spread_home_odds']})",
            "prob": r["spread_prob"],
            "ev": ev,
            "units": min(kelly(r["spread_prob"], r["spread_home_odds"]), 1.0)
        })

    # Over
    ev = ev_calc(r["over_prob"], r["over_odds"])
    if ev > 0:
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Total",
            "bet": f"OVER {r['total_line']} ({r['over_odds']})",
            "prob": r["over_prob"],
            "ev": ev,
            "units": min(kelly(r["over_prob"], r["over_odds"]), 1.0)
        })

card = pd.DataFrame(plays)

# ===== CAP LOGIC =====
card = card.sort_values("ev", ascending=False)

final = []
for matchup in card["matchup"].unique():
    sub = card[card["matchup"]==matchup]
    # allow max 2 per game
    final.append(sub.head(2))

final_card = pd.concat(final)

# Discord text
final_card["discord_text"] = (
    final_card["matchup"] + "\n" +
    final_card["bet"] + " — " +
    final_card["units"].round(2).astype(str) + "u\n" +
    "Sim Win%: " + (final_card["prob"]*100).round(1).astype(str) +
    "% | EV: " + (final_card["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ Sims:", SIMS)
print("✅ Final plays:", len(final_card))

final_card[["discord_text"]]

In [ ]:
import pandas as pd
import numpy as np

# Requires: games_df (fresh ET slate pull) and team_overall (PPP baseline)
if "team_overall" not in globals():
    raise NameError("team_overall not found. Run your PPP baseline build cell first.")

g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

# Fill missing team stats with league means (keeps run stable even if a team didn’t appear in history window)
for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")
    g[c] = g[c].fillna(g[c].mean())

games_df = g

print("✅ PPP columns merged into slate")
print(games_df[["away_team","home_team","h_poss","a_poss","h_ppp_for","a_ppp_for"]].head())

In [ ]:
# ======================================
# FULL STRICT ENGINE CELL — NBA (drop-in)
# Requires already defined: games_df (ET-slate odds), team_overall (PPP baseline), SLATE_DATE (YYYY-MM-DD)
# Uses fixed spread sign logic (home_line/home_odds + away_line/away_odds as-is)
# Outputs: Sim Win% + Market% + EV% + units + discord_text
# Rules: max 2 plays per game; no ML+Spread same team; totals allowed with side
# ======================================

import numpy as np
import pandas as pd

# ---- SETTINGS (keep consistent) ----
SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

HOME_EDGE_PTS = 2.0

BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

B2B_PENALTY_PTS = 1.5          # applied to mean_margin
BLOWOUT_SPREAD_TH = 8.0        # if abs(spread)>=8, compress margin + shave total
BLOWOUT_MARGIN_MULT = 0.90
BLOWOUT_TOTAL_BUMP  = -1.5

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200               # filter extreme negative prices if desired

rng = np.random.default_rng(42)

# ---- Helpers ----
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds: float) -> float:
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def ev_unit(prob: float, odds: float) -> float:
    dec = american_to_decimal(odds)
    return prob*(dec-1) - (1-prob)

def kelly_fraction(prob: float, odds: float) -> float:
    dec = american_to_decimal(odds)
    b = dec - 1
    if b <= 0:
        return 0.0
    return max((prob*dec - 1)/b, 0.0)

def to_units(prob: float, odds: float) -> float:
    # half-Kelly, scaled into 0.25–1.0u band
    f = 0.5 * kelly_fraction(prob, odds)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ---- 0) Preconditions ----
if "games_df" not in globals():
    raise NameError("games_df not found (run the ET-slate odds pull first).")
if "team_overall" not in globals():
    raise NameError("team_overall not found (run the PPP baseline build first).")

g = games_df.copy()

# Ensure numeric
num_cols = [
    "home_ml","away_ml",
    "spread_home_line","spread_home_odds","spread_away_line","spread_away_odds",
    "total_line","over_odds","under_odds"
]
for c in num_cols:
    if c in g.columns:
        g[c] = pd.to_numeric(g[c], errors="coerce")

# ---- 1) Merge PPP baseline into slate if missing ----
need_cols = {"h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"}
if not need_cols.issubset(set(g.columns)):
    g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
        "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
        "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")
        g[c] = g[c].fillna(g[c].mean())

# ---- 2) Projection layer (PPP/tempo proxy) ----
g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_pts_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_pts_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_pts_proj"] - g["away_pts_proj"] + HOME_EDGE_PTS
g["mean_total_model"]  = g["home_pts_proj"] + g["away_pts_proj"]

# ---- 3) Market blend (25/75) ----
g["market_margin"] = -g["spread_home_line"]
g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * g["total_line"]

# ---- 4) Context layer: B2B + blowout ----
# B2B flags should exist if you ran the B2B cell; if not, default False
if "home_b2b" not in g.columns:
    g["home_b2b"] = False
if "away_b2b" not in g.columns:
    g["away_b2b"] = False

g["mean_margin"] = g["mean_margin"] - g["home_b2b"].astype(int)*B2B_PENALTY_PTS + g["away_b2b"].astype(int)*B2B_PENALTY_PTS

blow = g["spread_home_line"].abs() >= BLOWOUT_SPREAD_TH
g.loc[blow, "mean_margin"] = g.loc[blow, "mean_margin"] * BLOWOUT_MARGIN_MULT
g.loc[blow, "mean_total"]  = g.loc[blow, "mean_total"]  + BLOWOUT_TOTAL_BUMP

# ---- 5) Sims ----
df = g.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# ---- 6) Market implied (for output) ----
# ML: vig-removed implied
ml_base = df.dropna(subset=["home_ml","away_ml"]).copy()
if len(ml_base):
    ml_base["home_imp"] = ml_base["home_ml"].apply(american_to_prob)
    ml_base["away_imp"] = ml_base["away_ml"].apply(american_to_prob)
    vig = (ml_base["home_imp"] + ml_base["away_imp"]).replace(0, np.nan)
    ml_base["home_mkt"] = ml_base["home_imp"]/vig
else:
    ml_base = df.copy()
    ml_base["home_mkt"] = np.nan

# Spread: use price implied for selected side (not vig-removed; consistent with prior)
# Total: same

# ---- 7) Build candidates (ML/Spread/Total) ----
plays = []

# ML: home only (consistent with prior runs)
for _, r in ml_base.iterrows():
    ev = ev_unit(r["p_home_win"], r["home_ml"])
    if (ev > 0) and (r["home_ml"] >= MAX_JUICE):
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "ML",
            "pick_team": r["home_team"],
            "bet": f"{r['home_team']} ML ({fmt_odds(r['home_ml'])})",
            "p_win": float(r["p_home_win"]),
            "p_mkt": float(r["home_mkt"]) if pd.notna(r["home_mkt"]) else np.nan,
            "ev": float(ev),
            "units": to_units(float(r["p_home_win"]), float(r["home_ml"])),
            "commence_time": r.get("commence_time", None),
        })

# Spread: choose better side using EV; FIXED sign using feed lines
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
if len(sp):
    sp["ev_home"] = sp.apply(lambda r: ev_unit(r["p_home_cover"], r["spread_home_odds"]), axis=1)
    sp["ev_away"] = sp.apply(lambda r: ev_unit(r["p_away_cover"], r["spread_away_odds"]), axis=1)

    for _, r in sp.iterrows():
        if r["ev_home"] <= 0 and r["ev_away"] <= 0:
            continue

        if r["ev_home"] >= r["ev_away"]:
            team = r["home_team"]
            line = r["spread_home_line"]
            odds = r["spread_home_odds"]
            pwin = r["p_home_cover"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_home"]
        else:
            team = r["away_team"]
            line = r["spread_away_line"]
            odds = r["spread_away_odds"]
            pwin = r["p_away_cover"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_away"]

        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Spread",
            "pick_team": team,
            "bet": f"{team} {fmt_line(line)} ({fmt_odds(odds)})",
            "p_win": float(pwin),
            "p_mkt": float(pmkt),
            "ev": float(evv),
            "units": to_units(float(pwin), float(odds)),
            "commence_time": r.get("commence_time", None),
        })

# Total: choose better side by EV
to = df.dropna(subset=["over_odds","under_odds"]).copy()
if len(to):
    to["ev_over"]  = to.apply(lambda r: ev_unit(r["p_over"],  r["over_odds"]), axis=1)
    to["ev_under"] = to.apply(lambda r: ev_unit(r["p_under"], r["under_odds"]), axis=1)

    for _, r in to.iterrows():
        if r["ev_over"] <= 0 and r["ev_under"] <= 0:
            continue

        if r["ev_over"] >= r["ev_under"]:
            side = "OVER"
            odds = r["over_odds"]
            pwin = r["p_over"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_over"]
        else:
            side = "UNDER"
            odds = r["under_odds"]
            pwin = r["p_under"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_under"]

        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Total",
            "pick_team": "",
            "bet": f"{side} {r['total_line']} ({fmt_odds(odds)})",
            "p_win": float(pwin),
            "p_mkt": float(pmkt),
            "ev": float(evv),
            "units": to_units(float(pwin), float(odds)),
            "commence_time": r.get("commence_time", None),
        })

card = pd.DataFrame(plays)
if card.empty:
    print("⚠ No +EV plays found.")
    card

# ---- 8) Filter + sort ----
card = card[(card["units"] > 0)].copy()
card = card.sort_values("ev", ascending=False).reset_index(drop=True)

# ---- 9) Exposure rules ----
# (a) max 2 plays per matchup
kept = []
counts = {}
for _, r in card.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1
card2 = pd.DataFrame(kept).reset_index(drop=True)

# (b) no ML + Spread on same team in same matchup -> keep higher EV, keep totals
out_rows = []
for matchup, grp in card2.groupby("matchup", sort=False):
    ml_rows = grp[grp["market"]=="ML"]
    sp_rows = grp[grp["market"]=="Spread"]
    if len(ml_rows) and len(sp_rows):
        if ml_rows.iloc[0]["pick_team"] == sp_rows.iloc[0]["pick_team"] and ml_rows.iloc[0]["pick_team"] != "":
            keep_idx = grp["ev"].astype(float).idxmax()
            grp = grp.loc[[keep_idx] + grp.index[grp["market"]=="Total"].tolist()]
            grp = grp.sort_values("ev", ascending=False)
    out_rows.append(grp)

final_card = pd.concat(out_rows, ignore_index=True).sort_values("ev", ascending=False).reset_index(drop=True)

# ---- 10) Discord text ----
final_card["discord_text"] = (
    final_card["matchup"] + "\n" +
    final_card["bet"] + " — " + final_card["units"].astype(str) + "u\n" +
    "Sim Win%: " + final_card["p_win"].apply(pct) +
    " | Market: " + final_card["p_mkt"].apply(lambda x: "NA" if pd.isna(x) else pct(x)) +
    "\nEV: " + (final_card["ev"]*100).round(1).astype(str) + "%\n"
)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Final plays:", len(final_card))
print(final_card[["discord_text"]].to_string(index=False))

# ---- 11) Export ----
try:
    out_name = f"nba_stable_{globals().get('SLATE_DATE','slate')}_FULL_STRICT_CARD.csv"
    final_card.to_csv(out_name, index=False)
    print("✅ Exported:", out_name)
except Exception as e:
    print("⚠ Export failed:", e)

In [ ]:
import os, numpy as np, pandas as pd
from datetime import datetime, timedelta
import pytz, requests

# Reset key objects so we don't mix old state
for k in ["games_df","df","final_card","injury_overrides","b2b_map","blowout_map"]:
    if k in globals():
        del globals()[k]

# Settings (match last run)
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

print("✅ Settings loaded:", SPORT, SLATE_DATE, "| sims:", SIMS, "| SD:", SD_MARGIN, SD_TOTAL, "| Blend:", BLEND_MODEL, BLEND_MARKET)

In [ ]:
ET = pytz.timezone("US/Eastern")
UTC = pytz.utc

start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et   = start_et + timedelta(days=1)
start_utc = start_et.astimezone(UTC)
end_utc   = end_et.astimezone(UTC)

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)
data = r.json()

rows = []
for g in data:
    home = g["home_team"]; away = g["away_team"]
    t = pd.to_datetime(g["commence_time"], utc=True)

    # use first bookmaker
    book = g["bookmakers"][0] if g.get("bookmakers") else None
    if not book:
        continue

    def mkt(key):
        for m in book["markets"]:
            if m["key"] == key: return m
        return None

    def pick(market, name):
        if not market: return None
        for o in market["outcomes"]:
            if o["name"] == name: return o
        return None

    h2h, spr, tot = mkt("h2h"), mkt("spreads"), mkt("totals")

    # ML
    home_ml = away_ml = None
    oh = pick(h2h, home); oa = pick(h2h, away)
    home_ml = oh["price"] if oh else None
    away_ml = oa["price"] if oa else None

    # Spreads
    sh_line = sh_odds = sa_line = sa_odds = None
    oh = pick(spr, home); oa = pick(spr, away)
    sh_line = oh["point"] if oh else None
    sh_odds = oh["price"] if oh else None
    sa_line = oa["point"] if oa else None
    sa_odds = oa["price"] if oa else None

    # Totals
    total_line = over_odds = under_odds = None
    oo = pick(tot, "Over"); ou = pick(tot, "Under")
    total_line = oo["point"] if oo else None
    over_odds  = oo["price"] if oo else None
    under_odds = ou["price"] if ou else None

    rows.append({
        "away_team": away,
        "home_team": home,
        "commence_time": t,
        "commence_et": t.tz_convert("US/Eastern"),
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": sh_line,
        "spread_home_odds": sh_odds,
        "spread_away_line": sa_line,
        "spread_away_odds": sa_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds,
    })

games_df = pd.DataFrame(rows)
num_cols = ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_line","spread_away_odds","total_line","over_odds","under_odds"]
for c in num_cols:
    games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

def implied_prob(american):
    if pd.isna(american): return np.nan
    a = float(american)
    return (-a)/(-a+100) if a < 0 else 100/(a+100)

games_df["home_ml_implied"] = games_df["home_ml"].apply(implied_prob)
games_df["away_ml_implied"] = games_df["away_ml"].apply(implied_prob)

print("✅ ET-slate pull complete. Games:", len(games_df))
games_df[["away_team","home_team","commence_et","home_ml","away_ml","spread_home_line","total_line"]].sort_values("commence_et")

In [ ]:
# ========= Helpers =========
def american_to_decimal(odds):
    o = float(odds)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def ev_unit(p_win, american_odds):
    d = american_to_decimal(american_odds)
    return p_win*(d-1) - (1-p_win)

def half_kelly(p_win, american_odds, max_u=1.0, min_u=0.25):
    d = american_to_decimal(american_odds)
    b = d - 1
    q = 1 - p_win
    f = (b*p_win - q)/b
    f = max(0.0, f) * 0.5
    # map fraction to unit scale (simple)
    u = f * 5.0
    return float(min(max_u, max(min_u, u)))

# ========= Projection Layer =========
g = games_df.copy()

# Market margin from spread (home perspective)
g["market_margin"] = -g["spread_home_line"]

# If you already built mean_margin/mean_total earlier in the notebook, keep them.
# Otherwise start from market as baseline and blend (matches our last run behavior when projection tables are missing).
if "mean_margin" not in g.columns:
    g["mean_margin"] = g["market_margin"].copy()

if "mean_total" not in g.columns:
    g["mean_total"] = g["total_line"].copy()

# Blend (same as last run)
g["mean_margin"] = BLEND_MODEL*g["mean_margin"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total"]  + BLEND_MARKET*g["total_line"]

# ========= Back-to-back + Blowout risk hooks =========
# NOTE: keep logic identical to last run unless you provide overrides.
# You can manually populate these maps if you have news:
b2b_map = {}      # e.g. {"Boston Celtics": -0.5}  # points off margin
blowout_map = {}  # e.g. {"Houston Rockets": -0.3} # total adjustment

g["b2b_adj_home"] = g["home_team"].map(b2b_map).fillna(0.0)
g["b2b_adj_away"] = g["away_team"].map(b2b_map).fillna(0.0)

# apply b2b to margin (home better if away b2b, worse if home b2b)
g["mean_margin"] = g["mean_margin"] + (g["b2b_adj_away"] - g["b2b_adj_home"])

# blowout impacts totals only (optional)
g["mean_total"] = g["mean_total"] + g["home_team"].map(blowout_map).fillna(0.0) + g["away_team"].map(blowout_map).fillna(0.0)

# ========= Monte Carlo Sims =========
g = g.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

rng = np.random.default_rng(42)
margins = rng.normal(g["mean_margin"].to_numpy()[:,None], SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(g["mean_total"].to_numpy()[:,None],  SD_TOTAL,  size=(len(g), SIMS))

g["p_home_win"] = (margins > 0).mean(axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

# spread cover prob (home spread line: home -x covers if margin > x; home +x covers if margin > -x)
g["p_home_cover"] = (margins > g["spread_home_line"].to_numpy()[:,None]).mean(axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

# totals
g["p_over"]  = (totals > g["total_line"].to_numpy()[:,None]).mean(axis=1)
g["p_under"] = 1 - g["p_over"]

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")

# ========= Build plays =========
plays = []

for i,row in g.iterrows():
    matchup = f"{row['away_team']} at {row['home_team']}"

    # ML
    if pd.notna(row["home_ml"]):
        p = float(row["p_home_win"])
        ev = ev_unit(p, row["home_ml"])
        plays.append({"matchup":matchup,"market":"ML","bet":f"{row['home_team']} ML ({int(row['home_ml'])})",
                      "p_win":p,"p_mkt":float(row["home_ml_implied"]), "ev":ev, "odds":row["home_ml"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})
    if pd.notna(row["away_ml"]):
        p = float(row["p_away_win"])
        ev = ev_unit(p, row["away_ml"])
        plays.append({"matchup":matchup,"market":"ML","bet":f"{row['away_team']} ML ({int(row['away_ml'])})",
                      "p_win":p,"p_mkt":float(row["away_ml_implied"]), "ev":ev, "odds":row["away_ml"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})

    # Spread (use away line for away bet)
    if pd.notna(row["spread_home_line"]) and pd.notna(row["spread_home_odds"]):
        # Home spread bet: home line as shown
        p = float(row["p_home_cover"])
        ev = ev_unit(p, row["spread_home_odds"])
        sign = "+" if row["spread_home_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{row['home_team']} {sign}{row['spread_home_line']} ({int(row['spread_home_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["spread_home_odds"]), "ev":ev, "odds":row["spread_home_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})
    if pd.notna(row["spread_away_line"]) and pd.notna(row["spread_away_odds"]):
        p = float(row["p_away_cover"])
        ev = ev_unit(p, row["spread_away_odds"])
        sign = "+" if row["spread_away_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{row['away_team']} {sign}{row['spread_away_line']} ({int(row['spread_away_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["spread_away_odds"]), "ev":ev, "odds":row["spread_away_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})

    # Totals
    if pd.notna(row["total_line"]) and pd.notna(row["over_odds"]):
        p = float(row["p_over"])
        ev = ev_unit(p, row["over_odds"])
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"OVER {row['total_line']} ({int(row['over_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["over_odds"]), "ev":ev, "odds":row["over_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})
    if pd.notna(row["total_line"]) and pd.notna(row["under_odds"]):
        p = float(row["p_under"])
        ev = ev_unit(p, row["under_odds"])
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"UNDER {row['total_line']} ({int(row['under_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["under_odds"]), "ev":ev, "odds":row["under_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})

plays_df = pd.DataFrame(plays)

# Keep +EV only (same behavior as last)
plays_df = plays_df[plays_df["ev"] > 0].copy()

# Units (half Kelly) with 0.25–1.0u bounds
plays_df["units"] = plays_df.apply(lambda r: half_kelly(r["p_win"], r["odds"]), axis=1)

# ========= Exposure cap =========
# Rule: max 2 picks per game; allow side+total; avoid ML+spread same team in same game
plays_df = plays_df.sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

final_rows = []
by_game = {}

for _,r in plays_df.iterrows():
    game = r["matchup"]
    by_game.setdefault(game, [])

    # max 2 per game
    if len(by_game[game]) >= 2:
        continue

    # block ML+SPREAD same team in same game
    if r["market"] in ["ML","SPREAD"]:
        team = None
        if r["market"] == "ML":
            team = r["bet"].split(" ML")[0]
        else:
            # spread bet starts with team name
            team = r["bet"].split(" (")[0]
            # team name is everything before last space (line)
            team = " ".join(team.split(" ")[:-1])

        for existing in by_game[game]:
            if existing["market"] in ["ML","SPREAD"]:
                ex_team = existing.get("team_tag")
                if ex_team == team:
                    team = team  # no-op
                    break
        else:
            pass

    # side+total is ok; ML+spread same team is blocked:
    if r["market"] in ["ML","SPREAD"]:
        team_tag = None
        if r["market"] == "ML":
            team_tag = r["bet"].split(" ML")[0]
        else:
            team_tag = " ".join(r["bet"].split(" (")[0].split(" ")[:-1])

        conflict = any((ex["market"] in ["ML","SPREAD"] and ex.get("team_tag")==team_tag) for ex in by_game[game])
        if conflict:
            continue
        r = r.to_dict()
        r["team_tag"] = team_tag
        by_game[game].append(r)
        final_rows.append(r)
    else:
        r = r.to_dict()
        r["team_tag"] = ""
        by_game[game].append(r)
        final_rows.append(r)

final_card = pd.DataFrame(final_rows).sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

# ========= Discord Text =========
def fmt_pct(x): return f"{100*x:.1f}%"

final_card["discord_text"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {r['units']:.2f}u\n"
    f"Sim Win%: {fmt_pct(r['p_win'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*r['ev']:.1f}%\n", axis=1)

print("✅ Final plays:", len(final_card))
final_card[["discord_text"]].head(50)

In [ ]:
# ======================================
# OFFICIAL NBA INJURY REPORT -> TEAM STATUS MAP
# (drop-in, no hunting)
# ======================================

import pandas as pd, re, requests
from datetime import datetime
import pdfplumber

# Latest official PDF for today (you can swap time if needed)
INJURY_PDF_URL = "https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-02_02_00PM.pdf"

pdf_path = "/content/nba_injury_report_2026-03-02.pdf"
with open(pdf_path, "wb") as f:
    f.write(requests.get(INJURY_PDF_URL, timeout=30).content)

# Teams on our slate
slate_teams = set(games_df["home_team"]).union(set(games_df["away_team"]))

# Parse PDF text
lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        txt = page.extract_text() or ""
        for ln in txt.split("\n"):
            lines.append(ln.strip())

# Build a simple table of (team, player, status)
# We capture: "... <Team> <Player Name> <Status> ..."
# Statuses in NBA report typically include: Out, Doubtful, Questionable, Probable, Available
status_pat = re.compile(r"\b(Out|Doubtful|Questionable|Probable|Available)\b", re.IGNORECASE)

rows = []
for ln in lines:
    m = status_pat.search(ln)
    if not m:
        continue
    status = m.group(1).title()

    # try to locate which slate team appears in line
    team_hit = None
    for t in slate_teams:
        if t in ln:
            team_hit = t
            break
    if not team_hit:
        continue

    # crude player extraction: text between team name and status word
    after_team = ln.split(team_hit, 1)[-1].strip()
    player_part = after_team.split(status, 1)[0].strip(" -;:")
    player = player_part if player_part else "UNKNOWN"

    rows.append({"team": team_hit, "player": player, "status": status, "raw": ln})

inj_df = pd.DataFrame(rows).drop_duplicates(subset=["team","player","status"])

print("✅ Official injury rows (slate teams):", len(inj_df))
display(inj_df.head(30))

In [ ]:
# ======================================
# MANUAL INJURY OVERRIDES (CONSISTENT INPUT)
# ======================================

# Add confirmed statuses here from UnderdogNBA / official reports
# Format:
# "Team": {"OUT": n, "DOUBTFUL": n, "QUESTIONABLE": n}

injury_override = {
    # Example — replace with today's confirmed news
    # "Boston Celtics": {"OUT": 1, "QUESTIONABLE": 1},
    # "Milwaukee Bucks": {"OUT": 0, "QUESTIONABLE": 2},
}

STATUS_MARGIN = {
    "OUT": -2.5,
    "DOUBTFUL": -1.5,
    "QUESTIONABLE": -0.8,
}

STATUS_TOTAL = {
    "OUT": -0.8,
    "DOUBTFUL": -0.5,
    "QUESTIONABLE": -0.25,
}

team_margin_adj = {}
team_total_adj = {}

for team in set(games_df["home_team"]).union(set(games_df["away_team"])):
    team_margin_adj[team] = 0.0
    team_total_adj[team] = 0.0

for team, statuses in injury_override.items():
    for status, count in statuses.items():
        team_margin_adj[team] += STATUS_MARGIN.get(status, 0.0) * count
        team_total_adj[team]  += STATUS_TOTAL.get(status, 0.0) * count

print("✅ Injury adjustments applied:")
for t in team_margin_adj:
    if team_margin_adj[t] != 0:
        print(t, "margin:", team_margin_adj[t], "| total:", team_total_adj[t])

In [ ]:
import os, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta
import pytz

# Required
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"

# Must match last run
SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

# Blend (same as last run)
BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

print("✅ Settings:", SPORT, SLATE_DATE, "| Sims:", SIMS, "| SD:", SD_MARGIN, SD_TOTAL, "| Blend:", BLEND_MODEL, BLEND_MARKET)

In [ ]:
ET = pytz.timezone("US/Eastern")
UTC = pytz.utc

start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et   = start_et + timedelta(days=1)
start_utc = start_et.astimezone(UTC)
end_utc   = end_et.astimezone(UTC)

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)
data = r.json()

rows = []
for g in data:
    home = g["home_team"]; away = g["away_team"]
    t = pd.to_datetime(g["commence_time"], utc=True)

    # grab first bookmaker
    if not g.get("bookmakers"):
        continue
    book = g["bookmakers"][0]

    def mkt(key):
        for m in book["markets"]:
            if m["key"] == key: return m
        return None

    def pick(market, name):
        if not market: return None
        for o in market["outcomes"]:
            if o["name"] == name: return o
        return None

    h2h, spr, tot = mkt("h2h"), mkt("spreads"), mkt("totals")

    oh, oa = pick(h2h, home), pick(h2h, away)
    home_ml = oh["price"] if oh else None
    away_ml = oa["price"] if oa else None

    oh, oa = pick(spr, home), pick(spr, away)
    sh_line = oh["point"] if oh else None
    sh_odds = oh["price"] if oh else None
    sa_line = oa["point"] if oa else None
    sa_odds = oa["price"] if oa else None

    oo, ou = pick(tot, "Over"), pick(tot, "Under")
    total_line = oo["point"] if oo else None
    over_odds  = oo["price"] if oo else None
    under_odds = ou["price"] if ou else None

    rows.append({
        "away_team": away,
        "home_team": home,
        "commence_time": t,
        "commence_et": t.tz_convert("US/Eastern"),
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": sh_line,
        "spread_home_odds": sh_odds,
        "spread_away_line": sa_line,
        "spread_away_odds": sa_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds
    })

games_df = pd.DataFrame(rows)
num_cols = ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_line","spread_away_odds","total_line","over_odds","under_odds"]
for c in num_cols:
    games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

print("✅ Games pulled:", len(games_df))
games_df[["away_team","home_team","commence_et","home_ml","spread_home_line","total_line"]].sort_values("commence_et")

In [ ]:
def implied_prob(american):
    if pd.isna(american): return np.nan
    a = float(american)
    return (-a)/(-a+100) if a < 0 else 100/(a+100)

def american_to_decimal(odds):
    o = float(odds)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def ev_unit(p_win, american_odds):
    d = american_to_decimal(american_odds)
    return p_win*(d-1) - (1-p_win)

def half_kelly_units(p_win, american_odds, min_u=0.25, max_u=1.0):
    d = american_to_decimal(american_odds)
    b = d - 1
    q = 1 - p_win
    f = (b*p_win - q)/b
    f = max(0.0, f) * 0.5  # half kelly
    u = f * 5.0           # scale
    return float(min(max_u, max(min_u, u)))

In [ ]:
# Add the injury news you saw (counts only)
injury_override = {
    # "Boston Celtics": {"OUT": 1, "QUESTIONABLE": 1},
}

STATUS_MARGIN = {"OUT": -2.5, "DOUBTFUL": -1.5, "QUESTIONABLE": -0.8}
STATUS_TOTAL  = {"OUT": -0.8, "DOUBTFUL": -0.5, "QUESTIONABLE": -0.25}

teams = set(games_df["home_team"]).union(set(games_df["away_team"]))
team_margin_adj = {t: 0.0 for t in teams}
team_total_adj  = {t: 0.0 for t in teams}

for team, statuses in injury_override.items():
    for status, count in statuses.items():
        team_margin_adj[team] += STATUS_MARGIN.get(status, 0.0) * count
        team_total_adj[team]  += STATUS_TOTAL.get(status, 0.0) * count

print("✅ Injury adj ready (non-zero only):")
for t in sorted(teams):
    if team_margin_adj[t] != 0 or team_total_adj[t] != 0:
        print(t, "| margin:", team_margin_adj[t], "| total:", team_total_adj[t])

In [ ]:
g = games_df.copy()

# Market baseline
g["market_margin"] = -g["spread_home_line"].astype(float)

# Base projections (market-based unless you have a projection layer)
g["mean_margin"] = g["market_margin"].copy()
g["mean_total"]  = g["total_line"].astype(float).copy()

# Blend layer (same as last run)
g["mean_margin"] = BLEND_MODEL*g["mean_margin"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total"]  + BLEND_MARKET*g["total_line"].astype(float)

# Injury layer
g["inj_home"] = g["home_team"].map(team_margin_adj).fillna(0.0)
g["inj_away"] = g["away_team"].map(team_margin_adj).fillna(0.0)
g["mean_margin"] = g["mean_margin"] + (g["inj_away"] - g["inj_home"])

g["mean_total"] = g["mean_total"] \
    + g["home_team"].map(team_total_adj).fillna(0.0) \
    + g["away_team"].map(team_total_adj).fillna(0.0)

# Drop any incomplete rows
g = g.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# Monte Carlo
rng = np.random.default_rng(42)
margins = rng.normal(g["mean_margin"].to_numpy()[:,None], SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(g["mean_total"].to_numpy()[:,None],  SD_TOTAL,  size=(len(g), SIMS))

g["p_home_win"] = (margins > 0).mean(axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

g["p_home_cover"] = (margins > g["spread_home_line"].to_numpy()[:,None]).mean(axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

g["p_over"]  = (totals > g["total_line"].to_numpy()[:,None]).mean(axis=1)
g["p_under"] = 1 - g["p_over"]

# Build plays list
plays = []
for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # ML
    if pd.notna(r["home_ml"]):
        p = float(r["p_home_win"]); odds = r["home_ml"]
        plays.append({"matchup":matchup,"market":"ML","bet":f"{r['home_team']} ML ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})
    if pd.notna(r["away_ml"]):
        p = float(r["p_away_win"]); odds = r["away_ml"]
        plays.append({"matchup":matchup,"market":"ML","bet":f"{r['away_team']} ML ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})

    # Spread (already correct sign from API)
    if pd.notna(r["spread_home_odds"]):
        p = float(r["p_home_cover"]); odds = r["spread_home_odds"]
        sign = "+" if r["spread_home_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{r['home_team']} {sign}{r['spread_home_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})
    if pd.notna(r["spread_away_odds"]):
        p = float(r["p_away_cover"]); odds = r["spread_away_odds"]
        sign = "+" if r["spread_away_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{r['away_team']} {sign}{r['spread_away_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})

    # Totals
    if pd.notna(r["over_odds"]):
        p = float(r["p_over"]); odds = r["over_odds"]
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"OVER {r['total_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})
    if pd.notna(r["under_odds"]):
        p = float(r["p_under"]); odds = r["under_odds"]
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"UNDER {r['total_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})

plays_df = pd.DataFrame(plays)

# +EV only + unit sizing
plays_df = plays_df[plays_df["ev"] > 0].copy()
plays_df["units"] = plays_df.apply(lambda x: half_kelly_units(x["p_win"], x["odds"]), axis=1)

# Exposure cap: max 2 plays per game, allow side+total, block ML+spread same team
plays_df = plays_df.sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

final = []
picked = {}

def team_from_bet(bet, market):
    if market == "ML":
        return bet.split(" ML")[0]
    if market == "SPREAD":
        core = bet.split(" (")[0]
        return " ".join(core.split(" ")[:-1])
    return ""

for _, r in plays_df.iterrows():
    game = r["matchup"]
    picked.setdefault(game, [])

    if len(picked[game]) >= 2:
        continue

    t = team_from_bet(r["bet"], r["market"])
    if r["market"] in ["ML","SPREAD"]:
        if any((x["market"] in ["ML","SPREAD"] and x.get("team")==t) for x in picked[game]):
            continue

    entry = r.to_dict()
    entry["team"] = t
    picked[game].append(entry)
    final.append(entry)

final_card = pd.DataFrame(final).sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

# Discord text
final_card["discord_text"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {r['units']:.2f}u\n"
    f"Sim Win%: {100*r['p_win']:.1f}% | Market: {100*r['p_mkt']:.1f}%\n"
    f"EV: {100*r['ev']:.1f}%\n", axis=1)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Final plays:", len(final_card))
final_card[["discord_text"]].head(50)

In [ ]:
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

In [ ]:
 need = ["p_home_win","p_away_win","p_home_cover","p_over","ev","units","discord_text"]
missing = [c for c in need if c not in final_card.columns]

print("Rows in final_card:", len(final_card))
print("Missing required STRICT-layer columns:", missing)

print("Sims check (should exist):", "SIMS" in globals(), "| SIMS =", globals().get("SIMS"))

In [ ]:
# ======================================
# FINALIZE + EXPORT (FULL LAYERS INCLUDED)
# Paste at bottom and run once
# ======================================

import pandas as pd

# Safety checks
if "final_card" not in globals():
    raise ValueError("final_card not found. Re-run the FULL STRICT engine cell first.")
if "g" not in globals():
    raise ValueError("g (sim dataframe) not found. Re-run the FULL STRICT engine cell first.")
if "SLATE_DATE" not in globals():
    SLATE_DATE = "2026-03-02"

# Build matchup key in g if missing
if "matchup" not in g.columns:
    if not {"away_team","home_team"}.issubset(g.columns):
        raise ValueError("g is missing away_team/home_team needed to build matchup key.")
    g = g.copy()
    g["matchup"] = g["away_team"] + " at " + g["home_team"]

# Keep only the sim probability layers we need
sim_cols = ["matchup","p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]
missing_sim = [c for c in sim_cols if c not in g.columns]
if missing_sim:
    raise ValueError(f"g is missing sim columns: {missing_sim}. Re-run the FULL STRICT engine cell.")

sim_layers = g[sim_cols].drop_duplicates("matchup")

# Merge into final_card
final_card = final_card.merge(sim_layers, on="matchup", how="left")

# Confirm we have the required layers now
need = ["p_home_win","p_away_win","p_home_cover","p_over"]
missing_after = [c for c in need if c not in final_card.columns or final_card[c].isna().all()]
print("✅ Merge complete. Missing/empty after merge:", missing_after)

# Export
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_COMPLETE.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

# Quick preview
display(final_card.head(25))

In [ ]:
# ======================================
# REBUILD DISCORD_TEXT WITH CORRECT SIM WIN%
# (uses merged p_home_win / p_home_cover / p_over, etc.)
# ======================================

def fmt_pct(x):
    return f"{100*float(x):.1f}%"

def detect_side_team(bet: str):
    # "Boston Celtics -2.0 (-110)" -> "Boston Celtics"
    return bet.split(" (")[0].rsplit(" ", 1)[0]

def detect_line(bet: str):
    # returns numeric line if present, else None
    core = bet.split(" (")[0]
    try:
        return float(core.rsplit(" ", 1)[-1])
    except:
        return None

def market_simwin(row):
    m = row["market"]
    bet = row["bet"]

    if m == "ML":
        team = bet.split(" ML")[0]
        if team == row["home_team"] if "home_team" in row else False:
            return row["p_home_win"]
        # safer: compare to matchup string
        home = row["matchup"].split(" at ")[1]
        away = row["matchup"].split(" at ")[0]
        return row["p_home_win"] if team == home else row["p_away_win"]

    if m == "SPREAD":
        side_team = detect_side_team(bet)
        home = row["matchup"].split(" at ")[1]
        away = row["matchup"].split(" at ")[0]
        return row["p_home_cover"] if side_team == home else row["p_away_cover"]

    if m == "TOTAL":
        if bet.startswith("OVER"):
            return row["p_over"]
        if bet.startswith("UNDER"):
            return row["p_under"]

    return row.get("p_win", np.nan)

# Ensure we have matchup + probability cols
need = ["matchup","market","bet","units","p_mkt","ev","p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]
missing = [c for c in need if c not in final_card.columns]
if missing:
    raise ValueError(f"final_card missing columns: {missing}")

# Add helper columns
final_card = final_card.copy()
final_card["sim_win_pct"] = final_card.apply(market_simwin, axis=1)

# Rebuild discord text
final_card["discord_text_fixed"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {float(r['units']):.2f}u\n"
    f"Sim Win%: {fmt_pct(r['sim_win_pct'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*float(r['ev']):.1f}%\n"
, axis=1)

print("\n=== DISCORD CARD (FIXED SIM WIN%) ===\n")
for t in final_card["discord_text_fixed"].tolist():
    print(t)

# Export fixed version too
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_COMPLETE_FIXEDTEXT.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

In [ ]:
g[["away_team","home_team","mean_margin","spread_home_line","mean_total"]]

In [ ]:
# ======================================
# HOTFIX AT BOTTOM: CORRECT SPREAD PROBS (NO HUNTING)
# - fixes p_home_cover / p_away_cover sign logic
# - updates SPREAD rows in final_card + discord text
# ======================================

import numpy as np
import pandas as pd

# ---- safety ----
if "g" not in globals():
    raise ValueError("Missing 'g' (sim dataframe). Re-run FULL STRICT engine cell first.")
if "final_card" not in globals():
    raise ValueError("Missing 'final_card'. Re-run FULL STRICT engine + merge/export cells first.")
if "margins" not in globals():
    raise ValueError("Missing 'margins' sim matrix. Re-run FULL STRICT engine cell first.")

# ---- ensure matchup in g ----
gg = g.copy()
if "matchup" not in gg.columns:
    gg["matchup"] = gg["away_team"] + " at " + gg["home_team"]

# ---- recompute correct cover probs ----
spread_home = gg["spread_home_line"].to_numpy()[:, None].astype(float)

# Home covers if (home - away) > -home_spread
gg["p_home_cover_fix"] = (margins > -spread_home).mean(axis=1)
gg["p_away_cover_fix"] = 1.0 - gg["p_home_cover_fix"]

# ---- helper funcs (if not already in notebook) ----
def implied_prob(american):
    if pd.isna(american): return np.nan
    a = float(american)
    return (-a)/(-a+100) if a < 0 else 100/(a+100)

def american_to_decimal(odds):
    o = float(odds)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def ev_unit(p_win, american_odds):
    d = american_to_decimal(american_odds)
    return p_win*(d-1) - (1-p_win)

def half_kelly_units(p_win, american_odds, min_u=0.25, max_u=1.0):
    d = american_to_decimal(american_odds)
    b = d - 1
    q = 1 - p_win
    f = (b*p_win - q)/b
    f = max(0.0, f) * 0.5
    u = f * 5.0
    return float(min(max_u, max(min_u, u)))

def detect_side_team(bet: str):
    # "Boston Celtics -2.0 (-110)" -> "Boston Celtics"
    return bet.split(" (")[0].rsplit(" ", 1)[0]

def fmt_pct(x):
    return f"{100*float(x):.1f}%"

# ---- map fixed cover probs into final_card SPREAD rows ----
fix_map = gg[["matchup","p_home_cover_fix","p_away_cover_fix"]].drop_duplicates("matchup")

fc = final_card.copy()
fc = fc.merge(fix_map, on="matchup", how="left")

# Determine whether spread bet is home side or away side, then assign corrected p_win
def corrected_spread_pwin(row):
    if row["market"] != "SPREAD":
        return row["p_win"]
    home = row["matchup"].split(" at ")[1]
    side_team = detect_side_team(row["bet"])
    return row["p_home_cover_fix"] if side_team == home else row["p_away_cover_fix"]

fc["p_win"] = fc.apply(corrected_spread_pwin, axis=1)

# Recompute EV/market/units ONLY for SPREAD rows
is_spread = fc["market"] == "SPREAD"
fc.loc[is_spread, "p_mkt"] = fc.loc[is_spread, "odds"].apply(implied_prob)
fc.loc[is_spread, "ev"] = fc.loc[is_spread].apply(lambda r: ev_unit(r["p_win"], r["odds"]), axis=1)
fc.loc[is_spread, "units"] = fc.loc[is_spread].apply(lambda r: half_kelly_units(r["p_win"], r["odds"]), axis=1)

# Rebuild discord text (correct win% by market)
def market_simwin(row):
    if row["market"] == "ML":
        team = row["bet"].split(" ML")[0]
        home = row["matchup"].split(" at ")[1]
        return row["p_home_win"] if team == home else row["p_away_win"]
    if row["market"] == "SPREAD":
        return row["p_win"]
    if row["market"] == "TOTAL":
        return row["p_over"] if row["bet"].startswith("OVER") else row["p_under"]
    return row["p_win"]

fc["sim_win_pct"] = fc.apply(market_simwin, axis=1)

fc["discord_text_fixed"] = fc.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {float(r['units']):.2f}u\n"
    f"Sim Win%: {fmt_pct(r['sim_win_pct'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*float(r['ev']):.1f}%\n"
, axis=1)

# Replace final_card
final_card = fc.drop(columns=["p_home_cover_fix","p_away_cover_fix"], errors="ignore")

print("✅ Spread prob sign hotfix applied.")
print("Preview:")
display(final_card[["matchup","market","bet","p_win","p_mkt","ev","units"]].head(20))

# Print Discord-ready card
print("\n=== DISCORD CARD (POST-HOTFIX) ===\n")
for t in final_card["discord_text_fixed"].tolist():
    print(t)

# Export
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_COMPLETE_HOTFIX.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

In [ ]:
# ======================================
# FINAL POSTABLE CARD: +EV ONLY + EXPOSURE CAP + DISCORD PRINT
# ======================================

import pandas as pd

fc = final_card.copy()

# Keep +EV only
fc = fc[fc["ev"] > 0].copy()

# Re-apply exposure rules (max 2 per game; allow side+total; block ML+spread same team)
def team_from_bet(bet, market):
    if market == "ML":
        return bet.split(" ML")[0]
    if market == "SPREAD":
        core = bet.split(" (")[0]
        return " ".join(core.split(" ")[:-1])
    return ""

fc = fc.sort_values(["ev","sim_win_pct"], ascending=False).reset_index(drop=True)

picked = {}
final = []

for _, r in fc.iterrows():
    game = r["matchup"]
    picked.setdefault(game, [])

    if len(picked[game]) >= 2:
        continue

    t = team_from_bet(r["bet"], r["market"])
    if r["market"] in ["ML","SPREAD"]:
        if any((x["market"] in ["ML","SPREAD"] and x.get("team") == t) for x in picked[game]):
            continue

    entry = r.to_dict()
    entry["team"] = t
    picked[game].append(entry)
    final.append(entry)

post_card = pd.DataFrame(final).sort_values(["ev","sim_win_pct"], ascending=False).reset_index(drop=True)

print("✅ POSTABLE plays:", len(post_card))
print("\n=== DISCORD CARD (POSTABLE / +EV ONLY) ===\n")
for t in post_card["discord_text_fixed"].tolist():
    print(t)

out_path = f"nba_stable_{SLATE_DATE}_POSTABLE_CARD.csv"
post_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

In [ ]:
# =========================
# NBA STABLE — STRICT RESET
# =========================

SLATE_DATE = "2026-03-03"
SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL = 18.0
BLEND_MODEL = 0.5
BLEND_MARKET = 0.5
HALF_KELLY = True

print("✅ STRICT settings locked")
print("Date:", SLATE_DATE, "| Sims:", SIMS)

In [ ]:
# =========================
# SET ODDS API KEY (STRICT)
# =========================

import os

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"
os.environ["ODDS_API_KEY"] = ODDS_API_KEY

print("✅ ODDS_API_KEY set")

In [ ]:
# =========================
# ET SLATE PULL (3/3)
# =========================

import pandas as pd
import requests
from datetime import datetime
import pytz

ET = pytz.timezone("US/Eastern")

start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et = start_et + pd.Timedelta(days=1)

start_utc = start_et.astimezone(pytz.utc)
end_utc = end_et.astimezone(pytz.utc)

print("ET window:", start_et, "to", end_et)
print("UTC window:", start_utc, "to", end_utc)

params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso"
}

r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds", params=params)
games = r.json()

rows = []
for g in games:
    commence = pd.to_datetime(g["commence_time"])
    if start_utc <= commence <= end_utc:
        home = g["home_team"]
        away = g["away_team"]

        book = g["bookmakers"][0]["markets"]

        data = {
            "away_team": away,
            "home_team": home,
            "commence_utc": commence
        }

        for m in book:
            if m["key"] == "h2h":
                for o in m["outcomes"]:
                    if o["name"] == home:
                        data["home_ml"] = o["price"]
                    else:
                        data["away_ml"] = o["price"]

            if m["key"] == "spreads":
                for o in m["outcomes"]:
                    if o["name"] == home:
                        data["spread_home_line"] = o["point"]
                        data["spread_home_odds"] = o["price"]

            if m["key"] == "totals":
                for o in m["outcomes"]:
                    if o["name"] == "Over":
                        data["total_line"] = o["point"]

        rows.append(data)

games_df = pd.DataFrame(rows)

print("✅ Games pulled:", len(games_df))
games_df.head()

In [ ]:
# =========================
# CELL A — COMPLETED SCORES (STRICT / VALID daysFrom)
# =========================
import pandas as pd, requests, time

SPORT = "basketball_nba"

def pull_completed_scores(days_from: int):
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/scores"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": days_from, "dateFormat": "iso"}
    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise Exception(r.text)
    data = r.json()
    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g["home_team"]
        away = g["away_team"]
        hs = g.get("scores", [{}])[0].get("score")
        as_ = g.get("scores", [{}, {}])[1].get("score") if len(g.get("scores", [])) > 1 else None
        if hs is None or as_ is None:
            continue
        rows.append({
            "home_team": home,
            "away_team": away,
            "home_score": float(hs),
            "away_score": float(as_),
            "commence_time": g.get("commence_time")
        })
    return pd.DataFrame(rows)

# Only use safe values; expand later if they work in your account
SAFE_DAYS = [3]
parts = []
for d in SAFE_DAYS:
    try:
        part = pull_completed_scores(d)
        print(f"✅ daysFrom {d}: {len(part)} games")
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed:", str(e)[:140])

games_hist = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["home_team","away_team","commence_time"])
print("✅ Unique completed games:", len(games_hist))
games_hist.head()

In [ ]:
# =========================
# CELL B — TEAM PPP/POSS TABLE (STRICT)
# =========================
import numpy as np
import pandas as pd

# Approx possessions from score environment using league-average points/poss proxy
# We use margin/total sims downstream; here we only need consistent team-strength signals.
games_hist["total_pts"] = games_hist["home_score"] + games_hist["away_score"]
games_hist["margin_home"] = games_hist["home_score"] - games_hist["away_score"]

# Possession proxy: normalize by league avg PPP ~ 1.12 (NBA typical)
LEAGUE_PPP = 1.12
games_hist["poss_proxy"] = games_hist["total_pts"] / LEAGUE_PPP

# Team points per poss proxy (for/against)
home_rows = pd.DataFrame({
    "team": games_hist["home_team"],
    "ppp_for": games_hist["home_score"] / games_hist["poss_proxy"],
    "ppp_against": games_hist["away_score"] / games_hist["poss_proxy"],
    "poss": games_hist["poss_proxy"]
})
away_rows = pd.DataFrame({
    "team": games_hist["away_team"],
    "ppp_for": games_hist["away_score"] / games_hist["poss_proxy"],
    "ppp_against": games_hist["home_score"] / games_hist["poss_proxy"],
    "poss": games_hist["poss_proxy"]
})

team_overall = pd.concat([home_rows, away_rows], ignore_index=True)\
    .groupby("team", as_index=False)\
    .agg(ppp_for=("ppp_for","mean"),
         ppp_against=("ppp_against","mean"),
         poss=("poss","mean"))

print("✅ team_overall built:", team_overall.shape)
team_overall.head()

In [ ]:
# =========================
# CELL C — MERGE + PROJECTIONS (STRICT)
# =========================
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

# Fill any missing teams with league means so the engine never breaks
for c in ["h_poss","a_poss","h_ppp_for","a_ppp_for","h_ppp_against","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["mean_total"] = (g["home_ppp_proj"] + g["away_ppp_proj"]) * g["poss_proj"]
g["mean_margin"] = (g["home_ppp_proj"] - g["away_ppp_proj"]) * g["poss_proj"]

# Market blend (same as last run)
g["market_margin"] = -g["spread_home_line"]
g["mean_margin"] = BLEND_MODEL*g["mean_margin"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total"]  + BLEND_MARKET*g["total_line"]

print("✅ Projections built")
print("Margin range:", float(g["mean_margin"].min()), "to", float(g["mean_margin"].max()))
print("Total range :", float(g["mean_total"].min()),  "to", float(g["mean_total"].max()))
g[["away_team","home_team","mean_margin","spread_home_line","mean_total","total_line","home_ml","away_ml"]].head()

In [ ]:
# =========================
# CELL D — FULL STRICT ENGINE (ML/SPREAD/TOTAL) + KELLY
# =========================
import numpy as np
import pandas as pd
from math import erf, sqrt

def american_to_prob(a):
    a = float(a)
    if a > 0:
        return 100.0/(a+100.0)
    return (-a)/((-a)+100.0)

def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1 + a/100.0
    return 1 + 100.0/(-a)

def kelly_fraction(p, dec_odds):
    b = dec_odds - 1
    q = 1 - p
    f = (b*p - q)/b
    return max(0.0, f)

rng = np.random.default_rng(7)

rows = []
for _,r in g.iterrows():
    # simulate margin and total
    sim_margin = rng.normal(loc=r["mean_margin"], scale=SD_MARGIN, size=SIMS)
    sim_total  = rng.normal(loc=r["mean_total"],  scale=SD_TOTAL,  size=SIMS)

    # Win prob (home wins if margin > 0)
    p_home_win = float((sim_margin > 0).mean())
    p_away_win = 1 - p_home_win

    # Spread cover: home spread line is usually negative for fav.
    sh = float(r["spread_home_line"])
    # Home covers if (home_score - away_score) > -sh
    p_home_cover = float((sim_margin > -sh).mean())
    p_away_cover = 1 - p_home_cover

    # Total
    tl = float(r["total_line"])
    p_over  = float((sim_total > tl).mean())
    p_under = 1 - p_over

    # Market implied
    # ML
    if pd.notna(r.get("home_ml")) and pd.notna(r.get("away_ml")):
        p_home_mkt = american_to_prob(r["home_ml"])
        p_away_mkt = american_to_prob(r["away_ml"])
    else:
        p_home_mkt = np.nan
        p_away_mkt = np.nan

    # Spread odds (use home odds for both sides when symmetrical; if away odds missing, reuse)
    ho = float(r.get("spread_home_odds", -110))
    ao = float(r.get("spread_away_odds", -110)) if "spread_away_odds" in r else ho

    p_sp_home_mkt = american_to_prob(ho)
    p_sp_away_mkt = american_to_prob(ao)

    # Total odds
    oo = float(r.get("over_odds", -110)) if "over_odds" in r else -110
    uo = float(r.get("under_odds", -110)) if "under_odds" in r else -110
    p_over_mkt = american_to_prob(oo)
    p_under_mkt = american_to_prob(uo)

    # Build candidates: ML, Spread, Total
    matchup = f"{r['away_team']} at {r['home_team']}"
    commence = r.get("commence_utc")

    # ML candidates (only if we have MLs)
    if pd.notna(r.get("home_ml")) and pd.notna(r.get("away_ml")):
        # Home ML
        dec = american_to_decimal(r["home_ml"])
        ev = p_home_win*(dec-1) - (1-p_home_win)
        f = kelly_fraction(p_home_win, dec)
        if HALF_KELLY: f *= 0.5
        rows.append({
            "matchup": matchup, "market": "ML", "team": r["home_team"],
            "bet": f"{r['home_team']} ML ({int(r['home_ml'])})",
            "p_win": p_home_win, "p_mkt": p_home_mkt, "ev": ev,
            "odds": r["home_ml"], "units": min(1.0, max(0.25, f))
        })
        # Away ML
        dec = american_to_decimal(r["away_ml"])
        ev = p_away_win*(dec-1) - (1-p_away_win)
        f = kelly_fraction(p_away_win, dec)
        if HALF_KELLY: f *= 0.5
        rows.append({
            "matchup": matchup, "market": "ML", "team": r["away_team"],
            "bet": f"{r['away_team']} ML ({int(r['away_ml'])})",
            "p_win": p_away_win, "p_mkt": p_away_mkt, "ev": ev,
            "odds": r["away_ml"], "units": min(1.0, max(0.25, f))
        })

    # Spread (home side)
    dec = american_to_decimal(ho)
    ev = p_home_cover*(dec-1) - (1-p_home_cover)
    f = kelly_fraction(p_home_cover, dec)
    if HALF_KELLY: f *= 0.5
    rows.append({
        "matchup": matchup, "market": "SPREAD", "team": r["home_team"],
        "bet": f"{r['home_team']} {sh:+g} ({int(ho)})",
        "p_win": p_home_cover, "p_mkt": p_sp_home_mkt, "ev": ev,
        "odds": ho, "units": min(1.0, max(0.25, f))
    })

    # Total Over/Under
    dec = american_to_decimal(oo)
    ev = p_over*(dec-1) - (1-p_over)
    f = kelly_fraction(p_over, dec)
    if HALF_KELLY: f *= 0.5
    rows.append({
        "matchup": matchup, "market": "TOTAL", "team": "OVER",
        "bet": f"OVER {tl:g} ({int(oo)})",
        "p_win": p_over, "p_mkt": p_over_mkt, "ev": ev,
        "odds": oo, "units": min(1.0, max(0.25, f))
    })

    dec = american_to_decimal(uo)
    ev = p_under*(dec-1) - (1-p_under)
    f = kelly_fraction(p_under, dec)
    if HALF_KELLY: f *= 0.5
    rows.append({
        "matchup": matchup, "market": "TOTAL", "team": "UNDER",
        "bet": f"UNDER {tl:g} ({int(uo)})",
        "p_win": p_under, "p_mkt": p_under_mkt, "ev": ev,
        "odds": uo, "units": min(1.0, max(0.25, f))
    })

final_card = pd.DataFrame(rows)

# +EV only
final_card = final_card[final_card["ev"] > 0].copy()

# Correlation cap: max 2 plays per game
final_card["game_key"] = final_card["matchup"]
final_card = final_card.sort_values(["ev","p_win"], ascending=False)\
    .groupby("game_key").head(2)\
    .reset_index(drop=True)

# Build Discord text
def fmt_line(x):
    return (f"{x['matchup']}\n"
            f"{x['bet']} — {x['units']:.2f}u\n"
            f"Sim Win%: {x['p_win']*100:.1f}% | Market: {x['p_mkt']*100:.1f}%\n"
            f"EV: {x['ev']*100:.1f}%\n")

final_card["discord_text"] = final_card.apply(fmt_line, axis=1)

print("✅ Sims:", SIMS, "| Plays:", len(final_card))
print(final_card[["discord_text"]].head(20))

# Export
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

In [ ]:
# =========================
# STRICT TRIM — QUALITY FILTER
# =========================

EV_FLOOR = 0.04   # 4% minimum edge

postable = final_card[final_card["ev"] > EV_FLOOR].copy()

# Keep max 2 per game (already capped, but re-apply cleanly)
postable = postable.sort_values(["ev","p_win"], ascending=False)\
    .groupby("game_key").head(2)\
    .reset_index(drop=True)

print("✅ Postable plays:", len(postable))
postable[["matchup","bet","p_win","ev","units"]]

In [ ]:
# =========================
# MARKET STABILIZER (STRICT PATCH)
# =========================

BLEND_MODEL = 0.35
BLEND_MARKET = 0.65

g["mean_margin"] = (
    BLEND_MODEL * g["mean_margin"] +
    BLEND_MARKET * (-g["spread_home_line"])
)

g["mean_total"] = (
    BLEND_MODEL * g["mean_total"] +
    BLEND_MARKET * g["total_line"]
)

print("✅ Stronger market anchor applied")
print("New margin range:",
      float(g["mean_margin"].min()),
      "to",
      float(g["mean_margin"].max()))

In [ ]:
# =========================
# BOTTOM RESET (HIST + TEAM OVERALL)
# =========================
import pandas as pd
import numpy as np

SAFE_DAYS = [3, 6, 10, 14, 21, 28]

parts = []
for d in SAFE_DAYS:
    try:
        part = pull_completed_scores(SPORT, d)  # uses your existing function
        print(f"✅ daysFrom {d}: {len(part)} games")
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed — skipping. Reason: {str(e)[:140]}")

if not parts:
    raise RuntimeError("No historical score pulls succeeded. Cannot rebuild team_overall.")

games_hist = pd.concat(parts, ignore_index=True)

# Standardize expected columns if present
keep_cols = [c for c in ["home_team","away_team","home_score","away_score","date","commence_time"] if c in games_hist.columns]
games_hist = games_hist[keep_cols].copy() if keep_cols else games_hist.copy()

# Drop obvious junk / duplicates
for c in ["home_score","away_score"]:
    if c in games_hist.columns:
        games_hist[c] = pd.to_numeric(games_hist[c], errors="coerce")

games_hist = games_hist.dropna(subset=[c for c in ["home_team","away_team","home_score","away_score"] if c in games_hist.columns])
games_hist = games_hist.drop_duplicates(subset=[c for c in ["home_team","away_team","date","commence_time"] if c in games_hist.columns])

print("✅ Unique historical games stored:", len(games_hist))
display(games_hist.head())

In [ ]:
# =========================
# BOTTOM RESET (HIST + TEAM OVERALL) — FIXED SIGNATURE
# =========================
import pandas as pd
import numpy as np

SAFE_DAYS = [3, 6, 10, 14, 21, 28]

parts = []
for d in SAFE_DAYS:
    try:
        # In THIS notebook, pull_completed_scores takes ONLY (days_from)
        part = pull_completed_scores(d)
        print(f"✅ daysFrom {d}: {len(part)} games")
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed — skipping. Reason: {str(e)[:140]}")

if not parts:
    raise RuntimeError("No historical score pulls succeeded. Cannot rebuild team_overall.")

games_hist = pd.concat(parts, ignore_index=True)

# Normalize columns (different notebooks may name them differently)
rename_map = {}
if "homeScore" in games_hist.columns: rename_map["homeScore"] = "home_score"
if "awayScore" in games_hist.columns: rename_map["awayScore"] = "away_score"
if "home_team" not in games_hist.columns and "homeTeam" in games_hist.columns: rename_map["homeTeam"] = "home_team"
if "away_team" not in games_hist.columns and "awayTeam" in games_hist.columns: rename_map["awayTeam"] = "away_team"
if rename_map:
    games_hist = games_hist.rename(columns=rename_map)

need = ["home_team","away_team","home_score","away_score"]
missing = [c for c in need if c not in games_hist.columns]
if missing:
    raise RuntimeError(f"games_hist missing required columns: {missing}. Columns are: {list(games_hist.columns)[:25]}")

for c in ["home_score","away_score"]:
    games_hist[c] = pd.to_numeric(games_hist[c], errors="coerce")

games_hist = games_hist.dropna(subset=["home_team","away_team","home_score","away_score"])
games_hist = games_hist.drop_duplicates(subset=["home_team","away_team","home_score","away_score"])

print("✅ Unique historical games stored:", len(games_hist))
display(games_hist.head())

# -------------------------
# TEAM_OVERALL BUILD (points-based, stable)
# -------------------------
POSS_PROXY = 99.0  # stable NBA proxy

home = (games_hist.groupby("home_team")
        .agg(ppg_for=("home_score","mean"),
             ppg_against=("away_score","mean"),
             games=("home_score","size"))
        .reset_index().rename(columns={"home_team":"team"}))

away = (games_hist.groupby("away_team")
        .agg(ppg_for=("away_score","mean"),
             ppg_against=("home_score","mean"),
             games=("away_score","size"))
        .reset_index().rename(columns={"away_team":"team"}))

team_overall = pd.concat([home, away], ignore_index=True)
team_overall = (team_overall.groupby("team", as_index=False)
                .apply(lambda x: pd.Series({
                    "ppg_for": np.average(x["ppg_for"], weights=x["games"]),
                    "ppg_against": np.average(x["ppg_against"], weights=x["games"]),
                    "games": x["games"].sum()
                }))
                .reset_index(drop=True))

team_overall["poss"] = POSS_PROXY
team_overall["ppp_for"] = team_overall["ppg_for"] / POSS_PROXY
team_overall["ppp_against"] = team_overall["ppg_against"] / POSS_PROXY

print("✅ team_overall built:", team_overall.shape)
display(team_overall.sort_values("games", ascending=False).head(12))

In [ ]:
# =========================
# MERGE PROJECTIONS INTO games_df + REBUILD MEANS
# =========================
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team","ppg_for","ppg_against","games"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team","ppg_for","ppg_against","games"], errors="ignore")

for col in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
    g[col] = pd.to_numeric(g[col], errors="coerce")
    g[col] = g[col].fillna(g[col].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["mean_total"]  = (g["home_ppp_proj"] + g["away_ppp_proj"]) * g["poss_proj"]
g["mean_margin"] = (g["home_ppp_proj"] - g["away_ppp_proj"]) * g["poss_proj"]  # + => home favored

games_df = g

print("✅ Projection layer rebuilt")
print("mean_margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("mean_total  range:", float(games_df["mean_total"].min()),  "to", float(games_df["mean_total"].max()))
display(games_df[["away_team","home_team","mean_margin","spread_home_line","mean_total","total_line"]].head(10))

In [ ]:
# =========================
# MARKET ANCHOR (CONSISTENT)
# =========================
BLEND_MODEL = 0.35
BLEND_MARKET = 0.65

games_df["mean_margin"] = BLEND_MODEL*games_df["mean_margin"] + BLEND_MARKET*(-games_df["spread_home_line"])
games_df["mean_total"]  = BLEND_MODEL*games_df["mean_total"]  + BLEND_MARKET*(games_df["total_line"])

print("✅ Anchor applied | margin range:",
      float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("✅ Anchor applied | total range:",
      float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

In [ ]:
# =========================
# STRICT IDENTICAL-RUN GATE (HARD FAIL IF NOT IDENTICAL)
# =========================

REQUIRED_FUNCS = [
    "pull_market_data",
    "pull_completed_scores",
]

REQUIRED_COLS_AFTER_PROJ = [
    "h_poss","a_poss",
    "h_ppp_for","h_ppp_against",
    "a_ppp_for","a_ppp_against",
    "poss_proj","home_ppp_proj","away_ppp_proj",
    "mean_margin","mean_total"
]

REQUIRED_COLS_AFTER_ENGINE = [
    "p_home_win","p_away_win","p_home_cover","p_over"
]

missing_funcs = [f for f in REQUIRED_FUNCS if f not in globals()]
if missing_funcs:
    raise RuntimeError(f"STOP — notebook missing required functions for IDENTICAL run: {missing_funcs}")

if "games_df" not in globals():
    raise RuntimeError("STOP — games_df not found. Run the market pull cell first.")

# check projection-layer columns
missing_proj = [c for c in REQUIRED_COLS_AFTER_PROJ if c not in games_df.columns]
if missing_proj:
    raise RuntimeError(
        "STOP — projection layer is NOT identical.\n"
        f"Missing columns: {missing_proj}\n"
        "This means we are NOT using the real possessions/PPP layer.\n"
        "Fix by restoring the original team metrics + projection cells from the last good notebook."
    )

# check strict-engine outputs if final_card exists
if "final_card" in globals() and isinstance(final_card, pd.DataFrame) and len(final_card):
    missing_eng = [c for c in REQUIRED_COLS_AFTER_ENGINE if c not in final_card.columns]
    if missing_eng:
        raise RuntimeError(
            "STOP — strict engine outputs missing.\n"
            f"Missing: {missing_eng}\n"
            "Re-run the FULL STRICT engine cell that computes win/cover/over probabilities."
        )

print("✅ IDENTICAL-RUN GATE PASSED — all required layers present.")

In [ ]:
# =========================
# RESTORE: pull_market_data (Odds API v4) + ET slate window
# =========================
import os, requests, pandas as pd
from datetime import datetime, timedelta
import pytz

def _american_to_implied(odds):
    try:
        o = float(odds)
    except:
        return None
    if o > 0:
        return 100.0 / (o + 100.0)
    else:
        return (-o) / ((-o) + 100.0)

def pull_market_data(
    sport: str,
    date: str,
    outside: bool = True,
    regions: str = "us",
    markets: str = "h2h,spreads,totals",
    odds_format: str = "american",
    preferred_book: str | None = None,
    tz_str: str = "America/Indiana/Indianapolis",
):
    """
    Pull Odds API markets for a single ET-slate date window [00:00 ET, 24:00 ET).
    Returns a games_df with columns used by our STRICT engine:
    away_team, home_team, commence_time, home_ml, away_ml,
    spread_home_line, spread_home_odds, spread_away_line, spread_away_odds,
    total_line, over_odds, under_odds,
    home_ml_implied, away_ml_implied
    """
    if not outside:
        raise RuntimeError("outside=False not supported for this restored cell. Set OUTSIDE_ON=True.")

    ODDS_API_KEY = os.getenv("ODDS_API_KEY")
    if not ODDS_API_KEY:
        raise ValueError("ODDS_API_KEY not found. Set it in env var first (os.environ['ODDS_API_KEY']=...).")

    # ET slate window → UTC window
    tz = pytz.timezone(tz_str)
    day_et = tz.localize(datetime.strptime(date, "%Y-%m-%d"))
    start_et = day_et
    end_et = day_et + timedelta(days=1)
    start_utc = start_et.astimezone(pytz.utc)
    end_utc = end_et.astimezone(pytz.utc)

    print(f"ET window: {start_et} to {end_et}")
    print(f"UTC window: {start_utc} to {end_utc}")

    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
        "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
        "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
    }

    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API failed: {r.status_code} | {r.text[:300]}")

    data = r.json()
    rows = []

    def pick_book(bookmakers):
        if not bookmakers:
            return None
        if preferred_book:
            for b in bookmakers:
                if b.get("key") == preferred_book or b.get("title","").lower() == preferred_book.lower():
                    return b
        # default: first bookmaker
        return bookmakers[0]

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        commence = g.get("commence_time")

        bm = pick_book(g.get("bookmakers", []))
        if not bm:
            continue

        # initialize
        home_ml = away_ml = None
        sh_line = sh_odds = sa_line = sa_odds = None
        total = over = under = None

        for m in bm.get("markets", []):
            k = m.get("key")
            outs = m.get("outcomes", [])

            if k == "h2h":
                for o in outs:
                    if o.get("name") == home:
                        home_ml = o.get("price")
                    elif o.get("name") == away:
                        away_ml = o.get("price")

            elif k == "spreads":
                for o in outs:
                    if o.get("name") == home:
                        sh_line = o.get("point")
                        sh_odds = o.get("price")
                    elif o.get("name") == away:
                        sa_line = o.get("point")
                        sa_odds = o.get("price")

            elif k == "totals":
                for o in outs:
                    if o.get("name","").lower() == "over":
                        total = o.get("point")
                        over = o.get("price")
                    elif o.get("name","").lower() == "under":
                        # total point should match
                        under = o.get("price")

        rows.append({
            "away_team": away,
            "home_team": home,
            "commence_time": commence,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": sh_line,
            "spread_home_odds": sh_odds,
            "spread_away_line": sa_line,
            "spread_away_odds": sa_odds,
            "total_line": total,
            "over_odds": over,
            "under_odds": under,
            "home_ml_implied": _american_to_implied(home_ml) if home_ml is not None else None,
            "away_ml_implied": _american_to_implied(away_ml) if away_ml is not None else None,
        })

    games_df = pd.DataFrame(rows)
    print(f"✅ pull_market_data: pulled {len(games_df)} games for {sport} on {date} (ET window)")

    if len(games_df):
        print("ML non-null:", int(games_df["home_ml"].notna().sum() + games_df["away_ml"].notna().sum())//2, "/", len(games_df))
        print("Spread non-null:", int(games_df["spread_home_line"].notna().sum()), "/", len(games_df))
        print("Total non-null:", int(games_df["total_line"].notna().sum()), "/", len(games_df))

    return games_df

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-03"
OUTSIDE_ON = True

In [ ]:
games_df = pull_market_data(
    sport=SPORT,
    date=SLATE_DATE,
    outside=OUTSIDE_ON
)

games_df.head()

In [ ]:
# =========================
# RESTORE: pull_completed_scores (Odds API v4 scores endpoint)
# =========================
import requests
import pandas as pd

def pull_completed_scores(days_from: int):
    """
    Pull completed NBA scores using Odds API v4.
    Returns a DataFrame with:
    home_team, away_team, home_score, away_score
    """

    ODDS_API_KEY = os.getenv("ODDS_API_KEY")
    if not ODDS_API_KEY:
        raise ValueError("ODDS_API_KEY not set.")

    url = "https://api.the-odds-api.com/v4/sports/basketball_nba/scores"

    params = {
        "apiKey": ODDS_API_KEY,
        "daysFrom": days_from
    }

    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise RuntimeError(f"Scores API failed: {r.status_code} | {r.text[:200]}")

    data = r.json()

    rows = []
    for g in data:
        if not g.get("completed"):
            continue

        scores = g.get("scores", [])
        if len(scores) != 2:
            continue

        rows.append({
            "home_team": g.get("home_team"),
            "away_team": g.get("away_team"),
            "home_score": scores[0].get("score") if scores[0].get("name") == g.get("home_team") else scores[1].get("score"),
            "away_score": scores[1].get("score") if scores[1].get("name") == g.get("away_team") else scores[0].get("score"),
        })

    df = pd.DataFrame(rows)
    print(f"✅ pull_completed_scores({days_from}) → {len(df)} games")
    return df

In [ ]:
# =========================
# REBUILD HISTORICAL DATA CLEAN
# =========================

parts = []
for d in [3, 6, 10, 14, 21, 28]:
    try:
        part = pull_completed_scores(d)
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed:", e)

if not parts:
    raise RuntimeError("No historical score pulls succeeded.")

games_hist = pd.concat(parts, ignore_index=True).drop_duplicates()

print("✅ Historical games rebuilt:", len(games_hist))
games_hist.head()

In [ ]:
# =========================
# PERMANENT FIX: historical rebuild (auto daysFrom cap) + team_overall + merge into games_df
# =========================
import numpy as np
import pandas as pd

def _compute_possessions(df, home_prefix="home_", away_prefix="away_"):
    """
    If we ever have boxscore components, use this.
    For OddsAPI scores-only, we won't. Placeholder kept for compatibility.
    """
    return None

def build_team_overall_from_scores(games_hist: pd.DataFrame):
    """
    Scores-only fallback: estimate team strength from scoring margins + totals.
    This matches what we did in the last NBA stable run when boxscore stats weren't available.
    Produces: team, poss, ppp_for, ppp_against
    """
    gh = games_hist.copy()

    # basic derived signals
    gh["home_margin"] = gh["home_score"] - gh["away_score"]
    gh["away_margin"] = -gh["home_margin"]
    gh["game_total"] = gh["home_score"] + gh["away_score"]

    # approximate possessions proxy (stable): use total points scaled
    # (we only need a consistent poss signal for the projection layer; sims handle variance)
    # clamp to reasonable NBA range
    gh["poss_proxy"] = (gh["game_total"] / 2.25).clip(90, 105)

    # home rows
    home_rows = pd.DataFrame({
        "team": gh["home_team"],
        "ppp_for": gh["home_score"] / gh["poss_proxy"],
        "ppp_against": gh["away_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })

    # away rows
    away_rows = pd.DataFrame({
        "team": gh["away_team"],
        "ppp_for": gh["away_score"] / gh["poss_proxy"],
        "ppp_against": gh["home_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })

    team_game = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = (
        team_game.groupby("team", as_index=False)
        .agg(poss=("poss","mean"), ppp_for=("ppp_for","mean"), ppp_against=("ppp_against","mean"))
    )

    print("✅ team_overall built:", team_overall.shape)
    return team_overall

def rebuild_hist_and_merge_into_games_df(days_try=(3,2,1)):
    # 1) Find max valid daysFrom
    best = None
    for d in days_try:
        try:
            df = pull_completed_scores(d)
            if len(df) >= 10:
                best = d
                break
        except Exception as e:
            print(f"⚠ daysFrom {d} failed:", e)

    if best is None:
        raise RuntimeError("No valid daysFrom found for scores endpoint.")

    print(f"✅ Using daysFrom={best} for historical rebuild (plan constraint safe).")

    # 2) Build historical sample
    games_hist = pull_completed_scores(best).drop_duplicates()
    if len(games_hist) < 10:
        raise RuntimeError(f"Not enough historical games ({len(games_hist)}) to build team_overall reliably.")

    # 3) Build team_overall (scores-only consistent layer)
    team_overall = build_team_overall_from_scores(games_hist)

    # 4) Merge into current slate games_df to produce STRICT layer columns
    if "games_df" not in globals():
        raise RuntimeError("games_df not found. Run the market pull cell first.")

    g = games_df.copy()

    # Merge home
    g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
        "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    # Merge away
    g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
        "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    # Fill any missing with league means (prevents NaNs breaking sims)
    for c in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
        if c in g.columns:
            g[c] = g[c].fillna(g[c].mean())

    # Write back
    globals()["games_hist"] = games_hist
    globals()["team_overall"] = team_overall
    globals()["games_df"] = g

    print("✅ Historical + team_overall merged into games_df.")
    print("Null check:",
          g[["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]].isna().sum().to_dict())
    return g

# Run it
games_df = rebuild_hist_and_merge_into_games_df()
games_df.head()

In [ ]:
# =========================
# HOTFIX: coerce OddsAPI scores to numeric + rebuild team_overall merge
# =========================
import pandas as pd
import numpy as np

def build_team_overall_from_scores(games_hist: pd.DataFrame):
    gh = games_hist.copy()

    # coerce scores to numeric (fix for 'str' subtraction error)
    gh["home_score"] = pd.to_numeric(gh["home_score"], errors="coerce")
    gh["away_score"] = pd.to_numeric(gh["away_score"], errors="coerce")

    gh = gh.dropna(subset=["home_score","away_score"]).copy()
    if gh.empty:
        raise RuntimeError("After coercion, no valid score rows remain.")

    gh["home_margin"] = gh["home_score"] - gh["away_score"]
    gh["away_margin"] = -gh["home_margin"]
    gh["game_total"] = gh["home_score"] + gh["away_score"]

    # possessions proxy (stable + bounded)
    gh["poss_proxy"] = (gh["game_total"] / 2.25).clip(90, 105)

    home_rows = pd.DataFrame({
        "team": gh["home_team"],
        "ppp_for": gh["home_score"] / gh["poss_proxy"],
        "ppp_against": gh["away_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })
    away_rows = pd.DataFrame({
        "team": gh["away_team"],
        "ppp_for": gh["away_score"] / gh["poss_proxy"],
        "ppp_against": gh["home_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })

    team_game = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = (
        team_game.groupby("team", as_index=False)
        .agg(poss=("poss","mean"), ppp_for=("ppp_for","mean"), ppp_against=("ppp_against","mean"))
    )
    print("✅ team_overall rebuilt (scores coerced):", team_overall.shape)
    return team_overall

# Re-run rebuild using your existing games_hist pulled already
if "games_hist" not in globals():
    raise RuntimeError("games_hist not found. Run the historical pull cell first.")

team_overall = build_team_overall_from_scores(games_hist)

g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

games_df = g

print("✅ Merged into games_df. Null check:",
      games_df[["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]].isna().sum().to_dict())

games_df.head()

In [ ]:
# ==============================
# NBA FULL STRICT ENGINE (BOTTOM CELL)
# - No hunting required
# - Builds all STRICT probability columns
# - Runs Monte Carlo sims (ML/Spread/Total)
# - Caps correlated exposure (max 2 picks per game)
# ==============================
import numpy as np
import pandas as pd
from math import erf, sqrt
from datetime import datetime

# ---------- SETTINGS (edit if needed) ----------
SIMS = int(globals().get("SIMS", 50000))
SD_MARGIN = float(globals().get("SD_MARGIN", 14.5))   # spread outcome volatility
SD_TOTAL  = float(globals().get("SD_TOTAL", 18.0))    # total outcome volatility

MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True
MAX_PICKS_PER_GAME = 2

SLATE_DATE = globals().get("SLATE_DATE", None) or globals().get("TARGET_DATE", None) or "slate"
EXPORT_NAME = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_BOTTOM.csv"

# ---------- HELPERS ----------
def american_to_decimal(odds):
    odds = float(odds)
    if odds > 0:
        return 1.0 + (odds / 100.0)
    else:
        return 1.0 + (100.0 / abs(odds))

def american_to_implied_prob(odds):
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return abs(odds) / (abs(odds) + 100.0)

def norm_cdf(x):
    # standard normal CDF using erf
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def kelly_fraction(p, dec_odds):
    # f* = (bp - q)/b where b = dec-1, q=1-p
    b = dec_odds - 1.0
    q = 1.0 - p
    f = (b * p - q) / b if b > 0 else 0.0
    return max(0.0, f)

def clip_units(u):
    if u <= 0:
        return 0.0
    return float(np.clip(u, MIN_U, MAX_U))

def _pick_sign_fix(row):
    """
    Ensure spread bet text + probability mapping is consistent with spread_home_line meaning.
    We treat spread_home_line as: home spread (e.g. -2 means home favored by 2).
    """
    return row

# ---------- INPUT CHECK ----------
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run the market pull / ET slate lock first.")

g = games_df.copy()

need_cols = ["home_team","away_team","spread_home_line","total_line"]
missing = [c for c in need_cols if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}")

# if commence time exists, keep it
if "commence_et" in g.columns:
    g["commence_time"] = g["commence_et"].astype(str)
elif "commence_utc" in g.columns:
    g["commence_time"] = g["commence_utc"].astype(str)
else:
    g["commence_time"] = ""

# ---------- PROJECTION LAYER ----------
# If your notebook already has model projections (mean_margin / mean_total), use them.
# Otherwise default to market anchors (spread/total) as the mean.
if "mean_margin" not in g.columns:
    # mean_margin is HOME margin (home_score - away_score)
    g["mean_margin"] = -g["spread_home_line"].astype(float)

if "mean_total" not in g.columns:
    g["mean_total"] = g["total_line"].astype(float)

# ---------- MONTE CARLO SIMS ----------
rng = np.random.default_rng(7)

# simulate home margin and total per game
n = len(g)
margins = rng.normal(loc=g["mean_margin"].to_numpy(), scale=SD_MARGIN, size=(SIMS, n))
totals  = rng.normal(loc=g["mean_total"].to_numpy(),  scale=SD_TOTAL,  size=(SIMS, n))

# win probs
p_home_win = (margins > 0).mean(axis=0)
p_away_win = 1.0 - p_home_win

g["p_home_win"] = p_home_win
g["p_away_win"] = p_away_win

# spread cover probs:
# Home covers if (home_margin - spread_home_line) > 0
spread_home = g["spread_home_line"].to_numpy(dtype=float)
p_home_cover = ((margins - spread_home) > 0).mean(axis=0)
p_away_cover = 1.0 - p_home_cover
g["p_home_cover"] = p_home_cover
g["p_away_cover"] = p_away_cover

# total probs
total_line = g["total_line"].to_numpy(dtype=float)
p_over = (totals > total_line).mean(axis=0)
p_under = 1.0 - p_over
g["p_over"] = p_over
g["p_under"] = p_under

# ---------- BUILD MARKET ROWS (ML / SPREAD / TOTAL) ----------
rows = []

# ML rows (if ml exists; otherwise skip)
has_ml = ("home_ml" in g.columns) and ("away_ml" in g.columns)
if has_ml:
    for i, r in g.iterrows():
        # HOME ML
        if pd.notna(r.get("home_ml")):
            odds = float(r["home_ml"])
            rows.append({
                "matchup": f'{r["away_team"]} at {r["home_team"]}',
                "game_key": f'{r["away_team"]} @ {r["home_team"]}',
                "market": "ML",
                "team": r["home_team"],
                "bet": f'{r["home_team"]} ML ({int(odds) if odds.is_integer() else odds})',
                "odds": odds,
                "p_win": float(r["p_home_win"]),
                "p_mkt": float(american_to_implied_prob(odds)),
                "commence_time": r["commence_time"],
            })
        # AWAY ML
        if pd.notna(r.get("away_ml")):
            odds = float(r["away_ml"])
            rows.append({
                "matchup": f'{r["away_team"]} at {r["home_team"]}',
                "game_key": f'{r["away_team"]} @ {r["home_team"]}',
                "market": "ML",
                "team": r["away_team"],
                "bet": f'{r["away_team"]} ML ({int(odds) if odds.is_integer() else odds})',
                "odds": odds,
                "p_win": float(r["p_away_win"]),
                "p_mkt": float(american_to_implied_prob(odds)),
                "commence_time": r["commence_time"],
            })

# SPREAD rows (use spread_home_line + both prices if present, else assume -110)
for i, r in g.iterrows():
    sh = float(r["spread_home_line"])
    home_odds = float(r["spread_home_odds"]) if "spread_home_odds" in g.columns and pd.notna(r.get("spread_home_odds")) else -110.0
    away_odds = float(r["spread_away_odds"]) if "spread_away_odds" in g.columns and pd.notna(r.get("spread_away_odds")) else -110.0

    # Home spread bet (e.g. -2 means home -2)
    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "SPREAD",
        "team": r["home_team"],
        "bet": f'{r["home_team"]} {sh:+g} ({int(home_odds)})'.replace("+", ""),
        "odds": home_odds,
        "p_win": float(r["p_home_cover"]),
        "p_mkt": float(american_to_implied_prob(home_odds)),
        "commence_time": r["commence_time"],
    })

    # Away spread is the opposite sign
    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "SPREAD",
        "team": r["away_team"],
        "bet": f'{r["away_team"]} {(-sh):+g} ({int(away_odds)})'.replace("+", ""),
        "odds": away_odds,
        "p_win": float(r["p_away_cover"]),
        "p_mkt": float(american_to_implied_prob(away_odds)),
        "commence_time": r["commence_time"],
    })

# TOTAL rows (assume over_odds/under_odds else -110)
for i, r in g.iterrows():
    tl = float(r["total_line"])
    over_odds  = float(r["over_odds"])  if "over_odds"  in g.columns and pd.notna(r.get("over_odds"))  else -110.0
    under_odds = float(r["under_odds"]) if "under_odds" in g.columns and pd.notna(r.get("under_odds")) else -110.0

    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "TOTAL",
        "team": "OVER",
        "bet": f'OVER {tl:g} ({int(over_odds)})',
        "odds": over_odds,
        "p_win": float(r["p_over"]),
        "p_mkt": float(american_to_implied_prob(over_odds)),
        "commence_time": r["commence_time"],
    })

    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "TOTAL",
        "team": "UNDER",
        "bet": f'UNDER {tl:g} ({int(under_odds)})',
        "odds": under_odds,
        "p_win": float(r["p_under"]),
        "p_mkt": float(american_to_implied_prob(under_odds)),
        "commence_time": r["commence_time"],
    })

card = pd.DataFrame(rows)

# ---------- EV + KELLY UNITS ----------
card["dec_odds"] = card["odds"].astype(float).apply(american_to_decimal)
# Expected value as ROI: EV = p*(dec-1) - (1-p)
card["ev"] = card["p_win"] * (card["dec_odds"] - 1.0) - (1.0 - card["p_win"])

# Kelly sizing (half Kelly default)
card["kelly_f"] = card.apply(lambda x: kelly_fraction(x["p_win"], x["dec_odds"]), axis=1)
if HALF_KELLY:
    card["kelly_f"] = 0.5 * card["kelly_f"]

# Map Kelly fraction to "units" (1.0u = 5% bankroll baseline)
# This keeps sizing consistent across slates.
BANKROLL_UNIT = 0.05
card["units_raw"] = card["kelly_f"] / BANKROLL_UNIT
card["units"] = card["units_raw"].apply(clip_units)

# ---------- FILTER POSTABLE (+EV ONLY) ----------
postable = card[(card["ev"] > 0) & (card["units"] > 0)].copy()

# ---------- CORRELATED EXPOSURE CAP (MAX 2 PICKS PER GAME) ----------
# Rank by a blend: EV then win prob (so we keep strong +EV but not only longshots)
postable["rank_score"] = (postable["ev"] * 0.65) + (postable["p_win"] * 0.35)
postable = postable.sort_values("rank_score", ascending=False)

kept = []
counts = {}
for _, r in postable.iterrows():
    k = r["game_key"]
    counts.setdefault(k, 0)
    if counts[k] >= MAX_PICKS_PER_GAME:
        continue
    kept.append(r)
    counts[k] += 1

final_card = pd.DataFrame(kept).copy()
final_card = final_card.sort_values(["rank_score"], ascending=False)

# ---------- DISCORD TEXT ----------
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_ev(x):  return f"{100*x:.1f}%"

final_card["discord_text"] = final_card.apply(
    lambda r: (
        f'{r["matchup"]}\n'
        f'{r["bet"]} — {r["units"]:.2f}u\n'
        f'Sim Win%: {fmt_pct(r["p_win"])} | Market: {fmt_pct(r["p_mkt"])}\n'
        f'EV: {fmt_ev(r["ev"])}\n'
    ),
    axis=1
)

# ---------- EXPORT ----------
final_card.to_csv(EXPORT_NAME, index=False)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print(f"✅ Final postable plays: {len(final_card)} (cap {MAX_PICKS_PER_GAME} per game)")
print(f"✅ Exported: {EXPORT_NAME}\n")

print("=== DISCORD CARD ===\n")
print("\n".join(final_card["discord_text"].tolist()))

# Keep globals for downstream (dogs, top10, etc.)
globals()["final_card"] = final_card
globals()["strict_probs_df"] = g

In [ ]:
# ==============================
# STRICT MEAN RESET + MARKET ANCHOR (MATCH 3/2 BEHAVIOR)
# ==============================

if "games_df" not in globals():
    raise RuntimeError("games_df not found.")

g = games_df.copy()

# Base projection from team strength if available
if all(c in g.columns for c in ["h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against","h_poss","a_poss"]):
    g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
    g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
    g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

    g["model_margin_raw"] = (g["home_ppp_proj"] - g["away_ppp_proj"]) * g["poss_proj"]
    g["model_total_raw"] = (g["home_ppp_proj"] + g["away_ppp_proj"]) * g["poss_proj"]
else:
    raise RuntimeError("PPP / possession layer missing.")

# MARKET ANCHOR BLEND (same philosophy as 3/2)
ANCHOR_WEIGHT = 0.65  # heavier anchor prevents crazy edges

g["mean_margin"] = (
    ANCHOR_WEIGHT * (-g["spread_home_line"]) +
    (1 - ANCHOR_WEIGHT) * g["model_margin_raw"]
)

g["mean_total"] = (
    ANCHOR_WEIGHT * g["total_line"] +
    (1 - ANCHOR_WEIGHT) * g["model_total_raw"]
)

print("✅ Stronger market anchor applied")
print("New margin range:", g["mean_margin"].min(), "to", g["mean_margin"].max())

games_df = g

In [ ]:
# ==========================================
# STRONG MARKET ANCHOR (IDENTICAL STYLE FIX)
# ==========================================

MARKET_WEIGHT = 0.65  # heavier anchor
MODEL_WEIGHT  = 0.35

games_df["anchored_margin"] = (
    MODEL_WEIGHT  * games_df["mean_margin"] +
    MARKET_WEIGHT * (-games_df["spread_home_line"])
)

games_df["anchored_total"] = (
    MODEL_WEIGHT  * games_df["mean_total"] +
    MARKET_WEIGHT * games_df["total_line"]
)

# Replace original mean inputs
games_df["mean_margin"] = games_df["anchored_margin"]
games_df["mean_total"]  = games_df["anchored_total"]

print("✅ Strong market gravity applied.")
print("New margin range:",
      games_df["mean_margin"].min(),
      "to",
      games_df["mean_margin"].max())

In [ ]:
# =========================
# FULL STRICT ENGINE (BOTTOM RUNNER)
# =========================
import numpy as np
import pandas as pd

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0
MAX_PLAYS_PER_GAME = 2
MIN_UNIT = 0.25
MAX_UNIT = 1.00

def american_to_prob(odds):
    o = float(odds)
    if o == 0 or np.isnan(o): return np.nan
    if o > 0:  return 100.0 / (o + 100.0)
    return (-o) / ((-o) + 100.0)

def prob_to_ev(p, odds):
    o = float(odds)
    if np.isnan(p) or np.isnan(o): return np.nan
    if o > 0:
        b = o/100.0
    else:
        b = 100.0/(-o)
    return p*b - (1-p)

def half_kelly_units(p, odds):
    o = float(odds)
    if np.isnan(p) or np.isnan(o): return MIN_UNIT
    if o > 0:
        b = o/100.0
    else:
        b = 100.0/(-o)
    q = 1-p
    f = (b*p - q)/b
    f = max(0.0, f) * 0.5  # half Kelly
    # map to 0.25–1.0u
    u = MIN_UNIT + f*(MAX_UNIT-MIN_UNIT)
    return float(min(MAX_UNIT, max(MIN_UNIT, u)))

g = games_df.copy()

need_cols = ["away_team","home_team","mean_margin","mean_total","spread_home_line","total_line"]
missing = [c for c in need_cols if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required cols: {missing}")

# sims
rng = np.random.default_rng(7)
margins = rng.normal(loc=g["mean_margin"].values[:,None], scale=SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(loc=g["mean_total"].values[:,None],  scale=SD_TOTAL,  size=(len(g), SIMS))

# probabilities
g["p_home_win"]   = (margins > 0).mean(axis=1)
g["p_away_win"]   = 1 - g["p_home_win"]
g["p_home_cover"] = (margins > (-g["spread_home_line"].values[:,None])).mean(axis=1)  # home cover
g["p_away_cover"] = 1 - g["p_home_cover"]
g["p_over"]       = (totals > (g["total_line"].values[:,None])).mean(axis=1)
g["p_under"]      = 1 - g["p_over"]

# build candidates (ML / Spread / Total)
rows = []

for i, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # ML (choose side with odds available)
    if "home_ml" in g.columns and pd.notna(r.get("home_ml")):
        p = r["p_home_win"]
        odds = r["home_ml"]
        rows.append((matchup,"ML",f"{r['home_team']} ML ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["home_team"]))
    if "away_ml" in g.columns and pd.notna(r.get("away_ml")):
        p = r["p_away_win"]
        odds = r["away_ml"]
        rows.append((matchup,"ML",f"{r['away_team']} ML ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["away_team"]))

    # Spread (home line, away is opposite)
    if "spread_home_odds" in g.columns and pd.notna(r.get("spread_home_odds")):
        odds = r["spread_home_odds"]
        p = r["p_home_cover"]
        rows.append((matchup,"SPREAD",f"{r['home_team']} {r['spread_home_line']} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["home_team"]))
    if "spread_away_odds" in g.columns and pd.notna(r.get("spread_away_odds")):
        odds = r["spread_away_odds"]
        p = r["p_away_cover"]
        away_line = -r["spread_home_line"]
        rows.append((matchup,"SPREAD",f"{r['away_team']} {away_line} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["away_team"]))

    # Totals
    if "over_odds" in g.columns and pd.notna(r.get("over_odds")):
        odds = r["over_odds"]
        p = r["p_over"]
        rows.append((matchup,"TOTAL",f"OVER {r['total_line']} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,"OVER"))
    if "under_odds" in g.columns and pd.notna(r.get("under_odds")):
        odds = r["under_odds"]
        p = r["p_under"]
        rows.append((matchup,"TOTAL",f"UNDER {r['total_line']} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,"UNDER"))

card = pd.DataFrame(rows, columns=["matchup","market","bet","p_win","p_mkt","ev","odds","team"])
card = card.dropna(subset=["p_win","odds","ev"]).copy()

# +EV only
card = card[card["ev"] > 0].copy()

# units
card["units"] = card.apply(lambda x: half_kelly_units(x["p_win"], x["odds"]), axis=1)

# cap 2 per game
card = card.sort_values(["matchup","ev"], ascending=[True, False])
card = card.groupby("matchup").head(MAX_PLAYS_PER_GAME).copy()
card = card.sort_values("ev", ascending=False).reset_index(drop=True)

# discord text
def fmt_pct(x): return f"{100*x:.1f}%"
card["discord_text"] = card.apply(
    lambda x: f"{x['matchup']}\n{x['bet']} — {x['units']:.2f}u\nSim Win%: {fmt_pct(x['p_win'])} | Market: {fmt_pct(x['p_mkt'])}\nEV: {100*x['ev']:.1f}%\n",
    axis=1
)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print(f"✅ Final postable plays: {len(card)} (cap {MAX_PLAYS_PER_GAME} per game)")

# export
out_path = f"nba_stable_2026-03-03_FULL_STRICT_CARD_BOTTOM.csv"
card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

print("\n=== DISCORD CARD ===\n")
print("\n".join(card["discord_text"].tolist()))

In [ ]:
# ==============================
# STRICT STABILIZER PATCH
# ==============================

# 1) Increase anchor weight
ANCHOR_WEIGHT = 0.75

g = games_df.copy()

g["mean_margin"] = (
    ANCHOR_WEIGHT * (-g["spread_home_line"]) +
    (1 - ANCHOR_WEIGHT) * g["model_margin_raw"]
)

g["mean_total"] = (
    ANCHOR_WEIGHT * g["total_line"] +
    (1 - ANCHOR_WEIGHT) * g["model_total_raw"]
)

games_df = g

# 2) Raise variance slightly
SD_MARGIN = 15.5
SD_TOTAL = 18.5

print("✅ Stabilizer applied")
print("New margin range:", g["mean_margin"].min(), "to", g["mean_margin"].max())

In [ ]:
# ==========================================
# BLENDED RANKING: +EV only, favor high p_win
# ==========================================

import numpy as np
import pandas as pd

# expects a dataframe named final_card with at least:
# matchup, bet, market, p_win, p_mkt, ev, units, discord_text
df = final_card.copy()

need = ["matchup","bet","market","p_win","p_mkt","ev","units","discord_text"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"final_card missing columns: {missing}")

# 1) +EV filter (hard gate)
df = df[df["ev"] > 0].copy()

# 2) Remove fake longshot spikes if you want (recommended)
# - eliminates super low win% dogs that look +EV from noise
df = df[(df["p_win"] >= 0.08) & (df["p_win"] <= 0.80)].copy()

# 3) Normalize EV to 0..1 (clip so outliers don't dominate)
ev_clip = df["ev"].clip(lower=0, upper=df["ev"].quantile(0.90))
df["ev_norm"] = (ev_clip - ev_clip.min()) / (ev_clip.max() - ev_clip.min() + 1e-9)

# 4) Normalize win prob to 0..1
df["pwin_norm"] = (df["p_win"] - df["p_win"].min()) / (df["p_win"].max() - df["p_win"].min() + 1e-9)

# 5) Blend weights (tune these)
W_PWIN = 0.65   # “most probable” priority
W_EV   = 0.35   # still rewards +EV

df["blend_score"] = W_PWIN * df["pwin_norm"] + W_EV * df["ev_norm"]

# 6) Correlation cap: max 2 picks per game
df["game_key"] = df["matchup"].astype(str)

df = df.sort_values("blend_score", ascending=False)
kept = []
counts = {}

for _, r in df.iterrows():
    k = r["game_key"]
    if counts.get(k, 0) >= 2:
        continue
    kept.append(r)
    counts[k] = counts.get(k, 0) + 1

df_blend = pd.DataFrame(kept).reset_index(drop=True)

print(f"✅ +EV plays after blend rank + cap: {len(df_blend)}")
df_blend[["matchup","bet","p_win","ev","units","blend_score"]].head(20)

In [ ]:
# =========================
# DISCORD CARD (BLENDED TOP)
# =========================

TOP_N = 10  # set 8 / 10 / 12

out = df_blend.head(TOP_N).copy()

print("=== DISCORD CARD (BLENDED: +EV + PROBABLE) ===\n")
print("\n".join(out["discord_text"].astype(str).tolist()))

# optional export
fname = f"nba_stable_{SLATE_DATE}_BLENDED_TOP{TOP_N}.csv"
out.to_csv(fname, index=False)
print(f"\n✅ Exported: {fname}")

In [ ]:
# ==========================================
# BLENDED RANKING v2 (PROBABILITY-LEANING)
# ==========================================

df = final_card.copy()

# +EV only
df = df[df["ev"] > 0].copy()

# Remove ultra longshots (keeps card sharp)
df = df[df["p_win"] >= 0.18].copy()

# Normalize EV (cap outliers)
ev_clip = df["ev"].clip(upper=df["ev"].quantile(0.85))
df["ev_norm"] = (ev_clip - ev_clip.min()) / (ev_clip.max() - ev_clip.min() + 1e-9)

# Win probability normalized
df["pwin_norm"] = (df["p_win"] - df["p_win"].min()) / (df["p_win"].max() - df["p_win"].min() + 1e-9)

# Stronger probability lean
W_PWIN = 0.75
W_EV   = 0.25

df["blend_score"] = W_PWIN * df["pwin_norm"] + W_EV * df["ev_norm"]

# Correlation cap
df = df.sort_values("blend_score", ascending=False)
kept = []
counts = {}

for _, r in df.iterrows():
    k = r["matchup"]
    if counts.get(k, 0) >= 2:
        continue
    kept.append(r)
    counts[k] = counts.get(k, 0) + 1

df_blend = pd.DataFrame(kept).reset_index(drop=True)

print(f"✅ Blended plays (probability leaning): {len(df_blend)}")
df_blend[["matchup","bet","p_win","ev","units","blend_score"]].head(15)

In [ ]:
TOP_N = 8

out = df_blend.head(TOP_N).copy()

print("=== DISCORD CARD (BLENDED v2: HIGHER HIT RATE) ===\n")
print("\n".join(out["discord_text"].tolist()))

In [ ]:
# ======================================
# HIGH HIT-RATE BRAND CARD (BOTTOM CELL)
# +EV only + probability-first ranking
# suppress longshot ML dogs
# cap 2 picks per game
# ======================================

import numpy as np
import pandas as pd
from datetime import datetime

# ---- EXPECTED INPUT ----
# final_card must exist and include at least:
# matchup, market, bet, odds (or AMERICAN_ODDS), p_win, p_mkt, ev, units, discord_text
df = final_card.copy()

# ---- SAFETY: normalize columns ----
if "odds" not in df.columns and "AMERICAN_ODDS" in df.columns:
    df["odds"] = df["AMERICAN_ODDS"]

need = ["matchup","market","bet","p_win","p_mkt","ev","units"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"final_card missing required columns: {missing}")

# ---- 1) +EV ONLY ----
df = df[df["ev"] > 0].copy()

# ---- 2) HIGH HIT-RATE FILTERS ----
# (A) Global minimum win prob (keeps card from being all lottery tickets)
PWIN_MIN = 0.52
df = df[df["p_win"] >= PWIN_MIN].copy()

# (B) Extra suppression for ML dogs (even if they barely clear PWIN_MIN somehow)
# If ML and american odds >= +200 => require higher p_win
ML_DOG_ODDS_CUTOFF = 200
ML_DOG_PWIN_MIN = 0.58

is_ml = df["market"].astype(str).str.upper().eq("ML")
is_dog = df["odds"].astype(float) >= ML_DOG_ODDS_CUTOFF
df = df[~(is_ml & is_dog & (df["p_win"] < ML_DOG_PWIN_MIN))].copy()

# (C) Optional: remove ultra-thin edges that clutter brand cards
EV_MIN = 0.02  # 2%+
df = df[df["ev"] >= EV_MIN].copy()

# ---- 3) PROBABILITY-FIRST SCORING ----
# Primary = p_win, Secondary = ev (soft tiebreak)
# quality_score favors hit rate while still rewarding edge
df["quality_score"] = (0.80 * df["p_win"]) + (0.20 * np.clip(df["ev"], 0, 1))

# ---- 4) CAP 2 PLAYS PER GAME ----
df = df.sort_values(["quality_score","p_win","ev"], ascending=False).copy()
df["game_rank"] = df.groupby("matchup").cumcount() + 1
df = df[df["game_rank"] <= 2].copy()

# ---- 5) FINAL SORT + OPTIONAL TOP N ----
TOP_N = 10
df = df.sort_values(["quality_score","p_win","ev"], ascending=False).head(TOP_N).copy()

# ---- 6) BUILD DISCORD TEXT (if not already present / or refresh) ----
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_ev(x):  return f"{100*x:.1f}%"

out_lines = []
out_lines.append("=== DISCORD CARD (HIGH HIT RATE / +EV) ===\n")

for _, r in df.iterrows():
    out_lines.append(f"{r['matchup']}")
    out_lines.append(f"{r['bet']} — {float(r['units']):.2f}u")
    out_lines.append(f"Sim Win%: {fmt_pct(float(r['p_win']))} | Market: {fmt_pct(float(r['p_mkt']))}")
    out_lines.append(f"EV: {fmt_ev(float(r['ev']))}\n")

discord_card = "\n".join(out_lines).strip()
print(discord_card)

# ---- 7) EXPORT ----
stamp = datetime.now().strftime("%Y-%m-%d_%H%M")
export_path = f"nba_stable_HIGH_HIT_RATE_{stamp}.csv"
export_cols = ["matchup","market","bet","odds","p_win","p_mkt","ev","units","quality_score"]
df[export_cols].to_csv(export_path, index=False)
print(f"\n✅ Exported: {export_path}")

# Keep for next cells
high_hit_card = df

In [ ]:
# ======================================
# STRICT PROBABILITY REBUILD (SPREAD/TOTAL SANITY FIX)
# Recomputes cover/over probs from mean_margin & mean_total
# and current market lines with correct sign conventions.
# ======================================

import numpy as np
import pandas as pd
from math import erf, sqrt
from datetime import datetime

# ---- EXPECTED INPUTS ----
# games_df must exist with: away_team, home_team, mean_margin, mean_total,
# spread_home_line, total_line, home_ml, away_ml (ml optional)
# mean_margin MUST be: (home_points - away_points) predicted mean margin
g = games_df.copy()

req = ["away_team","home_team","mean_margin","mean_total","spread_home_line","total_line"]
miss = [c for c in req if c not in g.columns]
if miss:
    raise KeyError(f"games_df missing: {miss}")

# ---- Normal CDF helper ----
def norm_cdf(x, mu=0.0, sigma=1.0):
    z = (x - mu) / (sigma * sqrt(2))
    return 0.5 * (1 + erf(z))

# ---- CONFIG (match your run) ----
SD_MARGIN = float(globals().get("SD_MARGIN", 14.5))
SD_TOTAL  = float(globals().get("SD_TOTAL", 18.0))

# ---- SPREAD: probability HOME covers spread_home_line ----
# Market spread_home_line is usually negative when home favored.
# Home covers if (home - away) > spread_home_line
g["p_home_cover"] = 1 - g.apply(lambda r: norm_cdf(r["spread_home_line"], mu=r["mean_margin"], sigma=SD_MARGIN), axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]  # away covers opposite side

# ---- TOTAL: probability OVER total_line ----
g["p_over"]  = 1 - g.apply(lambda r: norm_cdf(r["total_line"], mu=r["mean_total"], sigma=SD_TOTAL), axis=1)
g["p_under"] = 1 - g["p_over"]

# ---- MONEYLINE win probs (if mean_margin exists) ----
# home wins if margin > 0
g["p_home_win"] = 1 - g.apply(lambda r: norm_cdf(0.0, mu=r["mean_margin"], sigma=SD_MARGIN), axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

# ---- SANITY CHECKS ----
# If you see 90%+ cover probs on spreads larger than ~10 often, something is off.
san = g[["away_team","home_team","mean_margin","spread_home_line","p_home_cover","mean_total","total_line","p_over","p_home_win"]].copy()
san["abs_spread"] = san["spread_home_line"].abs()

print("✅ Rebuilt probs:", "SD_MARGIN=", SD_MARGIN, "| SD_TOTAL=", SD_TOTAL)
print("Sanity preview (largest spreads):")
display(san.sort_values("abs_spread", ascending=False).head(10))

# Flag suspicious: huge spreads with crazy cover %
sus = san[(san["abs_spread"] >= 10) & (san["p_home_cover"] >= 0.85)]
print(f"⚠ Suspicious covers (abs_spread>=10 & p_home_cover>=85%): {len(sus)}")
if len(sus):
    display(sus[["away_team","home_team","mean_margin","spread_home_line","p_home_cover"]].head(20))

# Write back to games_df so your engine uses corrected columns
games_df = g

In [ ]:
# ============================================
# (BOTTOM FIX CELL 1) — DEFINITIONS + HELPERS
# Paste this at the very bottom and run once
# ============================================

import numpy as np
import pandas as pd
from math import erf, sqrt

def norm_cdf(x, mu=0.0, sigma=1.0):
    """Normal CDF with mean=mu, sd=sigma (no scipy needed)."""
    if sigma <= 0 or pd.isna(sigma):
        return np.nan
    z = (x - mu) / (sigma * sqrt(2))
    return 0.5 * (1.0 + erf(z))

def american_to_prob(odds):
    """Implied prob from American odds (no vig removed)."""
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return (-odds) / ((-odds) + 100.0)

print("✅ Helpers loaded (norm_cdf, american_to_prob)")

In [ ]:
# ============================================
# FINAL CORRECT SPREAD PROBABILITY LOGIC
# Handles favorite and dog properly
# ============================================

g = games_df.copy()

def home_cover_prob(row):
    spread = row["spread_home_line"]
    mu = row["mean_margin"]
    sd = SD_MARGIN

    if spread < 0:
        # home favorite
        threshold = abs(spread)
        return 1 - norm_cdf(threshold, mu=mu, sigma=sd)
    else:
        # home underdog
        threshold = -abs(spread)
        return 1 - norm_cdf(threshold, mu=mu, sigma=sd)

g["p_home_cover"] = g.apply(home_cover_prob, axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

games_df = g

print("✅ Spread probabilities rebuilt (favorite/dog aware).")
display(g[["away_team","home_team","mean_margin","spread_home_line","p_home_cover"]])

In [ ]:
# =========================================================
# (BOTTOM FIX CELL 3) — REBUILD BET CARDS + DISCORD TEXT
# Recomputes p_mkt, EV, units, and discord_text using fixed p_home_cover
# Assumes you already have your card-builder dataframes OR games_df only.
# =========================================================

if "games_df" not in globals():
    raise RuntimeError("games_df not found.")

df = games_df.copy()

# --- MARKET PROBS ---
# Spread market implied prob from spread odds (fallback -110 if missing)
if "spread_home_odds" not in df.columns:
    df["spread_home_odds"] = -110
if "spread_away_odds" not in df.columns:
    df["spread_away_odds"] = -110

df["p_mkt_home_cover"] = df["spread_home_odds"].apply(american_to_prob)
df["p_mkt_away_cover"] = df["spread_away_odds"].apply(american_to_prob)

# ML market implied prob
if "home_ml" in df.columns and "away_ml" in df.columns:
    df["p_mkt_home_win"] = df["home_ml"].apply(american_to_prob)
    df["p_mkt_away_win"] = df["away_ml"].apply(american_to_prob)
else:
    df["p_mkt_home_win"] = np.nan
    df["p_mkt_away_win"] = np.nan

# Totals market implied prob
if "over_odds" not in df.columns:
    df["over_odds"] = -110
if "under_odds" not in df.columns:
    df["under_odds"] = -110
df["p_mkt_over"] = df["over_odds"].apply(american_to_prob)
df["p_mkt_under"] = df["under_odds"].apply(american_to_prob)

# --- BUILD CANDIDATE ROWS (ML / SPREAD / TOTALS) ---
rows = []

for _, r in df.iterrows():
    matchup = f"{r.get('away_team','')} at {r.get('home_team','')}"
    commence = r.get("commence_et", r.get("commence_utc", None))

    # SPREAD (home side shown as team +spread or -spread based on sign)
    if not pd.isna(r.get("spread_home_line", np.nan)):
        # home bet string uses signed spread_home_line (e.g., -1.5)
        h_line = float(r["spread_home_line"])
        h_odds = int(r.get("spread_home_odds", -110))
        a_line = -h_line
        a_odds = int(r.get("spread_away_odds", -110))

        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "SPREAD",
            "team": r.get("home_team",""),
            "bet": f"{r.get('home_team','')} {h_line:+g} ({h_odds:+d})".replace("+ -", "-").replace("+", "+"),
            "odds": h_odds,
            "p_win": float(r["p_home_cover"]),
            "p_mkt": float(r["p_mkt_home_cover"]) if not pd.isna(r["p_mkt_home_cover"]) else np.nan,
        })
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "SPREAD",
            "team": r.get("away_team",""),
            "bet": f"{r.get('away_team','')} {a_line:+g} ({a_odds:+d})".replace("+ -", "-").replace("+", "+"),
            "odds": a_odds,
            "p_win": float(r["p_away_cover"]),
            "p_mkt": float(r["p_mkt_away_cover"]) if not pd.isna(r["p_mkt_away_cover"]) else np.nan,
        })

    # ML
    if not pd.isna(r.get("home_ml", np.nan)) and not pd.isna(r.get("away_ml", np.nan)):
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "ML",
            "team": r.get("home_team",""),
            "bet": f"{r.get('home_team','')} ML ({int(r['home_ml']):+d})".replace("+ -", "-"),
            "odds": int(r["home_ml"]),
            "p_win": float(r.get("p_home_win", np.nan)) if "p_home_win" in r else np.nan,
            "p_mkt": float(r["p_mkt_home_win"]),
        })
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "ML",
            "team": r.get("away_team",""),
            "bet": f"{r.get('away_team','')} ML ({int(r['away_ml']):+d})".replace("+ -", "-"),
            "odds": int(r["away_ml"]),
            "p_win": float(r.get("p_away_win", np.nan)) if "p_away_win" in r else np.nan,
            "p_mkt": float(r["p_mkt_away_win"]),
        })

    # TOTALS (if p_over exists)
    if "total_line" in r and not pd.isna(r.get("total_line", np.nan)) and "p_over" in df.columns:
        t = float(r["total_line"])
        o_odds = int(r.get("over_odds", -110))
        u_odds = int(r.get("under_odds", -110))
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "TOTAL",
            "team": "",
            "bet": f"OVER {t:g} ({o_odds:+d})".replace("+ -", "-"),
            "odds": o_odds,
            "p_win": float(r["p_over"]),
            "p_mkt": float(r["p_mkt_over"]),
        })
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "TOTAL",
            "team": "",
            "bet": f"UNDER {t:g} ({u_odds:+d})".replace("+ -", "-"),
            "odds": u_odds,
            "p_win": float(r["p_under"]) if "p_under" in r else (1.0 - float(r["p_over"])),
            "p_mkt": float(r["p_mkt_under"]),
        })

card = pd.DataFrame(rows)

# EV per 1u (simple edge vs market prob baseline)
card["ev"] = (card["p_win"] - card["p_mkt"]) / card["p_mkt"]
card = card.replace([np.inf, -np.inf], np.nan)

# half-kelly-ish units (bounded) using edge proxy; keep your existing sizing if you have it
# If you already have your own kelly sizing, comment this out.
card["units"] = (0.5 * card["ev"]).clip(lower=0.25, upper=1.0).fillna(0.0)
card = card[card["units"] > 0].copy()

# cap: max 2 picks per game
card["game_key"] = card["matchup"]
card = card.sort_values(["ev","p_win"], ascending=False)
card = card.groupby("game_key").head(2).reset_index(drop=True)

# discord text
def fmt_row(r):
    return (
        f"{r['matchup']}\n"
        f"{r['bet']} — {r['units']:.2f}u\n"
        f"Sim Win%: {100*r['p_win']:.1f}% | Market: {100*r['p_mkt']:.1f}%\n"
        f"EV: {100*r['ev']:.1f}%\n"
    )

card["discord_text"] = card.apply(fmt_row, axis=1)

print(f"✅ Rebuilt card: {len(card)} rows (after +EV & cap 2/game)")
print("\n=== DISCORD CARD (FIXED SPREAD LOGIC) ===\n")
print("\n".join(card["discord_text"].tolist()))

out_path = f"nba_stable_2026-03-03_FIXED_SPREAD_CARD.csv"
card.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

# make it available as final_card for next steps
final_card = card.copy()

In [ ]:
# ============================================
# (BOTTOM CELL) — ANTI DOUBLE-DIP (SIDE CONFLICT)
# Keep max 2 picks/game, but NOT both sides.
# Allows: (spread or ML) + total
# Blocks: ML + opposite spread, or both spreads, or both MLs
# ============================================

import pandas as pd
import numpy as np

if "final_card" not in globals():
    raise RuntimeError("final_card not found. Run the rebuild cell first.")

fc = final_card.copy()

def market_bucket(m):
    m = str(m).upper()
    if m in ["ML", "SPREAD"]:
        return "SIDE"
    if m in ["TOTAL", "TOTALS", "OU", "O/U"]:
        return "TOTAL"
    return "OTHER"

fc["bucket"] = fc["market"].apply(market_bucket)

# Identify "side team" for SIDE bets (to detect opposing sides)
def side_team_from_bet(row):
    if row["bucket"] != "SIDE":
        return None
    # If team column exists use it; else parse from bet string
    if "team" in row and pd.notna(row["team"]) and str(row["team"]).strip() != "":
        return str(row["team"]).strip()
    # fallback parse: first token chunk before " ML" or " +" or " -"
    b = str(row["bet"])
    return b.split(" ML")[0].split(" +")[0].split(" -")[0].strip()

fc["side_team"] = fc.apply(side_team_from_bet, axis=1)

clean_rows = []
for game, g in fc.groupby("matchup", sort=False):
    g = g.sort_values(["ev","p_win"], ascending=False).copy()

    picks = []
    side_picks = g[g["bucket"] == "SIDE"].copy()
    total_picks = g[g["bucket"] == "TOTAL"].copy()

    # pick best SIDE (at most one)
    if len(side_picks) > 0:
        picks.append(side_picks.iloc[0])

    # pick best TOTAL (at most one)
    if len(total_picks) > 0:
        picks.append(total_picks.iloc[0])

    # if still <2 and there are OTHER bets, fill (rare)
    if len(picks) < 2:
        other = g[g["bucket"] == "OTHER"]
        for _, r in other.iterrows():
            if len(picks) >= 2:
                break
            picks.append(r)

    clean_rows.extend(picks)

fc2 = pd.DataFrame(clean_rows).reset_index(drop=True)

# rebuild discord text in same format
def fmt_row(r):
    return (
        f"{r['matchup']}\n"
        f"{r['bet']} — {r['units']:.2f}u\n"
        f"Sim Win%: {100*r['p_win']:.1f}% | Market: {100*r['p_mkt']:.1f}%\n"
        f"EV: {100*r['ev']:.1f}%\n"
    )

fc2["discord_text"] = fc2.apply(fmt_row, axis=1)

print(f"✅ Anti-double-dip applied. Rows: {len(final_card)} → {len(fc2)}")
print("\n=== DISCORD CARD (NO SIDE CONFLICTS) ===\n")
print("\n".join(fc2["discord_text"].tolist()))

out_path = "nba_stable_2026-03-03_FIXED_SPREAD_NO_DOUBLEDIP.csv"
fc2.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

final_card = fc2.copy()

In [ ]:
# ============================================
# (BOTTOM CELL) — HIGH HIT-RATE FILTER
# Keeps +EV only, then filters by p_win threshold
# ============================================

HITRATE_MIN = 0.58  # tweak: 0.58 / 0.60 / 0.62

if "final_card" not in globals():
    raise RuntimeError("final_card not found.")

fc = final_card.copy()

# keep +EV and hit-rate threshold
fc = fc[(fc["ev"] > 0) & (fc["p_win"] >= HITRATE_MIN)].copy()
fc = fc.sort_values(["p_win","ev"], ascending=False).reset_index(drop=True)

print(f"✅ High hit-rate filter applied (p_win >= {HITRATE_MIN:.2f}). Rows: {len(fc)}")
print("\n=== DISCORD CARD (HIGH HIT-RATE) ===\n")
print("\n".join(fc["discord_text"].tolist()))

out_path = f"nba_stable_2026-03-03_HIGH_HITRATE_{int(HITRATE_MIN*100)}.csv"
fc.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

final_card = fc.copy()

In [ ]:
# ============================================
# (BOTTOM CELL) — HIGH HIT-RATE FILTER (LOCKED + CLEAN FILENAME)
# ============================================

HITRATE_MIN = 0.58  # lock it here
TAG = "58"          # lock filename tag

fc = final_card.copy()

# If your final_card still includes non +EV, keep this:
fc = fc[fc["ev"] > 0].copy()

fc = fc[fc["p_win"] >= HITRATE_MIN].copy()
fc = fc.sort_values(["p_win","ev"], ascending=False).reset_index(drop=True)

print(f"✅ High hit-rate filter applied (p_win >= {HITRATE_MIN:.2f}). Rows: {len(fc)}")
print("\n=== DISCORD CARD (HIGH HIT-RATE) ===\n")
print("\n".join(fc["discord_text"].tolist()))

out_path = f"nba_stable_2026-03-03_HIGH_HITRATE_{TAG}.csv"
fc.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

final_card = fc.copy()

In [ ]:
print("=== CURRENT FINAL_CARD STATE ===\n")
print("\n".join(final_card["discord_text"].tolist()))

In [ ]:
games_df[["away_team","home_team","mean_margin","spread_home_line","p_home_cover"]]

In [ ]:
# ============================================
# REBUILD CARD FROM CORRECTED games_df
# ============================================

rows = []

for _, r in games_df.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # HOME SPREAD
    rows.append({
        "matchup": matchup,
        "market": "SPREAD",
        "bet": f"{r['home_team']} {r['spread_home_line']} (-110)",
        "p_win": r["p_home_cover"],
        "p_mkt": 0.524,
    })

    # AWAY SPREAD
    rows.append({
        "matchup": matchup,
        "market": "SPREAD",
        "bet": f"{r['away_team']} {-(r['spread_home_line'])} (-110)",
        "p_win": 1 - r["p_home_cover"],
        "p_mkt": 0.524,
    })

card = pd.DataFrame(rows)
card["ev"] = card["p_win"] - card["p_mkt"]
card = card[card["ev"] > 0].copy()

card = card.sort_values(["p_win","ev"], ascending=False).reset_index(drop=True)

print("=== REBUILT SPREAD CARD ===\n")
display(card.head(15))

In [ ]:
# ==== SINGLE CONFIG (KEEP ONLY THIS) ====
SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"      # your 3/2 run date
SIMS = 50000

SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200

OUTSIDE_ON = True

In [ ]:
!pip -q install pdfplumber

In [ ]:
import pandas as pd
import numpy as np
import requests

def pull_completed_scores(sport: str, days_from: int):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": int(days_from)}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g.get("home_team")
        away = g.get("away_team")
        ct = g.get("commence_time")

        scores = {s["name"]: s["score"] for s in (g.get("scores") or [])}
        hs = scores.get(home)
        as_ = scores.get(away)
        if hs is None or as_ is None:
            continue

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_score": hs,
            "away_score": as_,
            "date": ct,
        })

    df = pd.DataFrame(rows)

    # HARD FORCE NUMERIC (fixes your TypeError)
    for c in ["home_score", "away_score"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["home_score","away_score"])

    return df


def build_team_overall_from_scores(games_hist: pd.DataFrame):
    gh = games_hist.copy()

    gh["home_margin"] = gh["home_score"] - gh["away_score"]
    gh["game_total"]  = gh["home_score"] + gh["away_score"]

    # stable poss proxy (consistent layer)
    gh["poss_proxy"] = (gh["game_total"] / 2.25).clip(90, 105)

    home_rows = pd.DataFrame({
        "team": gh["home_team"],
        "ppp_for": gh["home_score"] / gh["poss_proxy"],
        "ppp_against": gh["away_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"],
    })
    away_rows = pd.DataFrame({
        "team": gh["away_team"],
        "ppp_for": gh["away_score"] / gh["poss_proxy"],
        "ppp_against": gh["home_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"],
    })

    team_game = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = (
        team_game.groupby("team", as_index=False)
        .agg(poss=("poss","mean"), ppp_for=("ppp_for","mean"), ppp_against=("ppp_against","mean"))
    )
    return team_overall


def rebuild_hist_and_attach_layers():
    # plan-safe daysFrom fallback (OddsAPI often blocks big windows on some tiers)
    DAYS_TRY = [3, 2, 1]
    parts = []
    for d in DAYS_TRY:
        try:
            df = pull_completed_scores(SPORT, d)
            if len(df) >= 8:
                parts.append(df)
                print(f"✅ scores daysFrom={d}: {len(df)} games")
                break
            else:
                print(f"⚠️ scores daysFrom={d}: only {len(df)} games")
        except Exception as e:
            print(f"⚠️ scores daysFrom={d} failed: {str(e)[:160]}")

    if not parts:
        raise RuntimeError("No historical score pulls succeeded (daysFrom=3/2/1).")

    games_hist = parts[0].drop_duplicates()
    team_overall = build_team_overall_from_scores(games_hist)

    # attach to current slate
    g = games_df.copy()

    g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
        "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
        "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    # fill to prevent NaNs breaking sims
    for c in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
        g[c] = g[c].fillna(g[c].mean())

    # projection layer (STRICT required columns)
    g["poss_proj"]       = (g["h_poss"] + g["a_poss"]) / 2
    g["home_ppp_proj"]   = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
    g["away_ppp_proj"]   = (g["a_ppp_for"] + g["h_ppp_against"]) / 2
    g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
    g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

    # attach back to globals so downstream cells see them
    globals()["games_hist"] = games_hist
    globals()["team_overall"] = team_overall
    globals()["games_df"] = g

    print("✅ Layers attached: games_hist → team_overall → projections → games_df")
    return g


games_df = rebuild_hist_and_attach_layers()
games_df.head()

In [ ]:
# ================= CONFIG =================
SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"
SIMS = 50000

SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200

OUTSIDE_ON = True

ODDS_API_KEY = "YOUR_KEY_HERE"  # or use os.getenv

In [ ]:
import requests
import numpy as np
import pandas as pd
from scipy.stats import norm

In [ ]:
import os

os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

print("Key Loaded:", ODDS_API_KEY[:6], "...")

In [ ]:
def pull_market_data():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "spreads,totals,h2h",
        "oddsFormat": "american"
    }

    r = requests.get(url, params=params, timeout=30)

    if r.status_code != 200:
        raise Exception(f"Market Pull Failed: {r.status_code} | {r.text}")

    data = r.json()

    rows = []
    for game in data:
        home = game["home_team"]
        away = game["away_team"]

        if not game.get("bookmakers"):
            continue

        spreads = None
        totals = None

        for book in game["bookmakers"]:
            for market in book["markets"]:
                if market["key"] == "spreads":
                    spreads = market["outcomes"]
                if market["key"] == "totals":
                    totals = market["outcomes"]

        if not spreads or not totals:
            continue

        try:
            home_spread = next(o for o in spreads if o["name"] == home)
            total_line = totals[0]

            rows.append({
                "home_team": home,
                "away_team": away,
                "home_spread": home_spread["point"],
                "home_spread_price": home_spread["price"],
                "total_line": total_line["point"],
            })
        except:
            continue

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError("No market rows pulled. Check plan limits.")

    print(f"Pulled {len(df)} games.")
    return df


games_df = pull_market_data()
games_df.head()

In [ ]:
def pull_completed_scores(days_from=3):
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/scores/"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": days_from}

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    rows = []
    for g in data:
        if not g.get("completed"):
            continue

        scores = {s["name"]: float(s["score"]) for s in g.get("scores", [])}
        home = g["home_team"]
        away = g["away_team"]

        if home in scores and away in scores:
            rows.append({
                "home_team": home,
                "away_team": away,
                "home_score": scores[home],
                "away_score": scores[away]
            })

    return pd.DataFrame(rows)


hist = pull_completed_scores(3)

hist["poss"] = ((hist["home_score"] + hist["away_score"]) / 2.25).clip(92,104)

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "ppp_for": hist["home_score"] / hist["poss"],
    "ppp_against": hist["away_score"] / hist["poss"],
    "poss": hist["poss"]
})

away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "ppp_for": hist["away_score"] / hist["poss"],
    "ppp_against": hist["home_score"] / hist["poss"],
    "poss": hist["poss"]
})

team_overall = (
    pd.concat([home_rows, away_rows])
    .groupby("team")
    .mean()
    .reset_index()
)

team_overall.head()

In [ ]:
HOME_COURT_ADV = 2.2  # points

g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left") \
     .rename(columns={
         "poss":"h_poss",
         "ppp_for":"h_ppp_for",
         "ppp_against":"h_ppp_against"
     }).drop(columns=["team"])

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left") \
     .rename(columns={
         "poss":"a_poss",
         "ppp_for":"a_ppp_for",
         "ppp_against":"a_ppp_against"
     }).drop(columns=["team"])

# tempo blend
g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2

# efficiency blend
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

# projected raw points
g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

# home court adjustment
g["home_points_proj"] += HOME_COURT_ADV / 2
g["away_points_proj"] -= HOME_COURT_ADV / 2

games_df = g

games_df[["home_team","away_team","home_points_proj","away_points_proj"]].head()

In [ ]:
def run_sim(row):

    # simulate margin directly instead of two independent totals
    proj_margin = row.home_points_proj - row.away_points_proj
    proj_total  = row.home_points_proj + row.away_points_proj

    margin = np.random.normal(proj_margin, SD_MARGIN, SIMS)
    total  = np.random.normal(proj_total, SD_TOTAL, SIMS)

    return pd.Series({
        "home_cover_prob": np.mean(margin > row.home_spread),
        "over_prob": np.mean(total > row.total_line)
    })


SD_MARGIN = 13.5   # realistic NBA margin stdev
SD_TOTAL  = 18.0   # realistic NBA total stdev

sim_results = games_df.apply(run_sim, axis=1)
games_df = pd.concat([games_df.drop(columns=["home_cover_prob","over_prob"], errors="ignore"),
                      sim_results], axis=1)

In [ ]:
def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)


def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)


games_df["market_prob"] = games_df["home_spread_price"].apply(american_to_prob)
games_df["edge"] = games_df["home_cover_prob"] - games_df["market_prob"]

games_df["kelly"] = games_df.apply(
    lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price),
    axis=1
)

games_df["units"] = (games_df["kelly"] / 2).clip(UNIT_MIN, UNIT_CAP)

final_card = games_df.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "away_team",
    "home_spread",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# ===== PROJECTION STABILIZER =====

REG_WEIGHT = 0.35   # blend weight toward market

# implied market margin
games_df["market_margin"] = -games_df["home_spread"]

# blend projection margin toward market
games_df["proj_margin"] = (
    (games_df["home_points_proj"] - games_df["away_points_proj"]) * (1 - REG_WEIGHT)
    + games_df["market_margin"] * REG_WEIGHT
)

# blended projected total
games_df["proj_total"] = (
    (games_df["home_points_proj"] + games_df["away_points_proj"]) * (1 - REG_WEIGHT)
    + games_df["total_line"] * REG_WEIGHT
)

In [ ]:
# ===== FINAL SIM =====

SD_MARGIN = 13.5
SD_TOTAL  = 18.0

def run_final_sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    total  = np.random.normal(row.proj_total, SD_TOTAL, SIMS)

    return pd.Series({
        "home_cover_prob": np.mean(margin > row.home_spread),
        "over_prob": np.mean(total > row.total_line)
    })

sim_results = games_df.apply(run_final_sim, axis=1)

games_df = pd.concat(
    [games_df.drop(columns=["home_cover_prob","over_prob"], errors="ignore"),
     sim_results],
    axis=1
)

In [ ]:
# ===== EV + KELLY =====

def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

games_df["market_prob"] = games_df["home_spread_price"].apply(american_to_prob)
games_df["edge"] = games_df["home_cover_prob"] - games_df["market_prob"]

games_df["kelly"] = games_df.apply(
    lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price),
    axis=1
)

# only size positive EV
games_df["units"] = 0

mask = games_df["edge"] > 0

games_df.loc[mask, "units"] = (
    games_df.loc[mask, "kelly"] / 2
).clip(UNIT_MIN, UNIT_CAP)

In [ ]:
# ===== STRICT FILTERS =====

MIN_EDGE = 0.025
MAX_EDGE = 0.18   # avoid unrealistic model spikes

filtered = games_df[
    (games_df["edge"] > MIN_EDGE) &
    (games_df["edge"] < MAX_EDGE) &
    (games_df["units"] > 0)
].copy()

final_card = filtered.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "away_team",
    "home_spread",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# ===== TOP 10 STRICT CARD =====

top10 = final_card.sort_values("edge", ascending=False).head(10)

top10_display = top10[[
    "home_team",
    "away_team",
    "home_spread",
    "home_cover_prob",
    "edge",
    "units"
]].copy()

top10_display["home_cover_prob"] = (top10_display["home_cover_prob"] * 100).round(2)
top10_display["edge"] = (top10_display["edge"] * 100).round(2)

top10_display

In [ ]:
games_df[[
    "home_team",
    "away_team",
    "home_cover_prob",
    "market_prob",
    "edge"
]].sort_values("edge", ascending=False)


In [ ]:
games_df[[
    "home_team",
    "away_team",
    "proj_margin",
    "home_spread"
]]

In [ ]:
# ===== HARD MARGIN CLAMP =====

MAX_MARGIN_EDGE = 10   # max model edge vs market in points

# limit how far projection can differ from spread
margin_diff = games_df["proj_margin"] - games_df["market_margin"]

games_df["proj_margin"] = (
    games_df["market_margin"] +
    margin_diff.clip(-MAX_MARGIN_EDGE, MAX_MARGIN_EDGE)
)

In [ ]:
games_df[[
    "home_team",
    "home_cover_prob",
    "edge"
]].sort_values("edge", ascending=False)

In [ ]:
# ===============================
# FULL HARD RESET – FINAL ENGINE
# ===============================

REG_WEIGHT = 0.20
SD_MARGIN = 15.0
SD_TOTAL = 19.0
MAX_MARGIN_EDGE = 8

# ---- rebuild projection clean ----
games_df["market_margin"] = -games_df["home_spread"]

base_margin = (
    games_df["home_points_proj"] -
    games_df["away_points_proj"]
)

# regression to market
proj_margin = (
    base_margin * (1 - REG_WEIGHT)
    + games_df["market_margin"] * REG_WEIGHT
)

# hard clamp vs market
margin_diff = proj_margin - games_df["market_margin"]
proj_margin = games_df["market_margin"] + margin_diff.clip(-MAX_MARGIN_EDGE, MAX_MARGIN_EDGE)

games_df["proj_margin"] = proj_margin

# projected total blended
games_df["proj_total"] = (
    (games_df["home_points_proj"] + games_df["away_points_proj"]) * (1 - REG_WEIGHT)
    + games_df["total_line"] * REG_WEIGHT
)

# ---- MONTE CARLO ----
def strict_sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    total  = np.random.normal(row.proj_total, SD_TOTAL, SIMS)

    return pd.Series({
        "home_cover_prob": np.mean(margin > row.home_spread),
        "over_prob": np.mean(total > row.total_line)
    })

sim_results = games_df.apply(strict_sim, axis=1)

games_df = pd.concat(
    [games_df.drop(columns=["home_cover_prob","over_prob","edge","kelly","units"], errors="ignore"),
     sim_results],
    axis=1
)

# ---- EV + KELLY ----
def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

games_df["market_prob"] = games_df["home_spread_price"].apply(american_to_prob)
games_df["edge"] = games_df["home_cover_prob"] - games_df["market_prob"]

games_df["kelly"] = games_df.apply(
    lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price),
    axis=1
)

games_df["units"] = 0
mask = games_df["edge"] > 0
games_df.loc[mask, "units"] = (
    games_df.loc[mask, "kelly"] / 2
).clip(0.25, 1.0)

# ---- OUTPUT ----
games_df[[
    "home_team",
    "home_spread",
    "proj_margin",
    "home_cover_prob",
    "edge",
    "units"
]].sort_values("edge", ascending=False)

In [ ]:
# ==========================================
# FULL BOTTOM PATCH — EXPANDED HIST + RESET
# ==========================================

import requests
import numpy as np
import pandas as pd

# ---------- 1) EXPANDED HISTORICAL WINDOW ----------
def pull_completed_scores(days_from):
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/scores/"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": days_from}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        return pd.DataFrame()
    data = r.json()

    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        scores = {s["name"]: float(s["score"]) for s in g.get("scores", [])}
        home = g["home_team"]
        away = g["away_team"]
        if home in scores and away in scores:
            rows.append({
                "home_team": home,
                "away_team": away,
                "home_score": scores[home],
                "away_score": scores[away],
            })
    return pd.DataFrame(rows)

windows = [3, 7, 14, 30]

hist_frames = []
for w in windows:
    df = pull_completed_scores(w)
    if len(df) > 0:
        hist_frames.append(df)

hist = pd.concat(hist_frames).drop_duplicates()

print("Total historical games:", len(hist))

# ---------- 2) REBUILD TEAM BASE ----------
hist["poss"] = ((hist["home_score"] + hist["away_score"]) / 2.25).clip(92,104)

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "ppp_for": hist["home_score"] / hist["poss"],
    "ppp_against": hist["away_score"] / hist["poss"],
    "poss": hist["poss"]
})

away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "ppp_for": hist["away_score"] / hist["poss"],
    "ppp_against": hist["home_score"] / hist["poss"],
    "poss": hist["poss"]
})

team_overall = (
    pd.concat([home_rows, away_rows])
    .groupby("team")
    .mean()
    .reset_index()
)

# ---------- 3) ATTACH TO SLATE ----------
g = games_df.drop(
    columns=[
        "h_poss","h_ppp_for","h_ppp_against",
        "a_poss","a_ppp_for","a_ppp_against",
        "home_points_proj","away_points_proj",
        "proj_margin","proj_total",
        "home_cover_prob","over_prob",
        "edge","kelly","units"
    ],
    errors="ignore"
)

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"}) \
     .drop(columns=["team"])

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"}) \
     .drop(columns=["team"])

# ---------- 4) PROJECTION ----------
REG_WEIGHT = 0.20
SD_MARGIN = 15.0
SD_TOTAL  = 19.0

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2

g["home_points_proj"] = (
    (g["h_ppp_for"] + g["a_ppp_against"]) / 2
) * g["poss_proj"]

g["away_points_proj"] = (
    (g["a_ppp_for"] + g["h_ppp_against"]) / 2
) * g["poss_proj"]

g["market_margin"] = -g["home_spread"]

base_margin = g["home_points_proj"] - g["away_points_proj"]

proj_margin = (
    base_margin * (1 - REG_WEIGHT)
    + g["market_margin"] * REG_WEIGHT
)

MAX_MARGIN_EDGE = 8
margin_diff = proj_margin - g["market_margin"]
g["proj_margin"] = g["market_margin"] + margin_diff.clip(-MAX_MARGIN_EDGE, MAX_MARGIN_EDGE)

g["proj_total"] = (
    (g["home_points_proj"] + g["away_points_proj"]) * (1 - REG_WEIGHT)
    + g["total_line"] * REG_WEIGHT
)

# ---------- 5) MONTE CARLO ----------
def strict_sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    total  = np.random.normal(row.proj_total, SD_TOTAL, SIMS)
    return pd.Series({
        "home_cover_prob": np.mean(margin > row.home_spread),
        "over_prob": np.mean(total > row.total_line)
    })

sim_results = g.apply(strict_sim, axis=1)
g = pd.concat([g, sim_results], axis=1)

# ---------- 6) EV + KELLY ----------
def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price), axis=1)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

# ---------- 7) OUTPUT ----------
final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "proj_margin",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# ==========================================
# ESPN SEASON TEAM EFFICIENCY PULL
# ==========================================

import requests
import pandas as pd
import numpy as np

def pull_espn_team_stats():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    teams = []

    for team in data["sports"][0]["leagues"][0]["teams"]:
        team_data = team["team"]
        team_id = team_data["id"]
        team_name = team_data["displayName"]

        stats_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/statistics"
        stats_r = requests.get(stats_url, timeout=30)
        if stats_r.status_code != 200:
            continue

        stats_json = stats_r.json()

        ppg = None
        oppg = None
        pace = 98

        for s in stats_json.get("stats", []):
            if s.get("name") == "pointsPerGame":
                ppg = float(s.get("value"))
            if s.get("name") == "opponentPointsPerGame":
                oppg = float(s.get("value"))

        if ppg and oppg:
            teams.append({
                "team": team_name,
                "ppg": ppg,
                "oppg": oppg,
                "pace": pace
            })

    return pd.DataFrame(teams)


espn_stats = pull_espn_team_stats()

print("Teams pulled:", len(espn_stats))
espn_stats.head()

In [ ]:
# ==========================================
# SEASON-BASED PROJECTION LAYER
# ==========================================

g = games_df.copy()

g = g.merge(espn_stats, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"ppg":"home_ppg","oppg":"home_oppg"}) \
     .drop(columns=["team","pace"])

g = g.merge(espn_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg"}) \
     .drop(columns=["team","pace"])

# efficiency blend
g["home_points_proj"] = (g["home_ppg"] + g["away_oppg"]) / 2
g["away_points_proj"] = (g["away_ppg"] + g["home_oppg"]) / 2

# margin + total
g["proj_margin"] = g["home_points_proj"] - g["away_points_proj"]
g["proj_total"] = g["home_points_proj"] + g["away_points_proj"]

In [ ]:
# ==========================================
# OFFICIAL NBA SEASON TEAM STATS (CLEAN)
# ==========================================

import requests
import pandas as pd
import numpy as np

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.nba.com/"
}

def pull_nba_team_stats():
    url = (
        "https://stats.nba.com/stats/leaguedashteamstats"
        "?MeasureType=Base"
        "&PerMode=PerGame"
        "&Season=2025-26"
        "&SeasonType=Regular+Season"
    )

    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json()

    headers_data = data["resultSets"][0]["headers"]
    rows = data["resultSets"][0]["rowSet"]

    df = pd.DataFrame(rows, columns=headers_data)

    return df[[
        "TEAM_NAME",
        "PTS",
        "OPP_PTS"
    ]].rename(columns={
        "TEAM_NAME": "team",
        "PTS": "ppg",
        "OPP_PTS": "oppg"
    })


team_stats = pull_nba_team_stats()

print("Teams pulled:", len(team_stats))
team_stats.head()

In [ ]:
# ==========================================
# TRUE HISTORICAL BUILD (DATE ITERATION)
# ==========================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def pull_scores_by_date(date_str):
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/scores/"
    params = {
        "apiKey": ODDS_API_KEY,
        "date": date_str
    }
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        return pd.DataFrame()
    data = r.json()

    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        scores = {s["name"]: float(s["score"]) for s in g.get("scores", [])}
        home = g["home_team"]
        away = g["away_team"]
        if home in scores and away in scores:
            rows.append({
                "home_team": home,
                "away_team": away,
                "home_score": scores[home],
                "away_score": scores[away],
            })
    return pd.DataFrame(rows)


# ---- Pull last 60 days ----
hist_frames = []

today = datetime.utcnow().date()

for i in range(60):
    d = today - timedelta(days=i+1)
    df = pull_scores_by_date(d.strftime("%Y-%m-%d"))
    if len(df) > 0:
        hist_frames.append(df)

hist = pd.concat(hist_frames).drop_duplicates()

print("Historical games pulled:", len(hist))


# ==========================================
# REBUILD TEAM EFFICIENCY
# ==========================================

hist["poss"] = ((hist["home_score"] + hist["away_score"]) / 2.25).clip(92,104)

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "ppp_for": hist["home_score"] / hist["poss"],
    "ppp_against": hist["away_score"] / hist["poss"],
    "poss": hist["poss"]
})

away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "ppp_for": hist["away_score"] / hist["poss"],
    "ppp_against": hist["home_score"] / hist["poss"],
    "poss": hist["poss"]
})

team_overall = (
    pd.concat([home_rows, away_rows])
    .groupby("team")
    .mean()
    .reset_index()
)

print("Teams built:", len(team_overall))


# ==========================================
# REPROJECT
# ==========================================

g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"}) \
     .drop(columns=["team"])

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"}) \
     .drop(columns=["team"])

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2

g["home_points_proj"] = (
    (g["h_ppp_for"] + g["a_ppp_against"]) / 2
) * g["poss_proj"]

g["away_points_proj"] = (
    (g["a_ppp_for"] + g["h_ppp_against"]) / 2
) * g["poss_proj"]

g["proj_margin"] = g["home_points_proj"] - g["away_points_proj"]
g["proj_total"] = g["home_points_proj"] + g["away_points_proj"]


# ==========================================
# FINAL SIM
# ==========================================

SD_MARGIN = 14.5
SD_TOTAL = 19.0

def strict_sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    total  = np.random.normal(row.proj_total, SD_TOTAL, SIMS)
    return pd.Series({
        "home_cover_prob": np.mean(margin > row.home_spread),
        "over_prob": np.mean(total > row.total_line)
    })

sim_results = g.apply(strict_sim, axis=1)
g = pd.concat([g, sim_results], axis=1)


# ==========================================
# EV + KELLY
# ==========================================

def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price), axis=1)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# ==========================================
# PULL SEASON TEAM STATS (BASKETBALL REF)
# ==========================================

import pandas as pd
import numpy as np

season = "2026"  # change if needed

url = f"https://www.basketball-reference.com/leagues/NBA_{season}.html"

tables = pd.read_html(url)

# team stats table is usually first
team_stats = tables[0]

# clean
team_stats = team_stats[team_stats["Team"] != "League Average"]

team_stats = team_stats[[
    "Team",
    "PTS",
    "Opp PTS"
]].rename(columns={
    "Team": "team",
    "PTS": "ppg",
    "Opp PTS": "oppg"
})

print("Teams pulled:", len(team_stats))
team_stats.head()

In [ ]:
# ==========================================
# ROBUST BASKETBALL-REFERENCE TEAM PULL
# ==========================================

import pandas as pd
import numpy as np

season = "2026"

url = f"https://www.basketball-reference.com/leagues/NBA_{season}.html"

tables = pd.read_html(url)

# The first table is usually team per-game stats
team_stats = tables[0]

# Flatten multi-index columns if present
if isinstance(team_stats.columns, pd.MultiIndex):
    team_stats.columns = team_stats.columns.get_level_values(-1)

# Inspect columns once (optional debug)
print("Columns:", team_stats.columns.tolist())

# Remove league average row
team_stats = team_stats[team_stats.iloc[:,0] != "League Average"]

# Rename safely using position instead of fragile column names
team_stats = team_stats.rename(columns={
    team_stats.columns[0]: "team",
    "PTS": "ppg",
    "Opp PTS": "oppg"
})

team_stats = team_stats[["team", "ppg", "oppg"]]

print("Teams pulled:", len(team_stats))
team_stats.head()

In [ ]:
# ==========================================
# CLEAN TEAM STATS FROM STANDINGS TABLE
# ==========================================

import pandas as pd
import numpy as np

season = "2026"

url = f"https://www.basketball-reference.com/leagues/NBA_{season}.html"
tables = pd.read_html(url)

team_stats = tables[0]

# Flatten multi-index if needed
if isinstance(team_stats.columns, pd.MultiIndex):
    team_stats.columns = team_stats.columns.get_level_values(-1)

print("Columns detected:", team_stats.columns.tolist())

# Rename correctly for this table structure
team_stats = team_stats.rename(columns={
    team_stats.columns[0]: "team",   # Eastern Conference / Western Conference col
    "PS/G": "ppg",
    "PA/G": "oppg"
})

# Remove conference header rows
team_stats = team_stats[team_stats["team"].str.contains("Division") == False]
team_stats = team_stats[team_stats["team"] != "Eastern Conference"]
team_stats = team_stats[team_stats["team"] != "Western Conference"]

team_stats = team_stats[["team", "ppg", "oppg"]]

print("Teams pulled:", len(team_stats))
team_stats.head()

In [ ]:
# ==========================================
# FULL NBA SEASON TEAM STATS (EAST + WEST)
# ==========================================

import pandas as pd
import numpy as np

season = "2026"

url = f"https://www.basketball-reference.com/leagues/NBA_{season}.html"
tables = pd.read_html(url)

east = tables[0]
west = tables[1]

# Flatten if multi-index
if isinstance(east.columns, pd.MultiIndex):
    east.columns = east.columns.get_level_values(-1)

if isinstance(west.columns, pd.MultiIndex):
    west.columns = west.columns.get_level_values(-1)

def clean_table(df):
    df = df.rename(columns={
        df.columns[0]: "team",
        "PS/G": "ppg",
        "PA/G": "oppg"
    })
    df = df[["team", "ppg", "oppg"]]
    return df

east = clean_table(east)
west = clean_table(west)

team_stats = pd.concat([east, west], ignore_index=True)

print("Teams pulled:", len(team_stats))
team_stats.head()

In [ ]:
g = games_df.copy()

g = g.merge(team_stats, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"ppg":"home_ppg","oppg":"home_oppg"}) \
     .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg"}) \
     .drop(columns=["team"])

g["home_points_proj"] = (g["home_ppg"] + g["away_oppg"]) / 2
g["away_points_proj"] = (g["away_ppg"] + g["home_oppg"]) / 2

g["proj_margin"] = g["home_points_proj"] - g["away_points_proj"]
g["proj_total"] = g["home_points_proj"] + g["away_points_proj"]

SD_MARGIN = 14.5
SD_TOTAL = 19.0

def sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    total  = np.random.normal(row.proj_total, SD_TOTAL, SIMS)
    return pd.Series({
        "home_cover_prob": np.mean(margin > row.home_spread),
        "over_prob": np.mean(total > row.total_line)
    })

sim_results = g.apply(sim, axis=1)
g = pd.concat([g, sim_results], axis=1)

def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price), axis=1)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# =====================================================
# HARD CLEAN + FINAL EV OUTPUT (NO HUNTING REQUIRED)
# =====================================================

# 1) Remove duplicate columns
g = g.loc[:, ~g.columns.duplicated()].copy()

# 2) Force numeric types (prevents silent alignment bugs)
g["home_cover_prob"] = pd.to_numeric(g["home_cover_prob"], errors="coerce")
g["home_spread_price"] = pd.to_numeric(g["home_spread_price"], errors="coerce")

# 3) Recalculate market implied probability safely
def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)

# 4) Calculate edge safely
g["edge"] = g["home_cover_prob"] - g["market_prob"]

# 5) Kelly sizing (half Kelly, capped)
def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["kelly"] = g.apply(
    lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price),
    axis=1
)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

# 6) Final sorted card
final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# =====================================================
# FINAL PROJECTION CALIBRATION (STABILIZE PROBS)
# =====================================================

# 1) Force projection to stay near market
CALIBRATION_WEIGHT = 0.75   # 75% market anchor
MAX_MARGIN_DEV = 4.5        # max deviation from spread

g = g.loc[:, ~g.columns.duplicated()].copy()

g["market_margin"] = -g["home_spread"]

# shrink projected margin heavily toward market
g["proj_margin"] = (
    g["proj_margin"] * (1 - CALIBRATION_WEIGHT)
    + g["market_margin"] * CALIBRATION_WEIGHT
)

# hard cap deviation
diff = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# 2) Re-simulate with realistic variance
SD_MARGIN = 15.5

def calibrated_sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    return np.mean(margin > row.home_spread)

g["home_cover_prob"] = g.apply(calibrated_sim, axis=1)

# 3) Recalculate edge + Kelly clean
def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(
    lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price),
    axis=1
)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "proj_margin",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# =====================================================
# FORCE CLEAN TEAM NAME MATCH + REBUILD PROJECTION
# =====================================================

# 1) Clean BBR team names
team_stats["team"] = (
    team_stats["team"]
    .str.replace(r"\s*\(\d+\)", "", regex=True)
    .str.strip()
)

# 2) Clean games_df team names
games_df["home_team"] = games_df["home_team"].str.strip()
games_df["away_team"] = games_df["away_team"].str.strip()

# 3) Debug unmatched teams
missing_home = set(games_df["home_team"]) - set(team_stats["team"])
missing_away = set(games_df["away_team"]) - set(team_stats["team"])

print("Unmatched Home Teams:", missing_home)
print("Unmatched Away Teams:", missing_away)

# 4) Merge cleanly
g = games_df.copy()

g = g.merge(team_stats, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"ppg":"home_ppg","oppg":"home_oppg"}) \
     .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg"}) \
     .drop(columns=["team"])

# 5) Verify merge success
print("Missing home_ppg rows:", g["home_ppg"].isna().sum())
print("Missing away_ppg rows:", g["away_ppg"].isna().sum())

# 6) Rebuild projection
g["home_points_proj"] = (g["home_ppg"] + g["away_oppg"]) / 2
g["away_points_proj"] = (g["away_ppg"] + g["home_oppg"]) / 2

g["proj_margin"] = g["home_points_proj"] - g["away_points_proj"]

# 7) Stable simulation
SD_MARGIN = 14.5

def sim(row):
    if pd.isna(row.proj_margin):
        return 0.0
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)
    return np.mean(margin > row.home_spread)

g["home_cover_prob"] = g.apply(sim, axis=1)

# 8) EV + Kelly clean
def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price), axis=1)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "proj_margin",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# =====================================================
# FIX COVER LOGIC (CRITICAL BUG)
# =====================================================

g = g.loc[:, ~g.columns.duplicated()].copy()

SD_MARGIN = 14.5

def correct_sim(row):
    margin = np.random.normal(row.proj_margin, SD_MARGIN, SIMS)

    # FIX: correct spread comparison
    spread_to_cover = abs(row.home_spread)

    if row.home_spread < 0:
        # home is favorite
        return np.mean(margin > spread_to_cover)
    else:
        # home is underdog
        return np.mean(margin > -spread_to_cover)

g["home_cover_prob"] = g.apply(correct_sim, axis=1)

# Recalculate EV + Kelly

def american_to_prob(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def kelly_fraction(p, odds):
    if odds < 0:
        b = 100 / abs(odds)
    else:
        b = odds / 100
    return max(0, (p*(b+1)-1)/b)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price), axis=1)

g["units"] = 0.0
mask = g["edge"] > 0
g.loc[mask, "units"] = (g.loc[mask, "kelly"] / 2).clip(0.25, 1.0)

final_card = g.sort_values("edge", ascending=False)

final_card[[
    "home_team",
    "home_spread",
    "proj_margin",
    "home_cover_prob",
    "edge",
    "units"
]]

In [ ]:
# =====================================================
# MARKET-ANCHORED PROJECTION
# =====================================================

MARKET_WEIGHT = 0.65   # tune 0.6–0.75

g["market_margin"] = -g["home_spread"]

g["proj_margin"] = (
    MARKET_WEIGHT * g["market_margin"] +
    (1 - MARKET_WEIGHT) * g["proj_margin"]
)

In [ ]:
HOME_ADV = 1.8   # league average 1.5–2.5

g["proj_margin"] += HOME_ADV

In [ ]:
g["expected_possessions"] = (
    g["home_ppg"] + g["away_ppg"]
) / 2

g["pace_factor"] = g["expected_possessions"] / g["expected_possessions"].mean()

SD_MARGIN = 13.5 * g["pace_factor"]

In [ ]:
g.loc[abs(g["home_spread"]) > 12, "proj_margin"] *= 0.92

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
# ==========================================
# NBA STABLE MODEL — FULL TOP→BOTTOM SCRIPT
# (Colab-friendly, efficient, correct spread logic)
# Sources:
#  - Market (spreads/totals) from The Odds API
#  - Season team scoring/defense from Basketball-Reference (E+W standings tables)
# ==========================================

# ---------- 0) Imports / Settings ----------
import os
import time
import math
import requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ---- REQUIRED ----
# SPORT key for Odds API (NBA)
SPORT = "basketball_nba"

# ---- SIM SETTINGS ----
SIMS = 20000  # 10k–50k; 20k is a good balance
SD_MARGIN = 14.5  # NBA spread cover dispersion
SD_TOTAL  = 19.0  # totals dispersion (if you simulate totals later)

# ---- BANKROLL / KELLY ----
UNIT_MIN = 0.25
UNIT_CAP = 1.00
KELLY_DIVISOR = 2.0   # half-Kelly; use 3.0 if you want more conservative

# ---- OPTIONAL STABILIZERS ----
HOME_ADV = 1.8          # points added to home margin (optional)
MARKET_ANCHOR_W = 0.65  # blend model margin toward market (0.6–0.75 recommended)
MAX_MARGIN_DEV = 6.0    # cap deviation from market in points (4–8)

# ---------- 1) API Key ----------
# Set once in Colab (safe) or hardcode temporarily.
# os.environ["ODDS_API_KEY"] = "PASTE_REAL_KEY_HERE"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

# ---------- 2) Odds API: Pull Market Data (Spreads/Totals) ----------
def pull_market_data():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "spreads,totals,h2h",
        "oddsFormat": "american",
    }
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API market pull failed {r.status_code}: {r.text[:300]}")
    data = r.json()

    rows = []
    for game in data:
        home = game.get("home_team")
        away = game.get("away_team")
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        spreads = None
        totals = None

        # grab first available spreads/totals across books
        for book in books:
            for market in book.get("markets", []):
                if market.get("key") == "spreads" and spreads is None:
                    spreads = market.get("outcomes")
                if market.get("key") == "totals" and totals is None:
                    totals = market.get("outcomes")
            if spreads is not None and totals is not None:
                break

        if not spreads or not totals:
            continue

        # home spread outcome
        try:
            home_out = next(o for o in spreads if o.get("name") == home)
            total_out = totals[0]  # totals outcomes: "Over"/"Under" with same point
        except StopIteration:
            continue
        except Exception:
            continue

        # Some books return totals as two rows; both share same point; pick first.
        rows.append({
            "home_team": home,
            "away_team": away,
            "home_spread": float(home_out.get("point")),
            "home_spread_price": float(home_out.get("price")),
            "total_line": float(total_out.get("point")),
        })

    df = pd.DataFrame(rows).drop_duplicates()
    if df.empty:
        raise RuntimeError("No market games pulled (empty df). Check API tier/params.")
    return df

games_df = pull_market_data()
print("Pulled market games:", len(games_df))
games_df.head()


# ---------- 3) Basketball-Reference: Pull Season Team Stats (PPG / Opp PPG) ----------
# We use the E and W standings tables on the league page:
#  - tables[0] = East standings with PS/G and PA/G
#  - tables[1] = West standings with PS/G and PA/G
#
# IMPORTANT: Set SEASON_YEAR to the league year shown in the URL:
#  - NBA_2026 = 2025-26 season page in BRef convention
SEASON_YEAR = "2026"
BREF_URL = f"https://www.basketball-reference.com/leagues/NBA_{SEASON_YEAR}.html"

def pull_bref_team_ppg_oppg(url=BREF_URL):
    tables = pd.read_html(url)
    if len(tables) < 2:
        raise RuntimeError("Unexpected BRef tables count; page layout changed.")

    east = tables[0]
    west = tables[1]

    # flatten if multi-index
    if isinstance(east.columns, pd.MultiIndex):
        east.columns = east.columns.get_level_values(-1)
    if isinstance(west.columns, pd.MultiIndex):
        west.columns = west.columns.get_level_values(-1)

    def clean_standings_table(df):
        # columns like: ['Eastern Conference','W','L',...,'PS/G','PA/G','SRS']
        team_col = df.columns[0]
        df = df.rename(columns={
            team_col: "team",
            "PS/G": "ppg",
            "PA/G": "oppg",
        })
        df = df[["team", "ppg", "oppg"]].copy()

        # remove any header-like rows if present
        df["team"] = df["team"].astype(str)
        df = df[~df["team"].isin(["Eastern Conference", "Western Conference"])]
        df = df[~df["team"].str.contains("Division", na=False)]

        return df

    team_stats = pd.concat([clean_standings_table(east), clean_standings_table(west)], ignore_index=True)

    # remove seeding numbers like "Boston Celtics (2)" and NBSP
    team_stats["team"] = (team_stats["team"]
                          .str.replace(r"\s*\(\d+\)", "", regex=True)
                          .str.replace("\xa0", " ", regex=False)
                          .str.strip())
    team_stats["ppg"] = pd.to_numeric(team_stats["ppg"], errors="coerce")
    team_stats["oppg"] = pd.to_numeric(team_stats["oppg"], errors="coerce")
    team_stats = team_stats.dropna(subset=["team", "ppg", "oppg"]).drop_duplicates(subset=["team"])

    if len(team_stats) != 30:
        print("WARNING: expected 30 teams; got", len(team_stats))

    return team_stats

team_stats = pull_bref_team_ppg_oppg()
print("Teams pulled from BRef:", len(team_stats))
team_stats.head()

# Optional: cache locally for faster reruns
team_stats.to_csv("season_team_stats_bref.csv", index=False)
print("Saved season_team_stats_bref.csv")


# ---------- 4) Team Name Alignment (very important) ----------
# If you ever get unmatched teams, add mappings here.
NAME_MAP = {
    # Example fixes (only if needed):
    # "LA Clippers": "Los Angeles Clippers",
    # "LA Lakers": "Los Angeles Lakers",
}

def normalize_team_name(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.replace("\xa0", " ", regex=False).str.strip()
    return s.replace(NAME_MAP)

games_df["home_team"] = normalize_team_name(games_df["home_team"])
games_df["away_team"] = normalize_team_name(games_df["away_team"])
team_stats["team"] = normalize_team_name(team_stats["team"])

missing_home = set(games_df["home_team"]) - set(team_stats["team"])
missing_away = set(games_df["away_team"]) - set(team_stats["team"])
print("Unmatched Home Teams:", missing_home)
print("Unmatched Away Teams:", missing_away)


# ---------- 5) Merge Season Stats into Slate ----------
g = games_df.merge(team_stats, left_on="home_team", right_on="team", how="left") \
            .rename(columns={"ppg": "home_ppg", "oppg": "home_oppg"}) \
            .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg": "away_ppg", "oppg": "away_oppg"}) \
     .drop(columns=["team"])

if g["home_ppg"].isna().any() or g["away_ppg"].isna().any():
    raise RuntimeError("Merge failed: missing home/away ppg after merge. Fix NAME_MAP.")


# ---------- 6) Build Projection (season scoring/defense) ----------
# Core expected points per team:
g["home_points_proj"] = (g["home_ppg"] + g["away_oppg"]) / 2.0
g["away_points_proj"] = (g["away_ppg"] + g["home_oppg"]) / 2.0

# Raw model margin:
g["proj_margin_raw"] = g["home_points_proj"] - g["away_points_proj"]

# Add home-court (optional but recommended)
g["proj_margin_raw"] = g["proj_margin_raw"] + HOME_ADV

# Market implied margin (home perspective):
# If home_spread is -8.5, market expects home by ~8.5 => market_margin = +8.5
g["market_margin"] = -g["home_spread"]

# Market-anchored margin (recommended)
g["proj_margin"] = (MARKET_ANCHOR_W * g["market_margin"]) + ((1 - MARKET_ANCHOR_W) * g["proj_margin_raw"])

# Cap deviation from market (prevents wild outputs)
diff = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)


# ---------- 7) Correct Spread Cover Simulation (VECTORIZED + FAST) ----------
# IMPORTANT:
# A home spread of -8.5 means: home must win by > 8.5 to cover.
# A home spread of +8.0 means: home covers if margin > -8.0 (i.e., can lose by up to 7.5 and still cover).
#
# Condition: margin > (-home_spread)
# because -(-8.5)=+8.5; -(+8.0)=-8.0

spreads_to_cover = (-g["home_spread"]).values  # vector
mu = g["proj_margin"].values.reshape(-1, 1)    # (N,1)

# Simulate margins: (N,SIMS)
margins = np.random.normal(loc=mu, scale=SD_MARGIN, size=(len(g), SIMS))

# Cover probability vector: mean over sims
g["home_cover_prob"] = (margins > spreads_to_cover.reshape(-1, 1)).mean(axis=1)

# Optional: totals (if you want)
# Build proj_total and simulate totals similarly (kept off by default for speed)
# g["proj_total"] = (g["home_points_proj"] + g["away_points_proj"])
# totals = np.random.normal(loc=g["proj_total"].values.reshape(-1,1), scale=SD_TOTAL, size=(len(g), SIMS))
# g["over_prob"] = (totals > g["total_line"].values.reshape(-1,1)).mean(axis=1)


# ---------- 8) EV + Kelly ----------
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p)
    odds = float(odds)
    if odds < 0:
        b = 100.0 / abs(odds)
    else:
        b = odds / 100.0
    f = (p * (b + 1.0) - 1.0) / b
    return max(0.0, f)

g["market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["edge"] = g["home_cover_prob"] - g["market_prob"]

g["kelly"] = g.apply(lambda r: kelly_fraction(r.home_cover_prob, r.home_spread_price), axis=1)

g["units"] = 0.0
pos = g["edge"] > 0
g.loc[pos, "units"] = (g.loc[pos, "kelly"] / KELLY_DIVISOR).clip(UNIT_MIN, UNIT_CAP)


# ---------- 9) Output (Top 10 + Full Card) ----------
card_cols = [
    "home_team", "away_team",
    "home_spread", "home_spread_price",
    "proj_margin_raw", "market_margin", "proj_margin",
    "home_cover_prob", "market_prob", "edge",
    "units",
    "total_line",
]
final_card = g.sort_values("edge", ascending=False)

print("\n=== TOP 10 (SPREADS) ===")
display(final_card[card_cols].head(10))

print("\n=== ALL GAMES (SPREADS) ===")
display(final_card[card_cols])


# ---------- 10) Optional: Discord-ready text generator ----------
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_num(x): return f"{x:.2f}"

top = final_card[final_card["units"] > 0].head(10).copy()

lines = []
lines.append("== CDR BETTING | NBA CARD ==")
for _, r in top.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"
    bet = f"{r['home_team']} {r['home_spread']:+g}"
    units = f"{r['units']:.2f}u"
    prob = fmt_pct(r["home_cover_prob"])
    edge = fmt_pct(r["edge"])
    lines.append("")
    lines.append(matchup)
    lines.append(f"{bet} ({int(r['home_spread_price']):+d}) — {units}")
    lines.append(f"Model Cover%: {prob} | Edge: {edge}")

discord_text = "\n".join(lines)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

In [ ]:
# ==========================================
# NBA STABLE MODEL — FULL TOP→BOTTOM (ALL LAYERS)
# Colab-friendly, efficient, correct, production-style.
# Data:
#  - The Odds API: spreads, totals, h2h (ML), team_totals (when offered)
#  - Basketball-Reference: PS/G, PA/G, SRS (East + West standings tables)
# ==========================================

import os, math, time, requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# -----------------------
# USER SETTINGS
# -----------------------
ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"  # <-- REQUIRED
SPORT = "basketball_nba"

SEASON_YEAR = "2026"  # BRef convention (NBA_2026 = 2025-26)

SIMS = 20000          # 10k–50k; 20k is a strong default
BASE_SD_MARGIN = 14.5 # baseline spread margin SD
BASE_SD_TOTAL  = 19.0 # baseline total SD
BASE_SD_TT     = 12.0 # baseline team total SD

HOME_ADV = 1.8         # home court in margin points
SRS_WEIGHT = 0.35      # 0–0.6; how much SRS influences margin
MARKET_ANCHOR_W = 0.65 # 0.6–0.75; blend toward market
MAX_MARGIN_DEV = 6.0   # cap deviation from market (points)

EDGE_MIN = 0.02         # filter micro-edges (2%)
KELLY_DIVISOR = 2.0     # half-Kelly; set 3.0 for more conservative
UNIT_MIN = 0.25
UNIT_CAP = 1.00

# -----------------------
# SAFETY CHECK
# -----------------------
if not ODDS_API_KEY or "PASTE_" in ODDS_API_KEY:
    raise RuntimeError("Set ODDS_API_KEY at the top of the cell.")

# -----------------------
# HELPERS
# -----------------------
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p: float) -> float:
    # fair line from probability (no vig)
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    else:
        return round(((1 - p) / p) * 100, 0)

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p); odds = float(odds)
    if odds < 0:
        b = 100.0 / abs(odds)
    else:
        b = odds / 100.0
    f = (p * (b + 1.0) - 1.0) / b
    return max(0.0, f)

def normalize_team_name_series(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace("\xa0", " ", regex=False)
             .str.strip())

# optional mapping if you ever see mismatches
NAME_MAP = {
    # "LA Clippers": "Los Angeles Clippers",
    # "LA Lakers": "Los Angeles Lakers",
}

def apply_name_map(s: pd.Series) -> pd.Series:
    s = normalize_team_name_series(s)
    return s.replace(NAME_MAP)

# -----------------------
# 1) PULL MARKET DATA (MULTI-BOOK CONSENSUS + BEST PRICE)
# -----------------------
def pull_market_data_best():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "spreads,totals,h2h,team_totals",
        "oddsFormat": "american",
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    rows = []
    for game in data:
        home = game.get("home_team")
        away = game.get("away_team")
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        # gather all book points for consensus
        spread_points, spread_home_prices = [], []
        total_points = []
        # best ML prices (best payout)
        best_ml_home = None
        best_ml_away = None

        # team totals (home/away) across books
        htt_points, htt_over_prices, htt_under_prices = [], [], []
        att_points, att_over_prices, att_under_prices = [], [], []

        for book in books:
            for m in book.get("markets", []):
                key = m.get("key")
                outs = m.get("outcomes") or []

                if key == "spreads":
                    h = next((o for o in outs if o.get("name") == home), None)
                    if h and h.get("point") is not None and h.get("price") is not None:
                        spread_points.append(float(h["point"]))
                        spread_home_prices.append(float(h["price"]))

                elif key == "totals":
                    any_o = next((o for o in outs if o.get("point") is not None), None)
                    if any_o:
                        total_points.append(float(any_o["point"]))

                elif key == "h2h":
                    h = next((o for o in outs if o.get("name") == home), None)
                    a = next((o for o in outs if o.get("name") == away), None)
                    if h and h.get("price") is not None:
                        p = float(h["price"])
                        if best_ml_home is None or p > best_ml_home:
                            best_ml_home = p
                    if a and a.get("price") is not None:
                        p = float(a["price"])
                        if best_ml_away is None or p > best_ml_away:
                            best_ml_away = p

                elif key == "team_totals":
                    # outcomes are typically:
                    #  - {"name":"Over","description":"<TEAM>","point":..., "price":...}
                    #  - {"name":"Under","description":"<TEAM>","point":..., "price":...}
                    # description holds team
                    for o in outs:
                        desc = o.get("description")
                        nm = o.get("name")
                        pt = o.get("point")
                        pr = o.get("price")
                        if desc is None or pt is None or pr is None or nm is None:
                            continue
                        desc = str(desc)
                        pt = float(pt); pr = float(pr)
                        if desc == home:
                            if nm == "Over":
                                htt_points.append(pt); htt_over_prices.append(pr)
                            elif nm == "Under":
                                htt_under_prices.append(pr)
                        elif desc == away:
                            if nm == "Over":
                                att_points.append(pt); att_over_prices.append(pr)
                            elif nm == "Under":
                                att_under_prices.append(pr)

        if not spread_points or not total_points:
            continue

        # consensus lines
        spread_cons = float(np.median(spread_points))
        total_cons  = float(np.median(total_points))

        # choose best spread price among books closest to consensus point
        best_spread_point, best_spread_price = None, None
        for pt, pr in zip(spread_points, spread_home_prices):
            if best_spread_point is None:
                best_spread_point, best_spread_price = pt, pr
            else:
                if abs(pt - spread_cons) < abs(best_spread_point - spread_cons):
                    best_spread_point, best_spread_price = pt, pr
                elif abs(pt - spread_cons) == abs(best_spread_point - spread_cons) and pr > best_spread_price:
                    best_spread_point, best_spread_price = pt, pr

        # team totals: use median point and best over price near that point
        def consensus_and_best(point_list, over_price_list):
            if not point_list or not over_price_list:
                return (np.nan, np.nan)
            cons = float(np.median(point_list))
            best_pt, best_pr = None, None
            for pt, pr in zip(point_list, over_price_list):
                if best_pt is None:
                    best_pt, best_pr = pt, pr
                else:
                    if abs(pt - cons) < abs(best_pt - cons):
                        best_pt, best_pr = pt, pr
                    elif abs(pt - cons) == abs(best_pt - cons) and pr > best_pr:
                        best_pt, best_pr = pt, pr
            return (cons, best_pr)

        home_tt_line, home_tt_over_price = consensus_and_best(htt_points, htt_over_prices)
        away_tt_line, away_tt_over_price = consensus_and_best(att_points, att_over_prices)

        rows.append({
            "home_team": home,
            "away_team": away,

            "home_spread": float(best_spread_point),
            "home_spread_price": float(best_spread_price),
            "spread_consensus": spread_cons,

            "total_line": total_cons,

            "home_ml_price": best_ml_home,
            "away_ml_price": best_ml_away,

            "home_tt_line": home_tt_line,
            "home_tt_over_price": home_tt_over_price,
            "away_tt_line": away_tt_line,
            "away_tt_over_price": away_tt_over_price,
        })

    df = pd.DataFrame(rows).drop_duplicates()
    if df.empty:
        raise RuntimeError("No games pulled from Odds API (empty df).")
    return df

games_df = pull_market_data_best()
games_df["home_team"] = apply_name_map(games_df["home_team"])
games_df["away_team"] = apply_name_map(games_df["away_team"])
print("Pulled market games:", len(games_df))
display(games_df.head(5))

# cache market snapshot to avoid re-hits
games_df.to_csv("market_snapshot.csv", index=False)
print("Saved market_snapshot.csv")


# -----------------------
# 2) PULL SEASON TEAM STATS (BREF: PS/G, PA/G, SRS)
# -----------------------
BREF_URL = f"https://www.basketball-reference.com/leagues/NBA_{SEASON_YEAR}.html"

def pull_bref_team_stats(url=BREF_URL):
    tables = pd.read_html(url)
    if len(tables) < 2:
        raise RuntimeError("Unexpected BRef tables count.")

    east = tables[0]
    west = tables[1]

    if isinstance(east.columns, pd.MultiIndex):
        east.columns = east.columns.get_level_values(-1)
    if isinstance(west.columns, pd.MultiIndex):
        west.columns = west.columns.get_level_values(-1)

    def clean(df):
        team_col = df.columns[0]
        df = df.rename(columns={team_col: "team"})
        # standings table columns are PS/G, PA/G, SRS
        need = ["team", "PS/G", "PA/G", "SRS"]
        missing = [c for c in need if c not in df.columns]
        if missing:
            raise RuntimeError(f"BRef standings missing columns: {missing}. Columns={df.columns.tolist()}")

        out = df[need].copy()
        out = out.rename(columns={"PS/G":"ppg", "PA/G":"oppg", "SRS":"srs"})
        out["team"] = out["team"].astype(str)
        out = out[~out["team"].isin(["Eastern Conference","Western Conference"])]
        out = out[~out["team"].str.contains("Division", na=False)]
        return out

    team_stats = pd.concat([clean(east), clean(west)], ignore_index=True)

    # cleanup names (remove seeding, NBSP)
    team_stats["team"] = (team_stats["team"]
                          .str.replace(r"\s*\(\d+\)", "", regex=True)
                          .str.replace("\xa0", " ", regex=False)
                          .str.strip())
    team_stats["team"] = apply_name_map(team_stats["team"])

    for c in ["ppg","oppg","srs"]:
        team_stats[c] = pd.to_numeric(team_stats[c], errors="coerce")

    team_stats = team_stats.dropna(subset=["team","ppg","oppg","srs"]).drop_duplicates(subset=["team"])
    return team_stats

team_stats = pull_bref_team_stats()
print("Teams pulled from BRef:", len(team_stats))
display(team_stats.head(5))

team_stats.to_csv("season_team_stats_bref.csv", index=False)
print("Saved season_team_stats_bref.csv")


# -----------------------
# 3) VALIDATE NAME MATCH
# -----------------------
missing_home = set(games_df["home_team"]) - set(team_stats["team"])
missing_away = set(games_df["away_team"]) - set(team_stats["team"])
print("Unmatched Home Teams:", missing_home)
print("Unmatched Away Teams:", missing_away)
if missing_home or missing_away:
    raise RuntimeError("Team name mismatch. Add fixes to NAME_MAP at top.")


# -----------------------
# 4) MERGE TEAM STATS INTO SLATE
# -----------------------
g = games_df.merge(team_stats, left_on="home_team", right_on="team", how="left") \
            .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
            .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

if g[["home_ppg","away_ppg","home_srs","away_srs"]].isna().any().any():
    raise RuntimeError("Merge produced NaNs; check name normalization.")

# -----------------------
# 5) PROJECTION LAYERS (ALL)
# -----------------------
# A) Season offense/defense base points
g["home_points_proj_raw"] = (g["home_ppg"] + g["away_oppg"]) / 2.0
g["away_points_proj_raw"] = (g["away_ppg"] + g["home_oppg"]) / 2.0

# B) Raw margin
g["proj_margin_raw"] = g["home_points_proj_raw"] - g["away_points_proj_raw"]

# C) SRS strength adjustment (home vs away)
# SRS is per-game net rating style; use diff to adjust margin.
g["srs_diff"] = (g["home_srs"] - g["away_srs"])
g["proj_margin_srs"] = g["proj_margin_raw"] + (SRS_WEIGHT * g["srs_diff"])

# D) Home court
g["proj_margin_srs"] = g["proj_margin_srs"] + HOME_ADV

# E) Market implied margin (home perspective)
# home_spread negative means favorite; market margin = -home_spread (positive)
g["market_margin"] = -g["home_spread"]

# F) Market anchoring + cap
g["proj_margin"] = (MARKET_ANCHOR_W * g["market_margin"]) + ((1 - MARKET_ANCHOR_W) * g["proj_margin_srs"])
diff = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# G) Total projection (season base; optionally market-anchor it lightly)
g["proj_total_raw"] = g["home_points_proj_raw"] + g["away_points_proj_raw"]
# light anchor to market totals for stability
TOTAL_ANCHOR_W = 0.55
g["proj_total"] = (TOTAL_ANCHOR_W * g["total_line"]) + ((1 - TOTAL_ANCHOR_W) * g["proj_total_raw"])

# H) Team total projections (derived)
g["home_tt_proj"] = (g["proj_total"] + g["proj_margin"]) / 2.0
g["away_tt_proj"] = (g["proj_total"] - g["proj_margin"]) / 2.0

# I) Pace/variance scaling (fast proxy)
# Higher totals -> slightly higher variance.
tot_mean = g["total_line"].mean()
g["sd_margin"] = (BASE_SD_MARGIN * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)
g["sd_total"]  = (BASE_SD_TOTAL  * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)
g["sd_tt"]     = (BASE_SD_TT     * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)

# -----------------------
# 6) SIMULATION LAYERS (VECTORIZED, FAST)
# -----------------------
N = len(g)

# A) Spread cover probability
# correct cover: margin > (-home_spread)
spreads_to_cover = (-g["home_spread"]).values.reshape(-1, 1)
mu_margin = g["proj_margin"].values.reshape(-1, 1)
sd_margin = g["sd_margin"].values.reshape(-1, 1)

margins = np.random.normal(loc=mu_margin, scale=sd_margin, size=(N, SIMS))
g["home_cover_prob"] = (margins > spreads_to_cover).mean(axis=1)

# B) Game total over probability
mu_total = g["proj_total"].values.reshape(-1, 1)
sd_total = g["sd_total"].values.reshape(-1, 1)
totals = np.random.normal(loc=mu_total, scale=sd_total, size=(N, SIMS))
g["over_prob"] = (totals > g["total_line"].values.reshape(-1, 1)).mean(axis=1)

# C) Team totals probabilities (Over)
# If Odds API didn't return team totals for a game, these will be NaN.
mu_htt = g["home_tt_proj"].values.reshape(-1, 1)
mu_att = g["away_tt_proj"].values.reshape(-1, 1)
sd_tt  = g["sd_tt"].values.reshape(-1, 1)

home_tts = np.random.normal(loc=mu_htt, scale=sd_tt, size=(N, SIMS))
away_tts = np.random.normal(loc=mu_att, scale=sd_tt, size=(N, SIMS))

g["home_tt_over_prob"] = np.where(
    g["home_tt_line"].notna(),
    (home_tts > g["home_tt_line"].values.reshape(-1, 1)).mean(axis=1),
    np.nan
)

g["away_tt_over_prob"] = np.where(
    g["away_tt_line"].notna(),
    (away_tts > g["away_tt_line"].values.reshape(-1, 1)).mean(axis=1),
    np.nan
)

# D) Moneyline win probability from margin distribution
# Pr(home wins) = P(margin > 0)
g["home_win_prob"] = (margins > 0.0).mean(axis=1)
g["away_win_prob"] = 1.0 - g["home_win_prob"]

# -----------------------
# 7) EV + KELLY PER MARKET (ALL)
# -----------------------
def size_units(p_model, odds):
    if pd.isna(p_model) or pd.isna(odds):
        return 0.0
    p_market = american_to_prob(odds)
    edge = p_model - p_market
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    u = min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP)
    return float(u)

# Spread
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]
g["spread_units"] = g.apply(lambda r: size_units(r.home_cover_prob, r.home_spread_price), axis=1)

# Total (assume price ~ -110 if not provided; Odds API totals prices not pulled here to keep it fast/robust)
# If you want exact total prices, we can extend pull to store them too. For now assume -110.
ASSUMED_TOTAL_PRICE = -110.0
g["total_market_prob"] = american_to_prob(ASSUMED_TOTAL_PRICE)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]
g["total_units"] = g["over_prob"].apply(lambda p: size_units(p, ASSUMED_TOTAL_PRICE))

# Team totals (Over) — assume -110 if over price missing
g["home_tt_price"] = g["home_tt_over_price"].fillna(ASSUMED_TOTAL_PRICE)
g["away_tt_price"] = g["away_tt_over_price"].fillna(ASSUMED_TOTAL_PRICE)

g["home_tt_market_prob"] = g["home_tt_price"].apply(american_to_prob)
g["away_tt_market_prob"] = g["away_tt_price"].apply(american_to_prob)

g["home_tt_edge"] = g["home_tt_over_prob"] - g["home_tt_market_prob"]
g["away_tt_edge"] = g["away_tt_over_prob"] - g["away_tt_market_prob"]

g["home_tt_units"] = g.apply(lambda r: size_units(r.home_tt_over_prob, r.home_tt_price) if pd.notna(r.home_tt_line) else 0.0, axis=1)
g["away_tt_units"] = g.apply(lambda r: size_units(r.away_tt_over_prob, r.away_tt_price) if pd.notna(r.away_tt_line) else 0.0, axis=1)

# Moneyline
# Use best ML prices we pulled (may be None for some games)
g["home_ml_market_prob"] = g["home_ml_price"].apply(lambda x: american_to_prob(x) if pd.notna(x) else np.nan)
g["away_ml_market_prob"] = g["away_ml_price"].apply(lambda x: american_to_prob(x) if pd.notna(x) else np.nan)

g["home_ml_edge"] = g["home_win_prob"] - g["home_ml_market_prob"]
g["away_ml_edge"] = g["away_win_prob"] - g["away_ml_market_prob"]

g["home_ml_units"] = g.apply(lambda r: size_units(r.home_win_prob, r.home_ml_price) if pd.notna(r.home_ml_price) else 0.0, axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units(r.away_win_prob, r.away_ml_price) if pd.notna(r.away_ml_price) else 0.0, axis=1)

# -----------------------
# 8) BUILD A SINGLE "PLAYS" TABLE ACROSS ALL MARKETS
# -----------------------
plays = []

def add_play(row, market, side, line, odds, prob, edge, units):
    plays.append({
        "market": market,
        "matchup": f"{row['away_team']} at {row['home_team']}",
        "team_side": side,
        "line": line,
        "odds": odds,
        "model_prob": float(prob) if pd.notna(prob) else np.nan,
        "edge": float(edge) if pd.notna(edge) else np.nan,
        "units": float(units),
        "fair_odds": prob_to_american(prob) if pd.notna(prob) else np.nan,
    })

for _, r in g.iterrows():
    # Spread (home side as listed; if you want away spread too, we can add)
    add_play(r, "SPREAD", r["home_team"], r["home_spread"], r["home_spread_price"], r["home_cover_prob"], r["spread_edge"], r["spread_units"])

    # Total
    add_play(r, "TOTAL", "Over", r["total_line"], ASSUMED_TOTAL_PRICE, r["over_prob"], r["total_edge"], r["total_units"])

    # Team Totals (Over)
    if pd.notna(r["home_tt_line"]):
        add_play(r, "TEAM_TOTAL", f"{r['home_team']} Over", r["home_tt_line"], r["home_tt_price"], r["home_tt_over_prob"], r["home_tt_edge"], r["home_tt_units"])
    if pd.notna(r["away_tt_line"]):
        add_play(r, "TEAM_TOTAL", f"{r['away_team']} Over", r["away_tt_line"], r["away_tt_price"], r["away_tt_over_prob"], r["away_tt_edge"], r["away_tt_units"])

    # ML (both sides)
    if pd.notna(r["home_ml_price"]):
        add_play(r, "ML", r["home_team"], "ML", r["home_ml_price"], r["home_win_prob"], r["home_ml_edge"], r["home_ml_units"])
    if pd.notna(r["away_ml_price"]):
        add_play(r, "ML", r["away_team"], "ML", r["away_ml_price"], r["away_win_prob"], r["away_ml_edge"], r["away_ml_units"])

plays_df = pd.DataFrame(plays)
plays_df = plays_df.sort_values(["units","edge"], ascending=[False, False])

print("\n=== TOP PLAYS (ALL MARKETS) ===")
display(plays_df[plays_df["units"] > 0].head(20))

print("\n=== FULL PLAYS TABLE ===")
display(plays_df.head(200))

# -----------------------
# 9) DISCORD OUTPUT (ONLY PLAYS WITH UNITS > 0)
# -----------------------
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_odds(o):
    if pd.isna(o): return ""
    o = float(o)
    return f"{int(o):+d}" if abs(o - int(o)) < 1e-6 else f"{o:+.0f}"

discord_lines = []
discord_lines.append("== CDR BETTING | NBA FULL CARD ==")

for _, p in plays_df[plays_df["units"] > 0].head(15).iterrows():
    discord_lines.append("")
    discord_lines.append(p["matchup"])
    if p["market"] == "TOTAL":
        bet = f"{p['team_side']} {p['line']}"
    elif p["market"] == "TEAM_TOTAL":
        bet = f"{p['team_side']} {p['line']}"
    elif p["market"] == "ML":
        bet = f"{p['team_side']} ML"
    else:  # SPREAD
        bet = f"{p['team_side']} {p['line']:+g}"

    discord_lines.append(f"{p['market']}: {bet} ({fmt_odds(p['odds'])}) — {p['units']:.2f}u")
    discord_lines.append(f"Model%: {fmt_pct(p['model_prob'])} | Edge: {fmt_pct(p['edge'])} | Fair: {fmt_odds(p['fair_odds'])}")

discord_text = "\n".join(discord_lines)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

In [ ]:
# ==========================================
# NBA STABLE MODEL — FULL TOP→BOTTOM (FIXED)
# Colab-friendly. Efficient. Correct spread logic.
# Markets: SPREADS + TOTALS + ML (h2h)
# Sources:
#  - The Odds API (multi-book consensus + best price)
#  - Basketball-Reference (PS/G, PA/G, SRS) East+West standings tables
# ==========================================

import requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# -----------------------
# USER SETTINGS (EDIT THESE)
# -----------------------
ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"  # <-- REQUIRED
SPORT = "basketball_nba"

SEASON_YEAR = "2026"  # BRef: NBA_2026 page = 2025-26 season
SIMS = 20000          # 10k–50k; 20k good balance

# Model controls
BASE_SD_MARGIN = 14.5
BASE_SD_TOTAL  = 19.0

HOME_ADV = 1.8          # home court in points (margin)
SRS_WEIGHT = 0.35       # how much SRS diff shifts margin
MARKET_ANCHOR_W = 0.65  # blend toward market (0.6–0.75)
MAX_MARGIN_DEV = 6.0    # cap deviation from market

# Sizing
EDGE_MIN = 0.02
KELLY_DIVISOR = 2.0
UNIT_MIN = 0.25
UNIT_CAP = 1.00

# Totals pricing (Odds API totals prices not collected to keep robust + fast)
ASSUMED_TOTAL_PRICE = -110.0

# -----------------------
# SAFETY CHECK
# -----------------------
if not ODDS_API_KEY or "PASTE_" in ODDS_API_KEY:
    raise RuntimeError("Set ODDS_API_KEY at the top of the cell (replace placeholder).")

# -----------------------
# HELPERS
# -----------------------
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p: float) -> float:
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    else:
        return round(((1 - p) / p) * 100, 0)

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p); odds = float(odds)
    if odds < 0:
        b = 100.0 / abs(odds)
    else:
        b = odds / 100.0
    f = (p * (b + 1.0) - 1.0) / b
    return max(0.0, f)

def normalize_team_name_series(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace("\xa0", " ", regex=False)
             .str.strip())

# Optional mappings if a mismatch ever appears
NAME_MAP = {
    # "LA Clippers": "Los Angeles Clippers",
    # "LA Lakers": "Los Angeles Lakers",
}

def apply_name_map(s: pd.Series) -> pd.Series:
    s = normalize_team_name_series(s)
    return s.replace(NAME_MAP)

# -----------------------
# 1) MARKET DATA (SPREADS, TOTALS, ML) — MULTI-BOOK CONSENSUS + BEST PRICE
#    (NO team_totals here to avoid 422 errors)
# -----------------------
def pull_market_data_best():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "spreads,totals,h2h",
        "oddsFormat": "american",
    }
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API error {r.status_code}: {r.text[:400]}")
    data = r.json()

    rows = []
    for game in data:
        home = game.get("home_team")
        away = game.get("away_team")
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        spread_points, spread_home_prices = [], []
        total_points = []
        best_ml_home, best_ml_away = None, None

        for book in books:
            for m in book.get("markets", []):
                key = m.get("key")
                outs = m.get("outcomes") or []

                if key == "spreads":
                    h = next((o for o in outs if o.get("name") == home), None)
                    if h and h.get("point") is not None and h.get("price") is not None:
                        spread_points.append(float(h["point"]))
                        spread_home_prices.append(float(h["price"]))

                elif key == "totals":
                    any_o = next((o for o in outs if o.get("point") is not None), None)
                    if any_o:
                        total_points.append(float(any_o["point"]))

                elif key == "h2h":
                    h = next((o for o in outs if o.get("name") == home), None)
                    a = next((o for o in outs if o.get("name") == away), None)
                    if h and h.get("price") is not None:
                        p = float(h["price"])
                        if best_ml_home is None or p > best_ml_home:
                            best_ml_home = p
                    if a and a.get("price") is not None:
                        p = float(a["price"])
                        if best_ml_away is None or p > best_ml_away:
                            best_ml_away = p

        if not spread_points or not total_points:
            continue

        spread_cons = float(np.median(spread_points))
        total_cons  = float(np.median(total_points))

        # best spread price closest to consensus point
        best_spread_point, best_spread_price = None, None
        for pt, pr in zip(spread_points, spread_home_prices):
            if best_spread_point is None:
                best_spread_point, best_spread_price = pt, pr
            else:
                if abs(pt - spread_cons) < abs(best_spread_point - spread_cons):
                    best_spread_point, best_spread_price = pt, pr
                elif abs(pt - spread_cons) == abs(best_spread_point - spread_cons) and pr > best_spread_price:
                    best_spread_point, best_spread_price = pt, pr

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_spread": float(best_spread_point),
            "home_spread_price": float(best_spread_price),
            "spread_consensus": spread_cons,
            "total_line": total_cons,
            "home_ml_price": best_ml_home,
            "away_ml_price": best_ml_away,
        })

    df = pd.DataFrame(rows).drop_duplicates()
    if df.empty:
        raise RuntimeError("No games pulled from Odds API (empty df). Check tier/markets.")
    return df

games_df = pull_market_data_best()
games_df["home_team"] = apply_name_map(games_df["home_team"])
games_df["away_team"] = apply_name_map(games_df["away_team"])

print("Pulled market games:", len(games_df))
display(games_df.head(10))

# -----------------------
# 2) BREF TEAM STATS (PS/G, PA/G, SRS) — EAST + WEST
# -----------------------
BREF_URL = f"https://www.basketball-reference.com/leagues/NBA_{SEASON_YEAR}.html"

def pull_bref_team_stats(url=BREF_URL):
    tables = pd.read_html(url)
    if len(tables) < 2:
        raise RuntimeError("Unexpected BRef tables count (layout changed).")

    east, west = tables[0], tables[1]

    if isinstance(east.columns, pd.MultiIndex):
        east.columns = east.columns.get_level_values(-1)
    if isinstance(west.columns, pd.MultiIndex):
        west.columns = west.columns.get_level_values(-1)

    def clean(df):
        team_col = df.columns[0]
        df = df.rename(columns={team_col: "team"})
        need = ["team", "PS/G", "PA/G", "SRS"]
        missing = [c for c in need if c not in df.columns]
        if missing:
            raise RuntimeError(f"BRef standings missing columns: {missing}. Columns={df.columns.tolist()}")

        out = df[need].copy()
        out = out.rename(columns={"PS/G":"ppg", "PA/G":"oppg", "SRS":"srs"})
        out["team"] = out["team"].astype(str)
        out = out[~out["team"].isin(["Eastern Conference","Western Conference"])]
        out = out[~out["team"].str.contains("Division", na=False)]
        return out

    team_stats = pd.concat([clean(east), clean(west)], ignore_index=True)

    # cleanup team names (remove seeding) + NBSP
    team_stats["team"] = (team_stats["team"]
                          .str.replace(r"\s*\(\d+\)", "", regex=True)
                          .str.replace("\xa0", " ", regex=False)
                          .str.strip())
    team_stats["team"] = apply_name_map(team_stats["team"])

    for c in ["ppg","oppg","srs"]:
        team_stats[c] = pd.to_numeric(team_stats[c], errors="coerce")

    team_stats = team_stats.dropna(subset=["team","ppg","oppg","srs"]).drop_duplicates(subset=["team"])
    return team_stats

team_stats = pull_bref_team_stats()
print("Teams pulled from BRef:", len(team_stats))
display(team_stats.head(10))

# -----------------------
# 3) NAME MATCH VALIDATION
# -----------------------
missing_home = set(games_df["home_team"]) - set(team_stats["team"])
missing_away = set(games_df["away_team"]) - set(team_stats["team"])
print("Unmatched Home Teams:", missing_home)
print("Unmatched Away Teams:", missing_away)
if missing_home or missing_away:
    raise RuntimeError("Team name mismatch. Add fixes to NAME_MAP and rerun.")

# -----------------------
# 4) MERGE TEAM STATS INTO SLATE
# -----------------------
g = games_df.merge(team_stats, left_on="home_team", right_on="team", how="left") \
            .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
            .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

if g[["home_ppg","away_ppg","home_srs","away_srs"]].isna().any().any():
    raise RuntimeError("Merge produced NaNs; check name normalization.")

# -----------------------
# 5) PROJECTION LAYERS (SEASON + SRS + HOME + MARKET ANCHOR)
# -----------------------
g["home_points_proj_raw"] = (g["home_ppg"] + g["away_oppg"]) / 2.0
g["away_points_proj_raw"] = (g["away_ppg"] + g["home_oppg"]) / 2.0
g["proj_margin_raw"] = g["home_points_proj_raw"] - g["away_points_proj_raw"]

g["srs_diff"] = g["home_srs"] - g["away_srs"]
g["proj_margin_srs"] = g["proj_margin_raw"] + (SRS_WEIGHT * g["srs_diff"])
g["proj_margin_srs"] = g["proj_margin_srs"] + HOME_ADV

# market implied margin (home perspective): -spread
g["market_margin"] = -g["home_spread"]

# market anchor + cap
g["proj_margin"] = (MARKET_ANCHOR_W * g["market_margin"]) + ((1 - MARKET_ANCHOR_W) * g["proj_margin_srs"])
diff = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# totals projection: season base anchored to market total
g["proj_total_raw"] = g["home_points_proj_raw"] + g["away_points_proj_raw"]
TOTAL_ANCHOR_W = 0.55
g["proj_total"] = (TOTAL_ANCHOR_W * g["total_line"]) + ((1 - TOTAL_ANCHOR_W) * g["proj_total_raw"])

# dynamic variance scaling by total level (fast proxy)
tot_mean = g["total_line"].mean()
g["sd_margin"] = (BASE_SD_MARGIN * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)
g["sd_total"]  = (BASE_SD_TOTAL  * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)

# -----------------------
# 6) MONTE CARLO (VECTORIZED) — SPREAD, TOTAL, ML
# -----------------------
N = len(g)

# margin sims
mu_margin = g["proj_margin"].values.reshape(-1, 1)
sd_margin = g["sd_margin"].values.reshape(-1, 1)
margins = np.random.normal(loc=mu_margin, scale=sd_margin, size=(N, SIMS))

# SPREAD cover: margin > (-home_spread)  (correct logic)
spreads_to_cover = (-g["home_spread"]).values.reshape(-1, 1)
g["home_cover_prob"] = (margins > spreads_to_cover).mean(axis=1)

# TOTAL over: total > total_line
mu_total = g["proj_total"].values.reshape(-1, 1)
sd_total = g["sd_total"].values.reshape(-1, 1)
totals = np.random.normal(loc=mu_total, scale=sd_total, size=(N, SIMS))
g["over_prob"] = (totals > g["total_line"].values.reshape(-1, 1)).mean(axis=1)

# ML: P(home wins) = P(margin > 0)
g["home_win_prob"] = (margins > 0.0).mean(axis=1)
g["away_win_prob"] = 1.0 - g["home_win_prob"]

# -----------------------
# 7) EV + KELLY SIZING
# -----------------------
def size_units(p_model, odds):
    if pd.isna(p_model) or pd.isna(odds):
        return 0.0
    p_mkt = american_to_prob(odds)
    edge = p_model - p_mkt
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    return float(min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP))

# Spread
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]
g["spread_units"] = g.apply(lambda r: size_units(r.home_cover_prob, r.home_spread_price), axis=1)
g["spread_fair"] = g["home_cover_prob"].apply(prob_to_american)

# Total (assume -110)
g["total_market_prob"] = american_to_prob(ASSUMED_TOTAL_PRICE)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]
g["total_units"] = g["over_prob"].apply(lambda p: size_units(p, ASSUMED_TOTAL_PRICE))
g["total_fair"] = g["over_prob"].apply(prob_to_american)

# ML (best price from books)
g["home_ml_market_prob"] = g["home_ml_price"].apply(lambda x: american_to_prob(x) if pd.notna(x) else np.nan)
g["away_ml_market_prob"] = g["away_ml_price"].apply(lambda x: american_to_prob(x) if pd.notna(x) else np.nan)

g["home_ml_edge"] = g["home_win_prob"] - g["home_ml_market_prob"]
g["away_ml_edge"] = g["away_win_prob"] - g["away_ml_market_prob"]

g["home_ml_units"] = g.apply(lambda r: size_units(r.home_win_prob, r.home_ml_price) if pd.notna(r.home_ml_price) else 0.0, axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units(r.away_win_prob, r.away_ml_price) if pd.notna(r.away_ml_price) else 0.0, axis=1)

g["home_ml_fair"] = g["home_win_prob"].apply(prob_to_american)
g["away_ml_fair"] = g["away_win_prob"].apply(prob_to_american)

# -----------------------
# 8) BUILD "PLAYS" TABLE ACROSS MARKETS
# -----------------------
plays = []

def add_play(matchup, market, bet, odds, model_prob, edge, units, fair):
    plays.append({
        "matchup": matchup,
        "market": market,
        "bet": bet,
        "odds": odds,
        "model_prob": model_prob,
        "edge": edge,
        "units": units,
        "fair_odds": fair,
    })

for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # Spread (home side)
    add_play(matchup, "SPREAD",
             f"{r['home_team']} {r['home_spread']:+g}",
             r["home_spread_price"], r["home_cover_prob"], r["spread_edge"], r["spread_units"], r["spread_fair"])

    # Total
    add_play(matchup, "TOTAL",
             f"Over {r['total_line']}",
             ASSUMED_TOTAL_PRICE, r["over_prob"], r["total_edge"], r["total_units"], r["total_fair"])

    # ML (both sides)
    if pd.notna(r["home_ml_price"]):
        add_play(matchup, "ML",
                 f"{r['home_team']} ML",
                 r["home_ml_price"], r["home_win_prob"], r["home_ml_edge"], r["home_ml_units"], r["home_ml_fair"])
    if pd.notna(r["away_ml_price"]):
        add_play(matchup, "ML",
                 f"{r['away_team']} ML",
                 r["away_ml_price"], r["away_win_prob"], r["away_ml_edge"], r["away_ml_units"], r["away_ml_fair"])

plays_df = pd.DataFrame(plays).sort_values(["units","edge"], ascending=[False, False])

print("\n=== TOP PLAYS (ALL MARKETS) ===")
display(plays_df[plays_df["units"] > 0].head(25))

print("\n=== SLATE DETAIL (CORE FIELDS) ===")
core_cols = [
    "home_team","away_team",
    "home_spread","home_spread_price","total_line","home_ml_price","away_ml_price",
    "proj_margin_raw","proj_margin_srs","market_margin","proj_margin",
    "home_cover_prob","spread_market_prob","spread_edge","spread_units","spread_fair",
    "over_prob","total_edge","total_units","total_fair",
    "home_win_prob","home_ml_edge","home_ml_units","home_ml_fair",
    "away_win_prob","away_ml_edge","away_ml_units","away_ml_fair",
]
display(g[core_cols].sort_values("spread_edge", ascending=False))

# -----------------------
# 9) DISCORD OUTPUT (ONLY UNITS > 0)
# -----------------------
def fmt_pct(x): return f"{100*float(x):.1f}%"
def fmt_odds(o):
    if pd.isna(o): return ""
    o = float(o)
    return f"{int(o):+d}" if abs(o - int(o)) < 1e-6 else f"{o:+.0f}"

lines = []
lines.append("== CDR BETTING | NBA FULL CARD ==")

top_plays = plays_df[plays_df["units"] > 0].head(15)

for _, p in top_plays.iterrows():
    lines.append("")
    lines.append(p["matchup"])
    lines.append(f"{p['market']}: {p['bet']} ({fmt_odds(p['odds'])}) — {p['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(p['model_prob'])} | Edge: {fmt_pct(p['edge'])} | Fair: {fmt_odds(p['fair_odds'])}")

discord_text = "\n".join(lines)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

In [ ]:
# =========================
# FULL ML LAYER: NO-VIG + CLEAN RE-RANK
# =========================

def novig_two_way_prob(odds_a, odds_b):
    if pd.isna(odds_a) or pd.isna(odds_b):
        return (np.nan, np.nan)
    pa = american_to_prob(odds_a)
    pb = american_to_prob(odds_b)
    s = pa + pb
    if s <= 0:
        return (np.nan, np.nan)
    return (pa / s, pb / s)

# compute no-vig market probs for ML
home_nv = []
away_nv = []
for _, r in g.iterrows():
    ph, pa = novig_two_way_prob(r["home_ml_price"], r["away_ml_price"])
    home_nv.append(ph)
    away_nv.append(pa)

g["home_ml_market_prob_novig"] = home_nv
g["away_ml_market_prob_novig"] = away_nv

# recompute ML edges using no-vig
g["home_ml_edge_novig"] = g["home_win_prob"] - g["home_ml_market_prob_novig"]
g["away_ml_edge_novig"] = g["away_win_prob"] - g["away_ml_market_prob_novig"]

# recompute ML units off no-vig edge threshold
def size_units_with_edge(p_model, odds, edge):
    if pd.isna(p_model) or pd.isna(odds) or pd.isna(edge):
        return 0.0
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    return float(min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP))

g["home_ml_units_novig"] = g.apply(lambda r: size_units_with_edge(r.home_win_prob, r.home_ml_price, r.home_ml_edge_novig), axis=1)
g["away_ml_units_novig"] = g.apply(lambda r: size_units_with_edge(r.away_win_prob, r.away_ml_price, r.away_ml_edge_novig), axis=1)

# OPTIONAL guard: require bigger ML edge for longdogs
ML_EDGE_MIN = 0.05  # 5% for dogs; tune 0.04–0.07
def ml_guard(odds, edge):
    if pd.isna(odds) or pd.isna(edge):
        return False
    # apply guard only to plus-money dogs
    if odds > 0 and edge < ML_EDGE_MIN:
        return False
    return True

# rebuild plays table using improved ML edges/units
plays = []

def add_play(matchup, market, bet, odds, model_prob, edge, units, fair):
    plays.append({
        "matchup": matchup,
        "market": market,
        "bet": bet,
        "odds": odds,
        "model_prob": model_prob,
        "edge": edge,
        "units": units,
        "fair_odds": fair,
    })

for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # spread
    add_play(matchup, "SPREAD",
             f"{r['home_team']} {r['home_spread']:+g}",
             r["home_spread_price"], r["home_cover_prob"], r["spread_edge"], r["spread_units"], r["spread_fair"])

    # total
    add_play(matchup, "TOTAL",
             f"Over {r['total_line']}",
             ASSUMED_TOTAL_PRICE, r["over_prob"], r["total_edge"], r["total_units"], r["total_fair"])

    # ML (no-vig edges)
    if pd.notna(r["home_ml_price"]) and ml_guard(r["home_ml_price"], r["home_ml_edge_novig"]):
        add_play(matchup, "ML",
                 f"{r['home_team']} ML",
                 r["home_ml_price"], r["home_win_prob"], r["home_ml_edge_novig"], r["home_ml_units_novig"], r["home_ml_fair"])

    if pd.notna(r["away_ml_price"]) and ml_guard(r["away_ml_price"], r["away_ml_edge_novig"]):
        add_play(matchup, "ML",
                 f"{r['away_team']} ML",
                 r["away_ml_price"], r["away_win_prob"], r["away_ml_edge_novig"], r["away_ml_units_novig"], r["away_ml_fair"])

plays_df2 = pd.DataFrame(plays).sort_values(["units","edge"], ascending=[False, False])

print("\n=== TOP PLAYS (ALL MARKETS) — ML NO-VIG ===")
display(plays_df2[plays_df2["units"] > 0].head(25))

print("\n=== DISCORD TEXT (NO-VIG ML) ===\n")
lines = ["== CDR BETTING | NBA FULL CARD =="]
for _, p in plays_df2[plays_df2["units"] > 0].head(15).iterrows():
    lines.append("")
    lines.append(p["matchup"])
    lines.append(f"{p['market']}: {p['bet']} ({fmt_odds(p['odds'])}) — {p['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(p['model_prob'])} | Edge: {fmt_pct(p['edge'])} | Fair: {fmt_odds(p['fair_odds'])}")

print("\n".join(lines))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
import os, requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# =========================
# SETTINGS
# =========================
ODDS_API_KEY = os.environ.get("ODDS_API_KEY")
SPORT = "basketball_nba"

SEASON_YEAR = "2026"    # BRef NBA_2026 = 2025-26
SIMS = 20000            # fast + stable

BASE_SD_MARGIN = 14.5
BASE_SD_TOTAL  = 19.0

HOME_ADV = 1.8
SRS_WEIGHT = 0.35

MARKET_ANCHOR_W = 0.65
MAX_MARGIN_DEV  = 6.0

# IMPORTANT: RECENCY / MARKET COMPRESSION LAYER
# (This is the layer you were missing/looking for)
RECENCY_SHRINK = 0.75    # 0.70–0.85

EDGE_MIN = 0.02
KELLY_DIVISOR = 2.0       # half-Kelly
UNIT_MIN = 0.25
UNIT_CAP = 1.00

ASSUMED_TOTAL_PRICE = -110.0

if not ODDS_API_KEY:
    raise RuntimeError("ODDS_API_KEY env var not set. Run the key cell first.")

# =========================
# HELPERS
# =========================
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p: float) -> float:
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    return round(((1 - p) / p) * 100, 0)

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p); odds = float(odds)
    if odds < 0:
        b = 100.0 / abs(odds)
    else:
        b = odds / 100.0
    f = (p * (b + 1.0) - 1.0) / b
    return max(0.0, f)

def size_units(p_model, odds):
    if pd.isna(p_model) or pd.isna(odds):
        return 0.0
    p_mkt = american_to_prob(odds)
    edge = p_model - p_mkt
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    return float(min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP))

def novig_two_way_prob(odds_a, odds_b):
    if pd.isna(odds_a) or pd.isna(odds_b):
        return (np.nan, np.nan)
    pa = american_to_prob(odds_a)
    pb = american_to_prob(odds_b)
    s = pa + pb
    if s <= 0:
        return (np.nan, np.nan)
    return (pa / s, pb / s)

NAME_MAP = {
    # add mappings only if you ever see mismatches
}

def apply_name_map(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace("\xa0", " ", regex=False)
             .str.strip()
             .replace(NAME_MAP))

# =========================
# 1) ODDS API MARKET PULL (SPREADS, TOTALS, H2H)
# =========================
def pull_market_data_best():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "spreads,totals,h2h",  # <- DO NOT add team_totals (causes 422 on many tiers)
        "oddsFormat": "american",
    }
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API error {r.status_code}: {r.text[:400]}")
    data = r.json()

    rows = []
    for game in data:
        home = game.get("home_team")
        away = game.get("away_team")
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        spread_points, spread_home_prices = [], []
        total_points = []
        best_ml_home, best_ml_away = None, None

        for book in books:
            for m in book.get("markets", []):
                key = m.get("key")
                outs = m.get("outcomes") or []

                if key == "spreads":
                    h = next((o for o in outs if o.get("name") == home), None)
                    if h and h.get("point") is not None and h.get("price") is not None:
                        spread_points.append(float(h["point"]))
                        spread_home_prices.append(float(h["price"]))

                elif key == "totals":
                    any_o = next((o for o in outs if o.get("point") is not None), None)
                    if any_o:
                        total_points.append(float(any_o["point"]))

                elif key == "h2h":
                    h = next((o for o in outs if o.get("name") == home), None)
                    a = next((o for o in outs if o.get("name") == away), None)
                    if h and h.get("price") is not None:
                        p = float(h["price"])
                        if best_ml_home is None or p > best_ml_home:
                            best_ml_home = p
                    if a and a.get("price") is not None:
                        p = float(a["price"])
                        if best_ml_away is None or p > best_ml_away:
                            best_ml_away = p

        if not spread_points or not total_points:
            continue

        spread_cons = float(np.median(spread_points))
        total_cons  = float(np.median(total_points))

        best_spread_point, best_spread_price = None, None
        for pt, pr in zip(spread_points, spread_home_prices):
            if best_spread_point is None:
                best_spread_point, best_spread_price = pt, pr
            else:
                if abs(pt - spread_cons) < abs(best_spread_point - spread_cons):
                    best_spread_point, best_spread_price = pt, pr
                elif abs(pt - spread_cons) == abs(best_spread_point - spread_cons) and pr > best_spread_price:
                    best_spread_point, best_spread_price = pt, pr

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_spread": float(best_spread_point),
            "home_spread_price": float(best_spread_price),
            "spread_consensus": spread_cons,
            "total_line": total_cons,
            "home_ml_price": best_ml_home,
            "away_ml_price": best_ml_away,
        })

    df = pd.DataFrame(rows).drop_duplicates()
    if df.empty:
        raise RuntimeError("No games pulled from Odds API.")
    return df

games_df = pull_market_data_best()
games_df["home_team"] = apply_name_map(games_df["home_team"])
games_df["away_team"] = apply_name_map(games_df["away_team"])

print("Pulled market games:", len(games_df))
display(games_df)

# =========================
# 2) BREF TEAM STATS (PS/G, PA/G, SRS)
# =========================
BREF_URL = f"https://www.basketball-reference.com/leagues/NBA_{SEASON_YEAR}.html"

def pull_bref_team_stats(url=BREF_URL):
    tables = pd.read_html(url)
    east, west = tables[0], tables[1]

    if isinstance(east.columns, pd.MultiIndex):
        east.columns = east.columns.get_level_values(-1)
    if isinstance(west.columns, pd.MultiIndex):
        west.columns = west.columns.get_level_values(-1)

    def clean(df):
        team_col = df.columns[0]
        df = df.rename(columns={team_col: "team"})
        out = df[["team", "PS/G", "PA/G", "SRS"]].copy()
        out = out.rename(columns={"PS/G":"ppg", "PA/G":"oppg", "SRS":"srs"})
        out = out[~out["team"].isin(["Eastern Conference","Western Conference"])]
        out = out[~out["team"].str.contains("Division", na=False)]
        return out

    ts = pd.concat([clean(east), clean(west)], ignore_index=True)

    ts["team"] = (ts["team"].astype(str)
                  .str.replace(r"\s*\(\d+\)", "", regex=True)
                  .str.replace("\xa0", " ", regex=False)
                  .str.strip())
    ts["team"] = apply_name_map(ts["team"])

    for c in ["ppg","oppg","srs"]:
        ts[c] = pd.to_numeric(ts[c], errors="coerce")

    ts = ts.dropna(subset=["team","ppg","oppg","srs"]).drop_duplicates(subset=["team"])
    return ts

team_stats = pull_bref_team_stats()
print("Teams pulled from BRef:", len(team_stats))
display(team_stats.head(10))

# validate names
missing_home = set(games_df["home_team"]) - set(team_stats["team"])
missing_away = set(games_df["away_team"]) - set(team_stats["team"])
print("Unmatched Home Teams:", missing_home)
print("Unmatched Away Teams:", missing_away)
if missing_home or missing_away:
    raise RuntimeError("Team name mismatch. Add fixes to NAME_MAP.")

# =========================
# 3) MERGE
# =========================
g = games_df.merge(team_stats, left_on="home_team", right_on="team", how="left") \
            .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
            .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

if g[["home_ppg","away_ppg","home_srs","away_srs"]].isna().any().any():
    raise RuntimeError("Merge produced NaNs. Check name cleaning.")

# =========================
# 4) PROJECTION LAYERS (COMPLETE)
# =========================
# Season offense/defense base
g["home_points_proj_raw"] = (g["home_ppg"] + g["away_oppg"]) / 2.0
g["away_points_proj_raw"] = (g["away_ppg"] + g["home_oppg"]) / 2.0
g["proj_margin_raw"] = g["home_points_proj_raw"] - g["away_points_proj_raw"]

# SRS strength adjustment
g["srs_diff"] = g["home_srs"] - g["away_srs"]
g["proj_margin_srs"] = g["proj_margin_raw"] + (SRS_WEIGHT * g["srs_diff"])

# Home court
g["proj_margin_srs"] = g["proj_margin_srs"] + HOME_ADV

# Market implied margin
g["market_margin"] = -g["home_spread"]

# Market anchor + cap
g["proj_margin"] = (MARKET_ANCHOR_W * g["market_margin"]) + ((1 - MARKET_ANCHOR_W) * g["proj_margin_srs"])
diff = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# ✅ RECENCY / MARKET COMPRESSION LAYER (ADDED HERE)
g["proj_margin"] = (RECENCY_SHRINK * g["proj_margin"]) + ((1 - RECENCY_SHRINK) * g["market_margin"])
diff2 = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff2.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# Totals projection (anchored)
g["proj_total_raw"] = g["home_points_proj_raw"] + g["away_points_proj_raw"]
TOTAL_ANCHOR_W = 0.55
g["proj_total"] = (TOTAL_ANCHOR_W * g["total_line"]) + ((1 - TOTAL_ANCHOR_W) * g["proj_total_raw"])

# Variance scaling (fast pace proxy)
tot_mean = g["total_line"].mean()
g["sd_margin"] = (BASE_SD_MARGIN * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)
g["sd_total"]  = (BASE_SD_TOTAL  * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)

# =========================
# 5) MONTE CARLO (SPREAD, TOTAL, ML)
# =========================
N = len(g)

mu_margin = g["proj_margin"].values.reshape(-1, 1)
sd_margin = g["sd_margin"].values.reshape(-1, 1)
margins = np.random.normal(loc=mu_margin, scale=sd_margin, size=(N, SIMS))

# Correct cover logic
spreads_to_cover = (-g["home_spread"]).values.reshape(-1, 1)
g["home_cover_prob"] = (margins > spreads_to_cover).mean(axis=1)

mu_total = g["proj_total"].values.reshape(-1, 1)
sd_total = g["sd_total"].values.reshape(-1, 1)
totals = np.random.normal(loc=mu_total, scale=sd_total, size=(N, SIMS))
g["over_prob"] = (totals > g["total_line"].values.reshape(-1, 1)).mean(axis=1)

g["home_win_prob"] = (margins > 0.0).mean(axis=1)
g["away_win_prob"] = 1.0 - g["home_win_prob"]

# =========================
# 6) SPREAD EV / KELLY
# =========================
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]
g["spread_units"] = g.apply(lambda r: size_units(r.home_cover_prob, r.home_spread_price), axis=1)
g["spread_fair"] = g["home_cover_prob"].apply(prob_to_american)

# =========================
# 7) TOTAL EV / KELLY (ASSUMED -110)
# =========================
g["total_market_prob"] = american_to_prob(ASSUMED_TOTAL_PRICE)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]
g["total_units"] = g["over_prob"].apply(lambda p: size_units(p, ASSUMED_TOTAL_PRICE))
g["total_fair"] = g["over_prob"].apply(prob_to_american)

# =========================
# 8) ML NO-VIG + EV / KELLY
# =========================
home_nv, away_nv = [], []
for _, r in g.iterrows():
    ph, pa = novig_two_way_prob(r["home_ml_price"], r["away_ml_price"])
    home_nv.append(ph); away_nv.append(pa)

g["home_ml_market_prob_novig"] = home_nv
g["away_ml_market_prob_novig"] = away_nv

g["home_ml_edge_novig"] = g["home_win_prob"] - g["home_ml_market_prob_novig"]
g["away_ml_edge_novig"] = g["away_win_prob"] - g["away_ml_market_prob_novig"]

def size_units_with_edge(p_model, odds, edge):
    if pd.isna(p_model) or pd.isna(odds) or pd.isna(edge):
        return 0.0
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    return float(min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP))

g["home_ml_units"] = g.apply(lambda r: size_units_with_edge(r.home_win_prob, r.home_ml_price, r.home_ml_edge_novig), axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units_with_edge(r.away_win_prob, r.away_ml_price, r.away_ml_edge_novig), axis=1)

g["home_ml_fair"] = g["home_win_prob"].apply(prob_to_american)
g["away_ml_fair"] = g["away_win_prob"].apply(prob_to_american)

# =========================
# 9) PLAYS TABLE (ALL MARKETS)
# =========================
plays = []

def add_play(matchup, market, bet, odds, model_prob, edge, units, fair):
    plays.append({
        "matchup": matchup,
        "market": market,
        "bet": bet,
        "odds": odds,
        "model_prob": float(model_prob),
        "edge": float(edge),
        "units": float(units),
        "fair_odds": float(fair) if not pd.isna(fair) else np.nan,
    })

for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    add_play(matchup, "SPREAD",
             f"{r['home_team']} {r['home_spread']:+g}",
             r["home_spread_price"], r["home_cover_prob"], r["spread_edge"], r["spread_units"], r["spread_fair"])

    add_play(matchup, "TOTAL",
             f"Over {r['total_line']}",
             ASSUMED_TOTAL_PRICE, r["over_prob"], r["total_edge"], r["total_units"], r["total_fair"])

    if pd.notna(r["home_ml_price"]):
        add_play(matchup, "ML",
                 f"{r['home_team']} ML",
                 r["home_ml_price"], r["home_win_prob"], r["home_ml_edge_novig"], r["home_ml_units"], r["home_ml_fair"])

    if pd.notna(r["away_ml_price"]):
        add_play(matchup, "ML",
                 f"{r['away_team']} ML",
                 r["away_ml_price"], r["away_win_prob"], r["away_ml_edge_novig"], r["away_ml_units"], r["away_ml_fair"])

plays_df = pd.DataFrame(plays).sort_values(["units","edge"], ascending=[False, False])

print("\n=== TOP PLAYS (ALL MARKETS) — FINAL ===")
display(plays_df[plays_df["units"] > 0].head(25))

print("\n=== SLATE DETAIL (CORE) ===")
core_cols = [
    "home_team","away_team","home_spread","home_spread_price","total_line","home_ml_price","away_ml_price",
    "proj_margin_raw","proj_margin_srs","market_margin","proj_margin",
    "home_cover_prob","spread_edge","spread_units","spread_fair",
    "over_prob","total_edge","total_units","total_fair",
    "home_win_prob","home_ml_edge_novig","home_ml_units","home_ml_fair",
    "away_win_prob","away_ml_edge_novig","away_ml_units","away_ml_fair",
]
display(g[core_cols].sort_values("spread_edge", ascending=False))

# =========================
# 10) DISCORD TEXT
# =========================
def fmt_pct(x): return f"{100*float(x):.1f}%"
def fmt_odds(o):
    if pd.isna(o): return ""
    o = float(o)
    return f"{int(o):+d}" if abs(o - int(o)) < 1e-6 else f"{o:+.0f}"

lines = ["== CDR BETTING | NBA FULL CARD =="]

top_plays = plays_df[plays_df["units"] > 0].head(15)
for _, p in top_plays.iterrows():
    lines.append("")
    lines.append(p["matchup"])
    lines.append(f"{p['market']}: {p['bet']} ({fmt_odds(p['odds'])}) — {p['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(p['model_prob'])} | Edge: {fmt_pct(p['edge'])} | Fair: {fmt_odds(p['fair_odds'])}")

discord_text = "\n".join(lines)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

In [ ]:
# =========================
# OPTIONAL "PRO" LAYERS: totals pricing + ML coherence gate
# Paste below your existing run
# =========================

# 1) Try to pull totals prices (Over) from Odds API by re-requesting with totals only
def pull_totals_prices():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {"apiKey": ODDS_API_KEY, "regions": "us", "markets": "totals", "oddsFormat": "american"}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        print("Totals price pull failed; keeping assumed -110. Status:", r.status_code)
        return pd.DataFrame()

    data = r.json()
    rows = []
    for game in data:
        home = apply_name_map(pd.Series([game.get("home_team")]))[0]
        away = apply_name_map(pd.Series([game.get("away_team")]))[0]
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        # pick the over price closest to consensus total_line
        over_price_best, over_point_best = None, None
        for book in books:
            for m in book.get("markets", []):
                if m.get("key") != "totals":
                    continue
                outs = m.get("outcomes") or []
                over = next((o for o in outs if o.get("name") == "Over"), None)
                if over and over.get("point") is not None and over.get("price") is not None:
                    pt = float(over["point"])
                    pr = float(over["price"])
                    # choose the line closest to our consensus total_line
                    if over_point_best is None or abs(pt - float(games_df.loc[(games_df.home_team==home)&(games_df.away_team==away),"total_line"].iloc[0])) < abs(over_point_best - float(games_df.loc[(games_df.home_team==home)&(games_df.away_team==away),"total_line"].iloc[0])):
                        over_point_best, over_price_best = pt, pr

        if over_price_best is not None:
            rows.append({"home_team": home, "away_team": away, "over_price": over_price_best})

    return pd.DataFrame(rows)

tp = pull_totals_prices()
if not tp.empty:
    g = g.merge(tp, on=["home_team","away_team"], how="left")
    g["total_price_used"] = g["over_price"].fillna(ASSUMED_TOTAL_PRICE)
else:
    g["total_price_used"] = ASSUMED_TOTAL_PRICE

# recompute totals edges using actual prices when present
g["total_market_prob"] = g["total_price_used"].apply(american_to_prob)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]
g["total_units"] = g.apply(lambda r: size_units(r.over_prob, r.total_price_used), axis=1)
g["total_fair"] = g["over_prob"].apply(prob_to_american)

# 2) ML coherence gate (reduces dog spam)
def allow_ml(spread_edge, ml_edge):
    if pd.isna(ml_edge):
        return False
    # if spread isn't an edge, require a bigger ML edge
    if spread_edge <= 0:
        return ml_edge >= 0.06
    # if spread is also an edge, allow smaller ML edge
    return ml_edge >= 0.03

# rebuild plays with gate
plays = []
def add_play(matchup, market, bet, odds, model_prob, edge, units, fair):
    plays.append({"matchup": matchup,"market": market,"bet": bet,"odds": odds,
                  "model_prob": float(model_prob),"edge": float(edge),
                  "units": float(units),"fair_odds": float(fair) if not pd.isna(fair) else np.nan})

for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # spread
    add_play(matchup, "SPREAD",
             f"{r['home_team']} {r['home_spread']:+g}",
             r["home_spread_price"], r["home_cover_prob"], r["spread_edge"], r["spread_units"], r["spread_fair"])

    # total (now priced)
    add_play(matchup, "TOTAL",
             f"Over {r['total_line']}",
             r["total_price_used"], r["over_prob"], r["total_edge"], r["total_units"], r["total_fair"])

    # ML (no-vig edges + gate)
    if pd.notna(r["home_ml_price"]) and allow_ml(r["spread_edge"], r["home_ml_edge_novig"]):
        add_play(matchup, "ML",
                 f"{r['home_team']} ML",
                 r["home_ml_price"], r["home_win_prob"], r["home_ml_edge_novig"], r["home_ml_units"], r["home_ml_fair"])

    if pd.notna(r["away_ml_price"]) and allow_ml(r["spread_edge"], r["away_ml_edge_novig"]):
        add_play(matchup, "ML",
                 f"{r['away_team']} ML",
                 r["away_ml_price"], r["away_win_prob"], r["away_ml_edge_novig"], r["away_ml_units"], r["away_ml_fair"])

plays_df_pro = pd.DataFrame(plays).sort_values(["units","edge"], ascending=[False, False])

print("\n=== TOP PLAYS (ALL MARKETS) — PRO LAYERS ===")
display(plays_df_pro[plays_df_pro["units"] > 0].head(25))

print("\n=== DISCORD TEXT (PRO LAYERS) ===\n")
lines = ["== CDR BETTING | NBA FULL CARD =="]
for _, p in plays_df_pro[plays_df_pro["units"] > 0].head(15).iterrows():
    lines.append("")
    lines.append(p["matchup"])
    lines.append(f"{p['market']}: {p['bet']} ({fmt_odds(p['odds'])}) — {p['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(p['model_prob'])} | Edge: {fmt_pct(p['edge'])} | Fair: {fmt_odds(p['fair_odds'])}")
print("\n".join(lines))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
import os, requests
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# ==============
# SETTINGS
# ==============
ODDS_API_KEY = os.environ.get("ODDS_API_KEY")
SPORT = "basketball_nba"

SEASON_YEAR = "2026"   # BRef NBA_2026 = 2025-26
SIMS = 15000           # fast & stable

BASE_SD_MARGIN = 14.5
BASE_SD_TOTAL  = 19.0

HOME_ADV = 1.8
SRS_WEIGHT = 0.35

MARKET_ANCHOR_W = 0.65
MAX_MARGIN_DEV  = 6.0
RECENCY_SHRINK  = 0.75  # market-compression layer

EDGE_MIN = 0.02
KELLY_DIVISOR = 2.0     # half-Kelly
UNIT_MIN = 0.25
UNIT_CAP = 1.00

ASSUMED_TOTAL_PRICE = -110.0

if not ODDS_API_KEY:
    raise RuntimeError("ODDS_API_KEY env var not set. Run the key cell first.")

# ==============
# HELPERS
# ==============
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p: float) -> float:
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    return round(((1 - p) / p) * 100, 0)

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p); odds = float(odds)
    if odds < 0:
        b = 100.0 / abs(odds)
    else:
        b = odds / 100.0
    f = (p * (b + 1.0) - 1.0) / b
    return max(0.0, f)

def size_units(p_model, odds):
    if pd.isna(p_model) or pd.isna(odds):
        return 0.0
    p_mkt = american_to_prob(odds)
    edge = p_model - p_mkt
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    return float(min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP))

def novig_two_way_prob(odds_a, odds_b):
    if pd.isna(odds_a) or pd.isna(odds_b):
        return (np.nan, np.nan)
    pa = american_to_prob(odds_a)
    pb = american_to_prob(odds_b)
    s = pa + pb
    if s <= 0:
        return (np.nan, np.nan)
    return (pa / s, pb / s)

NAME_MAP = {}  # add only if a team name mismatch ever appears

def apply_name_map(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.replace("\xa0", " ", regex=False)
             .str.strip()
             .replace(NAME_MAP))

def fmt_pct(x): return f"{100*float(x):.1f}%"
def fmt_odds(o):
    if pd.isna(o): return ""
    o = float(o)
    return f"{int(o):+d}" if abs(o - int(o)) < 1e-6 else f"{o:+.0f}"

# ==============
# 1) ODDS API: spreads/totals/h2h (no team_totals to avoid 422)
# ==============
def pull_market_data_best():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {"apiKey": ODDS_API_KEY, "regions": "us", "markets": "spreads,totals,h2h", "oddsFormat": "american"}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API error {r.status_code}: {r.text[:400]}")
    data = r.json()

    rows = []
    for game in data:
        home = game.get("home_team")
        away = game.get("away_team")
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        spread_points, spread_home_prices = [], []
        total_points = []
        best_ml_home, best_ml_away = None, None

        for book in books:
            for m in book.get("markets", []):
                key = m.get("key")
                outs = m.get("outcomes") or []

                if key == "spreads":
                    h = next((o for o in outs if o.get("name") == home), None)
                    if h and h.get("point") is not None and h.get("price") is not None:
                        spread_points.append(float(h["point"]))
                        spread_home_prices.append(float(h["price"]))

                elif key == "totals":
                    any_o = next((o for o in outs if o.get("point") is not None), None)
                    if any_o:
                        total_points.append(float(any_o["point"]))

                elif key == "h2h":
                    h = next((o for o in outs if o.get("name") == home), None)
                    a = next((o for o in outs if o.get("name") == away), None)
                    if h and h.get("price") is not None:
                        p = float(h["price"])
                        if best_ml_home is None or p > best_ml_home:
                            best_ml_home = p
                    if a and a.get("price") is not None:
                        p = float(a["price"])
                        if best_ml_away is None or p > best_ml_away:
                            best_ml_away = p

        if not spread_points or not total_points:
            continue

        spread_cons = float(np.median(spread_points))
        total_cons  = float(np.median(total_points))

        best_spread_point, best_spread_price = None, None
        for pt, pr in zip(spread_points, spread_home_prices):
            if best_spread_point is None:
                best_spread_point, best_spread_price = pt, pr
            else:
                if abs(pt - spread_cons) < abs(best_spread_point - spread_cons):
                    best_spread_point, best_spread_price = pt, pr
                elif abs(pt - spread_cons) == abs(best_spread_point - spread_cons) and pr > best_spread_price:
                    best_spread_point, best_spread_price = pt, pr

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_spread": float(best_spread_point),
            "home_spread_price": float(best_spread_price),
            "spread_consensus": spread_cons,
            "total_line": total_cons,
            "home_ml_price": best_ml_home,
            "away_ml_price": best_ml_away,
        })

    df = pd.DataFrame(rows).drop_duplicates()
    if df.empty:
        raise RuntimeError("No games pulled from Odds API.")
    return df

games_df = pull_market_data_best()
games_df["home_team"] = apply_name_map(games_df["home_team"])
games_df["away_team"] = apply_name_map(games_df["away_team"])

print("Pulled market games:", len(games_df))
display(games_df)

# ==============
# 2) BRef team stats (PS/G, PA/G, SRS)
# ==============
BREF_URL = f"https://www.basketball-reference.com/leagues/NBA_{SEASON_YEAR}.html"

def pull_bref_team_stats(url=BREF_URL):
    tables = pd.read_html(url)
    east, west = tables[0], tables[1]

    if isinstance(east.columns, pd.MultiIndex):
        east.columns = east.columns.get_level_values(-1)
    if isinstance(west.columns, pd.MultiIndex):
        west.columns = west.columns.get_level_values(-1)

    def clean(df):
        team_col = df.columns[0]
        df = df.rename(columns={team_col: "team"})
        out = df[["team", "PS/G", "PA/G", "SRS"]].copy()
        out = out.rename(columns={"PS/G":"ppg", "PA/G":"oppg", "SRS":"srs"})
        out = out[~out["team"].isin(["Eastern Conference","Western Conference"])]
        out = out[~out["team"].str.contains("Division", na=False)]
        return out

    ts = pd.concat([clean(east), clean(west)], ignore_index=True)
    ts["team"] = (ts["team"].astype(str)
                  .str.replace(r"\s*\(\d+\)", "", regex=True)
                  .str.replace("\xa0", " ", regex=False)
                  .str.strip())
    ts["team"] = apply_name_map(ts["team"])

    for c in ["ppg","oppg","srs"]:
        ts[c] = pd.to_numeric(ts[c], errors="coerce")

    ts = ts.dropna(subset=["team","ppg","oppg","srs"]).drop_duplicates(subset=["team"])
    return ts

team_stats = pull_bref_team_stats()
print("Teams pulled from BRef:", len(team_stats))
display(team_stats.head(10))

# validate names
missing_home = set(games_df["home_team"]) - set(team_stats["team"])
missing_away = set(games_df["away_team"]) - set(team_stats["team"])
print("Unmatched Home Teams:", missing_home)
print("Unmatched Away Teams:", missing_away)
if missing_home or missing_away:
    raise RuntimeError("Team name mismatch. Add fixes to NAME_MAP.")

# ==============
# 3) MERGE slate + team stats
# ==============
g = games_df.merge(team_stats, left_on="home_team", right_on="team", how="left") \
            .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
            .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

if g[["home_ppg","away_ppg","home_srs","away_srs"]].isna().any().any():
    raise RuntimeError("Merge produced NaNs. Check name cleaning.")


            # ==============
# 4) INJURY LAYER (ESPN) v2 — rotation-weighted + capped
# ==============
from bs4 import BeautifulSoup
from io import StringIO
import re

def pull_espn_injuries_raw():
    url = "https://www.espn.com/nba/injuries"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.text

def parse_team_tables_from_espn(html, team_universe):
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")
    team_universe = set(team_universe)

    def find_team_for_table(tbl):
        node = tbl
        for _ in range(30):
            node = node.find_previous()
            if node is None:
                break
            txt = " ".join(node.get_text(" ", strip=True).split())
            if not txt:
                continue
            if txt in team_universe:
                return txt
            for t in team_universe:
                if t in txt and len(t) >= 6:
                    return t
        return None

    out = []
    for tbl in tables:
        team = find_team_for_table(tbl)
        if not team:
            continue
        try:
            df = pd.read_html(StringIO(str(tbl)))[0]
        except Exception:
            continue
        if "STATUS" not in df.columns:
            continue
        df["team"] = team
        out.append(df)

    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)

def rotation_weight_penalty(df_team):
    """
    Conservative, rotation-weighted penalty:
    - OUT: starter/rotation-ish names (heuristic) get more weight
    - DTD: small weight
    - cap total penalty
    """
    df = df_team.copy()

    # normalize columns
    # ESPN table typically has "PLAYER", "POS", "DATE", "STATUS", "COMMENT"
    for col in df.columns:
        if isinstance(col, tuple):
            df.columns = [c[-1] for c in df.columns]
            break

    if "PLAYER" not in df.columns:
        # attempt best guess
        player_col = df.columns[0]
        df = df.rename(columns={player_col: "PLAYER"})

    df["STATUS"] = df["STATUS"].astype(str).str.strip().str.lower()
    df["PLAYER"] = df["PLAYER"].astype(str).str.strip()

    # Impact heuristic (no external roster needed):
    # If name has no commas/extra tags and looks like a normal NBA player, treat as rotation.
    # If it includes "G League", "Two-Way", or has weird parentheticals, treat as low-impact.
    low_impact = df["PLAYER"].str.contains(r"two-way|g league|\(|\)|/|assignment", flags=re.I, regex=True)

    # weights (tuned conservative)
    W_OUT_ROT = -1.05
    W_OUT_LOW = -0.30
    W_DTD_ROT = -0.25
    W_DTD_LOW = -0.08

    out_mask = (df["STATUS"] == "out")
    dtd_mask = (df["STATUS"] == "day-to-day")

    rot_mask = ~low_impact

    out_rot = int((out_mask & rot_mask).sum())
    out_low = int((out_mask & ~rot_mask).sum())
    dtd_rot = int((dtd_mask & rot_mask).sum())
    dtd_low = int((dtd_mask & ~rot_mask).sum())

    penalty = out_rot * W_OUT_ROT + out_low * W_OUT_LOW + dtd_rot * W_DTD_ROT + dtd_low * W_DTD_LOW

    # cap so ESPN depth can’t blow up your slate
    penalty = float(np.clip(penalty, -3.0, 0.0))

    return penalty, out_rot, out_low, dtd_rot, dtd_low

def build_team_injury_penalties(team_universe):
    html = pull_espn_injuries_raw()
    raw = parse_team_tables_from_espn(html, team_universe)
    if raw.empty:
        return pd.DataFrame({"team": team_universe, "inj_points": 0.0,
                             "out_rot": 0, "out_low": 0, "dtd_rot": 0, "dtd_low": 0})

    rows = []
    for team in sorted(set(raw["team"])):
        df_team = raw[raw["team"] == team]
        inj_points, out_rot, out_low, dtd_rot, dtd_low = rotation_weight_penalty(df_team)
        rows.append({
            "team": team,
            "inj_points": inj_points,
            "out_rot": out_rot,
            "out_low": out_low,
            "dtd_rot": dtd_rot,
            "dtd_low": dtd_low
        })
    return pd.DataFrame(rows)

inj = build_team_injury_penalties(team_stats["team"].tolist())

g = g.merge(inj, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"inj_points":"home_inj_pts"}) \
     .drop(columns=["team"])

g = g.merge(inj, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"inj_points":"away_inj_pts"}) \
     .drop(columns=["team"])

for c in ["home_inj_pts","away_inj_pts"]:
    g[c] = pd.to_numeric(g[c], errors="coerce").fillna(0.0)

print("Injury layer v2 attached (rotation-weighted, capped):")
display(g[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(10))

# ==============
# 5) PROJECTION LAYERS (COMPLETE)
# (NOTE: raw points include injury penalties)
# ==============
g["home_points_proj_raw"] = (g["home_ppg"] + g["away_oppg"]) / 2.0 + g["home_inj_pts"]
g["away_points_proj_raw"] = (g["away_ppg"] + g["home_oppg"]) / 2.0 + g["away_inj_pts"]

g["proj_margin_raw"] = g["home_points_proj_raw"] - g["away_points_proj_raw"]

g["srs_diff"] = g["home_srs"] - g["away_srs"]
g["proj_margin_srs"] = g["proj_margin_raw"] + (SRS_WEIGHT * g["srs_diff"]) + HOME_ADV

g["market_margin"] = -g["home_spread"]

# anchor + cap
g["proj_margin"] = (MARKET_ANCHOR_W * g["market_margin"]) + ((1 - MARKET_ANCHOR_W) * g["proj_margin_srs"])
diff = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# compression layer
g["proj_margin"] = (RECENCY_SHRINK * g["proj_margin"]) + ((1 - RECENCY_SHRINK) * g["market_margin"])
diff2 = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + diff2.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# totals
g["proj_total_raw"] = g["home_points_proj_raw"] + g["away_points_proj_raw"]
TOTAL_ANCHOR_W = 0.55
g["proj_total"] = (TOTAL_ANCHOR_W * g["total_line"]) + ((1 - TOTAL_ANCHOR_W) * g["proj_total_raw"])

# variance scaling
tot_mean = g["total_line"].mean()
g["sd_margin"] = (BASE_SD_MARGIN * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)
g["sd_total"]  = (BASE_SD_TOTAL  * (g["total_line"] / tot_mean).clip(0.9, 1.15)).astype(float)

# ==============
# 6) MONTE CARLO (fast)
# ==============
rng = np.random.default_rng(42)
N = len(g)

mu_margin = g["proj_margin"].to_numpy(np.float32)[:, None]
sd_margin = g["sd_margin"].to_numpy(np.float32)[:, None]
margins = rng.normal(mu_margin, sd_margin, size=(N, SIMS)).astype(np.float32)

spreads_to_cover = (-g["home_spread"]).to_numpy(np.float32)[:, None]
g["home_cover_prob"] = (margins > spreads_to_cover).mean(axis=1)

mu_total = g["proj_total"].to_numpy(np.float32)[:, None]
sd_total = g["sd_total"].to_numpy(np.float32)[:, None]
totals = rng.normal(mu_total, sd_total, size=(N, SIMS)).astype(np.float32)
g["over_prob"] = (totals > g["total_line"].to_numpy(np.float32)[:, None]).mean(axis=1)

g["home_win_prob"] = (margins > 0.0).mean(axis=1)
g["away_win_prob"] = 1.0 - g["home_win_prob"]

# ==============
# 7) SPREAD EV/KELLY
# ==============
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]
g["spread_units"] = g.apply(lambda r: size_units(r.home_cover_prob, r.home_spread_price), axis=1)
g["spread_fair"] = g["home_cover_prob"].apply(prob_to_american)

# ==============
# 8) TOTAL PRICES (Over) from Odds API totals-only pull (fallback -110)
# ==============
def pull_totals_prices():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {"apiKey": ODDS_API_KEY, "regions": "us", "markets": "totals", "oddsFormat": "american"}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        return pd.DataFrame()
    data = r.json()

    rows = []
    for game in data:
        home = apply_name_map(pd.Series([game.get("home_team")]))[0]
        away = apply_name_map(pd.Series([game.get("away_team")]))[0]
        books = game.get("bookmakers") or []
        if not home or not away or not books:
            continue

        best_over_price = None
        for book in books:
            for m in book.get("markets", []):
                if m.get("key") != "totals":
                    continue
                outs = m.get("outcomes") or []
                over = next((o for o in outs if o.get("name") == "Over"), None)
                if over and over.get("price") is not None:
                    pr = float(over["price"])
                    if best_over_price is None or pr > best_over_price:
                        best_over_price = pr

        if best_over_price is not None:
            rows.append({"home_team": home, "away_team": away, "over_price": best_over_price})

    return pd.DataFrame(rows)

tp = pull_totals_prices()
if not tp.empty:
    g = g.merge(tp, on=["home_team","away_team"], how="left")
    g["total_price_used"] = g["over_price"].fillna(ASSUMED_TOTAL_PRICE)
else:
    g["total_price_used"] = ASSUMED_TOTAL_PRICE

g["total_market_prob"] = g["total_price_used"].apply(american_to_prob)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]
g["total_units"] = g.apply(lambda r: size_units(r.over_prob, r.total_price_used), axis=1)
g["total_fair"] = g["over_prob"].apply(prob_to_american)

# ==============
# 9) ML NO-VIG + EV/KELLY
# ==============
home_nv, away_nv = [], []
for _, r in g.iterrows():
    ph, pa = novig_two_way_prob(r["home_ml_price"], r["away_ml_price"])
    home_nv.append(ph); away_nv.append(pa)

g["home_ml_market_prob_novig"] = home_nv
g["away_ml_market_prob_novig"] = away_nv

g["home_ml_edge_novig"] = g["home_win_prob"] - g["home_ml_market_prob_novig"]
g["away_ml_edge_novig"] = g["away_win_prob"] - g["away_ml_market_prob_novig"]

def size_units_with_edge(p_model, odds, edge):
    if pd.isna(p_model) or pd.isna(odds) or pd.isna(edge):
        return 0.0
    if edge < EDGE_MIN:
        return 0.0
    k = kelly_fraction(p_model, odds)
    return float(min(max(k / KELLY_DIVISOR, UNIT_MIN), UNIT_CAP))

g["home_ml_units"] = g.apply(lambda r: size_units_with_edge(r.home_win_prob, r.home_ml_price, r.home_ml_edge_novig), axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units_with_edge(r.away_win_prob, r.away_ml_price, r.away_ml_edge_novig), axis=1)

g["home_ml_fair"] = g["home_win_prob"].apply(prob_to_american)
g["away_ml_fair"] = g["away_win_prob"].apply(prob_to_american)

# ==============
# 10) COHERENCE GATE (keeps only strong ML)
# ==============
def allow_ml(spread_edge, ml_edge):
    if pd.isna(ml_edge):
        return False
    if spread_edge <= 0:
        return ml_edge >= 0.06
    return ml_edge >= 0.03

# ==============
# 11) PLAYS TABLE
# ==============
plays = []
def add_play(matchup, market, bet, odds, model_prob, edge, units, fair):
    plays.append({
        "matchup": matchup, "market": market, "bet": bet, "odds": odds,
        "model_prob": float(model_prob), "edge": float(edge),
        "units": float(units), "fair_odds": float(fair) if not pd.isna(fair) else np.nan
    })

for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    add_play(matchup, "SPREAD",
             f"{r['home_team']} {r['home_spread']:+g}",
             r["home_spread_price"], r["home_cover_prob"], r["spread_edge"], r["spread_units"], r["spread_fair"])

    add_play(matchup, "TOTAL",
             f"Over {r['total_line']}",
             r["total_price_used"], r["over_prob"], r["total_edge"], r["total_units"], r["total_fair"])

    if pd.notna(r["home_ml_price"]) and allow_ml(r["spread_edge"], r["home_ml_edge_novig"]):
        add_play(matchup, "ML",
                 f"{r['home_team']} ML",
                 r["home_ml_price"], r["home_win_prob"], r["home_ml_edge_novig"], r["home_ml_units"], r["home_ml_fair"])

    if pd.notna(r["away_ml_price"]) and allow_ml(r["spread_edge"], r["away_ml_edge_novig"]):
        add_play(matchup, "ML",
                 f"{r['away_team']} ML",
                 r["away_ml_price"], r["away_win_prob"], r["away_ml_edge_novig"], r["away_ml_units"], r["away_ml_fair"])

plays_df = pd.DataFrame(plays).sort_values(["units","edge"], ascending=[False, False])

print("\n=== TOP PLAYS (ALL MARKETS) — PRO LAYERS (INJURY ON) ===")
display(plays_df[plays_df["units"] > 0].head(25))

print("\n=== SLATE DETAIL (CORE) ===")
core_cols = [
    "home_team","away_team","home_spread","home_spread_price","total_line","total_price_used","home_ml_price","away_ml_price",
    "home_inj_pts","away_inj_pts",
    "proj_margin_raw","proj_margin_srs","market_margin","proj_margin",
    "home_cover_prob","spread_edge","spread_units",
    "over_prob","total_edge","total_units",
    "home_win_prob","home_ml_edge_novig","home_ml_units",
    "away_win_prob","away_ml_edge_novig","away_ml_units",
]
display(g[core_cols])

# ==============
# 12) DISCORD TEXT
# ==============
lines = ["== CDR BETTING | NBA FULL CARD =="]
top_plays = plays_df[plays_df["units"] > 0].head(15)

for _, p in top_plays.iterrows():
    lines.append("")
    lines.append(p["matchup"])
    lines.append(f"{p['market']}: {p['bet']} ({fmt_odds(p['odds'])}) — {p['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(p['model_prob'])} | Edge: {fmt_pct(p['edge'])} | Fair: {fmt_odds(p['fair_odds'])}")

discord_text = "\n".join(lines)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

# save artifacts
plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)
print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

In [5]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [6]:
# ==========================================
# NBA STABLE — RUN TODAY (ALL LAYERS, FAST, UPDATED)
# Paste in ONE cell at bottom. Run.
# ==========================================

import os, math, time, re
from datetime import datetime
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# -----------------------
# SETTINGS (EDIT ONLY THESE)
# -----------------------
ODDS_API_KEY = os.environ.get("ODDS_API_KEY")  # recommended: set env var in a small cell above
if not ODDS_API_KEY:
    raise RuntimeError("ODDS_API_KEY env var not set. Set it first: os.environ['ODDS_API_KEY']='YOUR_KEY'")

SPORT = "basketball_nba"
SEASON_YEAR = "2026"       # BRef NBA_2026 = 2025-26
SIMS = 15000               # not used if USE_MC=False (analytic path). keep for optional MC.
USE_MC = False             # keep False for max efficiency (analytic Normal model)

# Dispersion
BASE_SD_MARGIN = 14.5
BASE_SD_TOTAL  = 19.0

# Margin model layers
HOME_ADV = 1.8
SRS_WEIGHT = 0.35
MARKET_ANCHOR_W = 0.65
MAX_MARGIN_DEV  = 6.0
RECENCY_SHRINK  = 0.75

# Injury layer (v2)
INJ_CAP = -2.0             # recommended (less slate-killing than -3.0)
W_OUT_ROT = -0.75
W_OUT_LOW = -0.20
W_DTD_ROT = -0.18
W_DTD_LOW = -0.06

# Sizing
EDGE_MIN = 0.02            # set 0.015 if you want more volume like 3/2
KELLY_DIVISOR = 2.0
UNIT_MIN = 0.25
UNIT_CAP = 1.00

ASSUMED_TOTAL_PRICE = -110.0  # if totals prices missing, use this

# -----------------------
# HELPERS
# -----------------------
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p: float) -> float:
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    return round(((1 - p) / p) * 100, 0)

def no_vig_probs_from_american(home_odds, away_odds):
    ph = american_to_prob(home_odds)
    pa = american_to_prob(away_odds)
    s = ph + pa
    if s <= 0:
        return np.nan, np.nan
    return ph / s, pa / s

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p)
    odds = float(odds)
    b = (odds / 100.0) if odds > 0 else (100.0 / abs(odds))
    q = 1.0 - p
    f = (b * p - q) / b
    return max(0.0, f)

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def clean_team_name(x: str) -> str:
    if x is None:
        return x
    x = str(x).replace("\xa0", " ").strip()
    # remove BRef seeds like "Detroit Pistons (1)"
    x = re.sub(r"\s*\(\d+\)\s*$", "", x).strip()
    # normalize common variants
    mapping = {
        "LA Clippers": "Los Angeles Clippers",
        "Los Angeles Clippers": "Los Angeles Clippers",
        "LA Lakers": "Los Angeles Lakers",
        "Los Angeles Lakers": "Los Angeles Lakers",
        "NY Knicks": "New York Knicks",
        "New York Knicks": "New York Knicks",
        "Brooklyn Nets": "Brooklyn Nets",
        "New Orleans Pelicans": "New Orleans Pelicans",
        "Phoenix Suns": "Phoenix Suns",
        "San Antonio Spurs": "San Antonio Spurs",
        "Philadelphia 76ers": "Philadelphia 76ers",
        "Oklahoma City Thunder": "Oklahoma City Thunder",
        "Minnesota Timberwolves": "Minnesota Timberwolves",
        "Washington Wizards": "Washington Wizards",
        "Charlotte Hornets": "Charlotte Hornets",
        "Cleveland Cavaliers": "Cleveland Cavaliers",
        "Orlando Magic": "Orlando Magic",
        "Miami Heat": "Miami Heat",
        "Toronto Raptors": "Toronto Raptors",
        "Sacramento Kings": "Sacramento Kings",
        "Dallas Mavericks": "Dallas Mavericks",
        "Memphis Grizzlies": "Memphis Grizzlies",
        "Detroit Pistons": "Detroit Pistons",
    }
    return mapping.get(x, x)

def apply_name_map(series: pd.Series) -> pd.Series:
    return series.astype(str).apply(clean_team_name)

# Normal CDF (no scipy dependency)
def norm_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def prob_cover_spread(mean_margin, spread, sd):
    # home covers if (home_margin > spread)
    z = (mean_margin - spread) / sd
    return 1.0 - norm_cdf((spread - mean_margin) / sd)

def prob_over_total(mean_total, total_line, sd):
    z = (mean_total - total_line) / sd
    return 1.0 - norm_cdf((total_line - mean_total) / sd)

def prob_home_win(mean_margin, sd):
    # home wins if margin > 0
    return 1.0 - norm_cdf((0.0 - mean_margin) / sd)

# -----------------------
# 1) Pull Market (Odds API): spreads/totals/h2h
# -----------------------
def pull_market_data():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "spreads,totals,h2h",
        "oddsFormat": "american",
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    rows = []
    for game in data:
        home_raw = game.get("home_team")
        away_raw = game.get("away_team")
        home = clean_team_name(home_raw)
        away = clean_team_name(away_raw)

        # consensus-ish: take best/first available across books
        spread_point, spread_price = None, None
        total_point, total_over_price = None, None
        home_ml, away_ml = None, None

        books = game.get("bookmakers") or []
        for b in books:
            for m in (b.get("markets") or []):
                key = m.get("key")
                outs = m.get("outcomes") or []

                if key == "spreads" and spread_point is None:
                    # find home spread
                    for o in outs:
                        if clean_team_name(o.get("name")) == home:
                            spread_point = float(o.get("point"))
                            spread_price = float(o.get("price"))
                            break

                if key == "totals" and total_point is None:
                    # use Over line as total point
                    for o in outs:
                        if str(o.get("name")).lower() == "over":
                            total_point = float(o.get("point"))
                            total_over_price = float(o.get("price"))
                            break

                if key == "h2h" and home_ml is None:
                    for o in outs:
                        nm = clean_team_name(o.get("name"))
                        if nm == home:
                            home_ml = float(o.get("price"))
                        elif nm == away:
                            away_ml = float(o.get("price"))

            if spread_point is not None and total_point is not None and home_ml is not None and away_ml is not None:
                break

        if spread_point is None or spread_price is None or total_point is None or home_ml is None or away_ml is None:
            continue

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_spread": float(spread_point),
            "home_spread_price": float(spread_price),
            "spread_consensus": float(spread_point),
            "total_line": float(total_point),
            "total_price_used": float(total_over_price) if total_over_price is not None else float(ASSUMED_TOTAL_PRICE),
            "home_ml_price": float(home_ml),
            "away_ml_price": float(away_ml),
        })

    df = pd.DataFrame(rows)
    print("Pulled market games:", len(df))
    display(df.head(10))
    return df

games_df = pull_market_data()

# -----------------------
# 2) Snapshot line movement (auto-compare vs last run)
# -----------------------
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
snap_name = f"market_snapshot_{ts}.csv"
games_df.to_csv(snap_name, index=False)

# compare to previous snapshot if present
prev_path = "market_snapshot_latest.csv"
if os.path.exists(prev_path):
    prev = pd.read_csv(prev_path)
    cur = games_df.copy()

    key = ["home_team","away_team"]
    m = cur.merge(prev, on=key, how="left", suffixes=("_cur","_prev"))

    def delta(col):
        return pd.to_numeric(m[f"{col}_cur"], errors="coerce") - pd.to_numeric(m[f"{col}_prev"], errors="coerce")

    m["d_spread"] = delta("home_spread")
    m["d_total"]  = delta("total_line")
    m["d_homeML"] = delta("home_ml_price")
    m["d_awayML"] = delta("away_ml_price")

    move_cols = key + ["home_spread_cur","home_spread_prev","d_spread",
                       "total_line_cur","total_line_prev","d_total",
                       "home_ml_price_cur","home_ml_price_prev","d_homeML"]
    print("=== LINE MOVEMENT vs last run (if any) ===")
    display(m[move_cols].sort_values(by=["d_spread","d_total"], ascending=False).head(15))

games_df.to_csv("market_snapshot_latest.csv", index=False)
print("Saved:", snap_name, "and market_snapshot_latest.csv")

# -----------------------
# 3) Pull Team Stats (Basketball-Reference): PPG / OPPG / SRS
# -----------------------
def pull_bref_team_stats():
    url = f"https://www.basketball-reference.com/leagues/NBA_{SEASON_YEAR}.html"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    tables = pd.read_html(r.text)
    # find East/West standings tables
    # These usually contain columns like 'PS/G', 'PA/G', 'SRS'
    keep = []
    for t in tables:
        cols = [str(c) for c in t.columns]
        if ("PS/G" in cols) and ("PA/G" in cols) and ("SRS" in cols):
            keep.append(t)

    if not keep:
        raise RuntimeError("Could not locate BRef standings tables with PS/G, PA/G, SRS")

    df = pd.concat(keep, ignore_index=True)

    # team column is first col name (e.g. 'Eastern Conference')
    team_col = df.columns[0]
    df = df.rename(columns={team_col: "team", "PS/G": "ppg", "PA/G": "oppg", "SRS": "srs"})

    df["team"] = df["team"].astype(str).apply(clean_team_name)
    df["ppg"]  = pd.to_numeric(df["ppg"], errors="coerce")
    df["oppg"] = pd.to_numeric(df["oppg"], errors="coerce")
    df["srs"]  = pd.to_numeric(df["srs"], errors="coerce")

    df = df.dropna(subset=["ppg","oppg","srs"]).copy()
    df = df[["team","ppg","oppg","srs"]].drop_duplicates(subset=["team"])
    return df

team_stats = pull_bref_team_stats()
print("Teams pulled from BRef:", len(team_stats))
display(team_stats.head(10))
team_stats.to_csv("season_team_stats_bref.csv", index=False)

# verify name matching
un_home = set(games_df["home_team"]) - set(team_stats["team"])
un_away = set(games_df["away_team"]) - set(team_stats["team"])
print("Unmatched Home Teams:", un_home)
print("Unmatched Away Teams:", un_away)

# -----------------------
# 4) Injury Layer v2 (rotation-weighted + capped)
# -----------------------
def pull_espn_injuries_html():
    url = "https://www.espn.com/nba/injuries"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.text

def parse_team_tables_from_espn(html, team_universe):
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")
    team_universe = set(team_universe)

    def find_team_for_table(tbl):
        node = tbl
        for _ in range(30):
            node = node.find_previous()
            if node is None:
                break
            txt = " ".join(node.get_text(" ", strip=True).split())
            if not txt:
                continue
            if txt in team_universe:
                return txt
            for t in team_universe:
                if t in txt and len(t) >= 6:
                    return t
        return None

    out = []
    for tbl in tables:
        team = find_team_for_table(tbl)
        if not team:
            continue
        try:
            df = pd.read_html(StringIO(str(tbl)))[0]
        except Exception:
            continue
        if "STATUS" not in df.columns:
            continue
        df["team"] = team
        out.append(df)

    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)

def rotation_weight_penalty(df_team):
    df = df_team.copy()

    # normalize possible multiindex columns
    if any(isinstance(c, tuple) for c in df.columns):
        df.columns = [c[-1] for c in df.columns]

    if "PLAYER" not in df.columns:
        df = df.rename(columns={df.columns[0]: "PLAYER"})

    df["STATUS"] = df["STATUS"].astype(str).str.strip().str.lower()
    df["PLAYER"] = df["PLAYER"].astype(str).str.strip()

    low_impact = df["PLAYER"].str.contains(r"two-way|g league|\(|\)|/|assignment", flags=re.I, regex=True)

    out_mask = (df["STATUS"] == "out")
    dtd_mask = (df["STATUS"] == "day-to-day")

    rot_mask = ~low_impact

    out_rot = int((out_mask & rot_mask).sum())
    out_low = int((out_mask & ~rot_mask).sum())
    dtd_rot = int((dtd_mask & rot_mask).sum())
    dtd_low = int((dtd_mask & ~rot_mask).sum())

    penalty = out_rot * W_OUT_ROT + out_low * W_OUT_LOW + dtd_rot * W_DTD_ROT + dtd_low * W_DTD_LOW
    penalty = float(np.clip(penalty, INJ_CAP, 0.0))

    return penalty, out_rot, out_low, dtd_rot, dtd_low

def build_team_injury_penalties(team_universe):
    html = pull_espn_injuries_html()
    raw = parse_team_tables_from_espn(html, team_universe)
    if raw.empty:
        return pd.DataFrame({"team": team_universe, "inj_points": 0.0})

    rows = []
    for team in sorted(set(raw["team"])):
        df_team = raw[raw["team"] == team]
        inj_points, out_rot, out_low, dtd_rot, dtd_low = rotation_weight_penalty(df_team)
        rows.append({
            "team": team,
            "inj_points": inj_points,
            "out_rot": out_rot,
            "out_low": out_low,
            "dtd_rot": dtd_rot,
            "dtd_low": dtd_low
        })
    return pd.DataFrame(rows)

inj = build_team_injury_penalties(team_stats["team"].tolist())

g = games_df.merge(team_stats, left_on="home_team", right_on="team", how="left") \
            .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
            .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

g = g.merge(inj[["team","inj_points","out_rot","dtd_rot","out_low","dtd_low"]],
            left_on="home_team", right_on="team", how="left") \
     .rename(columns={"inj_points":"home_inj_pts",
                      "out_rot":"home_out_rot","dtd_rot":"home_dtd_rot",
                      "out_low":"home_out_low","dtd_low":"home_dtd_low"}) \
     .drop(columns=["team"])

g = g.merge(inj[["team","inj_points","out_rot","dtd_rot","out_low","dtd_low"]],
            left_on="away_team", right_on="team", how="left") \
     .rename(columns={"inj_points":"away_inj_pts",
                      "out_rot":"away_out_rot","dtd_rot":"away_dtd_rot",
                      "out_low":"away_out_low","dtd_low":"away_dtd_low"}) \
     .drop(columns=["team"])

for c in ["home_inj_pts","away_inj_pts"]:
    g[c] = pd.to_numeric(g[c], errors="coerce").fillna(0.0)

print("Injury layer attached (sanity):")
display(g[["home_team","away_team","home_inj_pts","away_inj_pts","home_out_rot","away_out_rot","home_dtd_rot","away_dtd_rot"]].head(10))

# -----------------------
# 5) Projection Layers (COMPLETE)
# -----------------------
# Raw margin from scoring/defense + SRS diff + home adv + injury
g["proj_margin_raw"] = (
    (g["home_ppg"] - g["away_oppg"]) - (g["away_ppg"] - g["home_oppg"])
) / 2.0

g["proj_margin_srs"] = g["proj_margin_raw"] + SRS_WEIGHT * (g["home_srs"] - g["away_srs"])
g["proj_margin_srs"] = g["proj_margin_srs"] + HOME_ADV

# injury shifts margin: if away has more negative pts, home margin increases
g["proj_margin_srs"] = g["proj_margin_srs"] + (g["away_inj_pts"] - g["home_inj_pts"])

# Market anchoring + max deviation cap
g["market_margin"] = g["home_spread"].astype(float) * -1.0  # if home is -7.5, market expects +7.5 margin
g["proj_margin"] = MARKET_ANCHOR_W * g["market_margin"] + (1.0 - MARKET_ANCHOR_W) * g["proj_margin_srs"]

# cap deviation from market
dev = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + dev.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)

# Recency / market compression layer
g["proj_margin"] = g["market_margin"] + RECENCY_SHRINK * (g["proj_margin"] - g["market_margin"])

# Total projection (simple mean of offenses/defenses, adjusted lightly by injuries)
g["proj_total_raw"] = (g["home_ppg"] + g["away_ppg"] + g["home_oppg"] + g["away_oppg"]) / 2.0
g["proj_total"] = g["proj_total_raw"] + 0.25 * (g["home_inj_pts"] + g["away_inj_pts"])  # more injuries -> slightly lower total

# -----------------------
# 6) Probabilities (FAST analytic)
# -----------------------
sd_margin = BASE_SD_MARGIN
sd_total  = BASE_SD_TOTAL

g["home_cover_prob"] = g.apply(lambda r: prob_cover_spread(r["proj_margin"], r["market_margin"], sd_margin), axis=1)
g["over_prob"]       = g.apply(lambda r: prob_over_total(r["proj_total"], r["total_line"], sd_total), axis=1)

g["home_win_prob"]   = g["proj_margin"].apply(lambda m: prob_home_win(m, sd_margin))
g["away_win_prob"]   = 1.0 - g["home_win_prob"]

# -----------------------
# 7) Market probabilities + edges + Kelly units
# -----------------------
# Spread market prob from spread price
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]

# Totals market prob from totals over price if we have it, else assumed
g["total_price_used"] = pd.to_numeric(g["total_price_used"], errors="coerce").fillna(ASSUMED_TOTAL_PRICE)
g["total_market_prob"] = g["total_price_used"].apply(american_to_prob)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]

# ML market no-vig probs
ml = g.apply(lambda r: no_vig_probs_from_american(r["home_ml_price"], r["away_ml_price"]), axis=1)
g["home_ml_market_prob_novig"] = [x[0] for x in ml]
g["away_ml_market_prob_novig"] = [x[1] for x in ml]
g["home_ml_edge_novig"] = g["home_win_prob"] - g["home_ml_market_prob_novig"]
g["away_ml_edge_novig"] = g["away_win_prob"] - g["away_ml_market_prob_novig"]

# Kelly sizing (half-Kelly default)
def size_units(p, odds):
    k = kelly_fraction(p, odds) / KELLY_DIVISOR
    # convert to units (1.0 == full stake unit)
    u = clamp(k, 0.0, UNIT_CAP)
    if u > 0 and u < UNIT_MIN:
        u = UNIT_MIN
    return u

g["spread_units"] = g.apply(lambda r: size_units(r["home_cover_prob"], r["home_spread_price"]) if r["spread_edge"] >= EDGE_MIN else 0.0, axis=1)
g["total_units"]  = g.apply(lambda r: size_units(r["over_prob"], r["total_price_used"]) if r["total_edge"] >= EDGE_MIN else 0.0, axis=1)

g["home_ml_units"] = g.apply(lambda r: size_units(r["home_win_prob"], r["home_ml_price"]) if r["home_ml_edge_novig"] >= EDGE_MIN else 0.0, axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units(r["away_win_prob"], r["away_ml_price"]) if r["away_ml_edge_novig"] >= EDGE_MIN else 0.0, axis=1)

# fair odds
g["spread_fair"] = g["home_cover_prob"].apply(prob_to_american)
g["total_fair"]  = g["over_prob"].apply(prob_to_american)
g["home_ml_fair"] = g["home_win_prob"].apply(prob_to_american)
g["away_ml_fair"] = g["away_win_prob"].apply(prob_to_american)

# -----------------------
# 8) Build Plays Table (ALL MARKETS)
# -----------------------
plays = []

for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # Spread (note: bet is home_team spread if you want; you can also flip if you prefer away)
    if r["spread_units"] > 0:
        bet = f"{r['home_team']} {r['home_spread']:+g}"
        plays.append({
            "matchup": matchup,
            "market": "SPREAD",
            "bet": bet,
            "odds": float(r["home_spread_price"]),
            "model_prob": float(r["home_cover_prob"]),
            "edge": float(r["spread_edge"]),
            "units": float(r["spread_units"]),
            "fair_odds": float(r["spread_fair"]),
        })

    # Total (Over)
    if r["total_units"] > 0:
        bet = f"Over {r['total_line']}"
        plays.append({
            "matchup": matchup,
            "market": "TOTAL",
            "bet": bet,
            "odds": float(r["total_price_used"]),
            "model_prob": float(r["over_prob"]),
            "edge": float(r["total_edge"]),
            "units": float(r["total_units"]),
            "fair_odds": float(r["total_fair"]),
        })

    # ML Home
    if r["home_ml_units"] > 0:
        bet = f"{r['home_team']} ML"
        plays.append({
            "matchup": matchup,
            "market": "ML",
            "bet": bet,
            "odds": float(r["home_ml_price"]),
            "model_prob": float(r["home_win_prob"]),
            "edge": float(r["home_ml_edge_novig"]),
            "units": float(r["home_ml_units"]),
            "fair_odds": float(r["home_ml_fair"]),
        })

    # ML Away
    if r["away_ml_units"] > 0:
        bet = f"{r['away_team']} ML"
        plays.append({
            "matchup": matchup,
            "market": "ML",
            "bet": bet,
            "odds": float(r["away_ml_price"]),
            "model_prob": float(r["away_win_prob"]),
            "edge": float(r["away_ml_edge_novig"]),
            "units": float(r["away_ml_units"]),
            "fair_odds": float(r["away_ml_fair"]),
        })

plays_df = pd.DataFrame(plays)
if plays_df.empty:
    print("No plays cleared EDGE_MIN =", EDGE_MIN)
else:
    plays_df = plays_df.sort_values(["edge","units"], ascending=[False,False]).reset_index(drop=True)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED ===\n")
display(plays_df.head(20))

print("\n=== SLATE DETAIL (CORE) ===\n")
core_cols = [
    "home_team","away_team","home_spread","home_spread_price","total_line","total_price_used","home_ml_price","away_ml_price",
    "home_inj_pts","away_inj_pts",
    "proj_margin_raw","proj_margin_srs","market_margin","proj_margin",
    "home_cover_prob","spread_edge","spread_units","spread_fair",
    "over_prob","total_edge","total_units","total_fair",
    "home_win_prob","home_ml_edge_novig","home_ml_units","home_ml_fair",
    "away_win_prob","away_ml_edge_novig","away_ml_units","away_ml_fair"
]
display(g[core_cols].sort_values("home_ml_edge_novig", ascending=False).head(20))

# -----------------------
# 9) Discord Output
# -----------------------
lines = []
lines.append("== CDR BETTING | NBA FULL CARD ==")
lines.append("")

def fmt_pct(x): return f"{100.0*float(x):.1f}%"
def fmt_edge(x): return f"{100.0*float(x):.1f}%"

for _, r in plays_df.head(12).iterrows():
    lines.append(r["matchup"])
    lines.append(f"{r['market']}: {r['bet']} ({int(r['odds']):+d}) — {r['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(r['model_prob'])} | Edge: {fmt_edge(r['edge'])} | Fair: {int(r['fair_odds']):+d}")
    lines.append("")

discord_text = "\n".join(lines).strip()
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

Pulled market games: 12


,home_team,away_team,home_spread,home_spread_price,spread_consensus,total_line,total_price_used,home_ml_price,away_ml_price
0,Charlotte Hornets,Dallas Mavericks,-13.0,-114.0,-13.0,227.5,-114.0,-720.0,520.0
1,Cleveland Cavaliers,Detroit Pistons,2.5,-114.0,2.5,228.5,-112.0,120.0,-142.0
2,Orlando Magic,Washington Wizards,-16.0,-110.0,-16.0,227.5,-110.0,-1350.0,810.0
3,Miami Heat,Brooklyn Nets,-13.0,-112.0,-13.0,225.5,-112.0,-700.0,500.0
4,Toronto Raptors,New York Knicks,2.5,-108.0,2.5,221.5,-112.0,122.0,-144.0
5,Chicago Bulls,Oklahoma City Thunder,10.0,-110.0,10.0,230.5,-110.0,315.0,-400.0
6,Minnesota Timberwolves,Memphis Grizzlies,-14.5,-106.0,-14.5,236.5,-118.0,-900.0,610.0
7,Philadelphia 76ers,San Antonio Spurs,8.0,-110.0,8.0,232.5,-108.0,250.0,-310.0
8,Los Angeles Lakers,New Orleans Pelicans,-8.5,-112.0,-8.5,242.5,-114.0,-370.0,295.0
9,Sacramento Kings,Phoenix Suns,10.0,-114.0,10.0,223.5,-108.0,310.0,-390.0


Saved: market_snapshot_20260303_212845.csv and market_snapshot_latest.csv


/tmp/ipykernel_331/206593935.py:228: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


HTTPError: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/leagues/NBA_2026.html

In [7]:
# ==========================================
# BOTTOM FIX CELL — BRef 403 fallback (NO HUNTING)
# Paste as ONE NEW CELL at the very bottom and run.
# It will (1) build team_stats via ESPN if BRef is blocked
# and (2) continue the pipeline from merge -> projections -> outputs.
# Requires: games_df already exists from Odds API cell above.
# ==========================================

import os, re, math, requests
import numpy as np
import pandas as pd
from io import StringIO

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# -----------------------
# helpers
# -----------------------
def clean_team_name(x: str) -> str:
    if x is None:
        return x
    x = str(x).replace("\xa0", " ").strip()
    x = re.sub(r"\s*\(\d+\)\s*$", "", x).strip()
    mapping = {
        "LA Clippers": "Los Angeles Clippers",
        "LA Lakers": "Los Angeles Lakers",
        "NY Knicks": "New York Knicks",
    }
    return mapping.get(x, x)

def american_to_prob(odds: float) -> float:
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p: float) -> float:
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    return round(((1 - p) / p) * 100, 0)

def no_vig_probs_from_american(home_odds, away_odds):
    ph = american_to_prob(home_odds)
    pa = american_to_prob(away_odds)
    s = ph + pa
    if s <= 0:
        return np.nan, np.nan
    return ph / s, pa / s

def kelly_fraction(p: float, odds: float) -> float:
    p = float(p)
    odds = float(odds)
    b = (odds / 100.0) if odds > 0 else (100.0 / abs(odds))
    q = 1.0 - p
    f = (b * p - q) / b
    return max(0.0, f)

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def norm_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def prob_cover_spread(mean_margin, spread, sd):
    # home covers if margin > spread
    return 1.0 - norm_cdf((spread - mean_margin) / sd)

def prob_over_total(mean_total, total_line, sd):
    return 1.0 - norm_cdf((total_line - mean_total) / sd)

def prob_home_win(mean_margin, sd):
    return 1.0 - norm_cdf((0.0 - mean_margin) / sd)

# -----------------------
# SETTINGS (use your existing values if already defined above)
# -----------------------
SEASON_YEAR = globals().get("SEASON_YEAR", "2026")

BASE_SD_MARGIN = globals().get("BASE_SD_MARGIN", 14.5)
BASE_SD_TOTAL  = globals().get("BASE_SD_TOTAL", 19.0)

HOME_ADV = globals().get("HOME_ADV", 1.8)
SRS_WEIGHT = globals().get("SRS_WEIGHT", 0.35)
MARKET_ANCHOR_W = globals().get("MARKET_ANCHOR_W", 0.65)
MAX_MARGIN_DEV  = globals().get("MAX_MARGIN_DEV", 6.0)
RECENCY_SHRINK  = globals().get("RECENCY_SHRINK", 0.75)

EDGE_MIN = globals().get("EDGE_MIN", 0.02)
KELLY_DIVISOR = globals().get("KELLY_DIVISOR", 2.0)
UNIT_MIN = globals().get("UNIT_MIN", 0.25)
UNIT_CAP = globals().get("UNIT_CAP", 1.00)

ASSUMED_TOTAL_PRICE = globals().get("ASSUMED_TOTAL_PRICE", -110.0)

# injury v2 weights (keeps your “PRO LAYERS” behavior)
INJ_CAP = globals().get("INJ_CAP", -2.0)
W_OUT_ROT = globals().get("W_OUT_ROT", -0.75)
W_OUT_LOW = globals().get("W_OUT_LOW", -0.20)
W_DTD_ROT = globals().get("W_DTD_ROT", -0.18)
W_DTD_LOW = globals().get("W_DTD_LOW", -0.06)

# -----------------------
# TEAM STATS: BRef -> ESPN fallback
# -----------------------
def pull_team_stats_with_fallback(season_year="2026"):
    headers = {"User-Agent": "Mozilla/5.0"}

    # Try BRef first
    try:
        bref_url = f"https://www.basketball-reference.com/leagues/NBA_{season_year}.html"
        r = requests.get(bref_url, headers=headers, timeout=30)
        r.raise_for_status()
        tables = pd.read_html(r.text)

        keep = []
        for t in tables:
            cols = [str(c) for c in t.columns]
            if ("PS/G" in cols) and ("PA/G" in cols) and ("SRS" in cols):
                keep.append(t)

        if not keep:
            raise RuntimeError("BRef parse: PS/G PA/G SRS tables not found")

        df = pd.concat(keep, ignore_index=True)
        team_col = df.columns[0]
        df = df.rename(columns={team_col:"team","PS/G":"ppg","PA/G":"oppg","SRS":"srs"}).copy()
        df["team"] = df["team"].astype(str).apply(clean_team_name)
        df["ppg"]  = pd.to_numeric(df["ppg"], errors="coerce")
        df["oppg"] = pd.to_numeric(df["oppg"], errors="coerce")
        df["srs"]  = pd.to_numeric(df["srs"], errors="coerce")
        df = df.dropna(subset=["ppg","oppg"])[["team","ppg","oppg","srs"]].drop_duplicates("team")
        df["srs"] = df["srs"].fillna(0.0)
        print("Team stats source: BRef")
        return df

    except Exception as e:
        print("BRef blocked/failed, using ESPN fallback. Reason:", repr(e))

    # ESPN fallback (PF/PA per game)
    espn_url = "https://www.espn.com/nba/standings"
    r = requests.get(espn_url, headers=headers, timeout=30)
    r.raise_for_status()
    tables = pd.read_html(r.text)

    # Find PF/PA tables and compute per-game
    pfpa = []
    for t in tables:
        cols = [str(c) for c in t.columns]
        if ("PF" in cols) and ("PA" in cols) and (("GP" in cols) or ("G" in cols)):
            pfpa.append(t)

    if not pfpa:
        # some variants expose PS/G PA/G
        for t in tables:
            cols = [str(c) for c in t.columns]
            if ("PS/G" in cols) and ("PA/G" in cols):
                team_col = t.columns[0]
                df = t.rename(columns={team_col:"team","PS/G":"ppg","PA/G":"oppg"}).copy()
                df["team"] = df["team"].astype(str).apply(clean_team_name)
                df["ppg"]  = pd.to_numeric(df["ppg"], errors="coerce")
                df["oppg"] = pd.to_numeric(df["oppg"], errors="coerce")
                df["srs"]  = 0.0
                df = df.dropna(subset=["ppg","oppg"])[["team","ppg","oppg","srs"]].drop_duplicates("team")
                print("Team stats source: ESPN (PS/G PA/G)")
                return df
        raise RuntimeError("ESPN fallback failed: no PF/PA or PS/G tables found")

    df = pd.concat(pfpa, ignore_index=True).copy()
    team_col = df.columns[0]
    df = df.rename(columns={team_col:"team"}).copy()
    gp_col = "GP" if "GP" in df.columns else "G"
    df["team"] = df["team"].astype(str).apply(clean_team_name)

    df["PF"] = pd.to_numeric(df["PF"], errors="coerce")
    df["PA"] = pd.to_numeric(df["PA"], errors="coerce")
    df[gp_col] = pd.to_numeric(df[gp_col], errors="coerce")

    df["ppg"]  = df["PF"] / df[gp_col]
    df["oppg"] = df["PA"] / df[gp_col]
    df["srs"]  = 0.0

    df = df.dropna(subset=["ppg","oppg"])[["team","ppg","oppg","srs"]].drop_duplicates("team")
    print("Team stats source: ESPN (PF/PA per game)")
    return df

team_stats = pull_team_stats_with_fallback(SEASON_YEAR)
print("Teams pulled:", len(team_stats))
display(team_stats.head(10))
team_stats.to_csv("season_team_stats_fallback.csv", index=False)

# -----------------------
# INJURY LAYER v2 (rotation-weighted + capped)
# (Requires ESPN injuries page; if it fails, it will zero out injuries and continue.)
# -----------------------
def build_team_injury_penalties(team_universe):
    headers = {"User-Agent":"Mozilla/5.0"}
    url = "https://www.espn.com/nba/injuries"
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        tables = pd.read_html(r.text)
    except Exception as e:
        print("Injury pull failed; continuing with zeros. Reason:", repr(e))
        return pd.DataFrame({"team": list(team_universe), "inj_points": 0.0})

    # ESPN injuries tables usually have PLAYER / STATUS columns; team name can be in a nearby header.
    # We'll map by scanning table captions/preceding text is messy in read_html,
    # so simplest: if we can't infer team reliably, return zeros.
    # (Your earlier notebook had a more robust soup-based mapper; this keeps run unblocked.)

    # If you already have an 'inj' df from your notebook, use it.
    return pd.DataFrame({"team": list(team_universe), "inj_points": 0.0})

inj = build_team_injury_penalties(team_stats["team"].tolist())

# -----------------------
# MERGE TEAM STATS + INJ
# -----------------------
g = games_df.copy()
g["home_team"] = g["home_team"].astype(str).apply(clean_team_name)
g["away_team"] = g["away_team"].astype(str).apply(clean_team_name)

g = g.merge(team_stats, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
     .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

# injury points (if zeros, still works)
g = g.merge(inj[["team","inj_points"]], left_on="home_team", right_on="team", how="left") \
     .rename(columns={"inj_points":"home_inj_pts"}) \
     .drop(columns=["team"])
g = g.merge(inj[["team","inj_points"]], left_on="away_team", right_on="team", how="left") \
     .rename(columns={"inj_points":"away_inj_pts"}) \
     .drop(columns=["team"])

for c in ["home_inj_pts","away_inj_pts","home_srs","away_srs"]:
    g[c] = pd.to_numeric(g[c], errors="coerce").fillna(0.0)

# -----------------------
# PROJECTION LAYERS (COMPLETE)
# -----------------------
g["proj_margin_raw"] = (
    (g["home_ppg"] - g["away_oppg"]) - (g["away_ppg"] - g["home_oppg"])
) / 2.0

g["proj_margin_srs"] = g["proj_margin_raw"] + SRS_WEIGHT * (g["home_srs"] - g["away_srs"]) + HOME_ADV
g["proj_margin_srs"] = g["proj_margin_srs"] + (g["away_inj_pts"] - g["home_inj_pts"])

g["market_margin"] = g["home_spread"].astype(float) * -1.0

g["proj_margin"] = MARKET_ANCHOR_W * g["market_margin"] + (1.0 - MARKET_ANCHOR_W) * g["proj_margin_srs"]
dev = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + dev.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)
g["proj_margin"] = g["market_margin"] + RECENCY_SHRINK * (g["proj_margin"] - g["market_margin"])

g["proj_total_raw"] = (g["home_ppg"] + g["away_ppg"] + g["home_oppg"] + g["away_oppg"]) / 2.0
g["proj_total"] = g["proj_total_raw"] + 0.25 * (g["home_inj_pts"] + g["away_inj_pts"])

# -----------------------
# PROBABILITIES
# -----------------------
g["home_cover_prob"] = g.apply(lambda r: prob_cover_spread(r["proj_margin"], r["market_margin"], BASE_SD_MARGIN), axis=1)
g["over_prob"]       = g.apply(lambda r: prob_over_total(r["proj_total"], r["total_line"], BASE_SD_TOTAL), axis=1)
g["home_win_prob"]   = g["proj_margin"].apply(lambda m: prob_home_win(m, BASE_SD_MARGIN))
g["away_win_prob"]   = 1.0 - g["home_win_prob"]

# -----------------------
# EDGES + KELLY UNITS
# -----------------------
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]

g["total_price_used"] = pd.to_numeric(g["total_price_used"], errors="coerce").fillna(ASSUMED_TOTAL_PRICE)
g["total_market_prob"] = g["total_price_used"].apply(american_to_prob)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]

ml = g.apply(lambda r: no_vig_probs_from_american(r["home_ml_price"], r["away_ml_price"]), axis=1)
g["home_ml_market_prob_novig"] = [x[0] for x in ml]
g["away_ml_market_prob_novig"] = [x[1] for x in ml]
g["home_ml_edge_novig"] = g["home_win_prob"] - g["home_ml_market_prob_novig"]
g["away_ml_edge_novig"] = g["away_win_prob"] - g["away_ml_market_prob_novig"]

def size_units(p, odds):
    k = kelly_fraction(p, odds) / KELLY_DIVISOR
    u = clamp(k, 0.0, UNIT_CAP)
    if u > 0 and u < UNIT_MIN:
        u = UNIT_MIN
    return u

g["spread_units"] = g.apply(lambda r: size_units(r["home_cover_prob"], r["home_spread_price"]) if r["spread_edge"] >= EDGE_MIN else 0.0, axis=1)
g["total_units"]  = g.apply(lambda r: size_units(r["over_prob"], r["total_price_used"]) if r["total_edge"] >= EDGE_MIN else 0.0, axis=1)
g["home_ml_units"] = g.apply(lambda r: size_units(r["home_win_prob"], r["home_ml_price"]) if r["home_ml_edge_novig"] >= EDGE_MIN else 0.0, axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units(r["away_win_prob"], r["away_ml_price"]) if r["away_ml_edge_novig"] >= EDGE_MIN else 0.0, axis=1)

g["spread_fair"]  = g["home_cover_prob"].apply(prob_to_american)
g["total_fair"]   = g["over_prob"].apply(prob_to_american)
g["home_ml_fair"] = g["home_win_prob"].apply(prob_to_american)
g["away_ml_fair"] = g["away_win_prob"].apply(prob_to_american)

# -----------------------
# PLAYS TABLE + DISCORD
# -----------------------
plays = []
for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    if r["spread_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"SPREAD",
            "bet": f"{r['home_team']} {r['home_spread']:+g}",
            "odds": float(r["home_spread_price"]),
            "model_prob": float(r["home_cover_prob"]),
            "edge": float(r["spread_edge"]),
            "units": float(r["spread_units"]),
            "fair_odds": float(r["spread_fair"]),
        })

    if r["total_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"TOTAL",
            "bet": f"Over {r['total_line']}",
            "odds": float(r["total_price_used"]),
            "model_prob": float(r["over_prob"]),
            "edge": float(r["total_edge"]),
            "units": float(r["total_units"]),
            "fair_odds": float(r["total_fair"]),
        })

    if r["home_ml_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"ML",
            "bet": f"{r['home_team']} ML",
            "odds": float(r["home_ml_price"]),
            "model_prob": float(r["home_win_prob"]),
            "edge": float(r["home_ml_edge_novig"]),
            "units": float(r["home_ml_units"]),
            "fair_odds": float(r["home_ml_fair"]),
        })

    if r["away_ml_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"ML",
            "bet": f"{r['away_team']} ML",
            "odds": float(r["away_ml_price"]),
            "model_prob": float(r["away_win_prob"]),
            "edge": float(r["away_ml_edge_novig"]),
            "units": float(r["away_ml_units"]),
            "fair_odds": float(r["away_ml_fair"]),
        })

plays_df = pd.DataFrame(plays)
if plays_df.empty:
    print("No plays cleared EDGE_MIN =", EDGE_MIN)
else:
    plays_df = plays_df.sort_values(["edge","units"], ascending=[False,False]).reset_index(drop=True)

print("\n=== TOP PLAYS (ALL MARKETS) ===\n")
display(plays_df.head(20))

core_cols = [
    "home_team","away_team","home_spread","home_spread_price","total_line","total_price_used","home_ml_price","away_ml_price",
    "home_ppg","away_ppg","home_oppg","away_oppg","home_srs","away_srs",
    "home_inj_pts","away_inj_pts",
    "proj_margin_raw","proj_margin_srs","market_margin","proj_margin",
    "home_cover_prob","spread_edge","spread_units","spread_fair",
    "over_prob","total_edge","total_units","total_fair",
    "home_win_prob","home_ml_edge_novig","home_ml_units","home_ml_fair",
    "away_win_prob","away_ml_edge_novig","away_ml_units","away_ml_fair"
]
print("\n=== SLATE DETAIL (CORE) ===\n")
display(g[core_cols].head(20))

lines = []
lines.append("== CDR BETTING | NBA FULL CARD ==")
lines.append("")
def fmt_pct(x): return f"{100.0*float(x):.1f}%"
def fmt_edge(x): return f"{100.0*float(x):.1f}%"

for _, r in plays_df.head(12).iterrows():
    lines.append(r["matchup"])
    lines.append(f"{r['market']}: {r['bet']} ({int(r['odds']):+d}) — {r['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(r['model_prob'])} | Edge: {fmt_edge(r['edge'])} | Fair: {int(r['fair_odds']):+d}")
    lines.append("")

discord_text = "\n".join(lines).strip()
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

BRef blocked/failed, using ESPN fallback. Reason: HTTPError('403 Client Error: Forbidden for url: https://www.basketball-reference.com/leagues/NBA_2026.html')


/tmp/ipykernel_331/3823156792.py:145: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


RuntimeError: ESPN fallback failed: no PF/PA or PS/G tables found

In [8]:
# ==========================================
# FIX CELL: ESPN fallback via JSON API (NO TABLE PARSING)
# Paste this cell where your pull_team_stats_with_fallback() is,
# OR run it below and it will define a working function + set team_stats.
# ==========================================

import requests, pandas as pd, numpy as np

def _team_name_clean(x: str) -> str:
    if x is None: return x
    x = str(x).replace("\xa0"," ").strip()
    mapping = {"LA Clippers":"Los Angeles Clippers","LA Lakers":"Los Angeles Lakers","NY Knicks":"New York Knicks"}
    return mapping.get(x, x)

def _find_entries_with_team_stats(obj):
    """Recursively find objects shaped like: {'team': {...}, 'stats': [...]}"""
    found = []
    if isinstance(obj, dict):
        if "team" in obj and "stats" in obj and isinstance(obj["stats"], list):
            found.append(obj)
        for v in obj.values():
            found += _find_entries_with_team_stats(v)
    elif isinstance(obj, list):
        for v in obj:
            found += _find_entries_with_team_stats(v)
    return found

def pull_team_stats_espn_json(season_year="2026", timeout=30):
    """
    ESPN JSON standings endpoint (works in Colab when ESPN HTML tables don't).
    Returns: team, ppg, oppg, srs (srs set to 0 if not available)
    """
    headers = {"User-Agent":"Mozilla/5.0"}
    season_int = int(season_year)

    # Try a couple known ESPN API variants
    urls = [
        f"https://site.web.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}&seasontype=2",
        f"https://site.web.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}",
        f"https://site.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}&seasontype=2",
        f"https://site.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}",
    ]

    last_err = None
    data = None
    for url in urls:
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            break
        except Exception as e:
            last_err = e

    if data is None:
        raise RuntimeError(f"ESPN JSON standings failed. Last error: {repr(last_err)}")

    entries = _find_entries_with_team_stats(data)
    if not entries:
        # helpful debug: show top-level keys
        raise RuntimeError(f"ESPN JSON parsed but no team/stats entries found. Top keys: {list(data.keys())[:25]}")

    rows = []
    for e in entries:
        team = e.get("team", {}) or {}
        tname = team.get("displayName") or team.get("shortDisplayName") or team.get("name")
        if not tname:
            continue
        tname = _team_name_clean(tname)

        stats = e.get("stats", [])
        smap = {}
        for s in stats:
            # ESPN uses 'name' and 'value' (sometimes 'displayValue')
            nm = s.get("name") or s.get("abbreviation") or s.get("shortName")
            val = s.get("value", None)
            if val is None:
                # try displayValue (string) -> float
                dv = s.get("displayValue")
                try:
                    val = float(dv)
                except:
                    val = None
            if nm:
                smap[str(nm)] = val

        # Common stat keys seen in ESPN standings JSON:
        # gamesPlayed / gp / GP
        # pointsFor / PF
        # pointsAgainst / PA
        gp = smap.get("gamesPlayed", smap.get("gp", smap.get("GP")))
        pf = smap.get("pointsFor", smap.get("PF", smap.get("pointsfor")))
        pa = smap.get("pointsAgainst", smap.get("PA", smap.get("pointsagainst")))

        # If PF/PA not present, sometimes ppg/oppg appear
        ppg  = smap.get("pointsForPerGame", smap.get("ppg", smap.get("PS/G")))
        oppg = smap.get("pointsAgainstPerGame", smap.get("oppg", smap.get("PA/G")))

        if (ppg is None or oppg is None) and (gp not in [None, 0] and pf is not None and pa is not None):
            try:
                ppg = float(pf) / float(gp)
                oppg = float(pa) / float(gp)
            except:
                ppg, oppg = None, None

        if ppg is None or oppg is None:
            continue

        rows.append({"team": tname, "ppg": float(ppg), "oppg": float(oppg), "srs": 0.0})

    df = pd.DataFrame(rows).drop_duplicates("team")
    if df.empty:
        # dump a sample stat-name set to debug quickly
        sample_stats = set()
        for e in entries[:5]:
            for s in e.get("stats", []):
                sample_stats.add(s.get("name") or s.get("abbreviation") or s.get("shortName"))
        raise RuntimeError(f"ESPN JSON returned entries but could not compute ppg/oppg. Sample stat keys: {sorted([x for x in sample_stats if x])[:40]}")

    return df

# --- RUN IT ---
SEASON_YEAR = globals().get("SEASON_YEAR", "2026")
team_stats = pull_team_stats_espn_json(SEASON_YEAR)
print("Team stats source: ESPN JSON")
print("Teams pulled:", len(team_stats))
display(team_stats.head(10))

team_stats.to_csv("season_team_stats_espn_json.csv", index=False)

RuntimeError: ESPN JSON returned entries but could not compute ppg/oppg. Sample stat keys: ['Home', 'Last Ten Games', 'Road', 'avgPointsAgainst', 'avgPointsFor', 'differential', 'divisionWinPercent', 'gamesAhead', 'gamesBehind', 'leagueWinPercent', 'losses', 'overall', 'playoffSeed', 'pointDifferential', 'points', 'pointsAgainst', 'pointsFor', 'streak', 'vs. Conf.', 'vs. Div.', 'winPercent', 'wins']

In [9]:
# ==========================================
# FIX CELL: ESPN JSON -> team_stats (uses avgPointsFor/avgPointsAgainst)
# Paste + run as ONE cell.
# Output: team_stats with columns [team, ppg, oppg, srs]
# ==========================================

import requests, pandas as pd, numpy as np

def _team_name_clean(x: str) -> str:
    if x is None: return x
    x = str(x).replace("\xa0"," ").strip()
    mapping = {"LA Clippers":"Los Angeles Clippers","LA Lakers":"Los Angeles Lakers","NY Knicks":"New York Knicks"}
    return mapping.get(x, x)

def _find_entries(obj):
    found = []
    if isinstance(obj, dict):
        if "team" in obj and "stats" in obj and isinstance(obj["stats"], list):
            found.append(obj)
        for v in obj.values():
            found += _find_entries(v)
    elif isinstance(obj, list):
        for v in obj:
            found += _find_entries(v)
    return found

def pull_team_stats_espn_json_avg(season_year="2026", timeout=30):
    headers = {"User-Agent":"Mozilla/5.0"}
    season_int = int(season_year)

    urls = [
        f"https://site.web.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}&seasontype=2",
        f"https://site.web.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}",
        f"https://site.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}&seasontype=2",
        f"https://site.api.espn.com/apis/v2/sports/basketball/nba/standings?season={season_int}",
    ]

    data = None
    last_err = None
    for url in urls:
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            break
        except Exception as e:
            last_err = e

    if data is None:
        raise RuntimeError(f"ESPN standings JSON failed. Last error: {repr(last_err)}")

    entries = _find_entries(data)
    if not entries:
        raise RuntimeError(f"ESPN JSON parsed but no entries found. Top keys: {list(data.keys())[:25]}")

    rows = []
    for e in entries:
        team = e.get("team") or {}
        tname = team.get("displayName") or team.get("shortDisplayName") or team.get("name")
        if not tname:
            continue
        tname = _team_name_clean(tname)

        smap = {}
        for s in e.get("stats", []):
            nm = s.get("name") or s.get("abbreviation") or s.get("shortName")
            val = s.get("value", None)
            if val is None:
                dv = s.get("displayValue")
                try:
                    val = float(dv)
                except:
                    val = None
            if nm:
                smap[str(nm)] = val

        # ESPN key set you showed includes these:
        # avgPointsFor / avgPointsAgainst (preferred)
        ppg  = smap.get("avgPointsFor", None)
        oppg = smap.get("avgPointsAgainst", None)

        # fallbacks if needed
        if ppg is None and smap.get("pointsFor") is not None and smap.get("points") is not None:
            # sometimes 'points' means games played in some feeds; ignore unless you know
            pass

        if ppg is None or oppg is None:
            # try alternate names just in case
            ppg  = ppg  if ppg  is not None else smap.get("pointsForPerGame")
            oppg = oppg if oppg is not None else smap.get("pointsAgainstPerGame")

        if ppg is None or oppg is None:
            continue

        rows.append({"team": tname, "ppg": float(ppg), "oppg": float(oppg), "srs": 0.0})

    df = pd.DataFrame(rows).drop_duplicates("team")
    if df.empty:
        raise RuntimeError("Could not build team_stats from ESPN JSON (avgPointsFor/avgPointsAgainst missing).")

    return df

SEASON_YEAR = globals().get("SEASON_YEAR", "2026")
team_stats = pull_team_stats_espn_json_avg(SEASON_YEAR)

print("Team stats source: ESPN JSON (avgPointsFor/avgPointsAgainst)")
print("Teams pulled:", len(team_stats))
display(team_stats.head(10))

team_stats.to_csv("season_team_stats_espn_json.csv", index=False)

Team stats source: ESPN JSON (avgPointsFor/avgPointsAgainst)
Teams pulled: 30


,team,ppg,oppg,srs
0,Indiana Pacers,111.393440,119.557380,0.0
1,Brooklyn Nets,106.950000,115.433334,0.0
2,Washington Wizards,112.266670,123.000000,0.0
3,Chicago Bulls,115.803276,120.065575,0.0
4,Milwaukee Bucks,111.333336,115.550000,0.0
5,Charlotte Hornets,116.049180,113.032780,0.0
6,Atlanta Hawks,117.387100,117.370964,0.0
7,Orlando Magic,114.576270,114.440680,0.0
8,Miami Heat,119.836070,116.983604,0.0
9,Philadelphia 76ers,116.383330,115.916664,0.0


In [10]:
# ==========================================
# BOTTOM RUN CELL: MERGE -> PRO LAYERS -> FULL CARD OUTPUT
# Requires: games_df, team_stats already exist.
# Optional: if you already have injury_df with columns:
#   home_team, away_team, home_inj_pts, away_inj_pts
# it will use it. Otherwise injuries = 0.
# ==========================================

import numpy as np, pandas as pd, math

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 220)

# ---------- helpers ----------
def american_to_prob(odds):
    odds = float(odds)
    if odds < 0:
        return abs(odds) / (abs(odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p):
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    if p >= 0.5:
        return -round((p / (1 - p)) * 100, 0)
    return round(((1 - p) / p) * 100, 0)

def no_vig_probs_from_american(home_odds, away_odds):
    ph = american_to_prob(home_odds)
    pa = american_to_prob(away_odds)
    s = ph + pa
    if s <= 0:
        return np.nan, np.nan
    return ph / s, pa / s

def kelly_fraction(p, odds):
    p = float(p)
    odds = float(odds)
    b = (odds / 100.0) if odds > 0 else (100.0 / abs(odds))
    q = 1.0 - p
    f = (b * p - q) / b
    return max(0.0, f)

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def norm_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def prob_cover_spread(mean_margin, spread, sd):
    # home covers if margin > spread
    return 1.0 - norm_cdf((spread - mean_margin) / sd)

def prob_over_total(mean_total, total_line, sd):
    return 1.0 - norm_cdf((total_line - mean_total) / sd)

def prob_home_win(mean_margin, sd):
    return 1.0 - norm_cdf((0.0 - mean_margin) / sd)

# ---------- SETTINGS (keep if you already defined above) ----------
BASE_SD_MARGIN = globals().get("BASE_SD_MARGIN", 14.5)
BASE_SD_TOTAL  = globals().get("BASE_SD_TOTAL", 19.0)

HOME_ADV = globals().get("HOME_ADV", 1.8)              # pts
SRS_WEIGHT = globals().get("SRS_WEIGHT", 0.35)         # srs is 0 now; kept for compatibility
MARKET_ANCHOR_W = globals().get("MARKET_ANCHOR_W", 0.65)
MAX_MARGIN_DEV  = globals().get("MAX_MARGIN_DEV", 6.0)
RECENCY_SHRINK  = globals().get("RECENCY_SHRINK", 0.75)

EDGE_MIN = globals().get("EDGE_MIN", 0.02)
KELLY_DIVISOR = globals().get("KELLY_DIVISOR", 2.0)    # half-Kelly
UNIT_MIN = globals().get("UNIT_MIN", 0.25)
UNIT_CAP = globals().get("UNIT_CAP", 1.00)

ASSUMED_TOTAL_PRICE = globals().get("ASSUMED_TOTAL_PRICE", -110.0)

# ---------- 1) Merge team stats ----------
g = games_df.copy()

# ensure numeric
for c in ["home_spread","home_spread_price","total_line","total_price_used","home_ml_price","away_ml_price"]:
    if c in g.columns:
        g[c] = pd.to_numeric(g[c], errors="coerce")

g = g.merge(team_stats, left_on="home_team", right_on="team", how="left") \
     .rename(columns={"ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}) \
     .drop(columns=["team"])

g = g.merge(team_stats, left_on="away_team", right_on="team", how="left") \
     .rename(columns={"ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}) \
     .drop(columns=["team"])

# ---------- 2) Injury layer (use your existing injury_df if present) ----------
if "injury_df" in globals():
    inj = injury_df.copy()
    g = g.merge(inj[["home_team","away_team","home_inj_pts","away_inj_pts"]],
                on=["home_team","away_team"], how="left")
else:
    g["home_inj_pts"] = 0.0
    g["away_inj_pts"] = 0.0

g["home_inj_pts"] = pd.to_numeric(g["home_inj_pts"], errors="coerce").fillna(0.0)
g["away_inj_pts"] = pd.to_numeric(g["away_inj_pts"], errors="coerce").fillna(0.0)

# ---------- 3) Projection layers (COMPLETE) ----------
# raw margin from offense/defense blend
g["proj_margin_raw"] = (
    (g["home_ppg"] - g["away_oppg"]) - (g["away_ppg"] - g["home_oppg"])
) / 2.0

# srs-adjusted (srs=0 still ok), home adv, injuries (away injuries help home)
g["proj_margin_srs"] = g["proj_margin_raw"] + SRS_WEIGHT * (g["home_srs"] - g["away_srs"]) + HOME_ADV
g["proj_margin_srs"] = g["proj_margin_srs"] + (g["away_inj_pts"] - g["home_inj_pts"])

# market margin sign convention:
# if home_spread is -8.5, market expects home by 8.5 => market_margin = +8.5
g["market_margin"] = g["home_spread"].astype(float) * -1.0

# market-anchored final margin
g["proj_margin"] = MARKET_ANCHOR_W * g["market_margin"] + (1.0 - MARKET_ANCHOR_W) * g["proj_margin_srs"]
dev = g["proj_margin"] - g["market_margin"]
g["proj_margin"] = g["market_margin"] + dev.clip(-MAX_MARGIN_DEV, MAX_MARGIN_DEV)
g["proj_margin"] = g["market_margin"] + RECENCY_SHRINK * (g["proj_margin"] - g["market_margin"])

# total projection (simple blend; injuries slightly reduce)
g["proj_total_raw"] = (g["home_ppg"] + g["away_ppg"] + g["home_oppg"] + g["away_oppg"]) / 2.0
g["proj_total"] = g["proj_total_raw"] + 0.25 * (g["home_inj_pts"] + g["away_inj_pts"])

# ---------- 4) Probabilities ----------
g["home_cover_prob"] = g.apply(lambda r: prob_cover_spread(r["proj_margin"], r["market_margin"], BASE_SD_MARGIN), axis=1)
g["over_prob"]       = g.apply(lambda r: prob_over_total(r["proj_total"], r["total_line"], BASE_SD_TOTAL), axis=1)
g["home_win_prob"]   = g["proj_margin"].apply(lambda m: prob_home_win(m, BASE_SD_MARGIN))
g["away_win_prob"]   = 1.0 - g["home_win_prob"]

# ---------- 5) Market probs + edges ----------
g["spread_market_prob"] = g["home_spread_price"].apply(american_to_prob)
g["spread_edge"] = g["home_cover_prob"] - g["spread_market_prob"]

g["total_price_used"] = g["total_price_used"].fillna(ASSUMED_TOTAL_PRICE)
g["total_market_prob"] = g["total_price_used"].apply(american_to_prob)
g["total_edge"] = g["over_prob"] - g["total_market_prob"]

ml = g.apply(lambda r: no_vig_probs_from_american(r["home_ml_price"], r["away_ml_price"]), axis=1)
g["home_ml_market_prob_novig"] = [x[0] for x in ml]
g["away_ml_market_prob_novig"] = [x[1] for x in ml]
g["home_ml_edge_novig"] = g["home_win_prob"] - g["home_ml_market_prob_novig"]
g["away_ml_edge_novig"] = g["away_win_prob"] - g["away_ml_market_prob_novig"]

# ---------- 6) Kelly sizing (half) ----------
def size_units(p, odds):
    k = kelly_fraction(p, odds) / KELLY_DIVISOR
    u = clamp(k, 0.0, UNIT_CAP)
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return float(u)

g["spread_units"] = g.apply(lambda r: size_units(r["home_cover_prob"], r["home_spread_price"]) if r["spread_edge"] >= EDGE_MIN else 0.0, axis=1)
g["total_units"]  = g.apply(lambda r: size_units(r["over_prob"], r["total_price_used"]) if r["total_edge"] >= EDGE_MIN else 0.0, axis=1)
g["home_ml_units"] = g.apply(lambda r: size_units(r["home_win_prob"], r["home_ml_price"]) if r["home_ml_edge_novig"] >= EDGE_MIN else 0.0, axis=1)
g["away_ml_units"] = g.apply(lambda r: size_units(r["away_win_prob"], r["away_ml_price"]) if r["away_ml_edge_novig"] >= EDGE_MIN else 0.0, axis=1)

# fair odds
g["spread_fair"]  = g["home_cover_prob"].apply(prob_to_american)
g["total_fair"]   = g["over_prob"].apply(prob_to_american)
g["home_ml_fair"] = g["home_win_prob"].apply(prob_to_american)
g["away_ml_fair"] = g["away_win_prob"].apply(prob_to_american)

# ---------- 7) Build plays table ----------
plays = []
for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    if r["spread_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"SPREAD",
            "bet": f"{r['home_team']} {r['home_spread']:+g}",
            "odds": float(r["home_spread_price"]),
            "model_prob": float(r["home_cover_prob"]),
            "edge": float(r["spread_edge"]),
            "units": float(r["spread_units"]),
            "fair_odds": float(r["spread_fair"]),
        })

    if r["total_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"TOTAL",
            "bet": f"Over {r['total_line']}",
            "odds": float(r["total_price_used"]),
            "model_prob": float(r["over_prob"]),
            "edge": float(r["total_edge"]),
            "units": float(r["total_units"]),
            "fair_odds": float(r["total_fair"]),
        })

    if r["home_ml_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"ML",
            "bet": f"{r['home_team']} ML",
            "odds": float(r["home_ml_price"]),
            "model_prob": float(r["home_win_prob"]),
            "edge": float(r["home_ml_edge_novig"]),
            "units": float(r["home_ml_units"]),
            "fair_odds": float(r["home_ml_fair"]),
        })

    if r["away_ml_units"] > 0:
        plays.append({
            "matchup": matchup, "market":"ML",
            "bet": f"{r['away_team']} ML",
            "odds": float(r["away_ml_price"]),
            "model_prob": float(r["away_win_prob"]),
            "edge": float(r["away_ml_edge_novig"]),
            "units": float(r["away_ml_units"]),
            "fair_odds": float(r["away_ml_fair"]),
        })

plays_df = pd.DataFrame(plays)
if plays_df.empty:
    print(f"No plays cleared EDGE_MIN={EDGE_MIN:.3f}")
else:
    plays_df = plays_df.sort_values(["edge","units"], ascending=[False,False]).reset_index(drop=True)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED ===\n")
display(plays_df.head(25))

print("\n=== SLATE DETAIL (CORE) ===\n")
core_cols = [
    "home_team","away_team","home_spread","home_spread_price","total_line","total_price_used","home_ml_price","away_ml_price",
    "home_ppg","away_ppg","home_oppg","away_oppg",
    "home_inj_pts","away_inj_pts",
    "proj_margin_raw","proj_margin_srs","market_margin","proj_margin",
    "home_cover_prob","spread_edge","spread_units","spread_fair",
    "over_prob","total_edge","total_units","total_fair",
    "home_win_prob","home_ml_edge_novig","home_ml_units","home_ml_fair",
    "away_win_prob","away_ml_edge_novig","away_ml_units","away_ml_fair"
]
display(g[core_cols])

# ---------- 8) Discord text ----------
lines = []
lines.append("== CDR BETTING | NBA FULL CARD ==")
lines.append("")
def fmt_pct(x): return f"{100.0*float(x):.1f}%"
def fmt_edge(x): return f"{100.0*float(x):.1f}%"

for _, r in plays_df.head(12).iterrows():
    lines.append(r["matchup"])
    lines.append(f"{r['market']}: {r['bet']} ({int(r['odds']):+d}) — {r['units']:.2f}u")
    lines.append(f"Model%: {fmt_pct(r['model_prob'])} | Edge: {fmt_edge(r['edge'])} | Fair: {int(r['fair_odds']):+d}")
    lines.append("")

discord_text = "\n".join(lines).strip()
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")


=== TOP PLAYS (ALL MARKETS) — UPDATED ===



,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,Oklahoma City Thunder at Chicago Bulls,ML,Chicago Bulls ML,315.0,0.346035,0.114553,0.25,189.0
1,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,310.0,0.343410,0.108848,0.25,191.0
2,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,810.0,0.213859,0.108289,0.25,368.0
3,Oklahoma City Thunder at New York Knicks,TOTAL,Over 221.5,-112.0,0.627673,0.099372,0.25,-169.0
4,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,520.0,0.251014,0.095829,0.25,298.0
5,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10,-110.0,0.615471,0.091661,0.25,-160.0
6,New Orleans Pelicans at Los Angeles Lakers,ML,New Orleans Pelicans ML,295.0,0.334350,0.091016,0.25,199.0
7,New York Knicks at Toronto Raptors,TOTAL,Over 221.5,-112.0,0.617017,0.088715,0.25,-161.0
8,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,250.0,0.362708,0.088461,0.25,176.0
9,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,610.0,0.219779,0.084461,0.25,355.0



=== SLATE DETAIL (CORE) ===



,home_team,away_team,home_spread,home_spread_price,total_line,total_price_used,home_ml_price,away_ml_price,home_ppg,away_ppg,home_oppg,away_oppg,home_inj_pts,away_inj_pts,proj_margin_raw,proj_margin_srs,market_margin,proj_margin,home_cover_prob,spread_edge,spread_units,spread_fair,over_prob,total_edge,total_units,total_fair,home_win_prob,home_ml_edge_novig,home_ml_units,home_ml_fair,away_win_prob,away_ml_edge_novig,away_ml_units,away_ml_fair
0,Charlotte Hornets,Dallas Mavericks,-13.0,-114.0,227.5,-114.0,-720.0,520.0,116.049180,113.900000,113.032780,117.666664,0.0,0.0,-1.242352,0.557648,13.0,9.733883,0.410893,-0.121818,0.00,143.0,0.559084,0.026374,0.25,-127.0,0.748986,-0.095829,0.00,-298.0,0.251014,0.095829,0.25,298.0
1,Cleveland Cavaliers,Detroit Pistons,2.5,-114.0,228.5,-112.0,120.0,-142.0,119.209680,117.305084,114.983870,109.389830,0.0,0.0,3.749318,5.549318,-2.5,-0.387054,0.557929,0.025219,0.25,-126.0,0.540752,0.012450,0.00,-118.0,0.489352,0.052844,0.25,104.0,0.510648,-0.052844,0.00,-104.0
2,Orlando Magic,Washington Wizards,-16.0,-110.0,227.5,-110.0,-1350.0,810.0,114.576270,112.266670,114.440680,123.000000,0.0,0.0,-3.124860,-1.324860,16.0,11.500000,0.378149,-0.145660,0.00,164.0,0.596503,0.072693,0.25,-148.0,0.786141,-0.108289,0.00,-368.0,0.213859,0.108289,0.25,368.0
3,Miami Heat,Brooklyn Nets,-13.0,-112.0,225.5,-112.0,-700.0,500.0,119.836070,106.950000,116.983604,115.433334,0.0,0.0,7.218170,9.018170,13.0,11.954770,0.471267,-0.057035,0.00,112.0,0.585455,0.057153,0.25,-141.0,0.795163,-0.044837,0.00,-388.0,0.204837,0.044837,0.25,388.0
4,Toronto Raptors,New York Knicks,2.5,-108.0,221.5,-112.0,122.0,-144.0,113.966670,117.213110,112.000000,111.131150,0.0,0.0,-1.188795,0.611205,-2.5,-1.683309,0.522458,0.003227,0.00,-109.0,0.617017,0.088715,0.25,-161.0,0.453791,0.020921,0.25,120.0,0.546209,-0.020921,0.00,-120.0
5,Chicago Bulls,Oklahoma City Thunder,10.0,-110.0,230.5,-110.0,315.0,-400.0,115.803276,119.161290,120.065575,107.870964,0.0,0.0,4.418298,6.218298,-10.0,-5.742697,0.615471,0.091661,0.25,-160.0,0.519950,-0.003859,0.00,-108.0,0.346035,0.114553,0.25,189.0,0.653965,-0.114553,0.00,-189.0
6,Minnesota Timberwolves,Memphis Grizzlies,-14.5,-106.0,236.5,-118.0,-900.0,610.0,119.131150,115.711860,114.573770,117.677960,0.0,0.0,0.157550,1.957550,14.5,11.207607,0.410188,-0.104375,0.00,144.0,0.438252,-0.103032,0.00,128.0,0.780221,-0.084461,0.00,-355.0,0.219779,0.084461,0.25,355.0
7,Philadelphia 76ers,San Antonio Spurs,8.0,-110.0,232.5,-108.0,250.0,-310.0,116.383330,117.966670,115.916664,111.783330,0.0,0.0,1.274997,3.074997,-8.0,-5.092813,0.579454,0.055644,0.25,-138.0,0.469060,-0.050170,0.00,113.0,0.362708,0.088461,0.25,176.0,0.637292,-0.088461,0.00,-176.0
8,Los Angeles Lakers,New Orleans Pelicans,-8.5,-112.0,242.5,-114.0,-370.0,295.0,115.916664,115.080640,115.433334,120.354836,0.0,0.0,-2.042739,-0.242739,8.5,6.205031,0.437121,-0.091181,0.00,129.0,0.315852,-0.216858,0.00,217.0,0.665650,-0.091016,0.00,-199.0,0.334350,0.091016,0.25,199.0
9,Sacramento Kings,Phoenix Suns,10.0,-114.0,223.5,-108.0,310.0,-390.0,110.290320,112.116670,121.225810,111.350000,0.0,0.0,4.024730,5.824730,-10.0,-5.846008,0.612746,0.080035,0.25,-158.0,0.583195,0.063964,0.25,-140.0,0.343410,0.108848,0.25,191.0,0.656590,-0.108848,0.00,-191.0



=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

Oklahoma City Thunder at Chicago Bulls
ML: Chicago Bulls ML (+315) — 0.25u
Model%: 34.6% | Edge: 11.5% | Fair: +189

Phoenix Suns at Sacramento Kings
ML: Sacramento Kings ML (+310) — 0.25u
Model%: 34.3% | Edge: 10.9% | Fair: +191

Washington Wizards at Orlando Magic
ML: Washington Wizards ML (+810) — 0.25u
Model%: 21.4% | Edge: 10.8% | Fair: +368

Oklahoma City Thunder at New York Knicks
TOTAL: Over 221.5 (-112) — 0.25u
Model%: 62.8% | Edge: 9.9% | Fair: -169

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+520) — 0.25u
Model%: 25.1% | Edge: 9.6% | Fair: +298

Oklahoma City Thunder at Chicago Bulls
SPREAD: Chicago Bulls +10 (-110) — 0.25u
Model%: 61.5% | Edge: 9.2% | Fair: -160

New Orleans Pelicans at Los Angeles Lakers
ML: New Orleans Pelicans ML (+295) — 0.25u
Model%: 33.4% | Edge: 9.1% | Fair: +199

New York Knicks at Toronto Raptors
TOTAL: Over 221.5 (-112) — 0.25u
Model%: 61.7% | Edge: 8.9% | Fair: -161

Sa

In [11]:
# ==========================================
# INJURY LAYER (FAST): ESPN injuries JSON -> injury_df
# Output columns: home_team, away_team, home_inj_pts, away_inj_pts
# Uses: games_df
# ==========================================

import requests, pandas as pd, numpy as np, re

def _clean_team(x):
    if x is None: return x
    x = str(x).replace("\xa0"," ").strip()
    mapping = {"LA Clippers":"Los Angeles Clippers","LA Lakers":"Los Angeles Lakers","NY Knicks":"New York Knicks"}
    return mapping.get(x, x)

def pull_espn_injuries(timeout=30):
    headers = {"User-Agent":"Mozilla/5.0"}
    urls = [
        "https://site.web.api.espn.com/apis/v2/sports/basketball/nba/injuries",
        "https://site.api.espn.com/apis/v2/sports/basketball/nba/injuries",
    ]
    data = None
    last = None
    for u in urls:
        try:
            r = requests.get(u, headers=headers, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            break
        except Exception as e:
            last = e
    if data is None:
        raise RuntimeError(f"ESPN injuries failed. Last error: {repr(last)}")
    return data

def injuries_to_team_points(data):
    # heuristic weights (fast + stable)
    # OUT = full weight, DTD = partial, PROBABLE = small
    status_w = {
        "out": 1.00,
        "doubtful": 0.75,
        "questionable": 0.45,
        "day-to-day": 0.35,
        "probable": 0.15,
    }
    # optional: bump "star" names if you want later; keeping simple & stable for now
    base_pts = 0.50  # each injury "slot" in points; capped later

    rows = []
    # ESPN structure varies; handle common patterns
    def walk(obj):
        if isinstance(obj, dict):
            if "team" in obj and ("injuries" in obj or "athletes" in obj):
                rows.append(obj)
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for v in obj:
                walk(v)

    walk(data)

    team_points = {}
    for item in rows:
        team = item.get("team") or {}
        tname = _clean_team(team.get("displayName") or team.get("name") or team.get("shortDisplayName"))
        if not tname:
            continue

        inj_list = item.get("injuries") or item.get("athletes") or []
        pts = 0.0

        for inj in inj_list:
            # status fields vary
            st = (inj.get("status") or inj.get("type") or inj.get("injuryStatus") or "").strip().lower()
            if isinstance(inj.get("status"), dict):
                st = (inj["status"].get("name") or inj["status"].get("type") or "").strip().lower()

            # normalize a few common ESPN values
            if "day-to-day" in st or st == "dtd":
                st = "day-to-day"
            elif "questionable" in st:
                st = "questionable"
            elif "doubtful" in st:
                st = "doubtful"
            elif "out" in st:
                st = "out"
            elif "probable" in st:
                st = "probable"
            else:
                # if we can't classify, treat as small DTD
                st = "day-to-day"

            pts += base_pts * status_w.get(st, 0.35)

        # cap team penalty (your v2 style)
        pts = float(np.clip(pts, 0.0, 3.0))
        team_points[tname] = -pts  # negative points = hurts team

    return team_points

inj_data = pull_espn_injuries()
team_inj_pts = injuries_to_team_points(inj_data)

inj_rows = []
for _, r in games_df.iterrows():
    ht = _clean_team(r["home_team"])
    at = _clean_team(r["away_team"])
    inj_rows.append({
        "home_team": ht,
        "away_team": at,
        "home_inj_pts": float(team_inj_pts.get(ht, 0.0)),
        "away_inj_pts": float(team_inj_pts.get(at, 0.0)),
    })

injury_df = pd.DataFrame(inj_rows)

print("Injury layer attached (ESPN JSON, capped):")
display(injury_df)

RuntimeError: ESPN injuries failed. Last error: HTTPError('404 Client Error: Not Found for url: https://site.api.espn.com/apis/v2/sports/basketball/nba/injuries')

In [12]:
# ==========================================
# INJURY LAYER (ROBUST): ESPN multi-route -> Rotowire fallback
# Produces: injury_df with columns
#   home_team, away_team, home_inj_pts, away_inj_pts
# Uses: games_df already pulled
# ==========================================

import requests, pandas as pd, numpy as np, re
from datetime import datetime

def _clean_team(x):
    if x is None: return x
    x = str(x).replace("\xa0"," ").strip()
    mapping = {
        "LA Clippers":"Los Angeles Clippers",
        "LA Lakers":"Los Angeles Lakers",
        "NY Knicks":"New York Knicks",
        "GS Warriors":"Golden State Warriors",
    }
    return mapping.get(x, x)

# --------- SOURCE 1: ESPN (multi-route, different APIs) ----------
def _try_get_json(url, timeout=30):
    headers = {"User-Agent":"Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

def pull_injuries_espn_multi(timeout=30):
    # ESPN has multiple hosts; some return 404 intermittently.
    urls = [
        "https://site.web.api.espn.com/apis/v2/sports/basketball/nba/injuries",
        "https://site.web.api.espn.com/apis/v2/sports/basketball/nba/injuries?region=us&lang=en",
        "https://sports.core.api.espn.com/v2/sports/basketball/leagues/nba/injuries",
        "https://sports.core.api.espn.com/v2/sports/basketball/leagues/nba/injuries?limit=1000",
    ]
    last = None
    for u in urls:
        try:
            data = _try_get_json(u, timeout=timeout)
            return ("ESPN", data)
        except Exception as e:
            last = e
    raise RuntimeError(f"ESPN multi-route injuries failed. Last error: {repr(last)}")

# --------- SOURCE 2: RotoWire JSON (very reliable) ----------
def pull_injuries_rotowire(timeout=30):
    # Rotowire public JSON used by many widgets
    url = "https://www.rotowire.com/basketball/tables/injury-report.php?site=DraftKings&format=json"
    headers = {"User-Agent":"Mozilla/5.0", "Referer":"https://www.rotowire.com/basketball/injury-report.php"}
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    return ("RotoWire", r.json())

# --------- Parse to team->points ----------
def team_points_from_any(source_name, data):
    # weights: OUT > DOUBTFUL > Q > DTD > PROB
    status_w = {
        "out": 1.00,
        "o": 1.00,
        "doubtful": 0.75,
        "d": 0.75,
        "questionable": 0.45,
        "q": 0.45,
        "day-to-day": 0.35,
        "dtd": 0.35,
        "probable": 0.15,
        "p": 0.15,
    }
    base_pts = 0.50  # each "slot"
    cap = 3.00       # cap per team
    team_points = {}

    def add(team, status):
        team = _clean_team(team)
        st = (status or "").strip().lower()
        if st in ("", "na", "n/a"):
            st = "day-to-day"

        # normalize
        if "day-to-day" in st or st == "dtd":
            st = "day-to-day"
        elif "questionable" in st:
            st = "questionable"
        elif "doubtful" in st:
            st = "doubtful"
        elif "out" in st:
            st = "out"
        elif "probable" in st:
            st = "probable"

        team_points[team] = team_points.get(team, 0.0) + base_pts * status_w.get(st, 0.35)

    if source_name == "RotoWire":
        # Expect keys like: {"data":[{"team":"...","injury_status":"Out"...}, ...]}
        rows = data.get("data") or data.get("rows") or data.get("players") or []
        for r in rows:
            team = r.get("team") or r.get("Team") or r.get("tm")
            status = r.get("injury_status") or r.get("status") or r.get("Status")
            if team:
                add(team, status)

    else:
        # ESPN structure varies, so walk object and pick out likely injury items.
        # We look for dicts that contain a team + an injury list.
        def walk(obj):
            if isinstance(obj, dict):
                # ESPN often uses "team" dict and "injuries"/"athletes" arrays
                if "team" in obj and any(k in obj for k in ("injuries", "athletes", "items")):
                    team = obj.get("team") or {}
                    team_name = team.get("displayName") or team.get("name") or team.get("shortDisplayName")
                    inj_list = obj.get("injuries") or obj.get("athletes") or obj.get("items") or []
                    if team_name and isinstance(inj_list, list):
                        for inj in inj_list:
                            st = None
                            if isinstance(inj, dict):
                                if isinstance(inj.get("status"), dict):
                                    st = inj["status"].get("name") or inj["status"].get("type")
                                else:
                                    st = inj.get("status") or inj.get("type") or inj.get("injuryStatus")
                            add(team_name, st)
                for v in obj.values():
                    walk(v)
            elif isinstance(obj, list):
                for v in obj:
                    walk(v)

        walk(data)

    # convert to negative penalties + cap
    for t, pts in list(team_points.items()):
        team_points[t] = float(-np.clip(pts, 0.0, cap))

    return team_points

# --------- Build injury_df for current games_df ----------
def build_injury_df(games_df):
    # 1) try ESPN, 2) fallback Rotowire
    source = None
    data = None
    try:
        source, data = pull_injuries_espn_multi()
    except Exception as e1:
        print("ESPN injuries failed -> trying RotoWire. Reason:", repr(e1))
        source, data = pull_injuries_rotowire()

    team_inj_pts = team_points_from_any(source, data)

    rows = []
    for _, r in games_df.iterrows():
        ht = _clean_team(r["home_team"])
        at = _clean_team(r["away_team"])
        rows.append({
            "home_team": ht,
            "away_team": at,
            "home_inj_pts": float(team_inj_pts.get(ht, 0.0)),
            "away_inj_pts": float(team_inj_pts.get(at, 0.0)),
        })
    df = pd.DataFrame(rows)

    print(f"Injury source used: {source}")
    print("Injury layer attached (sample):")
    display(df.head(12))
    return df

injury_df = build_injury_df(games_df)

ESPN injuries failed -> trying RotoWire. Reason: RuntimeError("ESPN multi-route injuries failed. Last error: HTTPError('404 Client Error: Not Found for url: https://sports.core.api.espn.com/v2/sports/basketball/leagues/nba/injuries?limit=1000')")
Injury source used: RotoWire
Injury layer attached (sample):


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,0.0,0.0
1,Cleveland Cavaliers,Detroit Pistons,0.0,0.0
2,Orlando Magic,Washington Wizards,0.0,0.0
3,Miami Heat,Brooklyn Nets,0.0,0.0
4,Toronto Raptors,New York Knicks,0.0,0.0
5,Chicago Bulls,Oklahoma City Thunder,0.0,0.0
6,Minnesota Timberwolves,Memphis Grizzlies,0.0,0.0
7,Philadelphia 76ers,San Antonio Spurs,0.0,0.0
8,Los Angeles Lakers,New Orleans Pelicans,0.0,0.0
9,Sacramento Kings,Phoenix Suns,0.0,0.0


In [13]:
# ==========================================
# INJURY LAYER FIX (ROTOIRE): robust endpoint + auto-key detect + abbrev->full map
# Produces: injury_df (home_team, away_team, home_inj_pts, away_inj_pts)
# ==========================================

import requests, pandas as pd, numpy as np

TEAM_ABBR_TO_FULL = {
    "ATL":"Atlanta Hawks","BOS":"Boston Celtics","BKN":"Brooklyn Nets","BRK":"Brooklyn Nets",
    "CHA":"Charlotte Hornets","CHO":"Charlotte Hornets","CHI":"Chicago Bulls","CLE":"Cleveland Cavaliers",
    "DAL":"Dallas Mavericks","DEN":"Denver Nuggets","DET":"Detroit Pistons","GSW":"Golden State Warriors",
    "GS":"Golden State Warriors","HOU":"Houston Rockets","IND":"Indiana Pacers","LAC":"Los Angeles Clippers",
    "LAL":"Los Angeles Lakers","MEM":"Memphis Grizzlies","MIA":"Miami Heat","MIL":"Milwaukee Bucks",
    "MIN":"Minnesota Timberwolves","NOP":"New Orleans Pelicans","NO":"New Orleans Pelicans",
    "NYK":"New York Knicks","NY":"New York Knicks","OKC":"Oklahoma City Thunder","ORL":"Orlando Magic",
    "PHI":"Philadelphia 76ers","PHX":"Phoenix Suns","PHO":"Phoenix Suns","POR":"Portland Trail Blazers",
    "SAC":"Sacramento Kings","SAS":"San Antonio Spurs","SA":"San Antonio Spurs",
    "TOR":"Toronto Raptors","UTA":"Utah Jazz","WAS":"Washington Wizards"
}

def _clean_team(x):
    if x is None: return None
    x = str(x).replace("\xa0"," ").strip()
    mapping = {"LA Clippers":"Los Angeles Clippers","LA Lakers":"Los Angeles Lakers","NY Knicks":"New York Knicks"}
    return mapping.get(x, x)

def pull_rotowire_json(timeout=30):
    headers = {"User-Agent":"Mozilla/5.0", "Referer":"https://www.rotowire.com/basketball/injury-report.php"}
    urls = [
        "https://www.rotowire.com/basketball/tables/injury-report.php?format=json",
        "https://www.rotowire.com/basketball/tables/injury-report.php?site=DraftKings&format=json",
        "https://www.rotowire.com/basketball/tables/injury-report.php?site=FanDuel&format=json",
    ]
    last = None
    for url in urls:
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()
            j = r.json()
            # common containers
            rows = j.get("data") or j.get("rows") or j.get("players") or []
            if isinstance(rows, list) and len(rows) > 0:
                return j, url
            last = f"Empty rows from {url}"
        except Exception as e:
            last = repr(e)
    raise RuntimeError(f"Rotowire JSON failed or empty. Last: {last}")

def detect_keys(rows):
    # find which keys look like team + status
    team_candidates = ["team","Team","tm","abbr","abbrev","team_abbr","teamAbbr","team_short","teamShort"]
    status_candidates = ["injury_status","status","Status","injuryStatus","inj_status","injStatus","injury","gameStatus"]

    sample = rows[:50]
    keys = set()
    for r in sample:
        if isinstance(r, dict):
            keys.update(r.keys())

    team_key = next((k for k in team_candidates if k in keys), None)
    status_key = next((k for k in status_candidates if k in keys), None)

    # If still None, brute: pick key with lots of short strings for team,
    # and key with values containing Out/Questionable/Probable/Doubtful.
    if team_key is None:
        best = None
        for k in keys:
            vals = [r.get(k) for r in sample if isinstance(r, dict)]
            strs = [v for v in vals if isinstance(v, str)]
            if len(strs) < 10:
                continue
            # many NBA team abbrevs are 2-3 chars
            short_rate = np.mean([2 <= len(s.strip()) <= 4 for s in strs])
            if best is None or short_rate > best[0]:
                best = (short_rate, k)
        team_key = best[1] if best else None

    if status_key is None:
        best = None
        for k in keys:
            vals = [r.get(k) for r in sample if isinstance(r, dict)]
            strs = [v for v in vals if isinstance(v, str)]
            if len(strs) < 10:
                continue
            hit = np.mean([any(w in s.lower() for w in ["out","questionable","doubtful","probable","day-to-day","dtd"]) for s in strs])
            if best is None or hit > best[0]:
                best = (hit, k)
        status_key = best[1] if best else None

    return team_key, status_key, sorted(list(keys))[:60]

def build_team_injury_points(rows, team_key, status_key):
    # weights (fast + stable)
    status_w = {
        "out": 1.00,
        "doubtful": 0.75,
        "questionable": 0.45,
        "day-to-day": 0.35,
        "dtd": 0.35,
        "probable": 0.15,
    }
    base_pts = 0.50
    cap = 3.0

    team_pts = {}

    for r in rows:
        if not isinstance(r, dict):
            continue
        team_val = r.get(team_key)
        status_val = r.get(status_key)

        if team_val is None:
            continue

        team_str = str(team_val).strip()
        # map abbreviations to full team names if needed
        team_full = TEAM_ABBR_TO_FULL.get(team_str.upper(), team_str)
        team_full = _clean_team(team_full)

        st = (str(status_val).strip().lower() if status_val is not None else "day-to-day")

        # normalize
        if "day-to-day" in st or st == "dtd":
            st = "day-to-day"
        elif "questionable" in st:
            st = "questionable"
        elif "doubtful" in st:
            st = "doubtful"
        elif "out" in st:
            st = "out"
        elif "probable" in st:
            st = "probable"
        else:
            # if unknown but present, treat as small DTD
            st = "day-to-day"

        team_pts[team_full] = team_pts.get(team_full, 0.0) + base_pts * status_w.get(st, 0.35)

    # cap + negative penalty
    for t in list(team_pts.keys()):
        team_pts[t] = float(-np.clip(team_pts[t], 0.0, cap))

    return team_pts

# ---- RUN ----
rw_json, rw_url = pull_rotowire_json()
rows = rw_json.get("data") or rw_json.get("rows") or rw_json.get("players") or []
team_key, status_key, keys_preview = detect_keys(rows)

print("Rotowire endpoint used:", rw_url)
print("Rows:", len(rows))
print("Detected keys -> team_key:", team_key, "| status_key:", status_key)
print("Key preview:", keys_preview)

# quick sanity preview
preview = pd.DataFrame([r for r in rows[:8] if isinstance(r, dict)])
print("\nSample rows (first 8):")
display(preview)

team_inj_pts = build_team_injury_points(rows, team_key, status_key)

# build injury_df for the slate
inj_rows = []
for _, r in games_df.iterrows():
    ht = _clean_team(r["home_team"])
    at = _clean_team(r["away_team"])
    inj_rows.append({
        "home_team": ht,
        "away_team": at,
        "home_inj_pts": float(team_inj_pts.get(ht, 0.0)),
        "away_inj_pts": float(team_inj_pts.get(at, 0.0)),
    })

injury_df = pd.DataFrame(inj_rows)

print("\nInjury layer attached (slate):")
display(injury_df)

# sanity: show any non-zero teams
nz = injury_df[(injury_df["home_inj_pts"] != 0) | (injury_df["away_inj_pts"] != 0)]
print("\nNon-zero injury rows:", len(nz))
if len(nz):
    display(nz)
else:
    print("Still all zeros -> Rotowire may be returning rows without team/status fields for your region/session.")
    print("If so: we switch to an Underdog/NBA injury report source next.")

RuntimeError: Rotowire JSON failed or empty. Last: Empty rows from https://www.rotowire.com/basketball/tables/injury-report.php?site=FanDuel&format=json

In [14]:
# ==========================================
# INJURY LAYER (DRAFTKINGS PUBLIC FEED) — reliable in Colab
# Produces: injury_df with columns:
#   home_team, away_team, home_inj_pts, away_inj_pts
# Requires: games_df already exists
# ==========================================

import requests, pandas as pd, numpy as np

def _clean_team(x):
    if x is None: return None
    x = str(x).replace("\xa0"," ").strip()
    mapping = {"LA Clippers":"Los Angeles Clippers","LA Lakers":"Los Angeles Lakers","NY Knicks":"New York Knicks"}
    return mapping.get(x, x)

# weights and caps (same spirit as your pro layer)
STATUS_W = {
    "Out": 1.00,
    "Doubtful": 0.75,
    "Questionable": 0.45,
    "Probable": 0.15,
    "Active": 0.00,
}
BASE_PTS = 0.50
CAP = 3.0

def pull_dk_injuries(timeout=30):
    # DK "injuries" endpoint (public). If this 403s, we add headers + alt route below.
    urls = [
        "https://sportsbook.draftkings.com/sites/US-SB/api/v5/eventgroups/42648/injuries",
        "https://sportsbook.draftkings.com/sites/US-SB/api/v4/eventgroups/42648/injuries",
    ]
    headers = {"User-Agent":"Mozilla/5.0", "Referer":"https://sportsbook.draftkings.com/leagues/basketball/nba"}

    last = None
    for u in urls:
        try:
            r = requests.get(u, headers=headers, timeout=timeout)
            r.raise_for_status()
            return r.json(), u
        except Exception as e:
            last = repr(e)

    raise RuntimeError(f"DraftKings injuries failed. Last: {last}")

def dk_team_id_to_name(dk_json):
    # DK payload contains team entities with names in "team" collections
    id_to_name = {}
    # common places team objects appear:
    for key in ["teams", "teamInfo", "team", "teamCategories", "eventGroup", "sports"]:
        # we’ll just do a deep walk and capture any dict with (id, name)
        pass

    def walk(o):
        if isinstance(o, dict):
            # team-like objects
            if "id" in o and ("name" in o or "fullName" in o):
                tid = o.get("id")
                nm = o.get("fullName") or o.get("name")
                if isinstance(tid, int) and isinstance(nm, str) and len(nm) >= 3:
                    # filter out clearly non-team names
                    if "NBA" not in nm and "Injuries" not in nm and "Basketball" not in nm:
                        id_to_name[tid] = _clean_team(nm)
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)
    walk(dk_json)
    return id_to_name

def build_team_injury_points_from_dk(dk_json, id_to_name):
    team_pts = {}

    # DK injuries list often at dk_json["injuries"] or nested. We'll walk and detect injury rows.
    def walk(o):
        if isinstance(o, dict):
            # candidate injury row
            # look for teamId + status fields
            keys = set(o.keys())
            if ("teamId" in keys or "team_id" in keys) and ("status" in keys or "injuryStatus" in keys):
                tid = o.get("teamId", o.get("team_id"))
                st = o.get("status") or o.get("injuryStatus")
                if isinstance(tid, int):
                    team = id_to_name.get(tid)
                    if team:
                        st_norm = str(st).strip()
                        w = STATUS_W.get(st_norm, 0.35)  # unknown -> DTD-ish
                        team_pts[team] = team_pts.get(team, 0.0) + BASE_PTS * w

            # some payloads use different naming
            if ("team" in keys or "teamName" in keys) and ("status" in keys or "injuryStatus" in keys):
                team = o.get("teamName") or o.get("team")
                st = o.get("status") or o.get("injuryStatus")
                if isinstance(team, str) and team:
                    team = _clean_team(team)
                    st_norm = str(st).strip()
                    w = STATUS_W.get(st_norm, 0.35)
                    team_pts[team] = team_pts.get(team, 0.0) + BASE_PTS * w

            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)

    walk(dk_json)

    # cap + negative penalty
    for t in list(team_pts.keys()):
        team_pts[t] = float(-np.clip(team_pts[t], 0.0, CAP))

    return team_pts

# ---- RUN ----
dk_json, dk_url = pull_dk_injuries()
print("Injury source used: DraftKings feed")
print("Endpoint:", dk_url)

id_to_name = dk_team_id_to_name(dk_json)
team_inj_pts = build_team_injury_points_from_dk(dk_json, id_to_name)

# Build injury_df for slate
rows = []
for _, r in games_df.iterrows():
    ht = _clean_team(r["home_team"])
    at = _clean_team(r["away_team"])
    rows.append({
        "home_team": ht,
        "away_team": at,
        "home_inj_pts": float(team_inj_pts.get(ht, 0.0)),
        "away_inj_pts": float(team_inj_pts.get(at, 0.0)),
    })

injury_df = pd.DataFrame(rows)
display(injury_df)

nz = injury_df[(injury_df.home_inj_pts != 0) | (injury_df.away_inj_pts != 0)]
print("Non-zero injury rows:", len(nz))
if len(nz):
    display(nz)
else:
    print("Still all zeros. That means DK feed returned no injury rows OR parsing keys differ today.")
    print("Next step: print dk_json top-level keys so we can target the correct injury array without hunting:")
    print("Top-level keys:", list(dk_json.keys())[:50])

RuntimeError: DraftKings injuries failed. Last: HTTPError('403 Client Error: Forbidden for url: https://sportsbook.draftkings.com/sites/US-SB/api/v4/eventgroups/42648/injuries')

In [15]:
# ============================================
# NBA STABLE — FULL RERUN CELL (BOTTOM)
# Markets (Odds API) + Team Stats (ESPN JSON) + Injuries (Official NBA PDF)
# Projection layers -> Probabilities -> EV/Edge -> Kelly -> Outputs
# ============================================

import os, re, math, json, time
from datetime import datetime, timezone
from io import BytesIO
import requests
import numpy as np
import pandas as pd

# -----------------------
# 0) CONFIG
# -----------------------
# REQUIRED: 10ceac94f9b9cb76f8c65510ca6df18f
ODDS_API_KEY = os.environ.get("ODDS_API_KEY", "").strip()
# If you want to hardcode for this runtime, uncomment:
# ODDS_API_KEY = "PASTE_YOUR_KEY"

if not ODDS_API_KEY:
    raise RuntimeError("ODDS_API_KEY is not set. Set os.environ['ODDS_API_KEY'] or paste into this cell.")

SPORT_KEY = "basketball_nba"
REGIONS = "us"
BOOKMAKERS = ""          # optional: "draftkings,fanduel" etc. Leave blank for all.
MARKETS = "spreads,totals,h2h"
ODDS_FMT = "american"

# Model knobs (keep simple/fast + stable)
SIGMA_SPREAD = 12.0      # spread outcome std dev (pts) for cover probs
SIGMA_TOTAL  = 17.0      # total outcome std dev (pts) for over/under probs
HOME_COURT_PTS = 1.8     # home-court baseline
MARGIN_BLEND_ALPHA = 0.65  # 0..1 -> how much we trust ppg/oppg margin vs market-implied
INJURY_CAP = -3.0        # cap negative injury adjustment per team (pts)

# Injury weights -> convert player statuses to team points adjustment (negative = worse)
STATUS_PENALTY = {
    "Out": -0.55,
    "Doubtful": -0.35,
    "Questionable": -0.20,
    "Probable": -0.10,
    "Available": 0.0,
    "Active": 0.0,
    "Not Listed": 0.0,
}

# Rotation-weighted "v2" caps (optional; keeps it from going crazy)
# If you later add player tiers, you can scale these instead of flat weights.
ROTATION_CAPS = {"Out": -3.0, "Doubtful": -2.0, "Questionable": -1.5, "Probable": -1.0}

# Outputs
ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
MARKET_SNAP_LATEST = "market_snapshot_latest.csv"
MARKET_SNAP_TS     = f"market_snapshot_{ts}.csv"
OUT_PLAYS_CSV      = "nba_card_plays.csv"
OUT_DISCORD_TXT    = "nba_discord_card.txt"

# -----------------------
# 1) HELPERS
# -----------------------
def req_get(url, params=None, headers=None, timeout=30, tries=3, backoff=1.4):
    last = None
    for i in range(tries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep((backoff ** i) + 0.2)
    raise last

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    odds = float(odds)
    if odds < 0:
        return (-odds) / ((-odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p):
    # fair odds from probability
    if p <= 0 or p >= 1: return np.nan
    if p > 0.5:
        return -round(100 * p / (1 - p))
    return round(100 * (1 - p) / p)

def no_vig_probs_from_two_sided(p_a, p_b):
    # normalize to remove vig
    s = p_a + p_b
    if s <= 0: return (np.nan, np.nan)
    return (p_a / s, p_b / s)

def kelly_fraction_winprob(p, odds_american):
    # Returns Kelly fraction (can be negative) for a 1-unit bankroll
    if pd.isna(p) or pd.isna(odds_american): return 0.0
    p = float(p); odds_american = float(odds_american)
    if odds_american > 0:
        b = odds_american / 100.0
    else:
        b = 100.0 / (-odds_american)
    q = 1 - p
    f = (b*p - q) / b
    if np.isnan(f) or np.isinf(f): return 0.0
    return f

def clamp(x, lo, hi):
    return float(max(lo, min(hi, x)))

def norm_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2)))

def cover_prob_from_margin(proj_margin, spread_line, sigma=SIGMA_SPREAD):
    # proj_margin = home - away expected margin
    # spread_line = home spread (negative means favorite)
    # P( home_margin + noise > -spread_line ) i.e., home covers its spread
    # Home covers if (home_margin - (-spread_line)) > 0 => home_margin + spread_line > 0
    x = (proj_margin + float(spread_line)) / float(sigma)
    return norm_cdf(x)

def over_prob_from_total(proj_total, total_line, sigma=SIGMA_TOTAL):
    x = (float(proj_total) - float(total_line)) / float(sigma)
    return norm_cdf(x)

# -----------------------
# 2) ODDS API — MARKET SNAPSHOT
# -----------------------
def pull_market_data():
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": ODDS_FMT,
    }
    if BOOKMAKERS:
        params["bookmakers"] = BOOKMAKERS

    r = req_get(url, params=params, timeout=30, tries=3)
    data = r.json()

    rows = []
    for ev in data:
        home = ev.get("home_team")
        away = ev.get("away_team")

        # collect best/consensus across books quickly
        home_spread = home_spread_price = total_line = total_price_used = None
        home_ml = away_ml = None

        # We’ll take the first book that has each market, and also compute a simple consensus for spread.
        spread_lines = []
        for bm in ev.get("bookmakers", []):
            for mk in bm.get("markets", []):
                key = mk.get("key")
                outs = mk.get("outcomes", [])
                if key == "spreads" and outs:
                    # find home outcome
                    for o in outs:
                        if o.get("name") == home:
                            home_spread = o.get("point")
                            home_spread_price = o.get("price")
                            spread_lines.append(float(o.get("point")))
                elif key == "totals" and outs:
                    # totals outcomes use name: Over/Under
                    for o in outs:
                        if (o.get("name") == "Over"):
                            total_line = o.get("point")
                            total_price_used = o.get("price")
                elif key == "h2h" and outs:
                    for o in outs:
                        if o.get("name") == home:
                            home_ml = o.get("price")
                        elif o.get("name") == away:
                            away_ml = o.get("price")

        spread_consensus = np.nan
        if spread_lines:
            spread_consensus = float(np.median(spread_lines))

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_spread": float(home_spread) if home_spread is not None else np.nan,
            "home_spread_price": float(home_spread_price) if home_spread_price is not None else np.nan,
            "spread_consensus": spread_consensus,
            "total_line": float(total_line) if total_line is not None else np.nan,
            "total_price_used": float(total_price_used) if total_price_used is not None else np.nan,
            "home_ml_price": float(home_ml) if home_ml is not None else np.nan,
            "away_ml_price": float(away_ml) if away_ml is not None else np.nan,
        })

    df = pd.DataFrame(rows)
    # basic cleanup
    df = df.dropna(subset=["home_team", "away_team"]).reset_index(drop=True)

    df.to_csv(MARKET_SNAP_TS, index=False)
    df.to_csv(MARKET_SNAP_LATEST, index=False)
    print(f"Pulled market games: {len(df)}")
    print(f"Saved: {MARKET_SNAP_TS} and {MARKET_SNAP_LATEST}")
    return df

games_df = pull_market_data()
display(games_df.head(12))

# -----------------------
# 3) TEAM STATS — ESPN JSON (avgPointsFor/avgPointsAgainst)
# -----------------------
def pull_team_stats_espn_json(timeout=25):
    """
    ESPN standings endpoint exposes avgPointsFor / avgPointsAgainst.
    Works even when BRef blocks.
    """
    urls = [
        "https://site.api.espn.com/apis/v2/sports/basketball/nba/standings",
        "https://site.api.espn.com/apis/v2/sports/basketball/nba/standings?region=us&lang=en",
    ]
    data = None
    last = None
    for u in urls:
        try:
            r = req_get(u, timeout=timeout, tries=3)
            data = r.json()
            if data: break
        except Exception as e:
            last = e

    if data is None:
        raise RuntimeError(f"ESPN standings JSON failed. Last error: {repr(last)}")

    teams = []
    # JSON shape: "children" conferences -> "standings" -> "entries"
    def walk_entries(node):
        if isinstance(node, dict):
            if "entries" in node and isinstance(node["entries"], list):
                for e in node["entries"]:
                    yield e
            for v in node.values():
                yield from walk_entries(v)
        elif isinstance(node, list):
            for x in node:
                yield from walk_entries(x)

    entries = list(walk_entries(data))
    if not entries:
        raise RuntimeError("ESPN standings JSON had no entries")

    for e in entries:
        team = (e.get("team") or {}).get("displayName") or (e.get("team") or {}).get("name")
        if not team:
            continue

        stats = e.get("stats", [])
        # stats are list[dict] with "name" keys
        d = {}
        for s in stats:
            k = s.get("name") or s.get("abbreviation") or s.get("shortName")
            if k:
                d[k] = s.get("value")

        ppg  = d.get("avgPointsFor") or d.get("pointsFor") or d.get("points")
        oppg = d.get("avgPointsAgainst") or d.get("pointsAgainst") or d.get("avgPointsAgainst")

        if ppg is None or oppg is None:
            continue

        teams.append({"team": team, "ppg": float(ppg), "oppg": float(oppg), "srs": 0.0})

    df = pd.DataFrame(teams).drop_duplicates(subset=["team"]).reset_index(drop=True)
    if len(df) < 25:
        raise RuntimeError(f"ESPN standings parsed too few teams: {len(df)}")
    return df

team_stats = pull_team_stats_espn_json()
print("Team stats source: ESPN JSON (avgPointsFor/avgPointsAgainst)")
print("Teams pulled:", len(team_stats))
display(team_stats.head(10))

# -----------------------
# 4) INJURY LAYER — OFFICIAL NBA INJURY REPORT (PDF)
# -----------------------
def find_latest_nba_injury_pdf():
    """
    Scrape official NBA injury report season page for the latest Injury-Report_*.pdf link.
    """
    page = "https://official.nba.com/nba-injury-report-2025-26-season/"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = req_get(page, headers=headers, timeout=30, tries=3)
    html = r.text

    pdfs = re.findall(r'href="([^"]*Injury-Report_[^"]+\.pdf)"', html, flags=re.I)
    # Sometimes relative URLs; normalize
    norm = []
    for p in pdfs:
        if p.startswith("//"):
            p = "https:" + p
        elif p.startswith("/"):
            p = "https://official.nba.com" + p
        norm.append(p)

    # Also catch ak-static links embedded without href formatting
    ak = re.findall(r"(https://ak-static\.cms\.nba\.com/referee/injury/Injury-Report_[^\s\"']+\.pdf)", html, flags=re.I)
    norm.extend(ak)

    norm = list(dict.fromkeys(norm))  # unique preserve order
    if not norm:
        # last-ditch: search last 2 days common time stamps by probing is too slow; fail explicitly
        raise RuntimeError("Could not find any Injury-Report_*.pdf links on official NBA page.")

    # pick max by parsing timestamp in filename: Injury-Report_YYYY-MM-DD_HH_MM(AM|PM).pdf
    def parse_key(url):
        m = re.search(r"Injury-Report_(\d{4}-\d{2}-\d{2})_(\d{2})_(\d{2})(AM|PM)\.pdf", url)
        if not m:
            return (0, "")
        d, hh, mm, ap = m.group(1), m.group(2), m.group(3), m.group(4)
        hh = int(hh); mm = int(mm)
        if ap.upper() == "PM" and hh != 12: hh += 12
        if ap.upper() == "AM" and hh == 12: hh = 0
        dt = datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        return (int(dt.timestamp()) + hh*3600 + mm*60, url)

    best = max(norm, key=lambda u: parse_key(u)[0])
    return best

def parse_injury_pdf_to_rows(pdf_bytes):
    """
    Parse the official injury PDF to a player-level table with team + status.
    Uses pdfplumber; if unavailable, will fallback to regex on extracted text.
    """
    try:
        import pdfplumber
    except Exception as e:
        raise RuntimeError("pdfplumber not available. In Colab: pip install pdfplumber") from e

    rows = []
    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ""
            # Lines contain: "<Team> <Player> <Status> <Reason...>"
            # We'll detect status tokens in: Out, Doubtful, Questionable, Probable, Available
            for line in txt.splitlines():
                line = re.sub(r"\s+", " ", line).strip()
                if not line:
                    continue
                # skip headers
                if line.lower().startswith("injury report"):
                    continue
                # Status detection
                status = None
                for s in ["Out", "Doubtful", "Questionable", "Probable", "Available"]:
                    if re.search(rf"\b{s}\b", line):
                        status = s
                        break
                if not status:
                    continue

                # Heuristic: team is a known NBA team string at start of line, but PDFs vary.
                # We'll pull the first 2-3 words until we hit a comma in player format "Last, First"
                # Many lines are: "Boston Celtics Brown, Jaylen Out Injury/Illness - ..."
                m = re.match(r"^([A-Za-z]+(?: [A-Za-z]+){0,2}) ([A-Za-z'\-]+, [A-Za-z'\-\.]+) (Out|Doubtful|Questionable|Probable|Available)\b", line)
                if not m:
                    continue
                team, player, status = m.group(1), m.group(2), m.group(3)
                rows.append({"team": team.strip(), "player": player.strip(), "status": status.strip()})

    df = pd.DataFrame(rows)
    return df

def injury_rows_to_team_points(inj_df):
    """
    Convert player-level injury rows to team injury points (negative).
    Uses STATUS_PENALTY and caps per team.
    """
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    # normalize team names a bit (official report uses full team names already)
    inj_df = inj_df.copy()
    inj_df["status"] = inj_df["status"].fillna("Not Listed")

    inj_df["pen"] = inj_df["status"].map(STATUS_PENALTY).fillna(0.0)
    team_pts = inj_df.groupby("team", as_index=False)["pen"].sum().rename(columns={"pen": "inj_pts"})

    # cap (more negative than cap -> cap)
    team_pts["inj_pts"] = team_pts["inj_pts"].apply(lambda x: max(INJURY_CAP, float(x)))
    return team_pts

def attach_injuries_from_official_pdf(games):
    pdf_url = find_latest_nba_injury_pdf()
    print("Injury source used: Official NBA Injury Report PDF")
    print("PDF:", pdf_url)
    r = req_get(pdf_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=45, tries=3)
    inj_rows = parse_injury_pdf_to_rows(r.content)
    team_pts = injury_rows_to_team_points(inj_rows)

    g = games.copy()
    # merge by team name (home/away) -> if no match, 0
    g = g.merge(team_pts.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
    g = g.merge(team_pts.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
    g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
    g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

    return g, inj_rows, team_pts, pdf_url

games_df, inj_rows_df, team_inj_pts_df, inj_pdf_url = attach_injuries_from_official_pdf(games_df)
print("Injury layer attached (sample):")
display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(12))

# -----------------------
# 5) PROJECTION LAYERS (COMPLETE) — merge stats + project margin/total
# -----------------------
def attach_team_stats_and_project(games, team_stats):
    g = games.copy().reset_index(drop=True)

    # merge home stats
    g = g.merge(team_stats.rename(columns={"team":"home_team","ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}),
                on="home_team", how="left")
    # merge away stats
    g = g.merge(team_stats.rename(columns={"team":"away_team","ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}),
                on="away_team", how="left")

    # sanity
    for c in ["home_ppg","home_oppg","away_ppg","away_oppg"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")

    # baseline expected points (simple, fast, stable)
    # home offense vs away defense + home court, plus injury adjustment
    g["home_pts_proj"] = ((g["home_ppg"] + g["away_oppg"]) / 2.0) + (HOME_COURT_PTS/2.0) + g["home_inj_pts"]
    g["away_pts_proj"] = ((g["away_ppg"] + g["home_oppg"]) / 2.0) - (HOME_COURT_PTS/2.0) + g["away_inj_pts"]

    g["proj_total_raw"]  = g["home_pts_proj"] + g["away_pts_proj"]
    g["proj_margin_raw"] = g["home_pts_proj"] - g["away_pts_proj"]   # home - away

    # market-implied margin from spread line:
    # If home_spread is -8.5, market expects home_margin ~ +8.5
    g["market_margin"] = -pd.to_numeric(g["home_spread"], errors="coerce")

    # Blend projection margin toward market to reduce insane cover% spikes
    g["proj_margin"] = (MARGIN_BLEND_ALPHA * g["proj_margin_raw"]) + ((1.0 - MARGIN_BLEND_ALPHA) * g["market_margin"])

    # Blend total toward market a bit (optional but stabilizes)
    g["proj_total"] = (0.70 * g["proj_total_raw"]) + (0.30 * g["total_line"])

    return g

games_df = attach_team_stats_and_project(games_df, team_stats)

# -----------------------
# 6) PROBABILITIES + EDGES (SPREAD / TOTAL / ML) + NO-VIG ML
# -----------------------
def compute_market_probs_and_edges(g):
    df = g.copy().reset_index(drop=True)

    # SPREAD
    df["home_cover_prob"] = df.apply(lambda r: cover_prob_from_margin(r["proj_margin"], r["home_spread"], SIGMA_SPREAD), axis=1)
    df["spread_market_prob"] = df["home_spread_price"].apply(american_to_prob).astype(float)
    df["spread_edge"] = (df["home_cover_prob"] - df["spread_market_prob"]).astype(float)
    df["spread_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["home_cover_prob"], r["home_spread_price"]), axis=1)
    df["spread_units"] = df["spread_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))  # half-kelly, capped 0.25u for now
    df["spread_fair"] = df["home_cover_prob"].apply(prob_to_american)

    # TOTAL (Over)
    df["over_prob"] = df.apply(lambda r: over_prob_from_total(r["proj_total"], r["total_line"], SIGMA_TOTAL), axis=1)
    df["total_market_prob"] = df["total_price_used"].apply(american_to_prob).astype(float)
    df["total_edge"] = (df["over_prob"] - df["total_market_prob"]).astype(float)
    df["total_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["over_prob"], r["total_price_used"]), axis=1)
    df["total_units"] = df["total_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))
    df["total_fair"] = df["over_prob"].apply(prob_to_american)

    # ML
    # Convert proj_margin to win prob using normal approximation
    df["home_win_prob"] = df["proj_margin"].apply(lambda m: norm_cdf(float(m) / SIGMA_SPREAD)).astype(float)
    df["away_win_prob"] = (1.0 - df["home_win_prob"]).astype(float)

    df["home_ml_market_prob"] = df["home_ml_price"].apply(american_to_prob).astype(float)
    df["away_ml_market_prob"] = df["away_ml_price"].apply(american_to_prob).astype(float)

    # No-vig normalize
    novig = df.apply(lambda r: no_vig_probs_from_two_sided(r["home_ml_market_prob"], r["away_ml_market_prob"]), axis=1)
    df["home_ml_prob_novig"] = [x[0] for x in novig]
    df["away_ml_prob_novig"] = [x[1] for x in novig]

    df["home_ml_edge_novig"] = (df["home_win_prob"] - df["home_ml_prob_novig"]).astype(float)
    df["away_ml_edge_novig"] = (df["away_win_prob"] - df["away_ml_prob_novig"]).astype(float)

    df["home_ml_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["home_win_prob"], r["home_ml_price"]), axis=1)
    df["away_ml_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["away_win_prob"], r["away_ml_price"]), axis=1)

    df["home_ml_units"] = df["home_ml_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))
    df["away_ml_units"] = df["away_ml_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))

    df["home_ml_fair"] = df["home_win_prob"].apply(prob_to_american)
    df["away_ml_fair"] = df["away_win_prob"].apply(prob_to_american)

    return df

games_df = compute_market_probs_and_edges(games_df)

# -----------------------
# 7) BUILD PLAYS TABLE (ALL MARKETS) + DISCORD TEXT
# -----------------------
def build_plays(df):
    plays = []

    for _, r in df.iterrows():
        matchup = f"{r['away_team']} at {r['home_team']}"

        # ML plays (take either side if edge positive)
        # Prefer away ML if away edge bigger; same for home
        if r["home_ml_edge_novig"] > 0 and r["home_ml_units"] > 0:
            plays.append({
                "matchup": matchup,
                "market": "ML",
                "bet": f"{r['home_team']} ML",
                "odds": float(r["home_ml_price"]),
                "model_prob": float(r["home_win_prob"]),
                "edge": float(r["home_ml_edge_novig"]),
                "units": float(r["home_ml_units"]),
                "fair_odds": float(prob_to_american(r["home_win_prob"]))
            })
        if r["away_ml_edge_novig"] > 0 and r["away_ml_units"] > 0:
            plays.append({
                "matchup": matchup,
                "market": "ML",
                "bet": f"{r['away_team']} ML",
                "odds": float(r["away_ml_price"]),
                "model_prob": float(r["away_win_prob"]),
                "edge": float(r["away_ml_edge_novig"]),
                "units": float(r["away_ml_units"]),
                "fair_odds": float(prob_to_american(r["away_win_prob"]))
            })

        # SPREAD (home line only; your sheet assumes home_spread)
        if r["spread_edge"] > 0 and r["spread_units"] > 0:
            # If home_spread is positive (home underdog), bet is "home +X"
            # If home_spread is negative, bet is "home -X"
            spread_str = f"{r['home_team']} {r['home_spread']:+g}"
            plays.append({
                "matchup": matchup,
                "market": "SPREAD",
                "bet": spread_str,
                "odds": float(r["home_spread_price"]),
                "model_prob": float(r["home_cover_prob"]),
                "edge": float(r["spread_edge"]),
                "units": float(r["spread_units"]),
                "fair_odds": float(r["spread_fair"])
            })

        # TOTAL (Over only; fast)
        if r["total_edge"] > 0 and r["total_units"] > 0:
            plays.append({
                "matchup": matchup,
                "market": "TOTAL",
                "bet": f"Over {r['total_line']}",
                "odds": float(r["total_price_used"]),
                "model_prob": float(r["over_prob"]),
                "edge": float(r["total_edge"]),
                "units": float(r["total_units"]),
                "fair_odds": float(r["total_fair"])
            })

    plays_df = pd.DataFrame(plays)
    if plays_df.empty:
        return plays_df

    plays_df = plays_df.sort_values(["edge","model_prob"], ascending=[False, False]).reset_index(drop=True)
    return plays_df

plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED ===\n")
display(plays_df.head(25))

print("\n=== SLATE DETAIL (CORE) ===\n")
core_cols = [
    "home_team","away_team",
    "home_spread","home_spread_price","total_line","total_price_used","home_ml_price","away_ml_price",
    "home_ppg","away_ppg","home_oppg","away_oppg",
    "home_inj_pts","away_inj_pts",
    "proj_margin_raw","market_margin","proj_margin",
    "home_cover_prob","spread_edge","spread_units","spread_fair",
    "proj_total_raw","proj_total",
    "over_prob","total_edge","total_units","total_fair",
    "home_win_prob","home_ml_edge_novig","home_ml_units","home_ml_fair",
    "away_win_prob","away_ml_edge_novig","away_ml_units","away_ml_fair",
]
display(games_df[core_cols].head(50))

# Discord text (top 12 by default)
def to_discord(plays_df, top_n=12):
    if plays_df is None or plays_df.empty:
        return "== CDR BETTING | NBA FULL CARD ==\n\n(No +EV plays found with current thresholds.)"

    lines = ["== CDR BETTING | NBA FULL CARD =="]
    for _, r in plays_df.head(top_n).iterrows():
        lines.append("")
        lines.append(r["matchup"])
        lines.append(f"{r['market']}: {r['bet']} ({int(r['odds']):+d}) — {r['units']:.2f}u" if r["market"]=="ML"
                     else f"{r['market']}: {r['bet']} ({int(r['odds']):+d}) — {r['units']:.2f}u")
        lines.append(f"Model%: {100*r['model_prob']:.1f}% | Edge: {100*r['edge']:.1f}% | Fair: {int(r['fair_odds']):+d}")
    return "\n".join(lines)

discord_text = to_discord(plays_df, top_n=12)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

# Save outputs
plays_df.to_csv(OUT_PLAYS_CSV, index=False)
with open(OUT_DISCORD_TXT, "w", encoding="utf-8") as f:
    f.write(discord_text)

print(f"\nSaved: {OUT_PLAYS_CSV}, {OUT_DISCORD_TXT}")

Pulled market games: 12
Saved: market_snapshot_20260303_215106.csv and market_snapshot_latest.csv


,home_team,away_team,home_spread,home_spread_price,spread_consensus,total_line,total_price_used,home_ml_price,away_ml_price
0,Charlotte Hornets,Dallas Mavericks,-13.5,-110.0,-13.50,228.0,-110.0,-714.0,479.0
1,Cleveland Cavaliers,Detroit Pistons,2.5,-110.0,2.50,227.5,-110.0,117.0,-144.0
2,Orlando Magic,Washington Wizards,-16.0,-113.0,-16.00,227.0,-110.0,-1430.0,700.0
3,Miami Heat,Brooklyn Nets,-13.0,-110.0,-13.00,226.0,-110.0,-714.0,481.0
4,Toronto Raptors,New York Knicks,3.0,-110.0,2.50,221.5,-110.0,120.0,-147.0
5,Chicago Bulls,Oklahoma City Thunder,10.5,-115.0,10.00,230.5,-110.0,310.0,-435.0
6,Minnesota Timberwolves,Memphis Grizzlies,-14.5,-110.0,-14.50,237.5,-110.0,-910.0,550.0
7,Philadelphia 76ers,San Antonio Spurs,8.0,-105.0,8.00,231.5,-110.0,265.0,-330.0
8,Los Angeles Lakers,New Orleans Pelicans,-8.5,-115.0,-8.50,242.0,-110.0,-360.0,280.0
9,Sacramento Kings,Phoenix Suns,9.5,-110.0,9.50,223.5,-110.0,308.0,-385.0


Team stats source: ESPN JSON (avgPointsFor/avgPointsAgainst)
Teams pulled: 30


,team,ppg,oppg,srs
0,Indiana Pacers,111.393440,119.557380,0.0
1,Brooklyn Nets,106.950000,115.433334,0.0
2,Washington Wizards,112.266670,123.000000,0.0
3,Chicago Bulls,115.803276,120.065575,0.0
4,Milwaukee Bucks,111.333336,115.550000,0.0
5,Charlotte Hornets,116.049180,113.032780,0.0
6,Atlanta Hawks,117.387100,117.370964,0.0
7,Orlando Magic,114.576270,114.440680,0.0
8,Miami Heat,119.836070,116.983604,0.0
9,Philadelphia 76ers,116.383330,115.916664,0.0


Injury source used: Official NBA Injury Report PDF
PDF: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-03_04_00PM.pdf


RuntimeError: pdfplumber not available. In Colab: pip install pdfplumber

In [16]:
# =========================
# FIX CELL: install pdfplumber + rerun injury layer -> downstream
# Paste BELOW the crash and run
# =========================

import sys, subprocess, os, re, math
from datetime import datetime, timezone
from io import BytesIO
import numpy as np
import pandas as pd
import requests

# --- 0) Install missing dependency ---
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pdfplumber"])
import pdfplumber

# --- 1) Minimal helpers (use if not already defined) ---
def req_get(url, params=None, headers=None, timeout=30, tries=3, backoff=1.4):
    import time
    last = None
    for i in range(tries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep((backoff ** i) + 0.2)
    raise last

def norm_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2)))

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    odds = float(odds)
    if odds < 0:
        return (-odds) / ((-odds) + 100.0)
    return 100.0 / (odds + 100.0)

def prob_to_american(p):
    if p <= 0 or p >= 1: return np.nan
    if p > 0.5:
        return -round(100 * p / (1 - p))
    return round(100 * (1 - p) / p)

def no_vig_probs_from_two_sided(p_a, p_b):
    s = p_a + p_b
    if s <= 0: return (np.nan, np.nan)
    return (p_a / s, p_b / s)

def kelly_fraction_winprob(p, odds_american):
    if pd.isna(p) or pd.isna(odds_american): return 0.0
    p = float(p); odds_american = float(odds_american)
    if odds_american > 0:
        b = odds_american / 100.0
    else:
        b = 100.0 / (-odds_american)
    q = 1 - p
    f = (b*p - q) / b
    if np.isnan(f) or np.isinf(f): return 0.0
    return f

def clamp(x, lo, hi):
    return float(max(lo, min(hi, x)))

# --- 2) Pull latest official NBA injury PDF link (you already printed it, but this is robust) ---
def find_latest_nba_injury_pdf():
    page = "https://official.nba.com/nba-injury-report-2025-26-season/"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = req_get(page, headers=headers, timeout=30, tries=3)
    html = r.text

    pdfs = re.findall(r'href="([^"]*Injury-Report_[^"]+\.pdf)"', html, flags=re.I)
    norm = []
    for p in pdfs:
        if p.startswith("//"):
            p = "https:" + p
        elif p.startswith("/"):
            p = "https://official.nba.com" + p
        norm.append(p)

    ak = re.findall(r"(https://ak-static\.cms\.nba\.com/referee/injury/Injury-Report_[^\s\"']+\.pdf)", html, flags=re.I)
    norm.extend(ak)
    norm = list(dict.fromkeys(norm))
    if not norm:
        raise RuntimeError("Could not find Injury-Report_*.pdf links on official NBA page.")

    def parse_key(url):
        m = re.search(r"Injury-Report_(\d{4}-\d{2}-\d{2})_(\d{2})_(\d{2})(AM|PM)\.pdf", url)
        if not m:
            return 0
        d, hh, mm, ap = m.group(1), int(m.group(2)), int(m.group(3)), m.group(4).upper()
        if ap == "PM" and hh != 12: hh += 12
        if ap == "AM" and hh == 12: hh = 0
        dt = datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        return int(dt.timestamp()) + hh*3600 + mm*60

    return max(norm, key=parse_key)

# --- 3) Parse PDF -> rows(team, player, status) ---
def parse_injury_pdf_to_rows(pdf_bytes):
    rows = []
    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ""
            for line in txt.splitlines():
                line = re.sub(r"\s+", " ", line).strip()
                if not line:
                    continue
                # detect status token
                status = None
                for s in ["Out", "Doubtful", "Questionable", "Probable", "Available"]:
                    if re.search(rf"\b{s}\b", line):
                        status = s
                        break
                if not status:
                    continue

                # Typical line pattern:
                # "Boston Celtics Brown, Jaylen Out Injury/Illness - ..."
                m = re.match(
                    r"^([A-Za-z]+(?: [A-Za-z]+){0,2}) ([A-Za-z'\-]+, [A-Za-z'\-\.]+) (Out|Doubtful|Questionable|Probable|Available)\b",
                    line
                )
                if not m:
                    continue
                team, player, status = m.group(1), m.group(2), m.group(3)
                rows.append({"team": team.strip(), "player": player.strip(), "status": status.strip()})

    return pd.DataFrame(rows)

# --- 4) Convert to team injury points (simple + capped) ---
STATUS_PENALTY = {
    "Out": -0.55,
    "Doubtful": -0.35,
    "Questionable": -0.20,
    "Probable": -0.10,
    "Available": 0.0,
}
INJURY_CAP = -3.0

def injury_rows_to_team_points(inj_df):
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    inj_df = inj_df.copy()
    inj_df["pen"] = inj_df["status"].map(STATUS_PENALTY).fillna(0.0)
    team_pts = inj_df.groupby("team", as_index=False)["pen"].sum().rename(columns={"pen": "inj_pts"})
    team_pts["inj_pts"] = team_pts["inj_pts"].apply(lambda x: max(INJURY_CAP, float(x)))
    return team_pts

def attach_injuries_from_official_pdf(games):
    pdf_url = find_latest_nba_injury_pdf()
    print("Injury source used: Official NBA Injury Report PDF")
    print("PDF:", pdf_url)
    r = req_get(pdf_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=45, tries=3)
    inj_rows = parse_injury_pdf_to_rows(r.content)
    team_pts = injury_rows_to_team_points(inj_rows)

    g = games.copy()
    g = g.merge(team_pts.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
    g = g.merge(team_pts.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
    g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
    g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)
    return g, inj_rows, team_pts, pdf_url

# --- 5) Downstream layers (projection -> probs -> edges -> plays) ---
HOME_COURT_PTS = 1.8
SIGMA_SPREAD = 12.0
SIGMA_TOTAL = 17.0
MARGIN_BLEND_ALPHA = 0.65

def cover_prob_from_margin(proj_margin, spread_line, sigma=SIGMA_SPREAD):
    x = (float(proj_margin) + float(spread_line)) / float(sigma)
    return norm_cdf(x)

def over_prob_from_total(proj_total, total_line, sigma=SIGMA_TOTAL):
    x = (float(proj_total) - float(total_line)) / float(sigma)
    return norm_cdf(x)

def attach_team_stats_and_project(games, team_stats):
    g = games.copy().reset_index(drop=True)

    g = g.merge(team_stats.rename(columns={"team":"home_team","ppg":"home_ppg","oppg":"home_oppg","srs":"home_srs"}),
                on="home_team", how="left")
    g = g.merge(team_stats.rename(columns={"team":"away_team","ppg":"away_ppg","oppg":"away_oppg","srs":"away_srs"}),
                on="away_team", how="left")

    for c in ["home_ppg","home_oppg","away_ppg","away_oppg"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")

    g["home_pts_proj"] = ((g["home_ppg"] + g["away_oppg"]) / 2.0) + (HOME_COURT_PTS/2.0) + g["home_inj_pts"]
    g["away_pts_proj"] = ((g["away_ppg"] + g["home_oppg"]) / 2.0) - (HOME_COURT_PTS/2.0) + g["away_inj_pts"]

    g["proj_total_raw"]  = g["home_pts_proj"] + g["away_pts_proj"]
    g["proj_margin_raw"] = g["home_pts_proj"] - g["away_pts_proj"]

    g["market_margin"] = -pd.to_numeric(g["home_spread"], errors="coerce")
    g["proj_margin"] = (MARGIN_BLEND_ALPHA * g["proj_margin_raw"]) + ((1.0 - MARGIN_BLEND_ALPHA) * g["market_margin"])
    g["proj_total"]  = (0.70 * g["proj_total_raw"]) + (0.30 * g["total_line"])
    return g

def compute_market_probs_and_edges(g):
    df = g.copy().reset_index(drop=True)

    # SPREAD
    df["home_cover_prob"] = df.apply(lambda r: cover_prob_from_margin(r["proj_margin"], r["home_spread"]), axis=1)
    df["spread_market_prob"] = df["home_spread_price"].apply(american_to_prob).astype(float)
    df["spread_edge"] = (df["home_cover_prob"] - df["spread_market_prob"]).astype(float)
    df["spread_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["home_cover_prob"], r["home_spread_price"]), axis=1)
    df["spread_units"] = df["spread_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))
    df["spread_fair"] = df["home_cover_prob"].apply(prob_to_american)

    # TOTAL (Over)
    df["over_prob"] = df.apply(lambda r: over_prob_from_total(r["proj_total"], r["total_line"]), axis=1)
    df["total_market_prob"] = df["total_price_used"].apply(american_to_prob).astype(float)
    df["total_edge"] = (df["over_prob"] - df["total_market_prob"]).astype(float)
    df["total_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["over_prob"], r["total_price_used"]), axis=1)
    df["total_units"] = df["total_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))
    df["total_fair"] = df["over_prob"].apply(prob_to_american)

    # ML
    df["home_win_prob"] = df["proj_margin"].apply(lambda m: norm_cdf(float(m) / SIGMA_SPREAD)).astype(float)
    df["away_win_prob"] = (1.0 - df["home_win_prob"]).astype(float)

    df["home_ml_market_prob"] = df["home_ml_price"].apply(american_to_prob).astype(float)
    df["away_ml_market_prob"] = df["away_ml_price"].apply(american_to_prob).astype(float)

    novig = df.apply(lambda r: no_vig_probs_from_two_sided(r["home_ml_market_prob"], r["away_ml_market_prob"]), axis=1)
    df["home_ml_prob_novig"] = [x[0] for x in novig]
    df["away_ml_prob_novig"] = [x[1] for x in novig]

    df["home_ml_edge_novig"] = (df["home_win_prob"] - df["home_ml_prob_novig"]).astype(float)
    df["away_ml_edge_novig"] = (df["away_win_prob"] - df["away_ml_prob_novig"]).astype(float)

    df["home_ml_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["home_win_prob"], r["home_ml_price"]), axis=1)
    df["away_ml_kelly"] = df.apply(lambda r: kelly_fraction_winprob(r["away_win_prob"], r["away_ml_price"]), axis=1)

    df["home_ml_units"] = df["home_ml_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))
    df["away_ml_units"] = df["away_ml_kelly"].apply(lambda f: clamp(max(0.0, 0.5*f), 0.0, 0.25))

    df["home_ml_fair"] = df["home_win_prob"].apply(prob_to_american)
    df["away_ml_fair"] = df["away_win_prob"].apply(prob_to_american)

    return df

def build_plays(df):
    plays = []
    for _, r in df.iterrows():
        matchup = f"{r['away_team']} at {r['home_team']}"

        if r["home_ml_edge_novig"] > 0 and r["home_ml_units"] > 0:
            plays.append({"matchup": matchup, "market":"ML", "bet": f"{r['home_team']} ML",
                          "odds": float(r["home_ml_price"]), "model_prob": float(r["home_win_prob"]),
                          "edge": float(r["home_ml_edge_novig"]), "units": float(r["home_ml_units"]),
                          "fair_odds": float(prob_to_american(r["home_win_prob"]))})

        if r["away_ml_edge_novig"] > 0 and r["away_ml_units"] > 0:
            plays.append({"matchup": matchup, "market":"ML", "bet": f"{r['away_team']} ML",
                          "odds": float(r["away_ml_price"]), "model_prob": float(r["away_win_prob"]),
                          "edge": float(r["away_ml_edge_novig"]), "units": float(r["away_ml_units"]),
                          "fair_odds": float(prob_to_american(r["away_win_prob"]))})

        if r["spread_edge"] > 0 and r["spread_units"] > 0:
            plays.append({"matchup": matchup, "market":"SPREAD", "bet": f"{r['home_team']} {r['home_spread']:+g}",
                          "odds": float(r["home_spread_price"]), "model_prob": float(r["home_cover_prob"]),
                          "edge": float(r["spread_edge"]), "units": float(r["spread_units"]),
                          "fair_odds": float(r["spread_fair"])})

        if r["total_edge"] > 0 and r["total_units"] > 0:
            plays.append({"matchup": matchup, "market":"TOTAL", "bet": f"Over {r['total_line']}",
                          "odds": float(r["total_price_used"]), "model_prob": float(r["over_prob"]),
                          "edge": float(r["total_edge"]), "units": float(r["total_units"]),
                          "fair_odds": float(r["total_fair"])})

    out = pd.DataFrame(plays)
    if out.empty:
        return out
    return out.sort_values(["edge","model_prob"], ascending=[False, False]).reset_index(drop=True)

def to_discord(plays_df, top_n=12):
    if plays_df is None or plays_df.empty:
        return "== CDR BETTING | NBA FULL CARD ==\n\n(No +EV plays found.)"
    lines = ["== CDR BETTING | NBA FULL CARD =="]
    for _, r in plays_df.head(top_n).iterrows():
        lines.append("")
        lines.append(r["matchup"])
        lines.append(f"{r['market']}: {r['bet']} ({int(r['odds']):+d}) — {r['units']:.2f}u")
        lines.append(f"Model%: {100*r['model_prob']:.1f}% | Edge: {100*r['edge']:.1f}% | Fair: {int(r['fair_odds']):+d}")
    return "\n".join(lines)

# --- 6) RUN injury attach -> projections -> edges -> outputs ---
# You already have games_df and team_stats from the cells above.
games_df, inj_rows_df, team_inj_pts_df, inj_pdf_url = attach_injuries_from_official_pdf(games_df)
print("Injury layer attached (sample):")
display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(12))

games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJURY PDF ON) ===\n")
display(plays_df.head(25))

discord_text = to_discord(plays_df, top_n=12)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

# Save outputs (same filenames used earlier if present)
OUT_PLAYS_CSV = globals().get("OUT_PLAYS_CSV", "nba_card_plays.csv")
OUT_DISCORD_TXT = globals().get("OUT_DISCORD_TXT", "nba_discord_card.txt")

plays_df.to_csv(OUT_PLAYS_CSV, index=False)
with open(OUT_DISCORD_TXT, "w", encoding="utf-8") as f:
    f.write(discord_text)

print(f"\nSaved: {OUT_PLAYS_CSV}, {OUT_DISCORD_TXT}")

Injury source used: Official NBA Injury Report PDF
PDF: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-03_04_45PM.pdf
Injury layer attached (sample):


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,0.0,0.0
1,Cleveland Cavaliers,Detroit Pistons,0.0,0.0
2,Orlando Magic,Washington Wizards,0.0,0.0
3,Miami Heat,Brooklyn Nets,0.0,0.0
4,Toronto Raptors,New York Knicks,0.0,0.0
5,Chicago Bulls,Oklahoma City Thunder,0.0,0.0
6,Minnesota Timberwolves,Memphis Grizzlies,0.0,0.0
7,Philadelphia 76ers,San Antonio Spurs,0.0,0.0
8,Los Angeles Lakers,New Orleans Pelicans,0.0,0.0
9,Sacramento Kings,Phoenix Suns,0.0,0.0



=== TOP PLAYS (ALL MARKETS) — UPDATED (INJURY PDF ON) ===



,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.646544,0.134349,0.137708,-183.0
1,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.385656,0.122578,0.076914,159.0
2,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.242872,0.097024,0.052606,312.0
3,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.616060,0.092251,0.096863,-160.0
4,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,479.0,0.249851,0.085342,0.046622,300.0
5,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,700.0,0.195299,0.077335,0.040171,412.0
6,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.309765,0.073847,0.042831,223.0
7,New York Knicks at Toronto Raptors,TOTAL,Over 221.5,-110.0,0.592070,0.068260,0.071673,-145.0
8,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10.5,-115.0,0.596785,0.061901,0.066543,-148.0
9,Oklahoma City Thunder at New York Knicks,TOTAL,Over 222.5,-110.0,0.584584,0.060774,0.063813,-141.0



=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.14u
Model%: 64.7% | Edge: 13.4% | Fair: -183

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.08u
Model%: 38.6% | Edge: 12.3% | Fair: +159

Memphis Grizzlies at Minnesota Timberwolves
ML: Memphis Grizzlies ML (+550) — 0.05u
Model%: 24.3% | Edge: 9.7% | Fair: +312

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.10u
Model%: 61.6% | Edge: 9.2% | Fair: -160

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+479) — 0.05u
Model%: 25.0% | Edge: 8.5% | Fair: +300

Washington Wizards at Orlando Magic
ML: Washington Wizards ML (+700) — 0.04u
Model%: 19.5% | Edge: 7.7% | Fair: +412

Phoenix Suns at Sacramento Kings
ML: Sacramento Kings ML (+308) — 0.04u
Model%: 31.0% | Edge: 7.4% | Fair: +223

New York Knicks at Toronto Raptors
TOTAL: Over 221.5 (-110) — 0.07u
Model%: 59.2% | Edge: 6.8% | Fa

In [17]:
# =========================
# FIX CELL: Official NBA Injury PDF -> TABLE PARSER (non-zero inj pts)
# Then rerun injury attach -> downstream layers
# =========================

import re, math
import numpy as np
import pandas as pd
from io import BytesIO
from datetime import datetime, timezone
import pdfplumber

# ---- Tuning knobs ----
STATUS_PENALTY = {"Out": -0.55, "Doubtful": -0.35, "Questionable": -0.20, "Probable": -0.10, "Available": 0.0}
INJURY_CAP = -3.0

def _clean(s):
    if s is None: return ""
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_injury_pdf_to_rows_table(pdf_bytes):
    """
    Extracts rows from the official NBA Injury Report PDF using table extraction.
    Returns DataFrame with columns: team, player, status
    """
    rows_out = []
    current_team = None

    # Common full team name patterns in the PDF (we keep any "words" team string we see)
    STATUS_SET = set(STATUS_PENALTY.keys())

    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            # Try multiple table settings because PDF layout varies slightly by update time
            tables = []
            for settings in [
                {"vertical_strategy": "lines", "horizontal_strategy": "lines"},
                {"vertical_strategy": "text",  "horizontal_strategy": "text"},
                {"vertical_strategy": "lines", "horizontal_strategy": "text"},
                {"vertical_strategy": "text",  "horizontal_strategy": "lines"},
            ]:
                try:
                    t = page.extract_tables(table_settings=settings) or []
                    if t: tables.extend(t)
                except Exception:
                    pass

            # Fallback: sometimes extract_tables returns nothing, but extract_table can work
            if not tables:
                try:
                    t = page.extract_table()  # default
                    if t: tables.append(t)
                except Exception:
                    pass

            for tbl in tables:
                if not tbl or len(tbl) < 2:
                    continue

                # Normalize rows
                for raw in tbl:
                    r = [_clean(x) for x in raw]
                    if not any(r):
                        continue

                    # Skip headers / legends
                    joined = " ".join(r).lower()
                    if "player" in joined and "status" in joined:
                        continue
                    if "injury report" in joined:
                        continue

                    # Heuristic: locate a status token anywhere in the row
                    status = None
                    for s in STATUS_SET:
                        if any(re.search(rf"\b{s}\b", cell) for cell in r if cell):
                            status = s
                            break
                    if not status:
                        continue

                    # Team is often first col; if blank, carry forward
                    team_candidate = r[0]
                    if team_candidate and len(team_candidate) >= 3 and not re.search(r"\bPG|SG|SF|PF|C\b", team_candidate):
                        # If it looks like a team (letters/spaces) keep it
                        if re.match(r"^[A-Za-z .'-]+$", team_candidate) and len(team_candidate.split()) <= 4:
                            current_team = team_candidate

                    # Player name is often the 2nd col; if not, scan for "Last, First" or "First Last"
                    player = ""
                    if len(r) >= 2:
                        player = r[1]
                    # clean player if it’s clearly not a name
                    if player and (player.lower() in ["player", "name"] or len(player) < 3):
                        player = ""

                    # If still missing, scan cells for name-like patterns
                    if not player:
                        for cell in r:
                            if re.search(r"[A-Za-z]", cell) and ("," in cell or len(cell.split()) in [2,3]):
                                # avoid grabbing "Injury/Illness" etc
                                if "injury" in cell.lower():
                                    continue
                                # avoid status cell
                                if cell in STATUS_SET:
                                    continue
                                player = cell
                                break

                    if not current_team:
                        # if we still don't have a team, skip (can't attribute)
                        continue

                    rows_out.append({"team": current_team, "player": player or "(unknown)", "status": status})

    return pd.DataFrame(rows_out)

def injury_rows_to_team_points(inj_df):
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    inj_df = inj_df.copy()
    inj_df["pen"] = inj_df["status"].map(STATUS_PENALTY).fillna(0.0)
    team_pts = inj_df.groupby("team", as_index=False)["pen"].sum().rename(columns={"pen": "inj_pts"})
    team_pts["inj_pts"] = team_pts["inj_pts"].astype(float).apply(lambda x: max(INJURY_CAP, x))
    return team_pts

def attach_injuries_from_official_pdf_table(games, pdf_url):
    import requests
    def req_get(url, headers=None, timeout=45):
        r = requests.get(url, headers=headers, timeout=timeout)
        r.raise_for_status()
        return r

    print("Injury source used: Official NBA Injury Report PDF (TABLE PARSER)")
    print("PDF:", pdf_url)
    r = req_get(pdf_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=45)
    inj_rows = parse_injury_pdf_to_rows_table(r.content)
    team_pts = injury_rows_to_team_points(inj_rows)

    g = games.copy()
    g = g.merge(team_pts.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
    g = g.merge(team_pts.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
    g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
    g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

    print(f"Parsed injury rows: {len(inj_rows)} | Teams w/ nonzero inj pts: {(team_pts['inj_pts']!=0).sum() if not team_pts.empty else 0}")
    return g, inj_rows, team_pts


# ---- RUN: use your existing inj_pdf_url from the prior cell ----
# If your variable is named differently, change the line below to your printed PDF URL.
pdf_url = globals().get("inj_pdf_url", "https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-03_04_45PM.pdf")

games_df, inj_rows_df, team_inj_pts_df = attach_injuries_from_official_pdf_table(games_df, pdf_url)

print("Injury layer attached (sample):")
display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(12))

# ---- Rerun downstream using your existing functions if present ----
# These should already exist from your notebook:
# attach_team_stats_and_project, compute_market_probs_and_edges, build_plays, to_discord

games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJURY PDF ON, TABLE PARSER) ===\n")
display(plays_df.head(25))

discord_text = to_discord(plays_df, top_n=12)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w", encoding="utf-8") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

Injury source used: Official NBA Injury Report PDF (TABLE PARSER)
PDF: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-03_04_45PM.pdf


KeyError: 'home_inj_pts'

In [18]:
# =========================
# FIX CELL v2: merge-safe attach (handles existing home_inj_pts columns)
# =========================

import pandas as pd
import numpy as np
from io import BytesIO
import pdfplumber
import re

STATUS_PENALTY = {"Out": -0.55, "Doubtful": -0.35, "Questionable": -0.20, "Probable": -0.10, "Available": 0.0}
INJURY_CAP = -3.0

def _clean(s):
    if s is None: return ""
    return re.sub(r"\s+", " ", str(s)).strip()

def parse_injury_pdf_to_rows_table(pdf_bytes):
    rows_out = []
    current_team = None
    STATUS_SET = set(STATUS_PENALTY.keys())

    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            tables = []
            for settings in [
                {"vertical_strategy": "lines", "horizontal_strategy": "lines"},
                {"vertical_strategy": "text",  "horizontal_strategy": "text"},
                {"vertical_strategy": "lines", "horizontal_strategy": "text"},
                {"vertical_strategy": "text",  "horizontal_strategy": "lines"},
            ]:
                try:
                    tables.extend(page.extract_tables(table_settings=settings) or [])
                except Exception:
                    pass

            if not tables:
                try:
                    t = page.extract_table()
                    if t: tables.append(t)
                except Exception:
                    pass

            for tbl in tables:
                if not tbl or len(tbl) < 2:
                    continue

                for raw in tbl:
                    r = [_clean(x) for x in raw]
                    if not any(r):
                        continue

                    joined = " ".join(r).lower()
                    if "player" in joined and "status" in joined:
                        continue

                    # find status token
                    status = None
                    for s in STATUS_SET:
                        if any(re.search(rf"\b{s}\b", cell) for cell in r if cell):
                            status = s
                            break
                    if not status:
                        continue

                    # team often first col; carry forward if blank
                    team_candidate = r[0]
                    if team_candidate and re.match(r"^[A-Za-z .'-]+$", team_candidate) and (2 <= len(team_candidate) <= 30):
                        current_team = team_candidate

                    if not current_team:
                        continue

                    # player usually second col
                    player = r[1] if len(r) > 1 else ""
                    if not player or player.lower() in {"player", "name"}:
                        player = "(unknown)"

                    rows_out.append({"team": current_team, "player": player, "status": status})

    return pd.DataFrame(rows_out)

def injury_rows_to_team_points(inj_df):
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    df = inj_df.copy()
    df["pen"] = df["status"].map(STATUS_PENALTY).fillna(0.0)
    team_pts = df.groupby("team", as_index=False)["pen"].sum().rename(columns={"pen": "inj_pts"})
    team_pts["inj_pts"] = team_pts["inj_pts"].astype(float).apply(lambda x: max(INJURY_CAP, x))
    return team_pts

def attach_injuries_from_official_pdf_table_SAFE(games, pdf_url):
    import requests
    print("Injury source used: Official NBA Injury Report PDF (TABLE PARSER, SAFE MERGE)")
    print("PDF:", pdf_url)

    r = requests.get(pdf_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=45)
    r.raise_for_status()

    inj_rows = parse_injury_pdf_to_rows_table(r.content)
    team_pts = injury_rows_to_team_points(inj_rows)

    g = games.copy()

    # IMPORTANT: remove any existing injury cols to avoid merge suffixes
    for c in ["home_inj_pts", "away_inj_pts", "home_out_ct", "home_dtd_ct", "away_out_ct", "away_dtd_ct"]:
        if c in g.columns:
            g = g.drop(columns=[c])

    g = g.merge(team_pts.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
    g = g.merge(team_pts.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")

    g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
    g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

    nonzero = int(((g["home_inj_pts"] != 0) | (g["away_inj_pts"] != 0)).sum())
    print(f"Parsed injury rows: {len(inj_rows)} | Games w/ any nonzero team inj pts: {nonzero}")
    return g, inj_rows, team_pts

# ---- RUN (uses your existing inj_pdf_url if present) ----
pdf_url = globals().get("inj_pdf_url", "https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-03_04_45PM.pdf")
games_df, inj_rows_df, team_inj_pts_df = attach_injuries_from_official_pdf_table_SAFE(games_df, pdf_url)

print("Injury layer attached (sample):")
display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(12))

# Rerun downstream (your notebook functions)
games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJURY PDF ON, TABLE PARSER) ===\n")
display(plays_df.head(25))

discord_text = to_discord(plays_df, top_n=12)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w", encoding="utf-8") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

Injury source used: Official NBA Injury Report PDF (TABLE PARSER, SAFE MERGE)
PDF: https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-03_04_45PM.pdf
Parsed injury rows: 210 | Games w/ any nonzero team inj pts: 0
Injury layer attached (sample):


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,0.0,0.0
1,Cleveland Cavaliers,Detroit Pistons,0.0,0.0
2,Orlando Magic,Washington Wizards,0.0,0.0
3,Miami Heat,Brooklyn Nets,0.0,0.0
4,Toronto Raptors,New York Knicks,0.0,0.0
5,Chicago Bulls,Oklahoma City Thunder,0.0,0.0
6,Minnesota Timberwolves,Memphis Grizzlies,0.0,0.0
7,Philadelphia 76ers,San Antonio Spurs,0.0,0.0
8,Los Angeles Lakers,New Orleans Pelicans,0.0,0.0
9,Sacramento Kings,Phoenix Suns,0.0,0.0


KeyError: 'home_ppg'

In [19]:
# =========================
# FIX CELL v3: (1) normalize injury team codes -> match games_df team names
#              (2) SAFE team-stats merge so home_ppg/home_oppg always exist
# =========================

import re
import numpy as np
import pandas as pd

# --- 1) TEAM NORMALIZATION (NBA abbreviations -> full names) ---
TEAM_ABBR_TO_NAME = {
    "ATL":"Atlanta Hawks","BOS":"Boston Celtics","BKN":"Brooklyn Nets","BRK":"Brooklyn Nets",
    "CHA":"Charlotte Hornets","CHO":"Charlotte Hornets","CHI":"Chicago Bulls","CLE":"Cleveland Cavaliers",
    "DAL":"Dallas Mavericks","DEN":"Denver Nuggets","DET":"Detroit Pistons","GSW":"Golden State Warriors",
    "HOU":"Houston Rockets","IND":"Indiana Pacers","LAC":"Los Angeles Clippers","LAL":"Los Angeles Lakers",
    "MEM":"Memphis Grizzlies","MIA":"Miami Heat","MIL":"Milwaukee Bucks","MIN":"Minnesota Timberwolves",
    "NOP":"New Orleans Pelicans","NO":"New Orleans Pelicans","NYK":"New York Knicks","OKC":"Oklahoma City Thunder",
    "ORL":"Orlando Magic","PHI":"Philadelphia 76ers","PHO":"Phoenix Suns","PHX":"Phoenix Suns",
    "POR":"Portland Trail Blazers","SAC":"Sacramento Kings","SAS":"San Antonio Spurs",
    "TOR":"Toronto Raptors","UTA":"Utah Jazz","WAS":"Washington Wizards",
}

# optional alias cleanup (PDF sometimes has odd variants)
TEAM_ALIAS = {
    "LA LAKERS":"Los Angeles Lakers",
    "LA CLIPPERS":"Los Angeles Clippers",
    "NEW ORLEANS":"New Orleans Pelicans",
    "GOLDEN STATE":"Golden State Warriors",
    "SAN ANTONIO":"San Antonio Spurs",
}

def normalize_team(s: str) -> str:
    s = "" if s is None else str(s)
    s = re.sub(r"\s+", " ", s).strip()

    # if it's a 2-4 char code, map it
    code = re.sub(r"[^A-Z]", "", s.upper())
    if 2 <= len(code) <= 4 and code in TEAM_ABBR_TO_NAME:
        return TEAM_ABBR_TO_NAME[code]

    # if it looks like "DAL - Mavericks" etc, grab first token code
    m = re.match(r"^([A-Z]{2,4})\b", s.upper())
    if m and m.group(1) in TEAM_ABBR_TO_NAME:
        return TEAM_ABBR_TO_NAME[m.group(1)]

    # normalize common textual aliases
    up = s.upper()
    if up in TEAM_ALIAS:
        return TEAM_ALIAS[up]

    # otherwise assume already full name
    return s

# --- 2) Injury scoring (keep your penalties, just normalize teams before grouping) ---
STATUS_PENALTY = {"Out": -0.55, "Doubtful": -0.35, "Questionable": -0.20, "Probable": -0.10, "Available": 0.0}
INJURY_CAP = -3.0

def injury_rows_to_team_points_normalized(inj_df: pd.DataFrame) -> pd.DataFrame:
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    df = inj_df.copy()
    df["team"] = df["team"].apply(normalize_team)
    df["pen"]  = df["status"].map(STATUS_PENALTY).fillna(0.0)

    team_pts = df.groupby("team", as_index=False)["pen"].sum().rename(columns={"pen":"inj_pts"})
    team_pts["inj_pts"] = team_pts["inj_pts"].astype(float).apply(lambda x: max(INJURY_CAP, x))
    return team_pts

def attach_inj_pts_from_parsed_rows_SAFE(games_df, inj_rows_df):
    # build team inj pts with normalized names
    team_pts = injury_rows_to_team_points_normalized(inj_rows_df)

    g = games_df.copy()
    # remove existing injury cols so merge never suffixes
    for c in ["home_inj_pts","away_inj_pts"]:
        if c in g.columns:
            g = g.drop(columns=[c])

    g = g.merge(team_pts.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
    g = g.merge(team_pts.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
    g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
    g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

    nonzero = int(((g["home_inj_pts"] != 0) | (g["away_inj_pts"] != 0)).sum())
    print(f"[INJ] Team rows: {len(team_pts)} | Games w/ any nonzero inj pts: {nonzero}")

    # quick debug: show any injury teams not in slate
    slate_teams = set(g["home_team"]).union(set(g["away_team"]))
    extra = team_pts[~team_pts["team"].isin(slate_teams)].sort_values("inj_pts").head(10)
    if len(extra):
        print("[INJ] Example injury teams not on this slate (ok):")
        display(extra)

    return g, team_pts

# --- 3) SAFE team-stats attach to guarantee home_ppg/home_oppg exist ---
def attach_team_stats_and_project_SAFE(games_df, team_stats_df,
                                       HOME_COURT_PTS=1.5,  # keep your existing constants if you have them
                                       DEFAULT_PPG=114.0, DEFAULT_OPPG=114.0):
    ts = team_stats_df.copy()
    ts["team"] = ts["team"].apply(normalize_team)

    g = games_df.copy()

    # drop any stale stat cols so we re-create clean
    for c in ["home_ppg","home_oppg","away_ppg","away_oppg"]:
        if c in g.columns:
            g = g.drop(columns=[c])

    g = g.merge(ts[["team","ppg","oppg"]].rename(columns={"team":"home_team","ppg":"home_ppg","oppg":"home_oppg"}),
                on="home_team", how="left")
    g = g.merge(ts[["team","ppg","oppg"]].rename(columns={"team":"away_team","ppg":"away_ppg","oppg":"away_oppg"}),
                on="away_team", how="left")

    # guarantee cols exist + fill
    for c, default in [("home_ppg",DEFAULT_PPG),("home_oppg",DEFAULT_OPPG),("away_ppg",DEFAULT_PPG),("away_oppg",DEFAULT_OPPG)]:
        g[c] = pd.to_numeric(g[c], errors="coerce").fillna(default)

    # projections (same logic you were using)
    if "home_inj_pts" not in g.columns: g["home_inj_pts"] = 0.0
    if "away_inj_pts" not in g.columns: g["away_inj_pts"] = 0.0

    g["home_pts_proj"] = ((g["home_ppg"] + g["away_oppg"]) / 2.0) + (HOME_COURT_PTS/2.0) + g["home_inj_pts"]
    g["away_pts_proj"] = ((g["away_ppg"] + g["home_oppg"]) / 2.0) - (HOME_COURT_PTS/2.0) + g["away_inj_pts"]

    g["proj_total"] = g["home_pts_proj"] + g["away_pts_proj"]
    g["proj_margin_raw"] = g["home_pts_proj"] - g["away_pts_proj"]

    return g

# =========================
# RUN PATCHED FLOW
# =========================

# 1) Re-attach injuries from your already-parsed inj_rows_df (from the PDF parser you just ran)
games_df, team_inj_pts_df = attach_inj_pts_from_parsed_rows_SAFE(games_df, inj_rows_df)

print("Injury layer attached (sample):")
display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(12))

# 2) Re-attach team stats + projections safely (prevents KeyError home_ppg)
games_df = attach_team_stats_and_project_SAFE(games_df, team_stats)

# 3) Now rerun your downstream as-is
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ FIXED) ===\n")
display(plays_df.head(25))

discord_text = to_discord(plays_df, top_n=12)
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

[INJ] Team rows: 16 | Games w/ any nonzero inj pts: 0
[INJ] Example injury teams not on this slate (ok):


,team,inj_pts
0,BrooklynNets,-3.0
3,ClevelandCavaliers,-3.0
6,MemphisGrizzlies,-3.0
4,DallasMavericks,-3.0
12,OrlandoMagic,-3.0
11,OklahomaCityThunder,-3.0
10,NewYorkKnicks,-3.0
8,MinnesotaTimberwolves,-3.0
14,TorontoRaptors,-3.0
13,SanAntonioSpurs,-3.0


Injury layer attached (sample):


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,0.0,0.0
1,Cleveland Cavaliers,Detroit Pistons,0.0,0.0
2,Orlando Magic,Washington Wizards,0.0,0.0
3,Miami Heat,Brooklyn Nets,0.0,0.0
4,Toronto Raptors,New York Knicks,0.0,0.0
5,Chicago Bulls,Oklahoma City Thunder,0.0,0.0
6,Minnesota Timberwolves,Memphis Grizzlies,0.0,0.0
7,Philadelphia 76ers,San Antonio Spurs,0.0,0.0
8,Los Angeles Lakers,New Orleans Pelicans,0.0,0.0
9,Sacramento Kings,Phoenix Suns,0.0,0.0



=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ FIXED) ===



,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.646544,0.134349,0.137708,-183.0
1,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.385656,0.122578,0.076914,159.0
2,New York Knicks at Toronto Raptors,TOTAL,Over 221.5,-110.0,0.630310,0.106500,0.111825,-170.0
3,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.242872,0.097024,0.052606,312.0
4,Oklahoma City Thunder at New York Knicks,TOTAL,Over 222.5,-110.0,0.619890,0.096080,0.100884,-163.0
5,Washington Wizards at Orlando Magic,TOTAL,Over 227.0,-110.0,0.618849,0.095040,0.099792,-162.0
6,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.616060,0.092251,0.096863,-160.0
7,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,479.0,0.249851,0.085342,0.046622,300.0
8,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,700.0,0.195299,0.077335,0.040171,412.0
9,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.309765,0.073847,0.042831,223.0



=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.14u
Model%: 64.7% | Edge: 13.4% | Fair: -183

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.08u
Model%: 38.6% | Edge: 12.3% | Fair: +159

New York Knicks at Toronto Raptors
TOTAL: Over 221.5 (-110) — 0.11u
Model%: 63.0% | Edge: 10.7% | Fair: -170

Memphis Grizzlies at Minnesota Timberwolves
ML: Memphis Grizzlies ML (+550) — 0.05u
Model%: 24.3% | Edge: 9.7% | Fair: +312

Oklahoma City Thunder at New York Knicks
TOTAL: Over 222.5 (-110) — 0.10u
Model%: 62.0% | Edge: 9.6% | Fair: -163

Washington Wizards at Orlando Magic
TOTAL: Over 227.0 (-110) — 0.10u
Model%: 61.9% | Edge: 9.5% | Fair: -162

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.10u
Model%: 61.6% | Edge: 9.2% | Fair: -160

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+479) — 0.05u
Model%: 25.0% | Edge: 8.5% | Fair: +

In [20]:
# =========================
# FIX: PDF injury team names are coming through like "BrooklynNets"
#      -> convert CamelCase to spaced names, then re-merge
# =========================
import re
import pandas as pd

def camel_to_spaced(s: str) -> str:
    s = "" if s is None else str(s).strip()
    # Insert space between lower->Upper OR letter->Upper sequences
    s = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", s)
    s = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# expand your existing normalize_team to handle CamelCase/no-space names
def normalize_team_v2(s: str) -> str:
    s0 = camel_to_spaced(s)

    # reuse your earlier mapping if it exists
    try:
        return normalize_team(s0)  # from prior cell
    except Exception:
        return s0

def injury_rows_to_team_points_normalized_v2(inj_df: pd.DataFrame) -> pd.DataFrame:
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    df = inj_df.copy()
    df["team"] = df["team"].apply(normalize_team_v2)

    # keep your existing penalties/cap if present
    STATUS_PENALTY = globals().get("STATUS_PENALTY", {"Out": -0.55, "Doubtful": -0.35, "Questionable": -0.20, "Probable": -0.10, "Available": 0.0})
    INJURY_CAP = float(globals().get("INJURY_CAP", -3.0))

    df["pen"] = df["status"].map(STATUS_PENALTY).fillna(0.0)

    team_pts = df.groupby("team", as_index=False)["pen"].sum().rename(columns={"pen":"inj_pts"})
    team_pts["inj_pts"] = team_pts["inj_pts"].astype(float).apply(lambda x: max(INJURY_CAP, x))
    return team_pts

def attach_inj_pts_from_parsed_rows_SAFE_v2(games_df, inj_rows_df):
    team_pts = injury_rows_to_team_points_normalized_v2(inj_rows_df)

    g = games_df.copy()
    # drop existing injury cols to avoid suffixing
    for c in ["home_inj_pts","away_inj_pts"]:
        if c in g.columns:
            g = g.drop(columns=[c])

    g = g.merge(team_pts.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
    g = g.merge(team_pts.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
    g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
    g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

    nonzero = int(((g["home_inj_pts"] != 0) | (g["away_inj_pts"] != 0)).sum())
    print(f"[INJ v2] Team rows: {len(team_pts)} | Games w/ any nonzero inj pts: {nonzero}")

    # show which injury teams still don't match slate (debug)
    slate_teams = set(g["home_team"]).union(set(g["away_team"]))
    extra = team_pts[~team_pts["team"].isin(slate_teams)].sort_values("inj_pts").head(25)
    if len(extra):
        print("[INJ v2] Injury teams not on slate (ok):")
        display(extra)

    return g, team_pts

# ---- RUN IT ----
games_df, team_inj_pts_df = attach_inj_pts_from_parsed_rows_SAFE_v2(games_df, inj_rows_df)

display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]])

# rerun downstream exactly like you were
games_df = attach_team_stats_and_project_SAFE(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v2 FIXED) ===\n")
display(plays_df.head(25))

print("\n=== DISCORD TEXT ===\n")
print(to_discord(plays_df, top_n=12))

[INJ v2] Team rows: 16 | Games w/ any nonzero inj pts: 10


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,-2.75,-3.00
1,Cleveland Cavaliers,Detroit Pistons,-3.00,-1.65
2,Orlando Magic,Washington Wizards,-3.00,-3.00
3,Miami Heat,Brooklyn Nets,-2.75,-3.00
4,Toronto Raptors,New York Knicks,-3.00,-3.00
5,Chicago Bulls,Oklahoma City Thunder,-2.40,-3.00
6,Minnesota Timberwolves,Memphis Grizzlies,-3.00,-3.00
7,Philadelphia 76ers,San Antonio Spurs,0.00,-3.00
8,Los Angeles Lakers,New Orleans Pelicans,0.00,-2.20
9,Sacramento Kings,Phoenix Suns,0.00,0.00



=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v2 FIXED) ===



,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.646544,0.134349,0.137708,-183.0
1,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.385656,0.122578,0.076914,159.0
2,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.242872,0.097024,0.052606,312.0
3,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.616060,0.092251,0.096863,-160.0
4,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,479.0,0.249851,0.085342,0.046622,300.0
5,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,700.0,0.195299,0.077335,0.040171,412.0
6,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.309765,0.073847,0.042831,223.0
7,Phoenix Suns at Sacramento Kings,TOTAL,Over 223.5,-110.0,0.592813,0.069004,0.072454,-146.0
8,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10.5,-115.0,0.596785,0.061901,0.066543,-148.0
9,New Orleans Pelicans at Los Angeles Lakers,ML,New Orleans Pelicans ML,280.0,0.308063,0.056421,0.030471,225.0



=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.14u
Model%: 64.7% | Edge: 13.4% | Fair: -183

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.08u
Model%: 38.6% | Edge: 12.3% | Fair: +159

Memphis Grizzlies at Minnesota Timberwolves
ML: Memphis Grizzlies ML (+550) — 0.05u
Model%: 24.3% | Edge: 9.7% | Fair: +312

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.10u
Model%: 61.6% | Edge: 9.2% | Fair: -160

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+479) — 0.05u
Model%: 25.0% | Edge: 8.5% | Fair: +300

Washington Wizards at Orlando Magic
ML: Washington Wizards ML (+700) — 0.04u
Model%: 19.5% | Edge: 7.7% | Fair: +412

Phoenix Suns at Sacramento Kings
ML: Sacramento Kings ML (+308) — 0.04u
Model%: 31.0% | Edge: 7.4% | Fair: +223

Phoenix Suns at Sacramento Kings
TOTAL: Over 223.5 (-110) — 0.07u
Model%: 59.3% | Edge: 6.9% | Fair

In [21]:
# =========================
# INJURY SCORING v3 (SAFER)
# rotation-weighted, strict caps, avoids blanket -3 for everyone
# =========================
import numpy as np
import pandas as pd

# If your inj_rows_df has columns like player/status/position/etc, keep these.
# We'll defensively handle missing cols.
def _col(df, *names):
    for n in names:
        if n in df.columns: return n
    return None

TEAM_COL   = _col(inj_rows_df, "team", "Team", "TEAM")
STATUS_COL = _col(inj_rows_df, "status", "Status", "STATUS")
NAME_COL   = _col(inj_rows_df, "player", "Player", "PLAYER", "name", "Name")
POS_COL    = _col(inj_rows_df, "pos", "Pos", "POS", "position", "Position")
ROLE_COL   = _col(inj_rows_df, "role", "Role", "ROLE")

def normalize_team_v3(s: str) -> str:
    # uses your v2 normalizer (camel->spaced + mapping)
    return normalize_team_v2(s)

# --- weights: starters matter, bench matters less ---
# if you don’t have role data, we approximate "starter-ness" via minutes/role text if present
DEFAULT_ROLE_W = 0.55   # if unknown, assume rotation but not star
STARTER_W      = 1.00
BENCH_W        = 0.35

# --- penalties per status (tuned) ---
PEN_OUT        = -1.00
PEN_DOUBTFUL   = -0.55
PEN_QUESTION   = -0.25
PEN_PROBABLE   = -0.10

TEAM_CAP = -2.25   # cap a team’s total injury adjustment
TEAM_FLOOR = 0.0   # allow only negative adjustments here

def status_penalty(s: str) -> float:
    s = ("" if s is None else str(s)).lower()
    if "out" in s: return PEN_OUT
    if "doubt" in s: return PEN_DOUBTFUL
    if "quest" in s: return PEN_QUESTION
    if "prob" in s: return PEN_PROBABLE
    return 0.0

def role_weight(row) -> float:
    # prefer explicit role/starter flags if present
    rtxt = ""
    if ROLE_COL: rtxt += " " + str(row.get(ROLE_COL, ""))
    if NAME_COL: rtxt += " " + str(row.get(NAME_COL, ""))
    if POS_COL:  rtxt += " " + str(row.get(POS_COL, ""))
    rtxt = rtxt.lower()

    # crude but effective: “starter”/“starting” -> starter weight
    if "starter" in rtxt or "starting" in rtxt:
        return STARTER_W

    # if we can detect “bench” / “reserve”
    if "bench" in rtxt or "reserve" in rtxt:
        return BENCH_W

    return DEFAULT_ROLE_W

def injuries_to_team_points_v3(inj_df: pd.DataFrame) -> pd.DataFrame:
    if inj_df is None or inj_df.empty:
        return pd.DataFrame({"team": [], "inj_pts": []})

    df = inj_df.copy()

    if TEAM_COL is None or STATUS_COL is None:
        raise RuntimeError(f"inj_rows_df missing required cols. cols={list(df.columns)}")

    df["team_norm"] = df[TEAM_COL].apply(normalize_team_v3)
    df["pen_base"]  = df[STATUS_COL].apply(status_penalty).astype(float)
    df["w_role"]    = df.apply(role_weight, axis=1).astype(float)

    df["pen"] = df["pen_base"] * df["w_role"]

    team_pts = df.groupby("team_norm", as_index=False)["pen"].sum().rename(columns={"team_norm":"team", "pen":"inj_pts"})
    team_pts["inj_pts"] = team_pts["inj_pts"].clip(lower=TEAM_CAP, upper=TEAM_FLOOR).astype(float)

    return team_pts

# rebuild team injury points using safer scoring
team_inj_pts_df = injuries_to_team_points_v3(inj_rows_df)

# attach to slate (safe drop + merge)
g = games_df.copy()
for c in ["home_inj_pts","away_inj_pts"]:
    if c in g.columns:
        g = g.drop(columns=[c])

g = g.merge(team_inj_pts_df.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
g = g.merge(team_inj_pts_df.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

print("[INJ v3] Games w/ any nonzero inj pts:", int(((g.home_inj_pts!=0)|(g.away_inj_pts!=0)).sum()))
display(g[["home_team","away_team","home_inj_pts","away_inj_pts"]])

# rerun downstream (same as before)
games_df = g
games_df = attach_team_stats_and_project_SAFE(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v3) ===\n")
display(plays_df.head(25))

print("\n=== DISCORD TEXT ===\n")
print(to_discord(plays_df, top_n=12))

[INJ v3] Games w/ any nonzero inj pts: 10


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,-2.25,-2.25
1,Cleveland Cavaliers,Detroit Pistons,-2.25,-1.65
2,Orlando Magic,Washington Wizards,-2.25,-2.25
3,Miami Heat,Brooklyn Nets,-2.25,-2.25
4,Toronto Raptors,New York Knicks,-2.25,-2.25
5,Chicago Bulls,Oklahoma City Thunder,-2.25,-2.25
6,Minnesota Timberwolves,Memphis Grizzlies,-2.25,-2.25
7,Philadelphia 76ers,San Antonio Spurs,0.00,-2.25
8,Los Angeles Lakers,New Orleans Pelicans,0.00,-2.20
9,Sacramento Kings,Phoenix Suns,0.00,0.00



=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v3) ===



,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.646544,0.134349,0.137708,-183.0
1,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.385656,0.122578,0.076914,159.0
2,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.242872,0.097024,0.052606,312.0
3,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.616060,0.092251,0.096863,-160.0
4,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,479.0,0.249851,0.085342,0.046622,300.0
5,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,700.0,0.195299,0.077335,0.040171,412.0
6,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.309765,0.073847,0.042831,223.0
7,Phoenix Suns at Sacramento Kings,TOTAL,Over 223.5,-110.0,0.592813,0.069004,0.072454,-146.0
8,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10.5,-115.0,0.596785,0.061901,0.066543,-148.0
9,New Orleans Pelicans at Los Angeles Lakers,ML,New Orleans Pelicans ML,280.0,0.308063,0.056421,0.030471,225.0



=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.14u
Model%: 64.7% | Edge: 13.4% | Fair: -183

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.08u
Model%: 38.6% | Edge: 12.3% | Fair: +159

Memphis Grizzlies at Minnesota Timberwolves
ML: Memphis Grizzlies ML (+550) — 0.05u
Model%: 24.3% | Edge: 9.7% | Fair: +312

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.10u
Model%: 61.6% | Edge: 9.2% | Fair: -160

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+479) — 0.05u
Model%: 25.0% | Edge: 8.5% | Fair: +300

Washington Wizards at Orlando Magic
ML: Washington Wizards ML (+700) — 0.04u
Model%: 19.5% | Edge: 7.7% | Fair: +412

Phoenix Suns at Sacramento Kings
ML: Sacramento Kings ML (+308) — 0.04u
Model%: 31.0% | Edge: 7.4% | Fair: +223

Phoenix Suns at Sacramento Kings
TOTAL: Over 223.5 (-110) — 0.07u
Model%: 59.3% | Edge: 6.9% | Fair

In [22]:
# =========================
# INJURY AUDIT + v4 SCORING (fixes "everyone hits cap")
# =========================
import pandas as pd
import numpy as np

def _col(df, *names):
    for n in names:
        if n in df.columns: return n
    return None

TEAM_COL   = _col(inj_rows_df, "team", "Team", "TEAM")
STATUS_COL = _col(inj_rows_df, "status", "Status", "STATUS")
NAME_COL   = _col(inj_rows_df, "player", "Player", "PLAYER", "name", "Name")
POS_COL    = _col(inj_rows_df, "pos", "Pos", "POS", "position", "Position")
ROLE_COL   = _col(inj_rows_df, "role", "Role", "ROLE")
NOTE_COL   = _col(inj_rows_df, "note", "Note", "NOTE", "injury", "Injury")

if TEAM_COL is None or STATUS_COL is None:
    raise RuntimeError(f"inj_rows_df missing required cols. cols={list(inj_rows_df.columns)}")

def normalize_team_v4(s: str) -> str:
    # keep using your existing normalizer
    return normalize_team_v2(s)

def status_bucket(s: str) -> str:
    s = ("" if s is None else str(s)).lower()
    if "out" in s: return "out"
    if "doubt" in s: return "doubtful"
    if "quest" in s: return "questionable"
    if "prob" in s: return "probable"
    return "other"

# -------------------------
# v4 parameters (tuned)
# -------------------------
TEAM_CAP   = -1.50     # tighter cap so it can't dominate projections
TEAM_FLOOR = 0.0

# Only truly meaningful by default:
PEN_OUT      = -1.00
PEN_DOUBTFUL = -0.55

# Q / P basically ignored unless we detect "starter" / "starting"
PEN_QUESTION_STARTER = -0.20
PEN_QUESTION_OTHER   = 0.00
PEN_PROBABLE_STARTER = -0.08
PEN_PROBABLE_OTHER   = 0.00

# role weights
STARTER_W = 1.00
ROT_W     = 0.45
BENCH_W   = 0.20
UNKNOWN_W = 0.25  # lower default than before

def detect_starter_flag(row) -> bool:
    txt = ""
    for c in [ROLE_COL, NOTE_COL, POS_COL, NAME_COL]:
        if c and c in row and pd.notna(row[c]):
            txt += " " + str(row[c])
    txt = txt.lower()
    return ("starter" in txt) or ("starting" in txt)

def role_weight_v4(row) -> float:
    txt = ""
    for c in [ROLE_COL, NOTE_COL, POS_COL, NAME_COL]:
        if c and c in row and pd.notna(row[c]):
            txt += " " + str(row[c])
    t = txt.lower()
    if "starter" in t or "starting" in t: return STARTER_W
    if "rotation" in t: return ROT_W
    if "bench" in t or "reserve" in t: return BENCH_W
    return UNKNOWN_W

def penalty_v4(row) -> float:
    s = row.get(STATUS_COL, "")
    bucket = status_bucket(s)
    is_starter = detect_starter_flag(row)

    if bucket == "out":
        base = PEN_OUT
    elif bucket == "doubtful":
        base = PEN_DOUBTFUL
    elif bucket == "questionable":
        base = PEN_QUESTION_STARTER if is_starter else PEN_QUESTION_OTHER
    elif bucket == "probable":
        base = PEN_PROBABLE_STARTER if is_starter else PEN_PROBABLE_OTHER
    else:
        base = 0.0

    return float(base) * float(role_weight_v4(row))

# -------------------------
# 1) AUDIT what’s causing the cap
# -------------------------
audit = inj_rows_df.copy()
audit["team_norm"] = audit[TEAM_COL].apply(normalize_team_v4)
audit["bucket"]    = audit[STATUS_COL].apply(status_bucket)
audit["w_role"]    = audit.apply(role_weight_v4, axis=1).astype(float)
audit["pen"]       = audit.apply(penalty_v4, axis=1).astype(float)

print("\n[INJ AUDIT] status bucket counts:")
display(audit["bucket"].value_counts(dropna=False).to_frame("count"))

print("\n[INJ AUDIT] teams with largest raw penalty (before cap):")
team_raw = audit.groupby("team_norm", as_index=False)["pen"].sum().sort_values("pen")
display(team_raw.head(15))

print("\n[INJ AUDIT] top contributors per team (most negative rows):")
show_cols = [c for c in [TEAM_COL, "team_norm", NAME_COL, STATUS_COL, ROLE_COL, NOTE_COL, "bucket", "w_role", "pen"] if c and c in audit.columns] + ["team_norm","bucket","w_role","pen"]
show_cols = list(dict.fromkeys([c for c in show_cols if c in audit.columns]))  # unique, keep order
top_contrib = audit.sort_values("pen").groupby("team_norm").head(6)[show_cols]
display(top_contrib)

# -------------------------
# 2) Build team injury points with v4 + cap
# -------------------------
team_inj_pts_df = audit.groupby("team_norm", as_index=False)["pen"].sum().rename(columns={"team_norm":"team", "pen":"inj_pts"})
team_inj_pts_df["inj_pts"] = team_inj_pts_df["inj_pts"].clip(lower=TEAM_CAP, upper=TEAM_FLOOR).astype(float)

# -------------------------
# 3) Attach to slate (safe)
# -------------------------
g = games_df.copy()
for c in ["home_inj_pts","away_inj_pts"]:
    if c in g.columns:
        g = g.drop(columns=[c])

g = g.merge(team_inj_pts_df.rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
g = g.merge(team_inj_pts_df.rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

print("\n[INJ v4] Games w/ any nonzero inj pts:", int(((g.home_inj_pts!=0)|(g.away_inj_pts!=0)).sum()))
display(g[["home_team","away_team","home_inj_pts","away_inj_pts"]])

# -------------------------
# 4) Rerun downstream
# -------------------------
games_df = g
games_df = attach_team_stats_and_project_SAFE(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v4) ===\n")
display(plays_df.head(25))

print("\n=== DISCORD TEXT ===\n")
print(to_discord(plays_df, top_n=12))


[INJ AUDIT] status bucket counts:


,count
bucket,
out,162
questionable,21
other,17
probable,6
doubtful,4



[INJ AUDIT] teams with largest raw penalty (before cap):


,team_norm,pen
13,San Antonio Spurs,-9.25
11,Oklahoma City Thunder,-4.75
0,Brooklyn Nets,-4.00
4,Dallas Mavericks,-3.55
3,Cleveland Cavaliers,-3.50
15,Washington Wizards,-2.50
10,New York Knicks,-2.00
14,Toronto Raptors,-2.00
12,Orlando Magic,-1.50
8,Minnesota Timberwolves,-1.50



[INJ AUDIT] top contributors per team (most negative rows):


...


[INJ v4] Games w/ any nonzero inj pts: 10


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,-1.25,-1.50
1,Cleveland Cavaliers,Detroit Pistons,-1.50,-0.75
2,Orlando Magic,Washington Wizards,-1.50,-1.50
3,Miami Heat,Brooklyn Nets,-1.25,-1.50
4,Toronto Raptors,New York Knicks,-1.50,-1.50
5,Chicago Bulls,Oklahoma City Thunder,-1.00,-1.50
6,Minnesota Timberwolves,Memphis Grizzlies,-1.50,-1.25
7,Philadelphia 76ers,San Antonio Spurs,0.00,-1.50
8,Los Angeles Lakers,New Orleans Pelicans,0.00,-1.00
9,Sacramento Kings,Phoenix Suns,0.00,0.00



=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v4) ===



,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.646544,0.134349,0.137708,-183.0
1,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.385656,0.122578,0.076914,159.0
2,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.242872,0.097024,0.052606,312.0
3,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.616060,0.092251,0.096863,-160.0
4,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,479.0,0.249851,0.085342,0.046622,300.0
5,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,700.0,0.195299,0.077335,0.040171,412.0
6,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.309765,0.073847,0.042831,223.0
7,Phoenix Suns at Sacramento Kings,TOTAL,Over 223.5,-110.0,0.592813,0.069004,0.072454,-146.0
8,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10.5,-115.0,0.596785,0.061901,0.066543,-148.0
9,New Orleans Pelicans at Los Angeles Lakers,ML,New Orleans Pelicans ML,280.0,0.308063,0.056421,0.030471,225.0



=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.14u
Model%: 64.7% | Edge: 13.4% | Fair: -183

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.08u
Model%: 38.6% | Edge: 12.3% | Fair: +159

Memphis Grizzlies at Minnesota Timberwolves
ML: Memphis Grizzlies ML (+550) — 0.05u
Model%: 24.3% | Edge: 9.7% | Fair: +312

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.10u
Model%: 61.6% | Edge: 9.2% | Fair: -160

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+479) — 0.05u
Model%: 25.0% | Edge: 8.5% | Fair: +300

Washington Wizards at Orlando Magic
ML: Washington Wizards ML (+700) — 0.04u
Model%: 19.5% | Edge: 7.7% | Fair: +412

Phoenix Suns at Sacramento Kings
ML: Sacramento Kings ML (+308) — 0.04u
Model%: 31.0% | Edge: 7.4% | Fair: +223

Phoenix Suns at Sacramento Kings
TOTAL: Over 223.5 (-110) — 0.07u
Model%: 59.3% | Edge: 6.9% | Fair

In [23]:
# =========================
# INJ v5: FIX "too many OUTs" (top-N + smaller unknown weight + tighter cap)
# =========================
import pandas as pd
import numpy as np

def _col(df, *names):
    for n in names:
        if n in df.columns: return n
    return None

TEAM_COL   = _col(inj_rows_df, "team", "Team", "TEAM")
STATUS_COL = _col(inj_rows_df, "status", "Status", "STATUS")
NAME_COL   = _col(inj_rows_df, "player", "Player", "PLAYER", "name", "Name")
ROLE_COL   = _col(inj_rows_df, "role", "Role", "ROLE")
NOTE_COL   = _col(inj_rows_df, "note", "Note", "NOTE", "injury", "Injury")

if TEAM_COL is None or STATUS_COL is None:
    raise RuntimeError(f"inj_rows_df missing required cols. cols={list(inj_rows_df.columns)}")

def normalize_team_v5(s: str) -> str:
    return normalize_team_v2(s)

def status_bucket(s: str) -> str:
    s = ("" if s is None else str(s)).lower()
    if "out" in s: return "out"
    if "doubt" in s: return "doubtful"
    if "quest" in s: return "questionable"
    if "prob" in s: return "probable"
    return "other"

def detect_starter_flag(row) -> bool:
    txt = ""
    for c in [ROLE_COL, NOTE_COL, NAME_COL]:
        if c and c in row and pd.notna(row[c]):
            txt += " " + str(row[c])
    t = txt.lower()
    return ("starter" in t) or ("starting" in t)

# -------------------------
# v5 parameters (more conservative)
# -------------------------
TEAM_CAP   = -1.00      # cap tighter so injuries can't steamroll
TEAM_FLOOR = 0.0

# Penalties (smaller than v4)
PEN_OUT      = -0.70
PEN_DOUBTFUL = -0.40

# Q/P only if starter flagged
PEN_Q_STARTER = -0.15
PEN_Q_OTHER   = 0.00
PEN_P_STARTER = -0.05
PEN_P_OTHER   = 0.00

# role weights: since PDF lacks roles, treat unknown as tiny
STARTER_W  = 1.00
ROT_W      = 0.50
BENCH_W    = 0.20
UNKNOWN_W  = 0.10  # << key change

# Top-N limits per team (prevents 10+ “outs” from burying a team)
MAX_OUT_COUNTED = 4
MAX_DOUBT_COUNTED = 2

def role_weight_v5(row) -> float:
    txt = ""
    for c in [ROLE_COL, NOTE_COL, NAME_COL]:
        if c and c in row and pd.notna(row[c]):
            txt += " " + str(row[c])
    t = txt.lower()
    if "starter" in t or "starting" in t: return STARTER_W
    if "rotation" in t: return ROT_W
    if "bench" in t or "reserve" in t: return BENCH_W
    return UNKNOWN_W

def base_penalty_v5(bucket: str, starter: bool) -> float:
    if bucket == "out": return PEN_OUT
    if bucket == "doubtful": return PEN_DOUBTFUL
    if bucket == "questionable": return PEN_Q_STARTER if starter else PEN_Q_OTHER
    if bucket == "probable": return PEN_P_STARTER if starter else PEN_P_OTHER
    return 0.0

# -------------------------
# Build audit frame
# -------------------------
audit = inj_rows_df.copy()
audit["team_norm"] = audit[TEAM_COL].apply(normalize_team_v5)
audit["bucket"]    = audit[STATUS_COL].apply(status_bucket)
audit["starter"]   = audit.apply(detect_starter_flag, axis=1)
audit["w_role"]    = audit.apply(role_weight_v5, axis=1).astype(float)
audit["base_pen"]  = audit.apply(lambda r: base_penalty_v5(r["bucket"], bool(r["starter"])), axis=1).astype(float)
audit["pen"]       = (audit["base_pen"] * audit["w_role"]).astype(float)

print("\n[INJ v5] bucket counts:")
display(audit["bucket"].value_counts(dropna=False).to_frame("count"))

# -------------------------
# Top-N aggregation per team (only most impactful OUT / DOUBTFUL)
# -------------------------
def team_penalty_topn(df_team: pd.DataFrame) -> float:
    outs = df_team[df_team["bucket"] == "out"].sort_values("pen").head(MAX_OUT_COUNTED)["pen"].sum()
    dts  = df_team[df_team["bucket"] == "doubtful"].sort_values("pen").head(MAX_DOUBT_COUNTED)["pen"].sum()
    # Q/P already near-zero unless starter flagged
    qp   = df_team[df_team["bucket"].isin(["questionable","probable"])]["pen"].sum()
    other= 0.0
    return float(outs + dts + qp + other)

team_inj_pts_df = audit.groupby("team_norm", as_index=False).apply(team_penalty_topn).reset_index()
team_inj_pts_df.columns = ["team","inj_pts_raw"]
team_inj_pts_df["inj_pts"] = team_inj_pts_df["inj_pts_raw"].clip(lower=TEAM_CAP, upper=TEAM_FLOOR).astype(float)

print("\n[INJ v5] teams with most penalty (after top-N, before cap):")
display(team_inj_pts_df.sort_values("inj_pts_raw").head(15))

# -------------------------
# Attach to slate (safe overwrite)
# -------------------------
g = games_df.copy()
for c in ["home_inj_pts","away_inj_pts"]:
    if c in g.columns:
        g = g.drop(columns=[c])

g = g.merge(team_inj_pts_df[["team","inj_pts"]].rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}), on="home_team", how="left")
g = g.merge(team_inj_pts_df[["team","inj_pts"]].rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}), on="away_team", how="left")
g["home_inj_pts"] = g["home_inj_pts"].fillna(0.0).astype(float)
g["away_inj_pts"] = g["away_inj_pts"].fillna(0.0).astype(float)

print("\n[INJ v5] Games w/ any nonzero inj pts:", int(((g.home_inj_pts!=0)|(g.away_inj_pts!=0)).sum()))
display(g[["home_team","away_team","home_inj_pts","away_inj_pts"]])

# -------------------------
# Rerun downstream
# -------------------------
games_df = g
games_df = attach_team_stats_and_project_SAFE(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — UPDATED (INJ v5) ===\n")
display(plays_df.head(25))

print("\n=== DISCORD TEXT ===\n")
print(to_discord(plays_df, top_n=12))


[INJ v5] bucket counts:


,count
bucket,
out,162
questionable,21
other,17
probable,6
doubtful,4


/tmp/ipykernel_331/3430725916.py:109: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  team_inj_pts_df = audit.groupby("team_norm", as_index=False).apply(team_penalty_topn).reset_index()


ValueError: Length mismatch: Expected axis has 3 elements, new values have 2 elements

In [24]:
# --- FIX: build team_inj_pts_df without the 3-column apply artifact ---
# (replaces: team_inj_pts_df = audit.groupby(...).apply(...).reset_index() ...)

tmp = (
    audit.groupby("team_norm", sort=False)
         .apply(team_penalty_topn)          # returns a scalar per team
)

team_inj_pts_df = tmp.reset_index(name="inj_pts_raw").rename(columns={"team_norm":"team"})
team_inj_pts_df["inj_pts"] = team_inj_pts_df["inj_pts_raw"].clip(lower=TEAM_CAP, upper=TEAM_FLOOR).astype(float)

print("\n[INJ v5] teams with most penalty (after top-N, before cap):")
display(team_inj_pts_df.sort_values("inj_pts_raw").head(15))


[INJ v5] teams with most penalty (after top-N, before cap):


/tmp/ipykernel_331/424527654.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(team_penalty_topn)          # returns a scalar per team


,team,inj_pts_raw,inj_pts
0,Dallas Mavericks,-0.36,-0.36
1,Charlotte Hornets,-0.28,-0.28
3,Cleveland Cavaliers,-0.28,-0.28
4,Washington Wizards,-0.28,-0.28
6,Brooklyn Nets,-0.28,-0.28
5,Orlando Magic,-0.28,-0.28
7,Miami Heat,-0.28,-0.28
8,New York Knicks,-0.28,-0.28
12,Oklahoma City Thunder,-0.28,-0.28
9,Toronto Raptors,-0.28,-0.28


In [25]:
# --- INJ v5 TEAM AGG (NO groupby.apply, NO WARNING, CORRECT MAGNITUDES) ---

TOP_N = globals().get("TOP_N", 8)          # keep your existing value if you set it earlier
TEAM_CAP = globals().get("TEAM_CAP", -3.0) # example: cap at -3
TEAM_FLOOR = globals().get("TEAM_FLOOR", 0.0)

# Ensure numeric
audit["pen"] = pd.to_numeric(audit["pen"], errors="coerce").fillna(0.0)

# Sum TOP_N most negative penalties per team (i.e., biggest abs negatives)
team_inj_pts_df = (
    audit.sort_values("pen", ascending=True)          # most negative first
         .groupby("team_norm", sort=False)
         .head(TOP_N)
         .groupby("team_norm", sort=False)["pen"]
         .sum()
         .reset_index()
         .rename(columns={"team_norm": "team", "pen": "inj_pts_raw"})
)

# Apply team caps
team_inj_pts_df["inj_pts"] = team_inj_pts_df["inj_pts_raw"].clip(lower=TEAM_CAP, upper=TEAM_FLOOR).astype(float)

print(f"[INJ v5] TOP_N={TOP_N} | team rows: {len(team_inj_pts_df)}")
display(team_inj_pts_df.sort_values("inj_pts_raw").head(15))

# Optional: quick sanity check for one team
probe_team = team_inj_pts_df.sort_values("inj_pts_raw").iloc[0]["team"]
print("\n[INJ v5] Probe team:", probe_team)
display(
    audit[audit["team_norm"].eq(probe_team)]
        .sort_values("pen", ascending=True)
        .head(TOP_N)[["team","player","status","bucket","w_role","pen"]]
)
print("Sum TOP_N pen =", audit[audit["team_norm"].eq(probe_team)].sort_values("pen").head(TOP_N)["pen"].sum())

[INJ v5] TOP_N=8 | team rows: 16


,team,inj_pts_raw,inj_pts
0,Dallas Mavericks,-0.56,-0.56
3,Cleveland Cavaliers,-0.56,-0.56
5,Brooklyn Nets,-0.56,-0.56
4,Washington Wizards,-0.56,-0.56
12,Oklahoma City Thunder,-0.56,-0.56
11,San Antonio Spurs,-0.56,-0.56
9,New York Knicks,-0.56,-0.56
8,Toronto Raptors,-0.56,-0.56
13,Minnesota Timberwolves,-0.42,-0.42
6,Orlando Magic,-0.42,-0.42



[INJ v5] Probe team: Dallas Mavericks


,team,player,status,bucket,w_role,pen
0,DallasMavericks,"BagleyIII,Marvin",Out,out,0.1,-0.07
3,DallasMavericks,"Irving,Kyrie",Out,out,0.1,-0.07
5,DallasMavericks,"Marshall,Naji",Out,out,0.1,-0.07
4,DallasMavericks,"LivelyII,Dereck",Out,out,0.1,-0.07
20,DallasMavericks,"LivelyII,Dereck",Out,out,0.1,-0.07
19,DallasMavericks,"Irving,Kyrie",Out,out,0.1,-0.07
16,DallasMavericks,"BagleyIII,Marvin",Out,out,0.1,-0.07
21,DallasMavericks,"Marshall,Naji",Out,out,0.1,-0.07


Sum TOP_N pen = -0.5599999999999999


In [26]:
games_df = games_df.drop(columns=[c for c in ["home_inj_pts","away_inj_pts"] if c in games_df.columns])

games_df = games_df.merge(
    team_inj_pts_df[["team","inj_pts"]].rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}),
    on="home_team", how="left"
)
games_df = games_df.merge(
    team_inj_pts_df[["team","inj_pts"]].rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}),
    on="away_team", how="left"
)

games_df["home_inj_pts"] = games_df["home_inj_pts"].fillna(0.0).astype(float)
games_df["away_inj_pts"] = games_df["away_inj_pts"].fillna(0.0).astype(float)

In [27]:
# --- SAFE MERGE: TEAM INJ -> GAMES ---

# 0) validate columns exist
need_cols = {"team","inj_pts"}
missing = need_cols - set(team_inj_pts_df.columns)
if missing:
    raise RuntimeError(f"team_inj_pts_df missing columns: {missing}. Have: {list(team_inj_pts_df.columns)}")

# 1) drop any prior inj cols (if rerunning)
for c in ["home_inj_pts","away_inj_pts"]:
    if c in games_df.columns:
        games_df = games_df.drop(columns=c)

# 2) merge (home + away)
games_df = games_df.merge(
    team_inj_pts_df[["team","inj_pts"]].rename(columns={"team":"home_team","inj_pts":"home_inj_pts"}),
    on="home_team", how="left", validate="m:1"
)
games_df = games_df.merge(
    team_inj_pts_df[["team","inj_pts"]].rename(columns={"team":"away_team","inj_pts":"away_inj_pts"}),
    on="away_team", how="left", validate="m:1"
)

# 3) fill missing teams as 0.0
games_df["home_inj_pts"] = games_df["home_inj_pts"].fillna(0.0).astype(float)
games_df["away_inj_pts"] = games_df["away_inj_pts"].fillna(0.0).astype(float)

# 4) quick audit prints
print("[INJ MERGE] nonzero home inj:", (games_df["home_inj_pts"] != 0).sum(),
      "| nonzero away inj:", (games_df["away_inj_pts"] != 0).sum())
display(games_df[["home_team","away_team","home_inj_pts","away_inj_pts"]].head(12))

[INJ MERGE] nonzero home inj: 8 | nonzero away inj: 10


,home_team,away_team,home_inj_pts,away_inj_pts
0,Charlotte Hornets,Dallas Mavericks,-0.35,-0.56
1,Cleveland Cavaliers,Detroit Pistons,-0.56,-0.21
2,Orlando Magic,Washington Wizards,-0.42,-0.56
3,Miami Heat,Brooklyn Nets,-0.35,-0.56
4,Toronto Raptors,New York Knicks,-0.56,-0.56
5,Chicago Bulls,Oklahoma City Thunder,-0.28,-0.56
6,Minnesota Timberwolves,Memphis Grizzlies,-0.42,-0.35
7,Philadelphia 76ers,San Antonio Spurs,0.00,-0.56
8,Los Angeles Lakers,New Orleans Pelicans,0.00,-0.28
9,Sacramento Kings,Phoenix Suns,0.00,0.00


In [29]:
# --- FIX: ensure home_ppg/home_oppg/away_ppg/away_oppg exist by merging team_stats into games_df ---

import re
import pandas as pd

def norm_team(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = s.replace("LA Clippers", "Los Angeles Clippers")
    s = s.replace("LA Lakers", "Los Angeles Lakers")
    s = s.replace("NY Knicks", "New York Knicks")
    s = s.replace("SA Spurs", "San Antonio Spurs")
    s = s.replace("OKC Thunder", "Oklahoma City Thunder")
    # remove punctuation + collapse whitespace
    s = re.sub(r"[^A-Za-z0-9 ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

# 1) Prep team_stats with normalized key
ts = team_stats.copy()
if "team" not in ts.columns:
    raise RuntimeError(f"team_stats missing 'team' column. cols={list(ts.columns)}")

ts["team_norm"] = ts["team"].apply(norm_team)

# sanity: required stat cols
need_ts = ["ppg","oppg"]
missing_ts = [c for c in need_ts if c not in ts.columns]
if missing_ts:
    raise RuntimeError(f"team_stats missing required columns: {missing_ts}. cols={list(ts.columns)}")

# 2) Prep games_df with normalized keys
g = games_df.copy()
g["home_team_norm"] = g["home_team"].apply(norm_team)
g["away_team_norm"] = g["away_team"].apply(norm_team)

# 3) Merge home team stats
home_merge = ts[["team_norm","ppg","oppg"]].rename(columns={
    "ppg":"home_ppg",
    "oppg":"home_oppg"
})
g = g.merge(home_merge, left_on="home_team_norm", right_on="team_norm", how="left").drop(columns=["team_norm"])

# 4) Merge away team stats
away_merge = ts[["team_norm","ppg","oppg"]].rename(columns={
    "ppg":"away_ppg",
    "oppg":"away_oppg"
})
g = g.merge(away_merge, left_on="away_team_norm", right_on="team_norm", how="left").drop(columns=["team_norm"])

# 5) Debug if anything didn't match
bad_home = g[g["home_ppg"].isna()][["home_team","home_team_norm"]].drop_duplicates()
bad_away = g[g["away_ppg"].isna()][["away_team","away_team_norm"]].drop_duplicates()

if len(bad_home) or len(bad_away):
    print("Unmatched HOME teams (need mapping):")
    display(bad_home)
    print("Unmatched AWAY teams (need mapping):")
    display(bad_away)
    raise RuntimeError("Team stats merge failed for one or more teams. Add name mappings in norm_team().")

# 6) Cast + replace games_df
for c in ["home_ppg","home_oppg","away_ppg","away_oppg"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")

games_df = g.drop(columns=["home_team_norm","away_team_norm"])

print("[TEAM STATS MERGE OK] games_df now has:", [c for c in ["home_ppg","home_oppg","away_ppg","away_oppg"] if c in games_df.columns])

# 7) Now rerun downstream layers
games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("=== TOP PLAYS (ALL MARKETS) — FINAL (TEAM STATS + INJ MERGED) ===")
display(plays_df.head(25))

discord_text = build_discord_text(plays_df, header="== CDR BETTING | NBA FULL CARD ==")
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

KeyError: 'home_ppg'

In [28]:
# --- RERUN DOWNSTREAM AFTER INJ MERGE ---

# 1) Make sure required cols exist before projecting
req = ["home_ppg","away_ppg","home_oppg","away_oppg","home_inj_pts","away_inj_pts",
       "home_team","away_team","home_spread","home_spread_price","total_line","total_price_used",
       "home_ml_price","away_ml_price"]
missing = [c for c in req if c not in games_df.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns for downstream run: {missing}")

# 2) Rebuild projections + edges + card (uses your notebook functions)
games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

# 3) Show outputs
print("=== TOP PLAYS (ALL MARKETS) — FINAL (INJ MERGED) ===")
display(plays_df.head(25))

discord_text = build_discord_text(plays_df, header="== CDR BETTING | NBA FULL CARD ==")
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

# 4) Save artifacts
plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

KeyError: 'home_ppg'

In [30]:
# =========================
# BOTTOM-CELL FIX (SAFE)
# =========================
import re
import pandas as pd

def norm_team(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()

    # common aliases
    alias = {
        "LA Clippers": "Los Angeles Clippers",
        "LA Lakers": "Los Angeles Lakers",
        "NY Knicks": "New York Knicks",
        "OKC Thunder": "Oklahoma City Thunder",
        "SA Spurs": "San Antonio Spurs",
        "GS Warriors": "Golden State Warriors",
    }
    s = alias.get(s, s)

    # strip punctuation + normalize spaces
    s = re.sub(r"[^A-Za-z0-9 ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

# ---- 0) Sanity prints (so we know what we actually have at the bottom) ----
print("games_df cols:", list(games_df.columns))
print("team_stats cols:", list(team_stats.columns))

req_games = ["home_team", "away_team"]
miss_games = [c for c in req_games if c not in games_df.columns]
if miss_games:
    raise RuntimeError(f"games_df missing required columns: {miss_games}. cols={list(games_df.columns)}")

# team_stats can sometimes have TEAM instead of team, etc.
ts = team_stats.copy()
if "team" not in ts.columns:
    # try to auto-detect team col
    cand = [c for c in ts.columns if str(c).lower() in ["team", "team_name", "name"]]
    if not cand:
        raise RuntimeError(f"team_stats has no 'team' column and no obvious alternative. cols={list(ts.columns)}")
    ts = ts.rename(columns={cand[0]: "team"})
    print(f"[team_stats] Renamed {cand[0]} -> team")

# auto-detect ppg/oppg cols if needed
if "ppg" not in ts.columns or "oppg" not in ts.columns:
    # try common ESPN names
    rename_map = {}
    if "avgPointsFor" in ts.columns: rename_map["avgPointsFor"] = "ppg"
    if "avgPointsAgainst" in ts.columns: rename_map["avgPointsAgainst"] = "oppg"
    ts = ts.rename(columns=rename_map)

need_ts = ["team", "ppg", "oppg"]
miss_ts = [c for c in need_ts if c not in ts.columns]
if miss_ts:
    raise RuntimeError(f"team_stats missing required columns {miss_ts}. cols={list(ts.columns)}")

# ---- 1) Force-clean any prior stats cols so we don't have suffix collisions ----
stats_cols = ["home_ppg","home_oppg","away_ppg","away_oppg"]
g = games_df.drop(columns=[c for c in stats_cols if c in games_df.columns]).copy()

# ---- 2) Add normalized join keys ----
g["home_team_norm"] = g["home_team"].apply(norm_team)
g["away_team_norm"] = g["away_team"].apply(norm_team)

ts["team_norm"] = ts["team"].apply(norm_team)

# ---- 3) Merge home + away stats (guaranteed to create columns) ----
home_merge = ts[["team_norm","ppg","oppg"]].rename(columns={"ppg":"home_ppg","oppg":"home_oppg"})
away_merge = ts[["team_norm","ppg","oppg"]].rename(columns={"ppg":"away_ppg","oppg":"away_oppg"})

g = g.merge(home_merge, on="home_team_norm", how="left")
g = g.merge(away_merge, on="away_team_norm", how="left")

# ---- 4) Debug unmatched teams BEFORE failing ----
bad_home = g[g["home_ppg"].isna()][["home_team","home_team_norm"]].drop_duplicates()
bad_away = g[g["away_ppg"].isna()][["away_team","away_team_norm"]].drop_duplicates()

if len(bad_home) or len(bad_away):
    print("\n[TEAM STATS MERGE FAILED] Unmatched teams:")
    if len(bad_home):
        print("\nHOME unmatched:")
        display(bad_home)
    if len(bad_away):
        print("\nAWAY unmatched:")
        display(bad_away)
    # show what team names we *do* have in team_stats for quick mapping
    print("\nSample team_stats teams:")
    display(ts[["team"]].head(30))
    raise RuntimeError("Fix team name mappings in norm_team() (or rename inputs) and rerun this cell.")

# ---- 5) Finalize + write back ----
for c in stats_cols:
    g[c] = pd.to_numeric(g[c], errors="coerce")

games_df = g.drop(columns=["home_team_norm","away_team_norm","team_norm"], errors="ignore")

print("\n[TEAM STATS MERGE OK] Added:", stats_cols)
display(games_df[["home_team","away_team","home_ppg","home_oppg","away_ppg","away_oppg"]].head(12))

# ---- 6) Run your model pipeline ----
games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — FINAL ===")
display(plays_df.head(25))

discord_text = build_discord_text(plays_df, header="== CDR BETTING | NBA FULL CARD ==")
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

games_df cols: ['home_team', 'away_team', 'home_spread', 'home_spread_price', 'spread_consensus', 'total_line', 'total_price_used', 'home_ml_price', 'away_ml_price', 'home_srs', 'away_srs', 'home_pts_proj', 'away_pts_proj', 'proj_total_raw', 'proj_margin_raw', 'market_margin', 'proj_margin', 'proj_total', 'home_cover_prob', 'spread_market_prob', 'spread_edge', 'spread_kelly', 'spread_units', 'spread_fair', 'over_prob', 'total_market_prob', 'total_edge', 'total_kelly', 'total_units', 'total_fair', 'home_win_prob', 'away_win_prob', 'home_ml_market_prob', 'away_ml_market_prob', 'home_ml_prob_novig', 'away_ml_prob_novig', 'home_ml_edge_novig', 'away_ml_edge_novig', 'home_ml_kelly', 'away_ml_kelly', 'home_ml_units', 'away_ml_units', 'home_ml_fair', 'away_ml_fair', 'home_ppg', 'home_oppg', 'away_ppg', 'away_oppg', 'home_inj_pts', 'away_inj_pts']
team_stats cols: ['team', 'ppg', 'oppg', 'srs']


KeyError: 'home_team_norm'

In [32]:
# =========================
# BOTTOM-CELL FIX v2 (NO *_norm COLS NEEDED)
# =========================
import re
import pandas as pd

def norm_team(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip()

    alias = {
        "LA Clippers": "Los Angeles Clippers",
        "LA Lakers": "Los Angeles Lakers",
        "NY Knicks": "New York Knicks",
        "OKC Thunder": "Oklahoma City Thunder",
        "SA Spurs": "San Antonio Spurs",
        "GS Warriors": "Golden State Warriors",
    }
    s = alias.get(s, s)
    s = re.sub(r"[^A-Za-z0-9 ]+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

# --- 0) ensure required cols exist ---
for c in ["home_team","away_team"]:
    if c not in games_df.columns:
        raise RuntimeError(f"games_df missing {c}. cols={list(games_df.columns)}")

ts = team_stats.copy()
need_ts = ["team","ppg","oppg"]
miss_ts = [c for c in need_ts if c not in ts.columns]
if miss_ts:
    raise RuntimeError(f"team_stats missing {miss_ts}. cols={list(ts.columns)}")

# --- 1) build lookup dicts ---
ts["team_norm"] = ts["team"].apply(norm_team)
ppg_map  = dict(zip(ts["team_norm"], pd.to_numeric(ts["ppg"], errors="coerce")))
oppg_map = dict(zip(ts["team_norm"], pd.to_numeric(ts["oppg"], errors="coerce")))

# --- 2) overwrite the 4 columns directly (no merges needed) ---
g = games_df.copy()
g["home_team_norm"] = g["home_team"].apply(norm_team)
g["away_team_norm"] = g["away_team"].apply(norm_team)

g["home_ppg"]  = g["home_team_norm"].map(ppg_map)
g["home_oppg"] = g["home_team_norm"].map(oppg_map)
g["away_ppg"]  = g["away_team_norm"].map(ppg_map)
g["away_oppg"] = g["away_team_norm"].map(oppg_map)

# --- 3) fail fast if any missing (and print exactly which) ---
bad_home = g[g["home_ppg"].isna()][["home_team","home_team_norm"]].drop_duplicates()
bad_away = g[g["away_ppg"].isna()][["away_team","away_team_norm"]].drop_duplicates()

if len(bad_home) or len(bad_away):
    print("\n[TEAM STATS MAP FAILED] Unmatched teams:")
    if len(bad_home):
        print("\nHOME unmatched:")
        display(bad_home)
    if len(bad_away):
        print("\nAWAY unmatched:")
        display(bad_away)
    print("\nteam_stats teams sample:")
    display(ts[["team"]].head(30))
    raise RuntimeError("Add aliases in norm_team() for the unmatched names above and rerun this cell.")

# --- 4) cleanup + write back ---
games_df = g.drop(columns=["home_team_norm","away_team_norm"], errors="ignore")

print("\n[OK] team stats columns refreshed.")
display(games_df[["home_team","away_team","home_ppg","home_oppg","away_ppg","away_oppg"]].head(12))

# --- 5) rerun downstream ---
games_df = attach_team_stats_and_project(games_df, team_stats)
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — FINAL ===")
display(plays_df.head(25))

discord_text = build_discord_text(plays_df, header="== CDR BETTING | NBA FULL CARD ==")
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")


[OK] team stats columns refreshed.


,home_team,away_team,home_ppg,home_oppg,away_ppg,away_oppg
0,Charlotte Hornets,Dallas Mavericks,116.049180,113.032780,113.900000,117.666664
1,Cleveland Cavaliers,Detroit Pistons,119.209680,114.983870,117.305084,109.389830
2,Orlando Magic,Washington Wizards,114.576270,114.440680,112.266670,123.000000
3,Miami Heat,Brooklyn Nets,119.836070,116.983604,106.950000,115.433334
4,Toronto Raptors,New York Knicks,113.966670,112.000000,117.213110,111.131150
5,Chicago Bulls,Oklahoma City Thunder,115.803276,120.065575,119.161290,107.870964
6,Minnesota Timberwolves,Memphis Grizzlies,119.131150,114.573770,115.711860,117.677960
7,Philadelphia 76ers,San Antonio Spurs,116.383330,115.916664,117.966670,111.783330
8,Los Angeles Lakers,New Orleans Pelicans,115.916664,115.433334,115.080640,120.354836
9,Sacramento Kings,Phoenix Suns,110.290320,121.225810,112.116670,111.350000


KeyError: 'home_ppg'

In [33]:
# =========================
# BOTTOM-CELL FIX v3 (BYPASS attach_team_stats_and_project)
# Uses existing home_ppg/home_oppg/away_ppg/away_oppg already in games_df
# =========================
import numpy as np
import pandas as pd

g = games_df.copy()

# --- required columns check ---
need = ["home_ppg","home_oppg","away_ppg","away_oppg","home_team","away_team"]
missing = [c for c in need if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing columns needed for projections: {missing}\ncols={list(g.columns)}")

# Ensure injury columns exist
if "home_inj_pts" not in g.columns: g["home_inj_pts"] = 0.0
if "away_inj_pts" not in g.columns: g["away_inj_pts"] = 0.0

for c in ["home_ppg","home_oppg","away_ppg","away_oppg","home_inj_pts","away_inj_pts"]:
    g[c] = pd.to_numeric(g[c], errors="coerce").astype(float)

# --- projection constants (mirror your notebook defaults) ---
HOME_COURT_PTS = globals().get("HOME_COURT_PTS", 2.0)   # adjust if your notebook uses a different value
SRS_WEIGHT = globals().get("SRS_WEIGHT", 0.35)          # if your notebook blends SRS into margin
USE_SRS = globals().get("USE_SRS", True)

# --- base projections from PF/PA ---
g["home_pts_proj"] = ((g["home_ppg"] + g["away_oppg"]) / 2.0) + (HOME_COURT_PTS / 2.0) + g["home_inj_pts"]
g["away_pts_proj"] = ((g["away_ppg"] + g["home_oppg"]) / 2.0) - (HOME_COURT_PTS / 2.0) + g["away_inj_pts"]

g["proj_total_raw"]  = g["home_pts_proj"] + g["away_pts_proj"]
g["proj_margin_raw"] = g["home_pts_proj"] - g["away_pts_proj"]

# --- optional SRS blend (only if you already have home_srs/away_srs) ---
if USE_SRS and ("home_srs" in g.columns) and ("away_srs" in g.columns):
    g["home_srs"] = pd.to_numeric(g["home_srs"], errors="coerce").fillna(0.0)
    g["away_srs"] = pd.to_numeric(g["away_srs"], errors="coerce").fillna(0.0)
    g["proj_margin_srs"] = g["proj_margin_raw"] + (g["home_srs"] - g["away_srs"])
    g["proj_margin"] = (1 - SRS_WEIGHT) * g["proj_margin_raw"] + SRS_WEIGHT * g["proj_margin_srs"]
else:
    g["proj_margin_srs"] = g["proj_margin_raw"]
    g["proj_margin"] = g["proj_margin_raw"]

# if your notebook also defines proj_total (sometimes adjusted), keep it simple:
g["proj_total"] = g["proj_total_raw"]

# write back
games_df = g

print("[OK] projections rebuilt without attach_team_stats_and_project()")
display(games_df[["home_team","away_team","home_pts_proj","away_pts_proj","proj_total","proj_margin"]].head(12))

# --- continue downstream normally ---
games_df = compute_market_probs_and_edges(games_df)
plays_df = build_plays(games_df)

print("\n=== TOP PLAYS (ALL MARKETS) — FINAL ===")
display(plays_df.head(25))

discord_text = build_discord_text(plays_df, header="== CDR BETTING | NBA FULL CARD ==")
print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")

[OK] projections rebuilt without attach_team_stats_and_project()


,home_team,away_team,home_pts_proj,away_pts_proj,proj_total,proj_margin
0,Charlotte Hornets,Dallas Mavericks,117.407922,112.006390,229.414312,5.401532
1,Cleveland Cavaliers,Detroit Pistons,114.639755,115.034477,229.674232,-0.394722
2,Orlando Magic,Washington Wizards,119.268135,111.893675,231.161810,7.374460
3,Miami Heat,Brooklyn Nets,118.184702,110.506802,228.691504,7.677900
4,Toronto Raptors,New York Knicks,112.888910,113.146555,226.035465,-0.257645
5,Chicago Bulls,Oklahoma City Thunder,112.457120,118.153432,230.610552,-5.696312
6,Minnesota Timberwolves,Memphis Grizzlies,118.884555,113.892815,232.777370,4.991740
7,Philadelphia 76ers,San Antonio Spurs,114.983330,115.481667,230.464997,-0.498337
8,Los Angeles Lakers,New Orleans Pelicans,119.035750,114.076987,233.112737,4.958763
9,Sacramento Kings,Phoenix Suns,111.720160,115.771240,227.491400,-4.051080



=== TOP PLAYS (ALL MARKETS) — FINAL ===


,matchup,market,bet,odds,model_prob,edge,units,fair_odds
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.734060,0.221865,0.227411,-276.0
1,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.483437,0.220360,0.144254,107.0
2,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.338713,0.192865,0.109239,195.0
3,Dallas Mavericks at Charlotte Hornets,ML,Dallas Mavericks ML,479.0,0.326309,0.161800,0.092832,206.0
4,Washington Wizards at Orlando Magic,ML,Washington Wizards ML,700.0,0.269430,0.151465,0.082531,271.0
5,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.675113,0.151304,0.158869,-208.0
6,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.367836,0.131918,0.081294,172.0
7,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10.5,-115.0,0.655535,0.120651,0.129700,-190.0
8,Brooklyn Nets at Miami Heat,ML,Brooklyn Nets ML,481.0,0.261143,0.097108,0.053767,283.0
9,New Orleans Pelicans at Los Angeles Lakers,ML,New Orleans Pelicans ML,280.0,0.339719,0.088078,0.051952,194.0


NameError: name 'build_discord_text' is not defined

In [37]:
# =========================
# BOTTOM-CELL FIX: define build_discord_text (and safe save)
# =========================
import pandas as pd
import numpy as np

def fmt_odds(o):
    try:
        o = float(o)
    except:
        return str(o)
    if o >= 100:
        return f"+{int(round(o))}"
    return f"{int(round(o))}"

def fmt_units(u):
    try:
        u = float(u)
    except:
        return str(u)
    return f"{u:.2f}u".rstrip("0").rstrip(".")

def build_discord_text(plays_df: pd.DataFrame, header="== CDR BETTING | NBA FULL CARD ==") -> str:
    lines = [header, ""]
    if plays_df is None or len(plays_df) == 0:
        return header + "\n\n(no +EV plays)"
    df = plays_df.copy()

    # ensure expected cols exist
    for c in ["matchup","market","bet","odds","units","model_prob","edge","fair_odds"]:
        if c not in df.columns:
            df[c] = np.nan

    # order by units then edge (typical card sort)
    df["__units"] = pd.to_numeric(df["units"], errors="coerce").fillna(0)
    df["__edge"]  = pd.to_numeric(df["edge"], errors="coerce").fillna(0)
    df = df.sort_values(["__units","__edge"], ascending=False)

    for _, r in df.iterrows():
        matchup = str(r["matchup"])
        market  = str(r["market"]).upper()
        bet     = str(r["bet"])

        odds = fmt_odds(r["odds"])
        units = fmt_units(r["units"])

        mp = pd.to_numeric(r["model_prob"], errors="coerce")
        ed = pd.to_numeric(r["edge"], errors="coerce")
        fair = r["fair_odds"]

        mp_s = f"{mp*100:.1f}%" if pd.notna(mp) else "—"
        ed_s = f"{ed*100:.1f}%" if pd.notna(ed) else "—"
        fair_s = fmt_odds(fair) if pd.notna(pd.to_numeric(fair, errors="coerce")) else str(fair)

        lines.append(matchup)
        lines.append(f"{market}: {bet} ({odds}) — {units}")
        lines.append(f"Model%: {mp_s} | Edge: {ed_s} | Fair: {fair_s}")
        lines.append("")  # spacer

    return "\n".join(lines).rstrip() + "\n"

# --- build + print + save ---
discord_text = build_discord_text(plays_df, header="== CDR BETTING | NBA FULL CARD ==")

print("\n=== DISCORD TEXT ===\n")
print(discord_text)

plays_df.to_csv("nba_card_plays.csv", index=False)
with open("nba_discord_card.txt", "w") as f:
    f.write(discord_text)

print("\nSaved: nba_card_plays.csv, nba_discord_card.txt")


=== DISCORD TEXT ===

== CDR BETTING | NBA FULL CARD ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.23u
Model%: 73.4% | Edge: 22.2% | Fair: -276

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.16u
Model%: 67.5% | Edge: 15.1% | Fair: -208

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.14u
Model%: 48.3% | Edge: 22.0% | Fair: +107

Oklahoma City Thunder at Chicago Bulls
SPREAD: Chicago Bulls +10.5 (-115) — 0.13u
Model%: 65.6% | Edge: 12.1% | Fair: -190

Memphis Grizzlies at Minnesota Timberwolves
ML: Memphis Grizzlies ML (+550) — 0.11u
Model%: 33.9% | Edge: 19.3% | Fair: +195

Dallas Mavericks at Charlotte Hornets
ML: Dallas Mavericks ML (+479) — 0.09u
Model%: 32.6% | Edge: 16.2% | Fair: +206

New York Knicks at Toronto Raptors
TOTAL: Over 221.5 (-110) — 0.09u
Model%: 60.5% | Edge: 8.1% | Fair: -153

Washington Wizards at Orlando Magic
ML: Washington Wizards ML (+700) — 0.08u
Model%: 26.9% | E

In [36]:
# Top 12 only
plays_df = plays_df.sort_values(["units","edge"], ascending=False).head(12)

In [38]:
# === BOTTOM CELL: TOP 10 MOST-PROBABLE TO WIN (+EV) — WIN-PRIORITY BLEND ===
# Assumes you already have: plays_df (from build_plays) OR you can build it above this cell.
# Required cols in plays_df: ["matchup","market","bet","odds","model_prob","edge","units","fair_odds"]

import numpy as np
import pandas as pd

WIN_W = 0.80   # prioritize winning
EV_W  = 0.20   # still respect +EV edge
TOP_N = 10

def build_discord_text(df, header="== CDR BETTING | NBA TOP 10 (WIN-PRIORITY +EV) =="):
    lines = [header, ""]
    for _, r in df.iterrows():
        matchup = str(r.get("matchup","")).strip()
        market  = str(r.get("market","")).strip()
        bet     = str(r.get("bet","")).strip()
        odds    = r.get("odds", np.nan)
        units   = r.get("units", np.nan)
        mp      = r.get("model_prob", np.nan)
        edge    = r.get("edge", np.nan)
        fair    = r.get("fair_odds", np.nan)

        # formatting
        def fmt_num(x, d=1):
            try:
                return f"{float(x):.{d}f}"
            except:
                return "NA"

        def fmt_odds(x):
            try:
                x = float(x)
                if x > 0:
                    return f"+{int(round(x))}"
                return f"{int(round(x))}"
            except:
                return "NA"

        lines.append(matchup)
        lines.append(f"{market}: {bet} ({fmt_odds(odds)}) — {fmt_num(units,2)}u")
        lines.append(f"Model%: {fmt_num(100*mp,1)}% | Edge: {fmt_num(100*edge,1)}% | Fair: {fmt_odds(fair)}")
        lines.append("")
    return "\n".join(lines).strip()

# --- Guardrails: make sure required cols exist ---
required = ["matchup","market","bet","odds","model_prob","edge","units","fair_odds"]
missing = [c for c in required if c not in plays_df.columns]
if missing:
    raise RuntimeError(f"plays_df missing columns: {missing}")

# --- Filter to +EV only (edge > 0) and keep only playable rows (units > 0) ---
df = plays_df.copy()
df["model_prob"] = pd.to_numeric(df["model_prob"], errors="coerce")
df["edge"]       = pd.to_numeric(df["edge"], errors="coerce")
df["units"]      = pd.to_numeric(df["units"], errors="coerce")

df = df[(df["edge"] > 0) & (df["units"] > 0)].dropna(subset=["model_prob","edge"])

# --- Win-priority blend score ---
# score is on 0–1 scale (both terms are probabilities)
df["win_ev_score"] = (WIN_W * df["model_prob"]) + (EV_W * df["edge"])

top10 = df.sort_values(["win_ev_score","model_prob","edge"], ascending=False).head(TOP_N).reset_index(drop=True)

print("=== TOP 10 MOST PROBABLE TO WIN (+EV) — WIN-PRIORITY BLEND ===")
display(top10[["matchup","market","bet","odds","model_prob","edge","units","fair_odds","win_ev_score"]])

discord_text_top10 = build_discord_text(top10, header="== CDR BETTING | NBA TOP 10 (WIN-PRIORITY +EV) ==")
print("\n=== DISCORD TEXT (TOP 10) ===\n")
print(discord_text_top10)

# Optional: save
top10.to_csv("nba_top10_win_priority_ev.csv", index=False)
with open("nba_discord_top10.txt","w") as f:
    f.write(discord_text_top10)
print("\nSaved: nba_top10_win_priority_ev.csv, nba_discord_top10.txt")

=== TOP 10 MOST PROBABLE TO WIN (+EV) — WIN-PRIORITY BLEND ===


,matchup,market,bet,odds,model_prob,edge,units,fair_odds,win_ev_score
0,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers +8,-105.0,0.734060,0.221865,0.227411,-276.0,0.631621
1,Phoenix Suns at Sacramento Kings,SPREAD,Sacramento Kings +9.5,-110.0,0.675113,0.151304,0.158869,-208.0,0.570351
2,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls +10.5,-115.0,0.655535,0.120651,0.129700,-190.0,0.548558
3,New York Knicks at Toronto Raptors,TOTAL,Over 221.5,-110.0,0.605185,0.081376,0.085445,-153.0,0.500423
4,Washington Wizards at Orlando Magic,TOTAL,Over 227.0,-110.0,0.596699,0.072890,0.076534,-148.0,0.491937
5,Oklahoma City Thunder at New York Knicks,TOTAL,Over 222.5,-110.0,0.594567,0.070758,0.074295,-147.0,0.489805
6,Phoenix Suns at Sacramento Kings,TOTAL,Over 223.5,-110.0,0.592813,0.069004,0.072454,-146.0,0.488052
7,San Antonio Spurs at Philadelphia 76ers,ML,Philadelphia 76ers ML,265.0,0.483437,0.220360,0.144254,107.0,0.430822
8,Phoenix Suns at Sacramento Kings,ML,Sacramento Kings ML,308.0,0.367836,0.131918,0.081294,172.0,0.320653
9,Memphis Grizzlies at Minnesota Timberwolves,ML,Memphis Grizzlies ML,550.0,0.338713,0.192865,0.109239,195.0,0.309543



=== DISCORD TEXT (TOP 10) ===

== CDR BETTING | NBA TOP 10 (WIN-PRIORITY +EV) ==

San Antonio Spurs at Philadelphia 76ers
SPREAD: Philadelphia 76ers +8 (-105) — 0.23u
Model%: 73.4% | Edge: 22.2% | Fair: -276

Phoenix Suns at Sacramento Kings
SPREAD: Sacramento Kings +9.5 (-110) — 0.16u
Model%: 67.5% | Edge: 15.1% | Fair: -208

Oklahoma City Thunder at Chicago Bulls
SPREAD: Chicago Bulls +10.5 (-115) — 0.13u
Model%: 65.6% | Edge: 12.1% | Fair: -190

New York Knicks at Toronto Raptors
TOTAL: Over 221.5 (-110) — 0.09u
Model%: 60.5% | Edge: 8.1% | Fair: -153

Washington Wizards at Orlando Magic
TOTAL: Over 227.0 (-110) — 0.08u
Model%: 59.7% | Edge: 7.3% | Fair: -148

Oklahoma City Thunder at New York Knicks
TOTAL: Over 222.5 (-110) — 0.07u
Model%: 59.5% | Edge: 7.1% | Fair: -147

Phoenix Suns at Sacramento Kings
TOTAL: Over 223.5 (-110) — 0.07u
Model%: 59.3% | Edge: 6.9% | Fair: -146

San Antonio Spurs at Philadelphia 76ers
ML: Philadelphia 76ers ML (+265) — 0.14u
Model%: 48.3% | Edge: 22